# InkLingua on Kaggle (GPU T4 x2)

Style-disentangled, uncertainty-aware **handwriting -> text -> translation** with fragment-level risk-controlled self-distillation
(FRAG-RC), writer self-distillation, an LLM in the loop and a validation protocol in which the test split never decides anything.

This notebook is **self-contained**: section 2 writes the `inklingua/` package to disk (one cell per module), the rest configures and
runs it.  It is generated from the repository by `tools/build_kaggle_notebook.py`; the repository is the source of truth.

**How the two T4s are used.**  The recogniser trains on `cuda:0`; the LLM (Qwen2.5-1.5B, fp16, ~3 GB) sits on `cuda:1` for
translation and N-best rescoring.  T4s have no bf16, so training uses fp16 autocast with a gradient scaler (handled automatically).

**Before you run.**
1. Settings -> Accelerator: *GPU T4 x2*.  Settings -> Internet: *On* (the dataset and the LLM are downloaded on first use).
   Offline alternative: attach a Kaggle dataset with `train/validation/test.parquet` and set `DATA_SOURCE = "local"`.
2. A full IAM run (100 epochs, 30 % labels) takes a few hours on a T4.  Kaggle sessions are limited to ~9-12 h: the run **resumes**
   from `/kaggle/working/inklingua_runs/<run>/checkpoints/last.pt` if you re-run the notebook, so save the working directory
   (Save Version with outputs) and continue in a new session if needed.
3. Start with `QUICK_TEST = True` (a few minutes) to see every output once, then switch it off.

| section | what it does |
|---|---|
| 1 | environment: packages, GPUs |
| 2 | the package source (collapsed; one cell per module) |
| 3 | configuration knobs: mode, corpora, label budget, LLM |
| 4 | fast self-tests (~30 s) |
| 5 | train + evaluate (resumable) |
| 6 | results: headline numbers, curves, the interactive report, key figures |
| 7 | optional: suite (baselines + ablations), multi-corpus, fine-tuning, zero-shot adaptation |
| 8 | export the run directory as a zip |

Honesty note: nothing here has been validated on real IAM yet.  Whether unlabeled data helps at all is exactly what the **suite**
(section 7) measures: `labeled_only` vs `full` over several seeds.  IAM is distributed for non-commercial research.

## 1. Environment

In [ ]:
# ---- 1. Environment -------------------------------------------------------------------------------------------------------
import os, subprocess, sys, importlib

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

_missing = [p for p, m in (("editdistance", "editdistance"), ("opencv-python-headless", "cv2"), ("datasets", "datasets"),
                           ("transformers", "transformers"), ("accelerate", "accelerate"), ("sentencepiece", "sentencepiece"),
                           ("sacremoses", "sacremoses"), ("matplotlib", "matplotlib"), ("tqdm", "tqdm"))
            if importlib.util.find_spec(m) is None]
if _missing:
    print("installing:", _missing)
    _pip(*_missing)

import torch
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i} {p.name}  {p.total_memory / 2**30:.1f} GB  compute {p.major}.{p.minor}")
os.makedirs("inklingua", exist_ok=True)
ON_KAGGLE = os.path.isdir("/kaggle/working")
WORK_DIR = "/kaggle/working/inklingua_runs" if ON_KAGGLE else "./inklingua_runs"
print("work dir:", WORK_DIR)

## 2. The package source

One cell per module, written to `./inklingua/`.  Collapsed on purpose: read the repository for the code; run the cells (Run -> Run all cells of this section).

In [ ]:
%%writefile inklingua/__init__.py
"""
InkLingua v11 -- style-disentangled, uncertainty-aware handwriting-to-translation with fragment-level
risk-controlled self-distillation, writer self-distillation, an LLM in the loop, multi-dataset / multi-script
training and a split validation protocol.

The package is the v10 single-file script restructured into readable modules (one question per module) with the
defects of the audit (`docs/AUDIT_v10_to_v11.md`) fixed and the additions listed in `docs/CHANGELOG.md`.

    inklingua.config            every option (dataclasses); nothing is parsed from the command line
    inklingua.datasets_registry the verified free line-level corpora that can complement IAM (12 entries)
    inklingua.text              grapheme-cluster codec, right-to-left handling, detokenisation, CER / WER / chrF, bootstrap
    inklingua.decoding          CTC greedy / prefix-beam decoding
    inklingua.style             the five measured style descriptors (targets of the connected task)
    inklingua.data              preprocessing, sources (Hub / local parquet / synthetic), datasets, loaders, MixLine
    inklingua.model             InkLinguaNet, losses, EMA teacher, style-equivariant GPU augmentation
    inklingua.llm               the pretrained LLM: translation + N-best rescoring
    inklingua.riskcontrol       isotonic risk curves, thresholds, cluster-robust bounds, transfer calibration
    inklingua.pseudo            legacy line-level pseudo-labels (kept as the `legacy_pl` baseline)
    inklingua.frag              FRAG-RC: fragment-level, style-stratified, risk-controlled self-distillation
    inklingua.trainer           the training loop (+ protocol files, curves)
    inklingua.render            closed-loop style-matched rendering of the translation
    inklingua.explain           per-sample explanation panels, calibration and selective-prediction helpers
    inklingua.posthoc           validation-gated post-hoc options (view ensemble, AdaBN, WSD, space bonus)
    inklingua.interpret         the interpretability suite (probes, uncertainty decomposition, counterfactuals, ...)
    inklingua.inferencer        evaluation: metrics, structured output folders, translation, report
    inklingua.adapt             zero-shot adaptation to a new written language + fine-tuning on a new dataset
    inklingua.report            the self-contained run report (index.html + the numbers behind every panel)
    inklingua.suite             the experiment suite (baselines, ablations, seed-paired comparison)
    inklingua.selftest          fast invariant tests + the synthetic end-to-end smoke
    inklingua.main              entry point: `main(cfg)` dispatches on cfg.run.mode
"""
from .config import Config  # noqa: F401
from .main import main  # noqa: F401

__version__ = "11.0.0"
__all__ = ["Config", "main", "__version__"]

In [ ]:
%%writefile inklingua/config.py
"""
Configuration.  EVERY option lives here; `main()` never parses command-line arguments.  Edit the dataclass defaults, or
build a Config in a notebook cell and pass it to `main(cfg)`.

The most used entry points:
    Config()                         the default IAM-line experiment (30 % labels, FRAG-RC + WSD)
    Config.labeled_only()            THE baseline: same labeled lines, no unlabeled data
    Config.ablation("no_frag")       one component removed (see Config.ablation for the list)
    Config.multi(("iam", "rimes"))   train on several corpora of the registry at once (docs/DATASETS.md)
    Config.smoke()                   a tiny synthetic configuration for plumbing tests
    cfg.run.mode = "suite"           train / evaluate every variant for every seed (the experiment a paper needs)
"""
from __future__ import annotations

import copy
import dataclasses
import json
import os
from dataclasses import asdict, dataclass, field
from typing import Any, Dict, Optional, Tuple

IAM_LINE_REVISION = "fbdad97500ce54635c0d1ba306bf535cb40656cf"      # Teklia/IAM-line, last modified 2024-03-14 (verified 2026-11-04)
IAM_LINE_SIZES = (6482, 976, 2915)                                    # train / validation / test


def _default_work_dir() -> str:
    return "/kaggle/working/inklingua_runs" if os.path.isdir("/kaggle/working") else "./inklingua_runs"


# ==============================================================================
# Datasets
# ==============================================================================
@dataclass
class DatasetSpec:
    """One line-level corpus.  The registry (inklingua/datasets_registry.py) holds verified specs; a custom one can be built
    for any Hugging Face dataset (or local parquet folder) that has an image column and a text column."""
    key: str                                   # short name used in cfg.data.sources and in every per-domain table
    hf_dataset: str = ""                       # Hub id, e.g. "Teklia/IAM-line" ("" with local_dir = local parquet only)
    hf_revision: Optional[str] = None          # pinned commit (None = latest)
    local_dir: Optional[str] = None            # folder with train/validation/test .parquet (takes precedence over the Hub)
    language: str = "en"                       # ISO code of the WRITTEN language (prompts, LM scoring, per-domain tables)
    script: str = "Latin"                      # "Latin" | "Arabic" | "Han" | "Cyrillic" | "Devanagari" | ...
    rtl: bool = False                          # right-to-left script: transcriptions are stored in LOGICAL order and are converted to
                                               # visual order (left -> right as the image is read) for training; outputs are converted back
    detokenize: bool = False                   # transcriptions are IAM-tokenised ("house , and"): re-attach punctuation for the LLM
    image_col: Optional[str] = None            # None = auto-detect (image / img / line_image)
    text_col: Optional[str] = None             # None = auto-detect (text / transcription / label / sentence / gt)
    train_split: str = "train"
    val_split: str = "validation"
    test_split: Optional[str] = "test"         # None = this corpus contributes no test lines (e.g. NewsEye has no test split)
    expected_sizes: Optional[Tuple[int, ...]] = None   # (train, val[, test]) as published; checked when strict_checks is on
    max_lines: Optional[Dict[str, int]] = None  # optional cap per split, e.g. {"train": 20000}: a seeded random subset is used
    licence: str = ""                          # for the provenance file
    note: str = ""


# ==============================================================================
# Run / data / model / losses
# ==============================================================================
@dataclass
class RunConfig:
    name: str = "inklingua_iam"
    work_dir: str = field(default_factory=_default_work_dir)   # checkpoints, curves, inference outputs
    seed: int = 1337
    device: str = "auto"                       # "auto" | "cpu" | "cuda" | "cuda:1"
    do_train: bool = True
    do_infer: bool = True
    self_test: bool = True                     # run the fast invariant tests (inklingua.selftest.run_self_tests) before anything else
    smoke_test: bool = False                   # True ONLY for Config.smoke(): permits synthetic data, stamps outputs
    log_level: str = "INFO"
    # "single"        train + evaluate this configuration
    # "suite"         train / evaluate every variant for every seed and write the seed-paired comparison (suite_summary.*)
    # "suite_summary" only re-aggregate the finished runs of a suite (no training)
    # "finetune"      start from cfg.train.init_from (any InkLingua checkpoint), grow the alphabet to the new data, train, evaluate
    # "adapt" / "adapt_curve" / "source_calib"  zero-shot adaptation to a new written language from a folder of line images (AdaptConfig)
    # "report"        rebuild inference/<split>/12_report/index.html of an EXISTING run from its files (no model, no download)
    # "smoke"         the synthetic end-to-end smoke (train -> resume -> inference) in a temporary directory
    mode: str = "single"
    suite_preset: str = "paper"                # "quick" | "paper" | "all" | "custom" (then suite_variants is used)
    suite_variants: Tuple[str, ...] = ("labeled_only", "legacy_pl", "full")
    suite_seeds: Tuple[int, ...] = (1337, 1338, 1339)


@dataclass
class DataConfig:
    source: str = "hf"                         # "hf" (download) | "local" (parquet folder, see local_dir) | "synthetic" (smoke test only)
    sources: Tuple[str, ...] = ("iam",)        # registry keys (inklingua.datasets_registry.DATASETS); several = multi-corpus training
    custom_sources: Tuple[DatasetSpec, ...] = ()   # specs that are not in the registry
    hf_cache_dir: Optional[str] = None         # default: <work_dir>/data
    local_dir: Optional[str] = None            # source="local" with ONE source: the parquet folder of that source
    strict_checks: bool = True                 # True: wrong split sizes / image overlap between splits raise
    img_height: int = 64                       # must be a multiple of 16
    max_img_width: int = 2560                  # Teklia lines are up to ~5000 px wide at 128 px height
    min_img_width: int = 32
    pad_x: int = 8                             # blank margin (px) left/right of every line (keeps augmentations from cropping ink)
    max_text_len: int = 90                     # applied to LABELED lines only (IAM-line maximum is 80)
    labeled_fraction: float = 0.30             # fraction of the train lines whose text is used (the annotation budget)
    calib_fraction: float = 0.10               # share of that budget HELD OUT (never trained on) to calibrate the trust rule of FRAG-RC /
                                               # the legacy pseudo-label selection; 0 = none (FRAG-RC then has no rule)
    drop_nonlinguistic: bool = True            # drop LABELED train lines whose transcription has no letter / digit (e.g. "-----")
    limit_train: Optional[int] = None          # quick experiments (applied to the concatenation of the sources)
    limit_val: Optional[int] = None
    limit_test: Optional[int] = None
    cache_images: bool = True                  # keep the preprocessed uint8 lines in RAM (~1 GB for all of IAM-line)
    batch_size: int = 16
    eval_batch_size: int = 32
    num_workers: int = 2
    bucket_mega: int = 50                      # width-bucketing window (in batches)
    train_eval_subset: int = 400               # labeled train lines used for the "train" curves
    style_num_clusters: int = 10               # K style classes
    style_label_column: Optional[str] = None   # e.g. "writer" if your dataset has writer ids
    val_gate_fraction: float = 0.5             # [PROTOCOL] share of the validation FORM BLOCKS reserved for post-hoc decisions (`val_gate`); the
                                               # rest (`val_sel`) selects the checkpoint.  0 or 1 = no split (validation does both jobs)
    val_block: int = 20                        # consecutive validation lines per block (a form / writer proxy: lines are stored form by form)
    domain_balance: float = 0.0                # multi-corpus: 0 = lines sampled in proportion to corpus size; 1 = every corpus equally often
    synthetic_sizes: Tuple[int, int, int] = (600, 120, 120)
    synthetic_words: Tuple[int, int] = (3, 6)  # words per synthetic line (offline smoke tests only)


@dataclass
class ModelConfig:
    channels: Tuple[int, int, int, int] = (32, 64, 128, 256)
    d_model: int = 256
    enc_layers: int = 4
    heads: int = 8
    ffn: int = 1024
    dropout: float = 0.1
    layer_drop: float = 0.1                    # stochastic depth: drop probability of the LAST encoder layer, linear in depth; 0 = off
    conv_kernel: int = 7
    low_dim: int = 128                         # stroke-level code dimension
    codes_low: int = 256                       # stroke-level codebook size
    codes_high: int = 512                      # component-level codebook size
    vq_decay: float = 0.99
    vq_dead_thresh: float = 0.5
    style_dim: int = 128
    style_input: str = "residual"              # "residual" (z - quantised z) | "continuous"
    film: bool = True
    film_detach_style: bool = True
    film_max: float = 0.5                      # |gamma|, |beta| of the style FiLM are bounded by this value (tanh, no bias)
    lstm_prefix: bool = False                  # optional BiLSTM before the Transformer
    style_grad_scale: float = 0.1              # fraction of the style-task gradient that reaches the shared backbone
    mask_prob: float = 0.25                    # masked-code modelling (span masking)
    mask_span: int = 8


@dataclass
class LossConfig:
    aux_ctc: float = 0.3                       # CTC on the quantised content stream (follows the aux curriculum)
    style_reg: float = 1.0
    style_cls: float = 0.2
    style_cls_smoothing: float = 0.1
    leak: float = 0.1                          # [CFL] closed-form leakage penalty on the content stream
    consistency: float = 0.2                   # weak->strong frame-posterior consistency to the EMA teacher
    masked_code: float = 0.2
    decorrelate: float = 0.05
    vq_commit: float = 0.25
    ctc_smooth: float = 0.02                   # [CTC-LS] label smoothing on EMITTING frames only; 0 = off
    frag: float = 0.5                          # [FRAG-RC] weight of the fragment-level self-distillation (x aux_scale); 0 = off


# ==============================================================================
# Pseudo-labels (legacy, line level) and FRAG-RC (character level)
# ==============================================================================
@dataclass
class PseudoConfig:
    """LEGACY (v3-v9) line-level pseudo-labelling.  Off by default; Config.ablation('legacy_pl') turns it on as a baseline."""
    enabled: bool = False
    warmup_epochs: int = 25
    refresh_every: int = 3
    target_cer: float = 0.02                   # intended CER of the selected pseudo-labels
    risk_z: float = 2.0                        # z of the upper bound (calibrated by simulation, not a confidence level)
    min_calib: int = 40
    min_selected_calib: int = 15
    shadow_calibration: bool = True            # [SHADOW] calibration lines are pseudo-labelled and trained on like unlabeled lines
    risk_folds: int = 5
    risk_smoothing: bool = True                # isotonic risk curve + stop at the FIRST bound violation
    cov_guard: bool = True
    max_cov_growth: float = 0.10
    min_cov_keep: float = 0.50
    cov_ref_window: int = 5                    # [GUARD-REF]
    save_source_calibration: bool = True       # checkpoints/source_calibration.npz, the source side of the zero-shot adapter's [XL-RC]
    guard_cer_factor: float = 1.5
    rel_views: Tuple[str, ...] = ("gauss noise", "shear +0.08", "shear -0.08")
    fallback_tau_conf: float = 0.97
    min_stability: int = 1
    min_chars: int = 2
    lops_drop: float = 0.05
    min_weight: float = 0.50
    consistency: bool = True                   # [PCG] perturbation-consistency gate
    cons_views: Tuple[str, ...] = ("gauss noise", "shear +0.08")
    tau_cons: float = 0.10
    llm_refine: bool = True
    llm_target_cer: float = 0.06
    llm_budget: int = 150
    llm_beam: int = 5
    llm_beta: float = 0.30
    llm_rho: float = 1.0
    llm_max_edit_ratio: float = 0.25
    llm_min_tok_lp: float = -4.5
    llm_weight_scale: float = 0.7


@dataclass
class FragConfig:
    """[FRAG-RC] fragment-level, style-stratified, risk-controlled self-distillation (inklingua/frag.py)."""
    enabled: bool = True
    warmup_epochs: int = 20                    # supervised-only epochs before the first calibration
    refresh_every: int = 4                     # epochs between calibrations of the trust rule
    target_err: float = 0.02                   # intended error rate of the TRUSTED characters (wrong, or next to an omission), per cell
    risk_z: float = 2.0                        # z of the line-clustered upper bound (heuristic, not a confidence level)
    min_chars: int = 400                       # fewer calibration characters in a cell -> nothing is trusted there
    folds: int = 5                             # cross-fitting folds (by line)
    calib_passes: int = 2                      # augmentation draws per calibration line
    guard_frames: int = 1                      # frames around an untrusted character that are never trusted
    sharpen: float = 0.6                       # tau: target = softmax(log p / tau) on trusted frames (1 = plain distillation)
    anchor_weight: float = 0.1                 # weight of the plain-teacher target on the untrusted frames of unlabeled lines
    unlabeled_ratio: float = 1.5               # unlabeled / calibration lines per labeled line in an epoch
    stratum_emphasis: float = 0.5              # labeled lines of harder cells are visited (err_s / mean err)^gamma times more often (1..3); 0 = off
    diag_lines: int = 600                      # unlabeled lines scored against their hidden text at each refresh (DIAGNOSTIC ONLY)
    strata: int = 3                            # hardness tertiles of the label-free score -(z thin stroke + z tight word gaps)
    strata_per_domain: bool = True             # multi-corpus: Mondrian cells = corpus x hardness tertile (one threshold per cell)


# ==============================================================================
# Training / LLM / adaptation / inference
# ==============================================================================
@dataclass
class TrainConfig:
    epochs: int = 100
    lr: float = 6e-4
    min_lr_ratio: float = 0.02
    weight_decay: float = 1e-2
    warmup_steps: int = 600
    grad_clip: float = 5.0
    amp: bool = True
    strong_aug: bool = True
    aug_geom_prob: float = 0.5
    aug_morph_prob: float = 0.24
    aug_blur_prob: float = 0.3
    aug_shear_max: float = 0.10
    aug_vscale: Tuple[float, float] = (0.92, 1.08)
    aug_hscale: Tuple[float, float] = (0.85, 1.15)
    aug_hscale_prob: float = 0.5
    aug_elastic_px: float = 2.0
    aug_elastic_prob: float = 0.3
    aug_tilt_px: float = 3.0
    unlabeled_aux_ratio: float = 0.5           # lines WITHOUT a recognition target per line with one (when FRAG-RC is off)
    style_equivariant: bool = True             # [SES]
    ema_decay: float = 0.999
    aux_start_epoch: int = 3
    aux_ramp_epochs: int = 5
    vq_start_epoch: int = 3
    vq_anneal_epochs: int = 4
    patience: int = 25                         # early stopping on val_sel CER; 0 = never
    min_epochs: int = 30
    min_delta: float = 5e-4
    resume: bool = True
    extend_policy: str = "new_cycle"           # resume with a LARGER epochs: "new_cycle" (warm restart) | "error"
    restart_lr_scale: float = 0.3
    restart_warmup_steps: int = 200
    leak_queue: int = 1024
    leak_ridge: float = 0.3
    leak_slack: float = 0.02
    save_every: int = 5
    keep_last: int = 3
    noise_eval_samples: int = 300
    pseudo: PseudoConfig = field(default_factory=PseudoConfig)
    loss: LossConfig = field(default_factory=LossConfig)
    frag: FragConfig = field(default_factory=FragConfig)
    mixline_prob: float = 0.15                 # [MIXLINE] probability that two adjacent labeled lines of a batch are concatenated (0 = off)
    mixline_gap: Tuple[int, int] = (0, 24)
    # --- warm start / fine-tuning on a new corpus (mode "finetune", or any run) -----------------------------------------------
    init_from: str = ""                        # path of an InkLingua checkpoint (best.pt / last.pt / adapted.pt); "" = train from scratch
    init_lr_scale: float = 0.5                 # peak lr relative to cfg.train.lr when starting from init_from
    init_strict_alphabet: bool = False         # False: the alphabet may GROW (new characters appended as tied copies, [ALPHA-SPLIT])


@dataclass
class LLMConfig:
    backend: str = "causal"                    # "causal" | "seq2seq" | "dummy"
    device: str = "auto"                       # "auto" = the SECOND GPU when there are two (Kaggle T4 x2: recogniser on cuda:0, LLM on cuda:1),
                                               # otherwise the recogniser's device; or an explicit "cuda:1" / "cpu"
    model_name: str = "Qwen/Qwen2.5-1.5B-Instruct"
    seq2seq_models: Dict[str, str] = field(default_factory=lambda: {
        "fr": "Helsinki-NLP/opus-mt-en-fr", "es": "Helsinki-NLP/opus-mt-en-es",
        "de": "Helsinki-NLP/opus-mt-en-de", "it": "Helsinki-NLP/opus-mt-en-it",
        "pt": "Helsinki-NLP/opus-mt-tc-big-en-pt", "ru": "Helsinki-NLP/opus-mt-en-ru",
        "zh": "Helsinki-NLP/opus-mt-en-zh", "ar": "Helsinki-NLP/opus-mt-en-ar",
        "hi": "Helsinki-NLP/opus-mt-en-hi"})
    target_language: str = "fr"
    source_language: str = ""                  # "" = taken from the (first) dataset spec of cfg.data.sources
    max_new_tokens: int = 96
    nbest: int = 3
    word_conf_thresh: float = 0.70             # fallback only: the Inferencer calibrates it on validation words ([CAL-MARK])
    word_conf_agg: str = "mean"
    use_uncertainty_prompt: bool = False       # the '[?]' prompt LOST chrF against the plain prompt in the v7 IAM run
    score_batch_size: int = 8


@dataclass
class AdaptConfig:
    """[ZERO-SHOT] adaptation of a trained checkpoint to handwriting in a language / orthography the training data did not have
    (mode "adapt"; inklingua/adapt.py).  Nothing here is used by a normal run."""
    checkpoint: str = "best"
    source_calibration: str = ""
    lines_dir: str = ""
    anchors_file: str = ""
    eval_file: str = ""
    out_dir: str = ""
    target_languages: Tuple[str, ...] = ("fr",)
    charset: str = ""
    visual_base: Dict[str, str] = field(default_factory=dict)
    rtl: bool = False                          # the target orthography is written right to left (text files are in logical order)
    rounds: int = 4
    epochs_per_round: int = 3
    lr: float = 3e-4
    batch_size: int = 16
    unfreeze_layers: int = 1
    ema_decay: float = 0.9
    max_lines: Optional[int] = None
    target_cer: float = 0.03
    risk_z: float = 2.0
    min_selected: int = 15
    min_ess: float = 10.0
    zero_anchor_inflation: float = 1.5
    anchor_prior_edits: float = 20.0
    anchor_train_share: float = 0.5
    min_anchors_to_train: int = 40
    lex_beam: int = 4
    lex_vis_weight: float = 1.0
    lex_lm_weight: float = 1.0
    lex_rho: float = 0.0
    lex_tie_margin: float = 1.0
    lex_decided_prob: float = 0.98
    lex_max_options: int = 6
    lex_max_lines: int = 400
    translate_max: int = 200
    lex_final_lines: int = 2000
    anchor_limit: Optional[int] = None
    anchor_counts: Tuple[int, ...] = (0, 25, 100, 200)


@dataclass
class InferConfig:
    checkpoint: str = "best"                   # "best" | "last" | explicit path
    split: str = "test"
    max_eval_samples: Optional[int] = None
    beam: int = 5
    n_vis: int = 8
    n_translate: int = 160
    render_translation_max: int = 24
    ig_steps: int = 12
    occl_window: int = 24
    occl_stride: int = 12
    pl_diag_samples: int = 1500
    selective_val_lines: int = 976
    robustness_lines: int = 300
    translation_ablation: bool = True
    render_translation: bool = True
    html_report: bool = True
    llm_rescoring: bool = True
    rescore_beam: int = 8
    rescore_val_lines: int = 976
    rescore_test_lines: Optional[int] = None
    rescore_betas: Tuple[float, ...] = (0.0, 0.1, 0.2, 0.35, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0)
    rescore_extend_grid: bool = True
    rescore_rhos: Tuple[float, ...] = (0.0, 1.0, 2.0, 3.0, 4.0, 6.0)
    rescore_extend_rho: bool = True
    bootstrap: int = 2000
    bootstrap_block: int = 20                  # lines per resampling block (form-clustered bootstrap); 1 = plain line bootstrap
    mark_precision: float = 0.5                # [CAL-MARK]
    posthoc: bool = True
    tta: bool = True
    tta_clusters: int = 8
    tta_shrink: Tuple[float, ...] = (10.0, 30.0, 100.0)
    tta_min_lines: int = 10
    view_ensemble: bool = True
    view_sets: Tuple[Tuple[str, ...], ...] = (("shear +0.08", "shear -0.08"), ("thicker ink", "thinner ink"),
                                              ("shear +0.08", "shear -0.08", "thicker ink", "thinner ink"))
    space_bonus_grid: Tuple[float, ...] = (0.0, 0.5, 1.0, 1.5, 2.0, 3.0)
    error_analysis_lines: int = 3000
    wsd: bool = True                           # [WSD] writer self-distillation (transductive, label-free; adopted only on a val_gate gain)
    wsd_clusters: int = 8
    wsd_steps: Tuple[int, ...] = (20, 60)
    wsd_lr: float = 1e-4
    wsd_batch: int = 16
    wsd_min_lines: int = 24
    wsd_tau: float = 0.6
    wsd_anchor: float = 0.1
    report: bool = True                        # build 12_report/index.html at the end of the evaluation
    # --- the interpretability suite (inklingua/interpret.py); every analysis writes CSV/JSON + PNG into its own folder ---------
    interpret: bool = True
    probe_lines: int = 400                     # lines whose encoder states feed the layer-wise linear probes
    mc_dropout_samples: int = 8                # stochastic forward passes of the uncertainty decomposition (0 = off)
    uncertainty_lines: int = 600
    counterfactual_lines: int = 6              # lines whose characters are erased one by one
    neighbour_queries: int = 6                 # query lines of the style-space neighbour panel
    longline_pairs: int = 300                  # concatenated test pairs of the long-line stress test (0 = off)
    export_alignments: bool = True             # 01_recognition/alignments.jsonl: per-line CTC alignment (char, frames, confidence)


@dataclass
class Config:
    run: RunConfig = field(default_factory=RunConfig)
    data: DataConfig = field(default_factory=DataConfig)
    model: ModelConfig = field(default_factory=ModelConfig)
    train: TrainConfig = field(default_factory=TrainConfig)
    llm: LLMConfig = field(default_factory=LLMConfig)
    infer: InferConfig = field(default_factory=InferConfig)
    adapt: AdaptConfig = field(default_factory=AdaptConfig)

    # ---- derived paths -------------------------------------------------------
    @property
    def run_dir(self) -> str:
        return os.path.join(self.run.work_dir, self.run.name)

    @property
    def ckpt_dir(self) -> str:
        return os.path.join(self.run_dir, "checkpoints")

    @property
    def curve_dir(self) -> str:
        return os.path.join(self.run_dir, "curves")

    @property
    def infer_dir(self) -> str:
        """Evaluation outputs of the configured split: <run>/inference/<split>/ (one folder per split that was looked at)."""
        return os.path.join(self.run_dir, "inference", self.infer.split)

    @property
    def data_dir(self) -> str:
        return self.data.hf_cache_dir or os.path.join(self.run.work_dir, "data")

    def make_dirs(self) -> None:
        for d in (self.run_dir, self.ckpt_dir, self.curve_dir, self.infer_dir, self.data_dir):
            os.makedirs(d, exist_ok=True)

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

    def save(self, path: Optional[str] = None) -> None:
        path = path or os.path.join(self.run_dir, "config.json")
        with open(path, "w") as f:
            json.dump(self.to_dict(), f, indent=2)

    @classmethod
    def from_dict(cls, d: Dict[str, Any]) -> "Config":
        return dc_from_dict(cls, d)

    # ---- variants ------------------------------------------------------------------------------------------------------------
    def with_seed(self, seed: int) -> "Config":
        c = copy.deepcopy(self)
        c.run.seed = seed
        c.run.name = f"{self.run.name}_s{seed}"
        return c

    @classmethod
    def multi(cls, sources: Tuple[str, ...], name: Optional[str] = None) -> "Config":
        """Train on several registry corpora at once (e.g. ("iam", "rimes", "khatt")).  The alphabet is the union of the labeled
        lines, every line carries its corpus id, FRAG-RC uses one trust threshold per corpus x hardness cell, and every metric is also
        reported per corpus."""
        c = cls()
        c.data.sources = tuple(sources)
        c.run.name = name or "inklingua_" + "_".join(sources)
        return c

    @classmethod
    def full_supervised(cls) -> "Config":
        """Upper-bound baseline: all training lines labeled, no self-distillation."""
        c = cls()
        c.run.name = "inklingua_iam_fullsup"
        c.data.labeled_fraction, c.data.calib_fraction = 1.0, 0.0
        c.train.pseudo.enabled = c.train.frag.enabled = False
        return c

    @classmethod
    def labeled_only(cls, base: Optional["Config"] = None) -> "Config":
        """THE baseline: identical model, augmentation (incl. MixLine) and labeled lines, but the unlabeled lines are never used
        (no pseudo-labels, no FRAG-RC, no calibration split -- it even trains on the lines `full` holds out for calibration)."""
        c = copy.deepcopy(base) if base is not None else cls()
        c.run.name = f"{c.run.name}_labeled_only"
        c.train.pseudo.enabled = c.train.frag.enabled = False
        c.data.calib_fraction = 0.0
        return c

    @classmethod
    def make_variant(cls, name: str, base: Optional["Config"] = None) -> "Config":
        """'full' -> a copy of `base`; 'labeled_only' | 'full_supervised' | any ablation kind (see `ablation`)."""
        b = base if base is not None else cls()
        if name == "full":
            c = copy.deepcopy(b)
            c.run.name = f"{c.run.name}_full"
            return c
        if name == "labeled_only":
            return cls.labeled_only(b)
        if name == "full_supervised":
            c = copy.deepcopy(b)
            c.run.name = f"{c.run.name}_full_supervised"
            c.data.labeled_fraction, c.data.calib_fraction = 1.0, 0.0
            c.train.pseudo.enabled = c.train.frag.enabled = False
            return c
        return cls.ablation(name, b)

    ABLATIONS = ("no_pseudo", "no_frag", "legacy_pl", "no_mixline", "no_wsd", "no_consistency", "no_llm", "no_style_task", "no_ses",
                 "no_vq", "no_film", "no_confusion", "no_frame_cons", "no_hscale", "no_calibration", "no_mask", "no_aux_ctc",
                 "no_shadow", "no_risk_smoothing", "no_v9_reg", "no_posthoc", "no_strata", "no_sharpen", "ctc_only")

    @classmethod
    def ablation(cls, kind: str, base: Optional["Config"] = None) -> "Config":
        """One component removed.  kind in Config.ABLATIONS:
          no_pseudo (no unlabeled data at all), no_frag (no FRAG-RC; labeled + calibration images only), legacy_pl (v9 line-level
          pseudo-labels instead of FRAG-RC), no_mixline, no_wsd, no_strata (one global FRAG-RC threshold), no_sharpen (plain
          distillation targets), no_consistency / no_llm (legacy gates), no_style_task, no_ses, no_vq, no_film, no_confusion
          (= no leakage penalty / decorrelation), no_frame_cons, no_hscale, no_calibration, no_mask, no_aux_ctc, no_shadow,
          no_risk_smoothing, no_v9_reg (no layer drop / CTC label smoothing), no_posthoc, ctc_only (plain CTC Transformer)."""
        c = copy.deepcopy(base) if base is not None else cls()
        c.run.name = f"{c.run.name}_abl_{kind}"
        L, P, Fg = c.train.loss, c.train.pseudo, c.train.frag
        if kind == "no_pseudo":
            P.enabled = Fg.enabled = False
        elif kind == "no_frag":
            Fg.enabled = False
        elif kind == "legacy_pl":
            Fg.enabled, P.enabled = False, True
        elif kind == "no_mixline":
            c.train.mixline_prob = 0.0
        elif kind == "no_wsd":
            c.infer.wsd = False
        elif kind == "no_strata":
            Fg.strata, Fg.strata_per_domain, Fg.stratum_emphasis = 1, False, 0.0
        elif kind == "no_sharpen":
            Fg.sharpen = 1.0
        elif kind == "no_consistency":
            P.consistency = False
        elif kind == "no_llm":
            P.llm_refine = False
        elif kind == "no_style_task":
            L.style_reg = L.style_cls = L.leak = L.decorrelate = 0.0
            c.model.film = False
        elif kind == "no_ses":
            c.train.style_equivariant = False
        elif kind == "no_vq":
            c.train.vq_start_epoch = 10 ** 6
        elif kind == "no_film":
            c.model.film = False
        elif kind == "no_confusion":
            L.leak = L.decorrelate = 0.0
        elif kind == "no_frame_cons":
            L.consistency = 0.0
        elif kind == "no_hscale":
            c.train.aug_hscale_prob = 0.0
        elif kind == "no_calibration":
            c.data.calib_fraction = 0.0
        elif kind == "no_mask":
            L.masked_code = 0.0
        elif kind == "no_aux_ctc":
            L.aux_ctc = 0.0
        elif kind == "no_shadow":
            P.shadow_calibration = False
        elif kind == "no_risk_smoothing":
            P.risk_smoothing, P.cov_guard = False, False
        elif kind == "no_v9_reg":
            c.model.layer_drop, L.ctc_smooth = 0.0, 0.0
        elif kind == "no_posthoc":
            c.infer.posthoc = False
        elif kind == "ctc_only":
            P.enabled = Fg.enabled = False
            c.infer.wsd = False
            c.data.calib_fraction = 0.0
            L.aux_ctc = L.style_reg = L.style_cls = L.leak = L.consistency = L.masked_code = L.decorrelate = L.vq_commit = 0.0
            c.train.vq_start_epoch = 10 ** 6
            c.model.film = False
        else:
            raise ValueError(f"unknown ablation '{kind}' (choose from {cls.ABLATIONS})")
        return c

    @classmethod
    def smoke(cls, work_dir: str = "./inklingua_smoke") -> "Config":
        """Tiny SYNTHETIC configuration used to validate the whole pipeline offline.  Its numbers mean nothing scientifically;
        outputs are stamped as such."""
        c = cls()
        c.run.name, c.run.work_dir, c.run.smoke_test = "smoke", work_dir, True
        c.data.source, c.data.sources = "synthetic", ("synthetic",)
        c.data.img_height, c.data.max_img_width, c.data.pad_x = 32, 512, 4
        c.data.batch_size, c.data.eval_batch_size, c.data.num_workers = 16, 16, 0
        c.data.synthetic_sizes, c.data.labeled_fraction = (1200, 48, 48), 0.5
        c.data.synthetic_words = (2, 3)
        c.data.train_eval_subset = 48
        c.model.channels, c.model.d_model, c.model.heads = (8, 16, 32, 48), 64, 4
        c.model.ffn, c.model.enc_layers = 128, 2
        c.model.low_dim, c.model.codes_low, c.model.codes_high, c.model.style_dim = 32, 32, 64, 32
        c.train.epochs, c.train.lr, c.train.warmup_steps = 24, 2e-3, 60
        c.train.aug_tilt_px = 1.5
        c.train.aux_start_epoch, c.train.aux_ramp_epochs = 5, 5
        c.train.vq_start_epoch, c.train.vq_anneal_epochs = 5, 5
        c.train.save_every, c.train.noise_eval_samples = 4, 30
        c.train.pseudo.warmup_epochs, c.train.pseudo.refresh_every = 10, 3
        c.train.pseudo.min_stability, c.train.pseudo.target_cer, c.train.pseudo.min_calib = 0, 0.12, 30
        c.train.frag.warmup_epochs, c.train.frag.refresh_every, c.train.frag.min_chars, c.train.frag.diag_lines = 8, 4, 30, 40
        c.train.frag.target_err, c.train.mixline_prob = 0.12, 0.2
        c.data.val_block = 8
        c.infer.wsd_steps, c.infer.wsd_min_lines, c.infer.wsd_clusters, c.infer.wsd_batch = (3, 6), 6, 2, 8
        c.train.patience, c.train.leak_queue = 0, 256
        c.data.calib_fraction = 0.15
        c.train.pseudo.llm_refine = False
        c.llm.backend = "dummy"
        c.infer.n_vis, c.infer.n_translate, c.infer.ig_steps = 3, 4, 4
        c.infer.pl_diag_samples, c.infer.occl_window, c.infer.occl_stride = 60, 16, 8
        c.infer.llm_rescoring, c.infer.bootstrap = False, 200
        c.infer.probe_lines, c.infer.mc_dropout_samples, c.infer.uncertainty_lines = 48, 3, 48
        c.infer.counterfactual_lines, c.infer.neighbour_queries, c.infer.longline_pairs = 2, 2, 20
        return c


# ==============================================================================
# (de)serialisation helpers
# ==============================================================================
def _tupled(v):
    """JSON gives lists; the config uses tuples (also nested ones, e.g. view_sets)."""
    return tuple(_tupled(x) if isinstance(x, list) else x for x in v)


def dc_from_dict(cls, d: Dict[str, Any]):
    """Rebuild a (nested) dataclass from a plain dict (JSON round trip).  Unknown keys are ignored, so a config saved by an
    older version still loads."""
    kwargs = {}
    for f in dataclasses.fields(cls):
        if f.name not in d:
            continue
        v = d[f.name]
        dflt = f.default_factory() if f.default_factory is not dataclasses.MISSING else f.default  # type: ignore
        if dataclasses.is_dataclass(dflt) and isinstance(v, dict):
            v = dc_from_dict(type(dflt), v)
        elif f.name == "custom_sources" and isinstance(v, (list, tuple)):
            v = tuple(dc_from_dict(DatasetSpec, x) if isinstance(x, dict) else x for x in v)
        elif isinstance(dflt, tuple) and isinstance(v, list):
            v = _tupled(v)
        kwargs[f.name] = v
    return cls(**kwargs)


SUITE_PRESETS: Dict[str, Tuple[str, ...]] = {
    # the cheapest decisive experiment: does the unlabeled data help at all?
    "quick": ("labeled_only", "full"),
    # what a reviewer needs: both baselines, the previous method, and each v10/v11 component removed once
    "paper": ("labeled_only", "ctc_only", "legacy_pl", "no_frag", "no_wsd", "no_mixline", "no_strata", "full"),
    # everything (expensive)
    "all": ("labeled_only", "ctc_only", "full_supervised", "legacy_pl") + tuple(k for k in Config.ABLATIONS if k not in ("legacy_pl", "no_pseudo")) + ("full",),
}

In [ ]:
%%writefile inklingua/utils.py
"""Generic utilities: seeding, strict JSON, logging, devices, atomic checkpoints, running means."""
from __future__ import annotations

import logging
import math
import os
import random
import sys
from collections import defaultdict
from typing import Any, Dict, Optional

import numpy as np
import torch

LOG = logging.getLogger("inklingua")


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False        # variable line widths: autotuning would only add noise


def json_safe(o: Any) -> Any:
    """Recursively convert numpy types and replace NaN / inf by null (strict JSON)."""
    if isinstance(o, dict):
        return {str(k): json_safe(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [json_safe(v) for v in o]
    if isinstance(o, (np.floating, np.integer, np.bool_)):
        o = o.item()
    if isinstance(o, np.ndarray):
        return json_safe(o.tolist())
    if isinstance(o, float) and (math.isnan(o) or math.isinf(o)):
        return None
    return o


_LOG_STACK = []


def setup_logging(cfg, replace: bool = True) -> None:
    """Log to stdout and to <run_dir>/run.log.  The v10 self-tests called main() for a synthetic run, which replaced the
    handlers of the REAL run with handlers pointing into a temporary directory that was then deleted; here the previous handlers
    are remembered (`push`) and `restore_logging()` puts them back."""
    os.makedirs(cfg.run_dir, exist_ok=True)
    if replace:
        _LOG_STACK.append((list(LOG.handlers), LOG.level))
    LOG.setLevel(getattr(logging, cfg.run.log_level.upper(), logging.INFO))
    for h in list(LOG.handlers):
        LOG.removeHandler(h)
        try:
            h.close()
        except Exception:
            pass
    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s", "%H:%M:%S")
    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(fmt)
    fh = logging.FileHandler(os.path.join(cfg.run_dir, "run.log"))
    fh.setFormatter(fmt)
    LOG.addHandler(sh)
    LOG.addHandler(fh)
    LOG.propagate = False


def restore_logging() -> None:
    """Undo the last setup_logging (used after a nested run such as the smoke test)."""
    if not _LOG_STACK:
        return
    handlers, level = _LOG_STACK.pop()
    for h in list(LOG.handlers):
        LOG.removeHandler(h)
        try:
            h.close()
        except Exception:
            pass
    for h in handlers:
        LOG.addHandler(h)
    LOG.setLevel(level)


def resolve_device(name: str) -> torch.device:
    if name == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(name)


def resolve_llm_device(name: str, main_device: torch.device) -> torch.device:
    """Device of the LLM: "auto" puts it on the second GPU when two are available (the recogniser keeps the first), else next to the recogniser."""
    if name == "auto":
        if main_device.type == "cuda" and torch.cuda.device_count() > 1:
            other = 1 if (main_device.index or 0) == 0 else 0
            return torch.device(f"cuda:{other}")
        return main_device
    return torch.device(name)


def atomic_save(obj: Any, path: str) -> None:
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)


def torch_load(path: str, map_location="cpu") -> Any:
    """Our own checkpoints contain numpy arrays / lists -> weights_only=False is intended."""
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:  # very old torch
        return torch.load(path, map_location=map_location)


def to_device(batch: Dict[str, Any], device: torch.device) -> Dict[str, Any]:
    return {k: (v.to(device, non_blocking=True) if torch.is_tensor(v) else v) for k, v in batch.items()}


class Meter:
    """Weighted running means for a dict of scalars."""

    def __init__(self):
        self.sum: Dict[str, float] = defaultdict(float)
        self.cnt: Dict[str, float] = defaultdict(float)

    def add(self, key: str, value: Optional[float], n: float = 1.0) -> None:
        if value is None or (isinstance(value, float) and (math.isnan(value) or math.isinf(value))):
            return
        self.sum[key] += float(value) * n
        self.cnt[key] += n

    def mean(self, key: str, default: float = float("nan")) -> float:
        return self.sum[key] / self.cnt[key] if self.cnt[key] > 0 else default

    def means(self) -> Dict[str, float]:
        return {k: self.mean(k) for k in self.sum}

In [ ]:
%%writefile inklingua/text.py
"""
Text: grapheme-cluster codec, right-to-left handling, IAM detokenisation, edit distance, CER / WER / chrF, character
classes, form-clustered bootstrap intervals and the "what does the transcription explain" regression.

Script-agnostic by design (v11):
  * the CTC alphabet is made of GRAPHEME CLUSTERS (base character + combining marks), not code points, so Devanagari matras,
    Arabic diacritics or 'e + dot below + acute' are one output unit each (v10 split them, which broke CTC feasibility counts
    and made the zero-shot adapter's units inconsistent with the trained model);
  * right-to-left scripts: the recogniser reads the image left to right, so an RTL transcription (logical order) is converted
    to VISUAL order (reversed cluster by cluster) for training and metrics -- edit distance is invariant under reversing both
    strings, so CER / WER do not change -- and converted back for every human-facing output and the LLM (`to_logical`);
  * character classes use Unicode categories, so uncased letters (Arabic, Chinese, ...) are letters, not punctuation.
"""
from __future__ import annotations

import math
import re
import unicodedata
from collections import Counter
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np

try:
    import editdistance as _editdistance
except Exception:  # pragma: no cover
    _editdistance = None


# ---------------------------------------------------------------- normalisation, grapheme clusters, direction
def normalize_text(t: str) -> str:
    t = t.replace(" ", " ").replace("\t", " ").replace("\n", " ")
    return re.sub(r"\s+", " ", t).strip()


def grapheme_clusters(text: str) -> List[str]:
    """Base character + its combining marks = one output unit (NFC text: 'e\\u0323\\u0301' keeps the dot below and the acute
    together, which no precomposed code point exists for).  Zero-width joiners inside a cluster stay attached to it."""
    out: List[str] = []
    for ch in unicodedata.normalize("NFC", text):
        if out and (unicodedata.combining(ch) or ch in ("‍", "‌")):
            out[-1] += ch
        else:
            out.append(ch)
    return out


def to_visual(text: str, rtl: bool) -> str:
    """Logical -> visual order for an RTL script (the order in which the recogniser reads the image, left to right)."""
    return "".join(reversed(grapheme_clusters(text))) if rtl else text


def to_logical(text: str, rtl: bool) -> str:
    """Visual -> logical order (the inverse of to_visual; reversal is its own inverse)."""
    return to_visual(text, rtl)


# ---------------------------------------------------------------- IAM detokenisation (English only)
_DQ = ('"', "``", "''")
_CONTRACTION = {"n't", "'s", "'re", "'ve", "'ll", "'d", "'m"}


def _quote_roles(toks: List[str], pos: List[int]) -> List[str]:
    """Opening / closing role of the quote marks at `pos` (token positions of one quote family).  Quotes alternate; only the
    state at the start of the line is unknown (IAM lines cut a quotation anywhere), so it is inferred: with an odd number of
    marks either the first one closes a quotation that began on the previous line or the last one opens one that continues on
    the next line -- the last one opens if a word follows it and it is preceded by a comma / colon, another quote mark or a word."""
    if not pos:
        return []
    first_closes = False
    if len(pos) % 2 == 1:
        k = pos[-1]
        has_next = k + 1 < len(toks)
        prev = toks[k - 1] if k > 0 else ""
        last_opens = has_next and (k == 0 or prev in _DQ or prev == "'" or prev[-1:] in ",:;" or prev[-1:].isalnum())
        first_closes = not last_opens
    return [("close" if (j % 2 == 0) == first_closes else "open") for j in range(len(pos))]


def detokenize(t: str) -> str:
    """IAM transcriptions are tokenised ("house , and"; \" Left here \" ; It 's).  Re-attach punctuation, brackets, contractions
    and quotation marks the way the sentence would be written -- for the LLM prompt / LM scoring.  No character other than spaces
    is ever added, dropped or moved (tested).  Only applied to corpora whose DatasetSpec.detokenize is True."""
    toks = t.replace(" ", " ").split()
    n = len(toks)
    roles: List[str] = ["word"] * n
    for family in (_DQ, ("'",)):
        pos = [i for i, w in enumerate(toks) if w in family]
        for i, r in zip(pos, _quote_roles(toks, pos)):
            roles[i] = r
    for i, w in enumerate(toks):
        if roles[i] != "word":
            continue
        if w in _CONTRACTION or (w and all(c in ".,;:!?%)]}" for c in w)):
            roles[i] = "close"
        elif w and all(c in "([{$£" for c in w):
            roles[i] = "open"
    out, glue = "", True
    for w, r in zip(toks, roles):
        if r == "close":
            out += w
            glue = False
        elif r == "open":
            out += ("" if glue else " ") + w
            glue = True
        else:
            out += ("" if glue else " ") + w
            glue = False
    return out.strip()


# ---------------------------------------------------------------- the codec
class CharCodec:
    """Index 0 is the CTC blank; every other index is one GRAPHEME CLUSTER.  The alphabet is built from the transcripts of the
    LABELED training lines only (transcriptions of unlabeled lines are never read by the training pipeline); units that occur only
    in validation / test simply count as errors.
    `chars` is the joined string of the units (kept for compatibility with older checkpoints: re-clustering it gives the units back)."""

    BLANK = 0

    def __init__(self, units):
        if isinstance(units, str):
            units = grapheme_clusters(units)
        self.units: List[str] = list(units)
        self.c2i = {u: i + 1 for i, u in enumerate(self.units)}
        self.i2c = {i + 1: u for i, u in enumerate(self.units)}

    @classmethod
    def from_texts(cls, texts: Sequence[str]) -> "CharCodec":
        units = set()
        for t in texts:
            units.update(grapheme_clusters(t))
        return cls(sorted(units))

    @property
    def chars(self) -> str:
        return "".join(self.units)

    @property
    def num_classes(self) -> int:
        return len(self.units) + 1

    def encode(self, text: str) -> List[int]:
        return [self.c2i[u] for u in grapheme_clusters(text) if u in self.c2i]

    def decode_ids(self, ids: Sequence[int]) -> str:
        return "".join(self.i2c.get(int(i), "") for i in ids)

    def __eq__(self, other) -> bool:
        return isinstance(other, CharCodec) and other.units == self.units


def ctc_feasible(text: str, frames: int) -> bool:
    """A CTC alignment exists only if the text fits into the frames (repeated units need a blank in between)."""
    u = grapheme_clusters(text)
    need = len(u) + sum(1 for a, b in zip(u, u[1:]) if a == b)
    return len(u) > 0 and need <= frames


def ctc_min_frames(ids: Sequence[int]) -> int:
    return len(ids) + sum(1 for a, b in zip(ids, ids[1:]) if a == b)


# ---------------------------------------------------------------- metrics
def edit_distance(a: Sequence, b: Sequence) -> int:
    if _editdistance is not None:
        return int(_editdistance.eval(a, b))
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


def cer(ref: str, hyp: str) -> float:
    return edit_distance(ref, hyp) / max(len(ref), 1)


def wer(ref: str, hyp: str) -> float:
    r, h = ref.split(), hyp.split()
    return edit_distance(r, h) / max(len(r), 1)


def align_chars(ref: str, hyp: str) -> List[Tuple[str, Optional[int], Optional[int]]]:
    """Levenshtein alignment. Returns ops ('match'|'sub'|'ins'|'del', ref_idx, hyp_idx)."""
    n, m = len(ref), len(hyp)
    D = np.zeros((n + 1, m + 1), dtype=np.int32)
    D[:, 0] = np.arange(n + 1)
    D[0, :] = np.arange(m + 1)
    for i in range(1, n + 1):
        ri = ref[i - 1]
        for j in range(1, m + 1):
            D[i, j] = min(D[i - 1, j] + 1, D[i, j - 1] + 1, D[i - 1, j - 1] + (ri != hyp[j - 1]))
    ops, i, j = [], n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and D[i, j] == D[i - 1, j - 1] + (ref[i - 1] != hyp[j - 1]):
            ops.append(("match" if ref[i - 1] == hyp[j - 1] else "sub", i - 1, j - 1))
            i, j = i - 1, j - 1
        elif i > 0 and D[i, j] == D[i - 1, j] + 1:
            ops.append(("del", i - 1, None))
            i -= 1
        else:
            ops.append(("ins", None, j - 1))
            j -= 1
    return ops[::-1]


def chrf(hyp: str, ref: str, max_n: int = 6, beta: float = 2.0) -> float:
    """Character n-gram F-score (chrF, 0..100; Popovic 2015: precision and recall averaged over the n-gram orders 1..6, then one
    F-beta with beta = 2; spaces ignored).  sacreBLEU averages the F-score per order instead; the two differ by < 0.01 chrF on
    sentence pairs, so numbers are comparable but not byte-identical."""
    h, r = hyp.replace(" ", ""), ref.replace(" ", "")
    if not h or not r:
        return 100.0 if h == r else 0.0
    ps, rs = [], []
    for n in range(1, max_n + 1):
        H = Counter(h[i:i + n] for i in range(len(h) - n + 1))
        R = Counter(r[i:i + n] for i in range(len(r) - n + 1))
        if not H or not R:
            continue
        ov = sum((H & R).values())
        ps.append(ov / sum(H.values()))
        rs.append(ov / sum(R.values()))
    if not ps:
        return 0.0
    P, R_ = float(np.mean(ps)), float(np.mean(rs))
    if P + R_ == 0:
        return 0.0
    return 100.0 * (1 + beta ** 2) * P * R_ / (beta ** 2 * P + R_)


CHAR_CLASSES = ("lower", "upper", "letter", "digit", "punct", "space")


def char_class(c: str) -> str:
    """Script-agnostic character class: cased letters keep their case, uncased letters (Arabic, Han, ...) are 'letter'."""
    if c == " ":
        return "space"
    cat = unicodedata.category(c[0])
    if cat == "Ll":
        return "lower"
    if cat == "Lu" or cat == "Lt":
        return "upper"
    if cat.startswith("L") or cat.startswith("M"):
        return "letter"
    if cat.startswith("N"):
        return "digit"
    return "punct"


def is_linguistic(t: str) -> bool:
    """A reference with at least one letter or digit (form separators such as '-----' are not handwriting)."""
    return any(unicodedata.category(c).startswith(("L", "N")) for c in t)


# ---------------------------------------------------------------- bootstrap intervals (form-clustered)
MIN_BOOTSTRAP_GROUPS = 8


def group_index(groups: Optional[Sequence[int]], n: int) -> np.ndarray:
    """Consecutive group ids 0..G-1 of the lines (None -> every line is its own group)."""
    if groups is None:
        return np.arange(n)
    g = np.asarray(groups)
    if len(g) != n:
        raise ValueError(f"groups has {len(g)} entries for {n} lines")
    inv = np.unique(g, return_inverse=True)[1]
    if int(inv.max()) + 1 < MIN_BOOTSTRAP_GROUPS:       # resampling < 8 blocks is degenerate: fall back to lines (too narrow, not degenerate)
        return np.arange(n)
    return inv


def group_sums(gid: np.ndarray, *arrs: np.ndarray) -> List[np.ndarray]:
    G = int(gid.max()) + 1 if len(gid) else 0
    return [np.bincount(gid, weights=np.asarray(a, np.float64), minlength=G) for a in arrs]


def bootstrap_cer_ci(ed: np.ndarray, ref_len: np.ndarray, n: int = 1000, seed: int = 0,
                     groups: Optional[Sequence[int]] = None) -> Tuple[float, float]:
    """95% percentile-bootstrap CI of the corpus CER = sum(ed) / sum(ref_len).  groups=None resamples LINES.  Evaluation lines
    are stored form by form and the errors cluster by form / writer (v3 test run: lag-1 autocorrelation of the line CER 0.46,
    design effect 4), so a line-level bootstrap is about HALF as wide as it should be.  Pass groups = line_index // block to
    resample whole blocks (cluster bootstrap)."""
    ed, ref_len = np.asarray(ed, np.float64), np.asarray(ref_len, np.float64)
    if len(ed) == 0 or n <= 0:
        return float("nan"), float("nan")
    gid = group_index(groups, len(ed))
    E, L = group_sums(gid, ed, ref_len)
    rng = np.random.RandomState(seed)
    idx = rng.randint(0, len(E), size=(n, len(E)))
    vals = E[idx].sum(1) / np.maximum(L[idx].sum(1), 1.0)
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


def paired_bootstrap_delta(ed_a: np.ndarray, ed_b: np.ndarray, ref_len: np.ndarray, n: int = 1000, seed: int = 0,
                           groups: Optional[Sequence[int]] = None, n_tests: int = 1) -> Dict[str, float]:
    """Paired (cluster) bootstrap of CER(a) - CER(b) on the same lines (negative = a is better).  lo / hi are the 95 % limits;
    lo_adj / hi_adj the limits at 95 % / n_tests (Bonferroni): use hi_adj < 0 to adopt the best of n_tests candidates chosen on
    the SAME lines."""
    ed_a, ed_b, ref_len = (np.asarray(v, np.float64) for v in (ed_a, ed_b, ref_len))
    if len(ed_a) == 0 or n <= 0:
        nan = float("nan")
        return {"delta": nan, "lo": nan, "hi": nan, "lo_adj": nan, "hi_adj": nan, "p_a_better": nan}
    gid = group_index(groups, len(ed_a))
    Ea, Eb, L = group_sums(gid, ed_a, ed_b, ref_len)
    rng = np.random.RandomState(seed)
    idx = rng.randint(0, len(Ea), size=(n, len(Ea)))
    d = (Ea[idx].sum(1) - Eb[idx].sum(1)) / np.maximum(L[idx].sum(1), 1.0)
    tail = 2.5 / max(int(n_tests), 1)
    return {"delta": float((ed_a.sum() - ed_b.sum()) / max(ref_len.sum(), 1.0)), "lo": float(np.percentile(d, 2.5)),
            "hi": float(np.percentile(d, 97.5)), "lo_adj": float(np.percentile(d, tail)),
            "hi_adj": float(np.percentile(d, 100.0 - tail)), "p_a_better": float((d < 0).mean())}


def paired_mean_ci(a: Sequence[float], b: Sequence[float], n: int = 1000, seed: int = 0,
                   groups: Optional[Sequence[int]] = None) -> Dict[str, float]:
    """Mean of (a - b) over lines with a paired-bootstrap 95 % interval, plus how often a is better / worse (form-clustered)."""
    d = np.asarray(a, np.float64) - np.asarray(b, np.float64)
    if len(d) == 0:
        return {"delta": float("nan"), "lo": float("nan"), "hi": float("nan"), "n": 0}
    rng = np.random.RandomState(seed)
    if groups is None:
        bs = d[rng.randint(0, len(d), (n, len(d)))].mean(1)
    else:
        gid = group_index(groups, len(d))
        G = int(gid.max()) + 1
        sums, cnt = group_sums(gid, d, np.ones_like(d))
        pick = rng.randint(0, G, (n, G))
        bs = sums[pick].sum(1) / np.maximum(cnt[pick].sum(1), 1)
    return {"delta": float(d.mean()), "lo": float(np.percentile(bs, 2.5)), "hi": float(np.percentile(bs, 97.5)),
            "n": int(len(d)), "share_better": float((d > 1e-9).mean()), "share_worse": float((d < -1e-9).mean())}


# ---------------------------------------------------------------- what does the TRANSCRIPTION explain of a style descriptor?
def text_features(texts: Sequence[str], units: Sequence[str]) -> np.ndarray:
    """Bag-of-units description of transcriptions: unit proportions, log length, words / 10, mean word length / 10."""
    idx = {c: i for i, c in enumerate(units)}
    X = np.zeros((len(texts), len(units) + 3), np.float32)
    for r, t in enumerate(texts):
        cl = grapheme_clusters(t)
        n = max(len(cl), 1)
        for c in cl:
            j = idx.get(c)
            if j is not None:
                X[r, j] += 1.0 / n
        nw = max(len(t.split()), 1)
        X[r, -3], X[r, -2], X[r, -1] = math.log1p(len(cl)), nw / 10.0, (len(cl) / nw) / 10.0
    return X


def ridge_cv_r2(X: np.ndarray, Y: np.ndarray, OK: np.ndarray, folds: int = 5, lam: float = 0.1, seed: int = 0) -> np.ndarray:
    """Cross-validated linear R^2 of every column of Y from X (closed-form ridge, centred on the TRAINING fold).  Column j uses
    only the rows with OK[:, j]; the result is clipped to [0, 1]."""
    n = len(X)
    out = np.zeros(Y.shape[1], np.float32)
    if n < 2 * folds:
        return out
    fold = np.random.RandomState(seed).permutation(n) % folds
    Xs = (X - X.mean(0)) / (X.std(0) + 1e-6)
    for j in range(Y.shape[1]):
        ok = OK[:, j]
        if ok.sum() < 3 * folds:
            continue
        ybar, sse, sst = float(Y[ok, j].mean()), 0.0, 0.0
        for f in range(folds):
            tr, te = ok & (fold != f), ok & (fold == f)
            if te.sum() == 0 or tr.sum() < 5:
                continue
            xm, ym = Xs[tr].mean(0), float(Y[tr, j].mean())
            A = Xs[tr] - xm
            Wt = np.linalg.solve(A.T @ A + lam * len(A) * np.eye(A.shape[1]), A.T @ (Y[tr, j] - ym))
            sse += float(((Y[te, j] - ((Xs[te] - xm) @ Wt + ym)) ** 2).sum())
            sst += float(((Y[te, j] - ybar) ** 2).sum())
        if sst > 0:
            out[j] = float(np.clip(1.0 - sse / sst, 0.0, 1.0))
    return out


def text_baseline_r2(texts: Sequence[str], Z: np.ndarray, OK: np.ndarray, units: Sequence[str], seed: int = 0) -> np.ndarray:
    return ridge_cv_r2(text_features(texts, units), np.asarray(Z, np.float64), np.asarray(OK, bool), seed=seed)

In [ ]:
%%writefile inklingua/decoding.py
"""CTC decoding: greedy with per-character confidences, prefix beam search N-best, the word-gap (space) bonus."""
from __future__ import annotations

import math
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np


def greedy_decode(logp: np.ndarray) -> Tuple[List[int], List[float], List[int]]:
    """logp: [T, V] log-probabilities (valid frames only). Returns ids, per-char confidence (peak probability inside the
    emitting run) and the peak frame of every char."""
    if logp.shape[0] == 0:
        return [], [], []
    p = np.exp(logp)
    idx, mp = p.argmax(1), p.max(1)
    starts = np.flatnonzero(np.r_[True, idx[1:] != idx[:-1]])
    ends = np.r_[starts[1:], len(idx)]
    run_ids = idx[starts]
    run_max = np.maximum.reduceat(mp, starts)
    ids, confs, frames = [], [], []
    for k in np.flatnonzero(run_ids != 0):
        ids.append(int(run_ids[k]))
        confs.append(float(run_max[k]))
        frames.append(int(starts[k] + mp[starts[k]:ends[k]].argmax()))
    return ids, confs, frames


def greedy_runs(logp: np.ndarray) -> Tuple[List[int], np.ndarray, np.ndarray]:
    """Emitted ids with the first frame and the end frame (exclusive) of every emitting run (the CTC alignment)."""
    if logp.shape[0] == 0:
        return [], np.zeros(0, np.int64), np.zeros(0, np.int64)
    idx = logp.argmax(1)
    starts = np.flatnonzero(np.r_[True, idx[1:] != idx[:-1]])
    ends = np.r_[starts[1:], len(idx)]
    keep = idx[starts] != 0
    return [int(v) for v in idx[starts][keep]], starts[keep].astype(np.int64), ends[keep].astype(np.int64)


def seq_confidence(confs: Sequence[float]) -> float:
    if len(confs) == 0:
        return 0.0
    return float(np.exp(np.mean(np.log(np.clip(confs, 1e-6, 1.0)))))


def _lse(a: float, b: float) -> float:
    if a < b:
        a, b = b, a
    if b <= -1e29:
        return a
    return a + math.log1p(math.exp(b - a))


def ctc_prefix_beam_search(logp: np.ndarray, beam: int = 5, prune: float = -7.0) -> List[Tuple[Tuple[int, ...], float]]:
    """Standard CTC prefix beam search. Returns [(ids, log_prob)] sorted best-first.  (A 5-best prefix beam, not an exact N-best.)"""
    NEG = -1e30
    T, V = logp.shape
    beams: Dict[Tuple[int, ...], Tuple[float, float]] = {(): (0.0, NEG)}
    for t in range(T):
        row = logp[t].tolist()
        cand = [c for c in range(V) if row[c] > prune] or [int(np.argmax(row))]
        nxt: Dict[Tuple[int, ...], List[float]] = {}

        def slot(k):
            if k not in nxt:
                nxt[k] = [NEG, NEG]
            return nxt[k]

        for prefix, (pb, pnb) in beams.items():
            ptot = _lse(pb, pnb)
            last = prefix[-1] if prefix else None
            for c in cand:
                p = row[c]
                if c == 0:
                    s = slot(prefix)
                    s[0] = _lse(s[0], ptot + p)
                elif c == last:
                    s = slot(prefix + (c,))
                    s[1] = _lse(s[1], pb + p)
                    s2 = slot(prefix)
                    s2[1] = _lse(s2[1], pnb + p)
                else:
                    s = slot(prefix + (c,))
                    s[1] = _lse(s[1], ptot + p)
        top = sorted(nxt.items(), key=lambda kv: -_lse(kv[1][0], kv[1][1]))[:beam]
        beams = {k: (v[0], v[1]) for k, v in top}
    out = [(k, _lse(v[0], v[1])) for k, v in beams.items()]
    out.sort(key=lambda x: -x[1])
    return out


def decode_with_space_bonus(lp: np.ndarray, space_id: Optional[int], delta: float) -> List[int]:
    """Greedy CTC decoding after adding `delta` (nats) to the space class: a one-parameter correction of the recogniser's bias
    against word gaps (v3 test run: 637 deleted vs 231 inserted spaces).  delta = 0 is plain greedy."""
    if space_id is not None and delta != 0.0:
        lp = lp.copy()
        lp[:, space_id] += delta
    return greedy_decode(lp)[0]


def log_softmax_np(x: np.ndarray) -> np.ndarray:
    m = x.max(-1, keepdims=True)
    return x - (m + np.log(np.exp(x - m).sum(-1, keepdims=True)))

In [ ]:
%%writefile inklingua/style.py
"""
Classical style descriptors: the targets of the connected "style" task.  They are MEASUREMENTS on the line image (never
invented): a descriptor that cannot be measured on a line is NaN and is masked out of every loss and metric.  They are
script-agnostic (slant, stroke width, ink density, word gap, body ratio exist for every handwriting).
"""
from __future__ import annotations

import math

import cv2
import numpy as np

STYLE_NAMES = ["slant_deg", "stroke_width", "ink_density", "gap_ratio", "body_ratio"]
NUM_DESC = len(STYLE_NAMES)
SLANT_RANGE_DEG = 45.0
STYLE_FLOORS = np.array([1.0, 0.005, 0.01, 0.01, 0.02], np.float32)   # lower bounds of the per-descriptor std


def _runs(b: np.ndarray) -> np.ndarray:
    d = np.diff(np.r_[0, b.astype(np.int8), 0])
    return np.flatnonzero(d == -1) - np.flatnonzero(d == 1)


def estimate_slant_deg(mask: np.ndarray, max_deg: float = SLANT_RANGE_DEG, step: float = 1.0,
                       min_prominence: float = 0.01) -> float:
    """Slant (deg, > 0 = leaning right) of a binary ink mask: the shear that makes the vertical projection profile sharpest (sum of
    squared column counts).  Every candidate angle uses the SAME resampling (rows moved by whole pixels), so 0 deg enjoys no
    interpolation advantage; the optimum is refined by a parabola through the three best angles.  NaN at the border of the search
    range or when the profile energy is flat."""
    H = mask.shape[0]
    ys, xs = np.nonzero(mask)
    if ys.size < 30:
        return float("nan")
    dy = ys.astype(np.float64) - (H - 1) / 2.0
    xs = xs.astype(np.int64)
    angles = np.arange(-max_deg, max_deg + 1e-9, step)
    obj = np.empty(len(angles))
    for k, th in enumerate(angles):
        xp = xs + np.rint(math.tan(math.radians(th)) * dy).astype(np.int64)
        prof = np.bincount(xp - xp.min())
        obj[k] = float((prof.astype(np.float64) ** 2).sum())
    k = int(obj.argmax())
    med = float(np.median(obj))
    if k == 0 or k == len(angles) - 1 or (obj[k] - med) / max(med, 1e-9) < min_prominence:
        return float("nan")
    den = obj[k - 1] - 2.0 * obj[k] + obj[k + 1]
    delta = 0.0 if den >= -1e-12 else 0.5 * (obj[k - 1] - obj[k + 1]) / den
    return float(angles[k] + float(np.clip(delta, -0.5, 0.5)) * step)


def body_ratio_from_rows(rows: np.ndarray, H: int) -> float:
    """Height of the dense 'body' band (rows whose smoothed ink count reaches half of the maximum) / H."""
    s = np.convolve(rows.astype(np.float64), np.ones(3), mode="same")
    mx = s.max()
    if mx <= 0:
        return float("nan")
    return float((s >= 0.5 * mx).sum() / H)


def compute_style_descriptors(ink: np.ndarray) -> np.ndarray:
    """ink: float [H, W] in [0,1], background 0.  Returns float32 [5]: slant (deg) | stroke width / H | ink density in the text
    box | mean inter-word gap / H | body ratio.  NaN = not measurable on this line."""
    H, W = ink.shape
    out = np.full(NUM_DESC, np.nan, np.float32)
    mask = (ink > 0.45).astype(np.uint8)
    if int(mask.sum()) < 30:
        return out
    cols, rows = mask.sum(0), mask.sum(1)
    xs, ys = np.flatnonzero(cols), np.flatnonzero(rows)
    x0, x1, y0, y1 = int(xs[0]), int(xs[-1]) + 1, int(ys[0]), int(ys[-1]) + 1
    out[2] = float(mask[y0:y1, x0:x1].mean())
    dt = cv2.distanceTransform(mask, cv2.DIST_L2, 3)
    ridge = (dt > 0) & (dt >= cv2.dilate(dt, np.ones((3, 3), np.uint8)) - 1e-6)
    if ridge.any():
        out[1] = float(2.0 * dt[ridge].mean() / H)
    out[0] = estimate_slant_deg(mask)
    gaps = _runs(cols[x0:x1] == 0)
    gaps = gaps[gaps >= max(3, int(0.12 * H))]
    if gaps.size:
        out[3] = float(gaps.mean() / H)
    out[4] = body_ratio_from_rows(rows, H)
    return out


def kmeans_np(X: np.ndarray, k: int, iters: int = 60, seed: int = 0) -> np.ndarray:
    """Tiny k-means++ (numpy only). Returns centroids [k, D] ordered by their first coordinate (slant) -> reproducible labels."""
    rng = np.random.RandomState(seed)
    k = min(k, len(X))
    C = [X[rng.randint(len(X))]]
    for _ in range(1, k):
        d2 = np.min(((X[:, None, :] - np.stack(C)[None]) ** 2).sum(-1), axis=1)
        pr = d2 / d2.sum() if d2.sum() > 0 else np.ones(len(X)) / len(X)
        C.append(X[rng.choice(len(X), p=pr)])
    C = np.stack(C)
    for _ in range(iters):
        a = ((X[:, None, :] - C[None]) ** 2).sum(-1).argmin(1)
        newC = np.stack([X[a == j].mean(0) if (a == j).any() else C[j] for j in range(k)])
        if np.allclose(newC, C):
            break
        C = newC
    return C[np.argsort(C[:, 0])]


def assign_nearest(X: np.ndarray, C: np.ndarray) -> np.ndarray:
    return ((X[:, None, :] - C[None]) ** 2).sum(-1).argmin(1)

In [ ]:
%%writefile inklingua/datasets_registry.py
"""
Registry of FREE line-level handwriting corpora that share the Hugging Face layout of Teklia/IAM-line (an `image` column and a
`text` column, parquet files, train / validation / test splits).  Every entry was checked against the Hub API on 2026-11-04
(commit hash, columns, split sizes, licence tag); the pinned revision keeps a run reproducible.

Why: IAM has 6,482 training lines, all English, all Latin script.  The registry makes it a one-line change to train on more
data (`Config.multi(("iam", "rimes"))`), on another language, or on another script (Arabic: KHATT, Han: CASIA-HWDB2), and
every per-domain table of the evaluation then shows what each corpus contributes.

Licences: the Hub tags are MIT for the Teklia conversions; the UNDERLYING corpora have their own terms (IAM: non-commercial
research; CASIA: free for academic research; KHATT: free research use; RIMES: research) -- check them before publishing.
Non-Teklia corpora (not in the registry) that are worth knowing: Riksarkivet (Swedish, cc-by), aarhus-city-archives
historical-danish-handwriting (cc-by-4.0), Rahvusarhiiv/et_handwriting_recognition (Estonian, cc0), fhswf/german_handwriting
(afl-3.0), SSamDav/portuguese-handwriting-16th-19th (cc-by-4.0); their columns differ, so they need a custom DatasetSpec.
"""
from __future__ import annotations

from typing import Dict

from .config import IAM_LINE_REVISION, IAM_LINE_SIZES, DatasetSpec

DATASETS: Dict[str, DatasetSpec] = {
    "iam": DatasetSpec(
        key="iam", hf_dataset="Teklia/IAM-line", hf_revision=IAM_LINE_REVISION, language="en", script="Latin", detokenize=True,
        expected_sizes=IAM_LINE_SIZES, licence="Hub: MIT; IAM itself: non-commercial research",
        note="Modern English, 657 writers (no writer id in this conversion); the Aachen writer-independent split."),
    "rimes": DatasetSpec(
        key="rimes", hf_dataset="Teklia/RIMES-2011-line", hf_revision="ba3e6b5573094208b30a134e32d9b65dab18e74e", language="fr",
        script="Latin", expected_sizes=(10188, 1138, 778), licence="Hub: MIT; RIMES: research",
        note="Modern French letters (RIMES 2011 line level)."),
    "belfort": DatasetSpec(
        key="belfort", hf_dataset="Teklia/Belfort-line", hf_revision="1e1caa665d48aa6756313cc0f72416f628de34c3", language="fr",
        script="Latin", expected_sizes=(25800, 3102, 3819), licence="Hub: MIT",
        note="Historical French municipal registers (Belfort), 19th-20th c."),
    "esposalles": DatasetSpec(
        key="esposalles", hf_dataset="Teklia/Esposalles-line", hf_revision="5f8e7ceccb126528ebae15cb81c475403d176ee4", language="ca",
        script="Latin", expected_sizes=(2328, 742, 757), licence="Hub: MIT", note="17th-century Catalan marriage licences."),
    "himanis": DatasetSpec(
        key="himanis", hf_dataset="Teklia/Himanis-line", hf_revision="3793c910e6c087ebc1d575ab7384d8d60ad7eafd", language="la",
        script="Latin", expected_sizes=(18504, 2367, 2241), licence="Hub: MIT", note="Medieval Latin / French chancery registers."),
    "home_alcar": DatasetSpec(
        key="home_alcar", hf_dataset="Teklia/HOME-Alcar-line", hf_revision="edb8b023ebd6c0c92ad61adcdd550cfee6e34a18", language="la",
        script="Latin", expected_sizes=(59969, 7905, 6932), licence="Hub: MIT", note="Medieval Latin cartularies (HOME-Alcar)."),
    "newseye_de": DatasetSpec(
        key="newseye_de", hf_dataset="Teklia/NewsEye-Austrian-line", hf_revision="8d6b94c016b7cb455ad77c7ac67f80036ebfb752",
        language="de", script="Latin", test_split=None, expected_sizes=(51588, 4379), licence="Hub: MIT",
        note="Austrian German; NO test split on the Hub -> contributes train / validation lines only."),
    "norhand1": DatasetSpec(
        key="norhand1", hf_dataset="Teklia/NorHand-v1-line", hf_revision="2725c76fe2bd2c0f100c641621e21f4887cbb26c", language="nb",
        script="Latin", expected_sizes=(19653, 2286, 1793), licence="Hub: MIT", note="Norwegian letters and diaries, 19th-20th c."),
    "norhand3": DatasetSpec(
        key="norhand3", hf_dataset="Teklia/NorHand-v3-line", hf_revision="113280c8fbb7f6df7923ee15a7ba3c97710a8f1c", language="nb",
        script="Latin", expected_sizes=(222381, 22679, 1562), licence="Hub: MIT",
        note="Large Norwegian corpus (222k train lines): use max_lines to cap it."),
    "popp": DatasetSpec(
        key="popp", hf_dataset="Teklia/POPP-line", hf_revision="7ab15c7589c164fc13cb42adccf68e61c05b2945", language="fr",
        script="Latin", expected_sizes=(3835, 480, 479), licence="Hub: MIT", note="Paris census tables (POPP), 1926-1946."),
    "casia": DatasetSpec(
        key="casia", hf_dataset="Teklia/CASIA-HWDB2-line", hf_revision="2974a15ceab0657047a28d2a71b76bd7544d4663", language="zh",
        script="Han", expected_sizes=(33401, 8318, 10441), licence="Hub: MIT; CASIA: free for academic research",
        note="Chinese (CASIA-HWDB2.0-2.2 lines); thousands of output units -> expect a much larger alphabet."),
    "khatt": DatasetSpec(
        key="khatt", hf_dataset="Teklia/KHATT_v1.0", hf_revision="7da941f6705364c085107c75af4f75e5d421fa4e", language="ar",
        script="Arabic", rtl=True, expected_sizes=(4672, 963, 1038), licence="no Hub tag; KHATT: free research use",
        note="Modern Arabic (KHATT v1.0); text in LOGICAL order -> rtl=True converts it to visual order for the recogniser."),
    # the synthetic corpus of Config.smoke(): generated fake handwriting, plumbing tests only
    "synthetic": DatasetSpec(key="synthetic", hf_dataset="", language="en", script="Latin", note="SYNTHETIC smoke-test data."),
}


def get_spec(key: str, custom=()) -> DatasetSpec:
    for s in custom:
        if s.key == key:
            return s
    if key not in DATASETS:
        raise KeyError(f"unknown dataset '{key}'; registry keys: {sorted(DATASETS)}; or add a DatasetSpec to cfg.data.custom_sources")
    return DATASETS[key]

In [ ]:
%%writefile inklingua/data.py
"""
Data: preprocessing, sources (Hugging Face Hub / local parquet / synthetic), the line cache, the dataset class that serves BOTH
tasks (recognition + style), the collator with MixLine, width-bucketed batching and the data module that assembles one or
several corpora.

Guarantees that a reader of the paper must be able to trust (all tested in inklingua.selftest):
  * the hidden transcriptions of the unlabeled train lines never reach a batch of the training dataset;
  * the calibration lines (labeled, text held out) are images only for the training loop; their text is read by the trust rule
    and by diagnostics;
  * the alphabet is built from the labeled lines only;
  * no image occurs in two splits (content digest), split sizes match the published ones (strict by default);
  * right-to-left corpora are stored in visual order and displayed in logical order; the metrics are unchanged by that.
"""
from __future__ import annotations

import glob
import hashlib
import json
import math
import os
import random
import time
from collections import Counter
from typing import Any, Dict, List, Optional, Sequence, Tuple

import cv2
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont
from torch.utils.data import DataLoader, Dataset, Sampler

from .config import Config, DatasetSpec
from .datasets_registry import get_spec
from .style import NUM_DESC, STYLE_FLOORS, STYLE_NAMES, compute_style_descriptors, kmeans_np
from .text import CharCodec, ctc_min_frames, is_linguistic, normalize_text, text_baseline_r2, to_logical, to_visual
from .utils import LOG, json_safe

try:
    from tqdm.auto import tqdm
except Exception:  # pragma: no cover
    def tqdm(x, **kw):  # type: ignore
        return x


# ==============================================================================
# preprocessing
# ==============================================================================
def preprocess_line(img: Image.Image, H: int, max_w: int, min_w: int = 32, pad_x: int = 8) -> np.ndarray:
    """PIL line image -> float32 [H, W] with ink=1, background=0 (per-line contrast normalised).  Aspect ratio is preserved; a
    blank margin of `pad_x` pixels is added on both sides so that shear / rotation augmentations never crop ink."""
    g = np.asarray(img.convert("L"), dtype=np.float32)
    h, w = g.shape
    if h < 2 or w < 2:
        raise ValueError(f"degenerate line image of size {w}x{h}")
    s = min(H / h, max(8, max_w - 2 * pad_x) / w)
    nh, nw = max(1, min(H, int(round(h * s)))), max(1, int(round(w * s)))
    g = cv2.resize(g, (nw, nh), interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_LINEAR)
    bg, lo = np.percentile(g, 92), np.percentile(g, 1)
    ink = np.clip((bg - g) / max(bg - lo, 25.0), 0.0, 1.0)
    canvas = np.zeros((H, max(min_w, nw + 2 * pad_x)), np.float32)
    top = (H - nh) // 2
    canvas[top:top + nh, pad_x:pad_x + nw] = ink
    return canvas


def to_u8(ink: np.ndarray) -> np.ndarray:
    return np.round(np.clip(ink, 0.0, 1.0) * 255.0).astype(np.uint8)


# ==============================================================================
# sources
# ==============================================================================
class SplitSource:
    """Uniform access to one split of one corpus, independent of where it comes from."""

    def __len__(self) -> int: raise NotImplementedError
    def image(self, j: int) -> Image.Image: raise NotImplementedError
    def text(self, j: int) -> str: raise NotImplementedError
    def extra(self, j: int) -> Optional[str]: return None


class HFSource(SplitSource):
    """One split of a Hugging Face `datasets` object with an image and a text column."""

    IMAGE_COLS = ("image", "img", "line_image")
    TEXT_COLS = ("text", "transcription", "label", "sentence", "gt")

    def __init__(self, hf_ds, style_col: Optional[str] = None, image_col: Optional[str] = None, text_col: Optional[str] = None):
        from datasets import Image as HFImage
        self.ds = hf_ds
        cols = list(hf_ds.column_names)
        self.img_col = image_col or next((c for c in self.IMAGE_COLS if c in cols), None) or next(
            (c for c in cols if isinstance(hf_ds.features[c], HFImage)), None)
        self.txt_col = text_col or next((c for c in self.TEXT_COLS if c in cols), None)
        if self.img_col is None or self.txt_col is None:
            raise ValueError(f"cannot find an image and a text column among {cols} (features: {dict(hf_ds.features)})")
        if not isinstance(hf_ds.features[self.img_col], HFImage):
            self.ds = hf_ds = hf_ds.cast_column(self.img_col, HFImage())
        self._texts = [normalize_text(str(t)) for t in hf_ds[self.txt_col]]        # text column only -> no image decoding here
        self._extra = list(hf_ds[style_col]) if style_col and style_col in cols else None

    def __len__(self): return len(self._texts)

    def image(self, j):
        im = self.ds[int(j)][self.img_col]
        if not isinstance(im, Image.Image):
            raise TypeError(f"row {j}: expected a PIL image, got {type(im)}")
        return im

    def text(self, j): return self._texts[int(j)]
    def extra(self, j): return None if self._extra is None else str(self._extra[int(j)])


_SYN_WORDS = ("the of and to in is that it was for on are with as his they be at one have this from or had by "
              "hot word but what some we can out other were all there when up use your how said an each she "
              "which do their time if will way about many then them write would like so these her long make "
              "thing see him two has look more day could go come did number sound no most people my over know "
              "water than call first who may down side been now find head stand own page should country found "
              "answer school grow study still learn plant cover food sun four between state keep eye never last "
              "let thought city tree cross farm hard start might story saw far sea draw left late run").split()


def font_paths() -> List[str]:
    try:
        import matplotlib as mpl
        d = os.path.join(mpl.get_data_path(), "fonts", "ttf")
        names = ["DejaVuSans.ttf", "DejaVuSerif.ttf", "DejaVuSansMono.ttf", "DejaVuSans-Bold.ttf",
                 "DejaVuSans-Oblique.ttf", "DejaVuSerif-Italic.ttf"]
        return [os.path.join(d, n) for n in names if os.path.exists(os.path.join(d, n))]
    except Exception:
        return []


def load_font(path: Optional[str], size: int):
    try:
        if path:
            return ImageFont.truetype(path, size)
        return ImageFont.load_default(size=size)
    except Exception:
        return ImageFont.load_default()


def render_synthetic_line(text: str, rng: random.Random, fonts: List[str]) -> Image.Image:
    """Offline 'handwriting': varied fonts / slant / stroke / blur / noise (for smoke tests only)."""
    font = load_font(rng.choice(fonts) if fonts else None, rng.randint(40, 56))
    stroke = rng.choice([0, 0, 1, 2])
    ink = int(rng.uniform(0, 90))
    tmp = ImageDraw.Draw(Image.new("L", (8, 8)))
    x0, y0, x1, y1 = tmp.textbbox((0, 0), text, font=font, stroke_width=stroke)
    w = (x1 - x0) + 60
    canvas = Image.new("L", (w, 128), 255)
    ImageDraw.Draw(canvas).text((30 - x0, 64 - (y1 - y0) // 2 - y0), text, font=font, fill=ink, stroke_width=stroke, stroke_fill=ink)
    shear = rng.uniform(-0.35, 0.35)
    canvas = canvas.transform(canvas.size, Image.AFFINE, (1, shear, -shear * 64, 0, 1, 0), resample=Image.BILINEAR, fillcolor=255)
    arr = np.asarray(canvas, np.float32)
    arr = cv2.GaussianBlur(arr, (0, 0), rng.uniform(0.3, 1.1))
    arr = np.clip(arr + np.random.RandomState(rng.randint(0, 10 ** 6)).normal(0, rng.uniform(1, 6), arr.shape), 0, 255)
    return Image.fromarray(arr.astype(np.uint8))


class SyntheticSource(SplitSource):
    def __init__(self, n: int, seed: int, words: Tuple[int, int] = (3, 6)):
        rng, fonts = random.Random(seed), font_paths()
        self._texts = [" ".join(rng.choice(_SYN_WORDS) for _ in range(rng.randint(*words))) for _ in range(n)]
        self._imgs = [render_synthetic_line(t, rng, fonts) for t in self._texts]

    def __len__(self): return len(self._texts)
    def image(self, j): return self._imgs[int(j)]
    def text(self, j): return self._texts[int(j)]


class ListSource(SplitSource):
    """Tiny in-memory split rendered from a list of texts (self-tests)."""

    def __init__(self, texts: List[str], seed: int = 0):
        rng, fonts = random.Random(seed), font_paths()
        self._t = list(texts)
        self._im = [render_synthetic_line(t, rng, fonts) for t in texts]

    def __len__(self): return len(self._t)
    def image(self, j): return self._im[int(j)]
    def text(self, j): return self._t[int(j)]


# ==============================================================================
# the line cache
# ==============================================================================
def build_line_cache(src: SplitSource, indices: np.ndarray, H: int, max_w: int, min_w: int, pad_x: int, desc: str,
                     keep_ink: bool = True) -> Dict[str, Any]:
    """One decoding pass over the selected lines: preprocessed uint8 ink (optional), widths, style descriptors and a content digest
    (used to prove that no image occurs in two splits).  A corrupt image raises immediately -- lines are never skipped silently."""
    n = len(indices)
    width, style = np.zeros(n, np.int64), np.zeros((n, NUM_DESC), np.float32)
    ink: List[Optional[np.ndarray]] = []
    digest: List[str] = []
    for k, j in enumerate(tqdm(indices, desc=desc, leave=False)):
        try:
            u8 = to_u8(preprocess_line(src.image(int(j)), H, max_w, min_w, pad_x))
        except Exception as e:
            raise RuntimeError(f"{desc}: line {int(j)} could not be decoded: {e}") from e
        width[k] = u8.shape[1]
        style[k] = compute_style_descriptors(u8.astype(np.float32) / 255.0)
        digest.append(hashlib.blake2b(u8.tobytes(), digest_size=8).hexdigest())
        ink.append(u8 if keep_ink else None)
    return {"ink": ink, "width": width, "style": style, "digest": digest}


def _concat_caches(caches: List[Dict[str, Any]]) -> Dict[str, Any]:
    if not caches:
        return {"ink": [], "width": np.zeros(0, np.int64), "style": np.zeros((0, NUM_DESC), np.float32), "digest": []}
    return {"ink": sum((c["ink"] for c in caches), []), "width": np.concatenate([c["width"] for c in caches]),
            "style": np.concatenate([c["style"] for c in caches]), "digest": sum((c["digest"] for c in caches), [])}


def _subset_cache(cache: Dict[str, Any], keep: np.ndarray) -> Dict[str, Any]:
    return {"ink": [cache["ink"][i] for i in keep], "width": cache["width"][keep], "style": cache["style"][keep],
            "digest": [cache["digest"][i] for i in keep]}


# ==============================================================================
# the dataset class
# ==============================================================================
class HandwritingDataset(Dataset):
    """One item serves BOTH tasks: the recognition target `text` (+ per-sample CTC `weight`; 0 for lines without a usable label) and
    the style targets (5 standardised descriptors + style cluster id).  `oracle` holds the transcription of EVERY line in VISUAL
    order; for the unlabeled lines of the train split it is HIDDEN: returned by `reference()` / in batches only if `expose_hidden`,
    which is never the case for the training dataset.  `domain[i]` is the corpus of line i, `rtl[i]` its writing direction."""

    def __init__(self, sources: List[SplitSource], src_id: np.ndarray, indices: np.ndarray, ink: Optional[List[np.ndarray]],
                 width: np.ndarray, style: np.ndarray, style_ok: np.ndarray, style_raw: np.ndarray, cluster: np.ndarray,
                 oracle: List[str], labeled: np.ndarray, calib: Optional[np.ndarray], domain: np.ndarray, domain_specs: List[DatasetSpec],
                 img_h: int, max_w: int, min_w: int, pad_x: int, name: str, expose_hidden: bool = False):
        self.sources, self.src_id, self.indices, self.ink = sources, np.asarray(src_id), np.asarray(indices), ink
        self.width, self.style, self.cluster = width, style.astype(np.float32), cluster.astype(np.int64)
        self.style_ok, self.style_raw = np.asarray(style_ok, bool), np.asarray(style_raw, np.float32)
        self.oracle, self.labeled = list(oracle), np.asarray(labeled, bool)
        self.calib = np.zeros(len(self.oracle), bool) if calib is None else np.asarray(calib, bool)
        self.domain, self.domain_specs = np.asarray(domain, np.int64), list(domain_specs)
        self.rtl = np.array([self.domain_specs[d].rtl for d in self.domain], bool) if len(self.domain) else np.zeros(0, bool)
        self.img_h, self.max_w, self.min_w, self.pad_x, self.name = img_h, max_w, min_w, pad_x, name
        self.expose_hidden = expose_hidden
        self.tgt_text = [t if l else "" for t, l in zip(self.oracle, self.labeled)]
        self.tgt_weight = np.where(self.labeled, 1.0, 0.0).astype(np.float32)
        self.stratum = np.zeros(len(self.oracle), np.int64)       # difficulty cell (set by the data module)

    def __len__(self) -> int:
        return len(self.indices)

    @property
    def domain_names(self) -> List[str]:
        return [s.key for s in self.domain_specs]

    def reference(self, i: int) -> str:
        """Ground-truth transcription of line i (visual order), or "" if it is hidden."""
        return self.oracle[i] if (self.expose_hidden or self.labeled[i] or self.calib[i]) else ""

    def display(self, i: int, text: Optional[str] = None) -> str:
        """Human-readable (logical-order) form of a visual-order string of line i (default: its reference)."""
        return to_logical(self.oracle[i] if text is None else text, bool(self.rtl[i]))

    def detok(self, i: int) -> bool:
        return bool(self.domain_specs[int(self.domain[i])].detokenize)

    def language(self, i: int) -> str:
        return self.domain_specs[int(self.domain[i])].language

    def set_pseudo(self, texts: Sequence[str], weights: np.ndarray, include_calib: bool = False) -> None:
        """Install pseudo-labels for the unlabeled lines (labeled lines are never overwritten).  Calibration lines receive
        pseudo-labels only with include_calib=True ([SHADOW]); their true text is never a training target in either case."""
        for i in np.flatnonzero(~self.labeled if include_calib else (~self.labeled & ~self.calib)):
            self.tgt_text[i], self.tgt_weight[i] = texts[i], float(weights[i])

    def get_ink(self, i: int) -> np.ndarray:
        if self.ink is not None:
            return self.ink[i].astype(np.float32) / 255.0
        src = self.sources[int(self.src_id[i])]
        return to_u8(preprocess_line(src.image(int(self.indices[i])), self.img_h, self.max_w, self.min_w, self.pad_x)).astype(np.float32) / 255.0

    def __getitem__(self, i: int) -> Dict[str, Any]:
        ink = self.get_ink(i)
        return {"index": i, "image": torch.from_numpy(ink)[None], "width": ink.shape[1],
                "text": self.tgt_text[i], "weight": float(self.tgt_weight[i]),
                "oracle": self.oracle[i] if (self.expose_hidden or self.labeled[i]) else "",     # calibration lines stay hidden here
                "style": torch.from_numpy(self.style[i]), "style_ok": torch.from_numpy(self.style_ok[i]),
                "cluster": int(self.cluster[i]), "labeled": bool(self.labeled[i]), "stratum": int(self.stratum[i]),
                "domain": int(self.domain[i])}


# ==============================================================================
# collation (+ MixLine) and batching
# ==============================================================================
class Collator:
    def __init__(self, codec: CharCodec, pad_multiple: int = 4, mix_prob: float = 0.0, max_w: int = 2560,
                 gap: Tuple[int, int] = (0, 24), seed: int = 0):
        self.codec, self.pm = codec, pad_multiple
        self.mix_prob, self.max_w, self.gap, self.seed = float(mix_prob), int(max_w), tuple(gap), int(seed)

    def _mix(self, items: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """[MIXLINE] concatenate two adjacent LABELED lines of the SAME corpus (real images, exact label 'A B').  The style targets of a
        concatenation are masked (they describe one line), the style class is ignored.  Visual order is kept, so RTL lines concatenate
        correctly (the label of the right-hand image part comes last)."""
        rng = np.random.RandomState((int(sum(int(it["index"]) for it in items)) * 2654435761 + self.seed) % (2 ** 32))
        out: List[Dict[str, Any]] = []
        sep = " " if " " in self.codec.c2i else ""             # scripts written without word spaces are simply juxtaposed
        k = 0
        while k < len(items):
            a = items[k]
            b = items[k + 1] if k + 1 < len(items) else None
            if (b is not None and a["labeled"] and b["labeled"] and a["weight"] > 0 and b["weight"] > 0 and a["text"] and b["text"]
                    and a.get("domain", 0) == b.get("domain", 0) and rng.rand() < self.mix_prob):
                g = int(rng.randint(self.gap[0], self.gap[1] + 1))
                w = int(a["width"]) + g + int(b["width"])
                if w <= self.max_w:
                    img = torch.zeros(1, a["image"].shape[1], w)
                    img[:, :, :a["width"]] = a["image"][:, :, :a["width"]]
                    img[:, :, a["width"] + g:w] = b["image"][:, :, :b["width"]]
                    m = dict(a)
                    m.update(image=img, width=w, text=a["text"] + sep + b["text"], oracle=a["oracle"] + sep + b["oracle"],
                             style_ok=torch.zeros_like(a["style_ok"]), cluster=-100)
                    out.append(m)
                    k += 2
                    continue
            out.append(a)
            k += 1
        return out

    def __call__(self, items: List[Dict[str, Any]]) -> Dict[str, Any]:
        if self.mix_prob > 0 and len(items) > 1:
            items = self._mix(items)
        B, H = len(items), items[0]["image"].shape[1]
        Wm = int(math.ceil(max(it["width"] for it in items) / self.pm) * self.pm)
        imgs = torch.zeros(B, 1, H, Wm)
        tgt, tlen, minw = [], [], []
        for b, it in enumerate(items):
            imgs[b, :, :, :it["width"]] = it["image"]
            ids = self.codec.encode(it["text"]) if it["weight"] > 0 else []
            tgt += ids
            tlen.append(len(ids))
            minw.append(4 * ctc_min_frames(ids) if ids else 0)      # the horizontal-scale augmentation never goes below it
        return {"image": imgs, "width": torch.tensor([it["width"] for it in items]),
                "target": torch.tensor(tgt, dtype=torch.long), "target_len": torch.tensor(tlen, dtype=torch.long),
                "weight": torch.tensor([it["weight"] if tl > 0 else 0.0 for it, tl in zip(items, tlen)]),
                "style": torch.stack([it["style"] for it in items]),
                "style_ok": torch.stack([it["style_ok"] for it in items]),
                "min_width": torch.tensor(minw, dtype=torch.long),
                "cluster": torch.tensor([it["cluster"] for it in items], dtype=torch.long),
                "labeled": torch.tensor([it["labeled"] for it in items]),
                "index": torch.tensor([it["index"] for it in items], dtype=torch.long),
                "stratum": torch.tensor([it.get("stratum", 0) for it in items], dtype=torch.long),
                "domain": torch.tensor([it.get("domain", 0) for it in items], dtype=torch.long),
                "text": [it["text"] for it in items], "oracle": [it["oracle"] for it in items]}


class BucketBatchSampler(Sampler):
    """Batches lines of similar width together (less padding). Deterministic per (seed, epoch)."""

    def __init__(self, lengths: np.ndarray, batch_size: int, shuffle: bool, mega: int = 50, seed: int = 0,
                 indices: Optional[Sequence[int]] = None):
        self.lengths = np.asarray(lengths)
        self.indices = np.arange(len(lengths)) if indices is None else np.asarray(indices)
        self.bs, self.shuffle, self.mega, self.seed, self.epoch = batch_size, shuffle, mega, seed, 0

    def set_epoch(self, e: int) -> None:
        self.epoch = e

    def __len__(self) -> int:
        return int(math.ceil(len(self.indices) / self.bs))

    def __iter__(self):
        idx = self.indices.copy()
        rng = np.random.RandomState(self.seed + self.epoch)
        if self.shuffle:
            rng.shuffle(idx)
            chunks = [idx[i:i + self.bs * self.mega] for i in range(0, len(idx), self.bs * self.mega)]
            idx = np.concatenate([c[np.argsort(-self.lengths[c], kind="stable")] for c in chunks])
        else:
            idx = idx[np.argsort(-self.lengths[idx], kind="stable")]
        batches = [idx[i:i + self.bs].tolist() for i in range(0, len(idx), self.bs)]
        if self.shuffle:
            rng.shuffle(batches)
        yield from batches


# ==============================================================================
# the data module
# ==============================================================================
class HandwritingDataModule:
    """Downloads (Hub) or reads (local parquet) one or several corpora, checks their integrity, splits them, builds datasets and
    loaders.  There is NO fallback to synthetic data: if the real data cannot be obtained the run stops."""

    def __init__(self, cfg: Config, base_units: Optional[Sequence[str]] = None):
        """base_units: alphabet of a checkpoint to warm-start from (mode "finetune"): the new alphabet EXTENDS it (old indices keep their
        meaning, new units are appended) so that the checkpoint's output layer can be grown function-preservingly."""
        self.cfg, self.dc = cfg, cfg.data
        self.base_units = list(base_units) if base_units else None
        self.specs: List[DatasetSpec] = [get_spec(k, self.dc.custom_sources) for k in self.dc.sources]
        if not self.specs:
            raise ValueError("cfg.data.sources is empty")
        self.codec: Optional[CharCodec] = None
        self.style_mean = np.zeros(NUM_DESC, np.float32)
        self.style_std = np.ones(NUM_DESC, np.float32)
        self.n_clusters = self.dc.style_num_clusters
        self.centroids: Optional[np.ndarray] = None
        self.train_ds = self.val_ds = self.test_ds = None
        self.text_baseline_r2 = np.zeros(NUM_DESC, np.float32)
        self.leak_active = np.zeros(NUM_DESC, bool)
        self.labeled_idx = self.calib_idx = self.unlabeled_idx = self.train_eval_idx = np.zeros(0, np.int64)
        self.n_strata, self.strata_cuts = 1, {}            # strata_cuts: {domain: cut points}
        self.val_sel_idx = self.val_gate_idx = np.zeros(0, np.int64)
        self.using_synthetic = False
        self.provenance: Dict[str, Any] = {}

    @property
    def is_synthetic(self) -> bool:
        return self.using_synthetic or self.dc.source == "synthetic"

    @property
    def domain_names(self) -> List[str]:
        return [s.key for s in self.specs]

    @property
    def primary_language(self) -> str:
        return self.specs[0].language

    # ---------------------------------------------------------------- loading
    def _download(self, load_dataset, spec: DatasetSpec):
        err = None
        for attempt in range(3):
            try:
                LOG.info("Downloading / loading '%s' (revision %s, attempt %d) ...", spec.hf_dataset, spec.hf_revision or "latest", attempt + 1)
                return load_dataset(spec.hf_dataset, cache_dir=os.path.join(self.cfg.data_dir, "hf"), revision=spec.hf_revision)
            except Exception as e:  # network / hub errors
                err = e
                LOG.warning("attempt %d failed: %s", attempt + 1, e)
                time.sleep(3 * (attempt + 1))
        raise RuntimeError(f"could not obtain '{spec.hf_dataset}' ({err}).\nCheck the internet connection (on Kaggle: Settings -> "
                           "Internet on), or download the parquet files once and set DatasetSpec.local_dir / cfg.data.local_dir.  "
                           "There is deliberately NO synthetic fallback.")

    def _load_local(self, load_dataset, d: str, spec: DatasetSpec):
        if not d or not os.path.isdir(d):
            raise FileNotFoundError(f"local_dir='{d}' of dataset '{spec.key}' is not a directory")
        if os.path.exists(os.path.join(d, "dataset_dict.json")):
            from datasets import load_from_disk
            return load_from_disk(d)
        aliases = {spec.train_split: (spec.train_split, "train"), spec.val_split: (spec.val_split, "validation", "val", "valid", "dev")}
        if spec.test_split:
            aliases[spec.test_split] = (spec.test_split, "test")
        files: Dict[str, List[str]] = {}
        for split, al in aliases.items():
            f: List[str] = []
            for a in dict.fromkeys(al):
                f += glob.glob(os.path.join(d, f"{a}*.parquet")) + glob.glob(os.path.join(d, "**", f"{a}*.parquet"), recursive=True)
            if not f:
                raise FileNotFoundError(f"no '{split}*.parquet' file found under {d}")
            files[split] = sorted(set(f))
        return load_dataset("parquet", data_files=files, cache_dir=os.path.join(self.cfg.data_dir, "hf_local"))

    def _load_spec(self, spec: DatasetSpec) -> Dict[str, Optional[SplitSource]]:
        """{'train': src, 'val': src, 'test': src or None} for one corpus."""
        try:
            from datasets import load_dataset
        except Exception as e:
            raise ImportError(f"`datasets` is required for real data: pip install datasets ({e})") from e
        local = spec.local_dir or (self.dc.local_dir if (self.dc.source == "local" and len(self.specs) == 1) else None)
        if self.dc.source == "local" and local is None:
            raise ValueError(f"cfg.data.source='local' but dataset '{spec.key}' has no local_dir (set DatasetSpec.local_dir, or cfg.data.local_dir "
                             f"for a single source)")
        dsd = self._load_local(load_dataset, local, spec) if local else self._download(load_dataset, spec)
        names = list(dsd.keys())

        def pick(*ks):
            return next((k for k in ks if k in names), None)
        k_tr, k_va = pick(spec.train_split, "train"), pick(spec.val_split, "validation", "val", "valid", "dev")
        k_te = pick(spec.test_split, "test") if spec.test_split else None
        if k_tr is None or k_va is None or (spec.test_split and k_te is None):
            raise RuntimeError(f"dataset '{spec.key}': expected splits {spec.train_split} / {spec.val_split} / {spec.test_split}, found {names}. "
                               "(Re-using the validation split as test, or carving validation out of train silently, is not allowed.)")
        raw = [len(dsd[k_tr]), len(dsd[k_va])] + ([len(dsd[k_te])] if k_te else [])
        prov = {"dataset": spec.hf_dataset or local, "requested_revision": spec.hf_revision, "raw_sizes": raw, "language": spec.language,
                "script": spec.script, "rtl": spec.rtl, "licence": spec.licence, "local": bool(local)}
        LOG.info("[%s] splits: %s | columns: %s", spec.key, dict(zip(("train", "validation", "test"), raw)), dsd[k_tr].column_names)
        if not local and spec.hf_dataset:
            try:                                                   # record the exact commit that was used
                from huggingface_hub import HfApi
                prov["resolved_commit"] = HfApi().dataset_info(spec.hf_dataset, revision=spec.hf_revision).sha
            except Exception as e:
                prov["resolved_commit"] = None
                LOG.info("could not query the dataset commit hash (%s)", e)
        if spec.expected_sizes is not None and tuple(raw) != tuple(spec.expected_sizes):
            msg = f"dataset '{spec.key}': split sizes {raw} differ from the expected {tuple(spec.expected_sizes)} (wrong / truncated / other version?)"
            if self.dc.strict_checks:
                raise RuntimeError(msg + "  Set DatasetSpec.expected_sizes=None or cfg.data.strict_checks=False to override.")
            LOG.warning(msg)
        self.provenance.setdefault("sources", {})[spec.key] = prov
        col = self.dc.style_label_column
        mk = lambda k: HFSource(dsd[k], col, spec.image_col, spec.text_col)  # noqa: E731
        return {"train": mk(k_tr), "val": mk(k_va), "test": mk(k_te) if k_te else None}

    def _synthetic(self) -> Dict[str, Optional[SplitSource]]:
        self.using_synthetic = True
        n_tr, n_va, n_te = self.dc.synthetic_sizes
        s, w = self.cfg.run.seed, self.dc.synthetic_words
        LOG.warning("=" * 78)
        LOG.warning("SYNTHETIC SMOKE TEST: fake handwriting, for plumbing tests only -- NOT FOR PUBLICATION")
        LOG.warning("=" * 78)
        self.provenance.setdefault("sources", {})["synthetic"] = {"synthetic": True, "sizes": list(self.dc.synthetic_sizes)}
        return {"train": SyntheticSource(n_tr, s, w), "val": SyntheticSource(n_va, s + 1, w), "test": SyntheticSource(n_te, s + 2, w)}

    def _load_sources(self) -> List[Dict[str, Optional[SplitSource]]]:
        """One {'train','val','test'} dict per corpus (same order as self.specs).  Overridden by the self-tests."""
        dc = self.dc
        self.provenance.update({"source": dc.source, "sources": {}, "synthetic": dc.source == "synthetic"})
        if dc.source == "synthetic":
            if not self.cfg.run.smoke_test:
                raise RuntimeError("cfg.data.source='synthetic' is only permitted through Config.smoke() (run.smoke_test=True): a real run "
                                   "never trains on synthetic data.")
            return [self._synthetic() for _ in self.specs]
        if dc.source not in ("hf", "local"):
            raise ValueError(f"unknown cfg.data.source '{dc.source}'")
        out = []
        for spec in self.specs:
            if spec.key == "synthetic":
                raise RuntimeError("the 'synthetic' corpus is only available with cfg.data.source='synthetic' (Config.smoke())")
            out.append(self._load_spec(spec))
        return out

    # ---------------------------------------------------------------- prepare
    @staticmethod
    def _select(n: int, limit: Optional[int], seed: int) -> np.ndarray:
        idx = np.arange(n)
        if limit is not None and limit < len(idx):
            idx = np.sort(np.random.RandomState(seed).permutation(idx)[:limit])
        return idx

    def _integrity(self, cache: Dict[str, Dict[str, Any]], text: Dict[str, List[str]]) -> Dict[str, Any]:
        rep: Dict[str, Any] = {}
        dig = {k: set(c["digest"]) for k, c in cache.items()}
        for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
            n = len(dig[a] & dig[b])
            rep[f"image_overlap_{a}_{b}"] = n
            if n:
                msg = f"{n} identical line images occur in both '{a}' and '{b}' (data leakage)"
                if self.dc.strict_checks:
                    raise RuntimeError(msg)
                LOG.warning(msg)
        for k in text:
            rep[f"empty_text_{k}"] = int(sum(1 for t in text[k] if not t))
        return rep

    def prepare(self) -> "HandwritingDataModule":
        dc, seed, prov = self.dc, self.cfg.run.seed, self.provenance
        assert dc.img_height % 16 == 0, "img_height must be a multiple of 16"
        per_corpus = self._load_sources()
        # ---- gather every split of every corpus: caches, VISUAL-order texts, domain ids ---------------------------------------
        cache: Dict[str, Dict[str, Any]] = {}
        text: Dict[str, List[str]] = {}
        src_id: Dict[str, np.ndarray] = {}
        indices: Dict[str, np.ndarray] = {}
        domain: Dict[str, np.ndarray] = {}
        sources: Dict[str, List[SplitSource]] = {"train": [], "val": [], "test": []}
        for k in ("train", "val", "test"):
            caches, texts, sids, idxs, doms = [], [], [], [], []
            for d, (spec, srcs) in enumerate(zip(self.specs, per_corpus)):
                src = srcs.get(k)
                if src is None:
                    continue
                cap = (spec.max_lines or {}).get({"train": "train", "val": "validation", "test": "test"}[k])
                sel = self._select(len(src), cap, seed + 100 * d + {"train": 0, "val": 1, "test": 2}[k])
                caches.append(build_line_cache(src, sel, dc.img_height, dc.max_img_width, dc.min_img_width, dc.pad_x,
                                               f"decoding [{spec.key}/{k}]", dc.cache_images))
                texts += [to_visual(src.text(int(j)), spec.rtl) for j in sel]
                sids.append(np.full(len(sel), len(sources[k]), np.int64))
                sources[k].append(src)
                idxs.append(sel)
                doms.append(np.full(len(sel), d, np.int64))
            cache[k] = _concat_caches(caches)
            text[k] = texts
            src_id[k] = np.concatenate(sids) if sids else np.zeros(0, np.int64)
            indices[k] = np.concatenate(idxs) if idxs else np.zeros(0, np.int64)
            domain[k] = np.concatenate(doms) if doms else np.zeros(0, np.int64)
            limit = {"train": dc.limit_train, "val": dc.limit_val, "test": dc.limit_test}[k]
            if limit is not None and limit < len(text[k]):
                keep = self._select(len(text[k]), limit, seed + {"train": 0, "val": 1, "test": 2}[k])
                cache[k] = _subset_cache(cache[k], keep)
                text[k] = [text[k][i] for i in keep]
                src_id[k], indices[k], domain[k] = src_id[k][keep], indices[k][keep], domain[k][keep]
        if len(text["test"]) == 0:
            raise RuntimeError("no corpus contributes test lines (every DatasetSpec has test_split=None): nothing to evaluate on")
        prov["integrity"] = self._integrity(cache, text)
        LOG.info("Example (train line 0, %s): %r | preprocessed width %d px", self.specs[int(domain["train"][0])].key, text["train"][0],
                 int(cache["train"]["width"][0]))

        # ---- labeled budget / calibration lines / unlabeled lines (decided BEFORE any label-dependent filtering) -----------------
        n_tr = len(text["train"])
        perm = np.random.RandomState(seed).permutation(n_tr)
        n_lab = n_tr if dc.labeled_fraction >= 1.0 else max(1, int(round(dc.labeled_fraction * n_tr)))
        labeled = np.zeros(n_tr, bool)
        labeled[perm[:n_lab]] = True
        drop, why = np.zeros(n_tr, bool), Counter()
        for i in np.flatnonzero(labeled):                      # only LABELED lines may be filtered by their text
            t = text["train"][i]
            frames = int(math.ceil(cache["train"]["width"][i] / 4))
            reason = ("empty transcription" if len(t) == 0 else
                      "no letter or digit in the transcription" if dc.drop_nonlinguistic and not is_linguistic(t)
                      else "transcription longer than max_text_len" if len(t) > dc.max_text_len
                      else "too few CTC frames" if ctc_min_frames(list(t)) > frames else None)
            if reason:
                drop[i] = True
                why[reason] += 1
        keep = np.flatnonzero(~drop)
        tr_cache, tr_text, tr_lab = _subset_cache(cache["train"], keep), [text["train"][i] for i in keep], labeled[keep]
        tr_sid, tr_idx, tr_dom = src_id["train"][keep], indices["train"][keep], domain["train"][keep]
        if why:
            LOG.info("dropped %d labeled train lines: %s", int(drop.sum()), dict(why))

        # ---- alphabet from the LABELED budget only (grapheme clusters; calibration lines are part of the budget) ------------------
        self.codec = CharCodec.from_texts([t for t, l in zip(tr_text, tr_lab) if l])
        if self.base_units:                                   # warm start: keep the checkpoint's indices, append what the new corpus adds
            from .alphabet import extend_codec_units
            self.codec, _groups = extend_codec_units(self.base_units, self.codec.units)
            LOG.info("alphabet extends the checkpoint's %d units with %d new ones", len(self.base_units), len(self.codec.units) - len(self.base_units))
        LOG.info("Alphabet (%d units, labeled lines only): %s", len(self.codec.units), self.codec.chars[:200] + ("..." if len(self.codec.units) > 200 else ""))
        oov = {}
        for k in ("val", "test"):
            tot = max(sum(len(t) for t in text[k]), 1)
            oov[k] = sum(1 for t in text[k] for c in t if c not in self.codec.c2i) / tot
        if max(oov.values()) > 0:
            LOG.info("characters outside the alphabet (counted as errors): val %.4f%% test %.4f%%", 100 * oov["val"], 100 * oov["test"])

        # ---- part of the labeled budget is HELD OUT (never trained on) to calibrate the trust rule / selection ---------------------
        calib = np.zeros(len(keep), bool)
        if (self.cfg.train.pseudo.enabled or self.cfg.train.frag.enabled) and dc.calib_fraction > 0 and not tr_lab.all():
            pos = np.flatnonzero(tr_lab)
            n_c = int(round(dc.calib_fraction * len(pos)))
            if n_c > 0:
                calib[np.random.RandomState(seed + 11).permutation(pos)[:n_c]] = True
        tr_lab_fit = tr_lab & ~calib
        LOG.info("labeled budget %d lines = %d for training + %d held out for calibration", int(tr_lab.sum()), int(tr_lab_fit.sum()), int(calib.sum()))

        # ---- style normalisation + style classes (images only) -------------------------------------------------------------------
        meta = {"train": tr_cache, "val": cache["val"], "test": cache["test"]}
        raw_tr = tr_cache["style"]
        with np.errstate(all="ignore"):
            mu = np.nan_to_num(np.nanmean(raw_tr, 0), nan=0.0)
            sd = np.nan_to_num(np.nanstd(raw_tr, 0), nan=1.0)
        self.style_mean, self.style_std = mu.astype(np.float32), np.maximum(sd, STYLE_FLOORS).astype(np.float32)
        norm, ok_d, raw = {}, {}, {}
        for k in meta:
            r = meta[k]["style"]
            z = (r - self.style_mean) / self.style_std
            ok_d[k], raw[k] = np.isfinite(z), r.astype(np.float32)
            norm[k] = np.clip(np.nan_to_num(z, nan=0.0), -3.0, 3.0).astype(np.float32)
        nan_frac = {n: float(1.0 - ok_d["train"][:, j].mean()) for j, n in enumerate(STYLE_NAMES)}
        LOG.info("style descriptors not measurable (masked, not invented): %s", {k: f"{100 * v:.1f}%" for k, v in nan_frac.items()})
        extra = {"train": [sources["train"][s].extra(int(j)) for s, j in zip(tr_sid, tr_idx)],
                 "val": [sources["val"][s].extra(int(j)) for s, j in zip(src_id["val"], indices["val"])],
                 "test": [sources["test"][s].extra(int(j)) for s, j in zip(src_id["test"], indices["test"])]}
        if dc.style_label_column and extra["train"] and extra["train"][0] is not None:
            vocab = {w: i for i, w in enumerate(sorted(set(extra["train"])))}
            self.n_clusters = len(vocab)
            cl = {k: np.array([vocab.get(e, -100) for e in extra[k]]) for k in extra}
            LOG.info("Style classes = %d writers from column '%s'", self.n_clusters, dc.style_label_column)
        else:
            self.centroids = kmeans_np(norm["train"], dc.style_num_clusters, seed=seed)
            self.n_clusters = len(self.centroids)
            cl = {k: ((norm[k][:, None, :] - self.centroids[None]) ** 2).sum(-1).argmin(1) for k in norm}
            for k in cl:                                       # a class label from <= 3 measured descriptors is not trusted
                cl[k] = np.where(ok_d[k].sum(1) >= NUM_DESC - 1, cl[k], -100)

        # ---- [STRATA] label-free difficulty cells: thin stroke + tight word gaps = hard; cut points per corpus (train images only) ---
        fc = self.cfg.train.frag
        n_str = max(1, int(fc.strata))
        doms_all = {"train": tr_dom, "val": domain["val"], "test": domain["test"]}
        hard = {k: -(np.where(ok_d[k][:, 1], norm[k][:, 1], 0.0) + np.where(ok_d[k][:, 3], norm[k][:, 3], 0.0)) for k in norm}
        per_dom = bool(fc.strata_per_domain) and len(self.specs) > 1
        self.strata_cuts = {}
        for d in range(len(self.specs)):
            h = hard["train"][tr_dom == d] if per_dom else hard["train"]
            self.strata_cuts[d] = np.quantile(h, np.linspace(0, 1, n_str + 1)[1:-1]) if (n_str > 1 and len(h) > n_str) else np.zeros(0)
            if not per_dom:
                break
        strat = {}
        for k in hard:
            s_ = np.zeros(len(hard[k]), np.int64)
            for d in range(len(self.specs)):
                m = doms_all[k] == d
                cuts = self.strata_cuts[d if per_dom else 0]
                s_[m] = np.digitize(hard[k][m], cuts) + (d * n_str if per_dom else 0)
            strat[k] = s_
        self.n_strata = n_str * (len(self.specs) if per_dom else 1)
        self.labeled_idx = np.flatnonzero(tr_lab_fit)
        self.calib_idx = np.flatnonzero(calib)
        self.unlabeled_idx = np.flatnonzero(~tr_lab)
        self.train_eval_idx = np.sort(self.labeled_idx[np.random.RandomState(seed + 7).permutation(len(self.labeled_idx))[:dc.train_eval_subset]])
        LOG.info("Train lines: %d (labeled %d / calibration %d / unlabeled %d) | val %d | test %d | corpora %s | strata %d", len(tr_idx),
                 len(self.labeled_idx), len(self.calib_idx), len(self.unlabeled_idx), len(text["val"]), len(text["test"]), self.domain_names, self.n_strata)

        # ---- how much of each descriptor does the TRANSCRIPTION alone explain?  (labeled lines only) ----------------------------
        self.text_baseline_r2 = text_baseline_r2([tr_text[i] for i in self.labeled_idx], norm["train"][self.labeled_idx],
                                                 ok_d["train"][self.labeled_idx], self.codec.units, seed=seed)
        self.leak_active = (self.text_baseline_r2 < 0.25) & (ok_d["train"].mean(0) > 0.5)
        LOG.info("style explained by the transcription alone (R2): %s | leakage penalty acts on: %s",
                 {n: round(float(v), 2) for n, v in zip(STYLE_NAMES, self.text_baseline_r2)}, [n for n, a_ in zip(STYLE_NAMES, self.leak_active) if a_])

        common = dict(img_h=dc.img_height, max_w=dc.max_img_width, min_w=dc.min_img_width, pad_x=dc.pad_x)
        self.train_ds = HandwritingDataset(sources["train"], tr_sid, tr_idx, tr_cache["ink"] if dc.cache_images else None,
                                           tr_cache["width"], norm["train"], ok_d["train"], raw["train"], cl["train"],
                                           tr_text, tr_lab_fit, calib, tr_dom, self.specs, name="train", expose_hidden=False, **common)
        for k, attr in (("val", "val_ds"), ("test", "test_ds")):
            setattr(self, attr, HandwritingDataset(
                sources[k], src_id[k], indices[k], cache[k]["ink"] if dc.cache_images else None, cache[k]["width"], norm[k], ok_d[k],
                raw[k], cl[k], text[k], np.ones(len(text[k]), bool), None, domain[k], self.specs, name=k, expose_hidden=True, **common))
        self.train_ds.stratum, self.val_ds.stratum, self.test_ds.stratum = strat["train"], strat["val"], strat["test"]

        # ---- [PROTOCOL] validation = val_sel (checkpoint selection) + val_gate (every post-hoc decision), disjoint whole form blocks ----
        nv, blk = len(text["val"]), max(1, int(dc.val_block))
        if 0.0 < dc.val_gate_fraction < 1.0 and nv >= 4 * blk:
            nb = int(math.ceil(nv / blk))
            gate_blocks = np.zeros(nb, bool)
            gate_blocks[np.random.RandomState(seed + 23).permutation(nb)[:max(1, int(round(dc.val_gate_fraction * nb)))]] = True
            is_gate = gate_blocks[np.arange(nv) // blk]
            self.val_gate_idx, self.val_sel_idx = np.flatnonzero(is_gate), np.flatnonzero(~is_gate)
        else:
            self.val_gate_idx = self.val_sel_idx = np.arange(nv)
            if dc.val_gate_fraction > 0 and nv < 4 * blk:
                LOG.warning("validation has %d lines (< 4 blocks of %d): no val_sel / val_gate split", nv, blk)
        LOG.info("validation: %d lines -> val_sel %d (checkpoint selection) + val_gate %d (post-hoc decisions)%s", nv, len(self.val_sel_idx),
                 len(self.val_gate_idx), "" if len(self.val_sel_idx) != nv else "  [NOT split]")
        nonling = {k: int(sum(1 for t in text[k] if not is_linguistic(t))) for k in ("val", "test")}
        if any(nonling.values()):
            LOG.warning("evaluation lines whose transcription has no letter / digit: val %d, test %d (kept in the main metric; the report "
                        "also gives the CER without them)", nonling["val"], nonling["test"])

        prov.update({"n_train_used": int(len(tr_idx)), "n_labeled": int(len(self.labeled_idx)), "n_calibration": int(len(self.calib_idx)),
                     "n_unlabeled": int(len(self.unlabeled_idx)), "n_val": int(nv), "n_test": int(len(text["test"])),
                     "dropped_labeled": dict(why), "alphabet_units": self.codec.units, "alphabet_size": len(self.codec.units), "oov_rate": oov,
                     "labeled_fraction": dc.labeled_fraction, "calib_fraction": dc.calib_fraction, "seed": seed,
                     "labeled_lines_sha1": hashlib.sha1(np.c_[tr_dom[self.labeled_idx], tr_idx[self.labeled_idx]].tobytes()).hexdigest(),
                     "calibration_lines_sha1": hashlib.sha1(np.c_[tr_dom[self.calib_idx], tr_idx[self.calib_idx]].tobytes()).hexdigest(),
                     "nonlinguistic_eval_lines": nonling, "style_not_measurable_fraction": nan_frac,
                     "style_explained_by_text_r2": dict(zip(STYLE_NAMES, self.text_baseline_r2.tolist())),
                     "domains": self.domain_names, "n_strata": self.n_strata,
                     "lines_per_domain": {k: {s.key: int((doms_all[k] == d).sum()) for d, s in enumerate(self.specs)} for k in doms_all},
                     "versions": {"torch": torch.__version__, "numpy": np.__version__}})
        try:
            import datasets as _hfd
            prov["versions"]["datasets"] = _hfd.__version__
        except Exception:
            pass
        os.makedirs(self.cfg.run_dir, exist_ok=True)
        with open(os.path.join(self.cfg.run_dir, "provenance.json"), "w") as f:
            json.dump(json_safe(prov), f, indent=2, ensure_ascii=False)
        if self.is_synthetic:
            with open(os.path.join(self.cfg.run_dir, "SYNTHETIC_SMOKE_TEST_NOT_FOR_PUBLICATION.txt"), "w") as f:
                f.write("This run used generated fake handwriting. Its numbers have no scientific meaning.\n")
        return self

    # ---------------------------------------------------------------- loaders
    def denorm_style(self, v: np.ndarray) -> np.ndarray:
        return np.asarray(v) * self.style_std + self.style_mean

    def _loader(self, ds: HandwritingDataset, sampler: BucketBatchSampler, workers: Optional[int] = None, collator=None) -> DataLoader:
        nw = self.dc.num_workers if workers is None else workers
        return DataLoader(ds, batch_sampler=sampler, collate_fn=collator or Collator(self.codec), num_workers=nw,
                          pin_memory=torch.cuda.is_available(), persistent_workers=False)

    def train_loader(self, indices: Optional[Sequence[int]] = None) -> DataLoader:
        """`indices` = the lines of this epoch (may contain repeats); default: every train line once."""
        s = BucketBatchSampler(self.train_ds.width, self.dc.batch_size, True, self.dc.bucket_mega, seed=self.cfg.run.seed, indices=indices)
        tc = self.cfg.train
        return self._loader(self.train_ds, s, collator=Collator(self.codec, mix_prob=tc.mixline_prob, max_w=self.dc.max_img_width,
                                                                  gap=tc.mixline_gap, seed=self.cfg.run.seed))

    def eval_loader(self, ds: HandwritingDataset, indices: Optional[Sequence[int]] = None, batch_size: Optional[int] = None) -> DataLoader:
        s = BucketBatchSampler(ds.width, batch_size or self.dc.eval_batch_size, False, indices=indices)
        return self._loader(ds, s, workers=0 if ds.ink is not None else None)   # cached lines need no decoding workers

In [ ]:
%%writefile inklingua/model.py
"""
InkLinguaNet: CNN -> dual-granularity VQ bottleneck -> style-FiLM Transformer -> CTC, with the connected style task; the losses;
the EMA teacher; the style-equivariant GPU augmentation ([SES]) and the closed-form leakage penalty ([CFL]).
"""
from __future__ import annotations

import copy
import math
from typing import Any, Dict, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from .config import LossConfig, ModelConfig, TrainConfig
from .style import SLANT_RANGE_DEG


# ==============================================================================
# layers
# ==============================================================================
class MaskedBatchNorm2d(nn.Module):
    """BatchNorm whose training statistics are computed over REAL columns only (nn.BatchNorm2d also counts the zero padding of short
    lines, so its statistics depended on how much padding the batch contained).  `m` is a [B,1,1,W] boolean mask of real columns;
    padded columns are returned as exact zeros.  `collect` (AdaBN) accumulates exact sums for test-time adaptation."""

    def __init__(self, c: int, eps: float = 1e-5, momentum: float = 0.1):
        super().__init__()
        self.eps, self.momentum = eps, momentum
        self.weight = nn.Parameter(torch.ones(c))
        self.bias = nn.Parameter(torch.zeros(c))
        self.register_buffer("running_mean", torch.zeros(c))
        self.register_buffer("running_var", torch.ones(c))
        self.collect: Optional[Dict[str, Any]] = None

    def forward(self, x: torch.Tensor, m: torch.Tensor) -> torch.Tensor:
        xf, mf = x.float(), m.to(torch.float32)
        if self.collect is not None and not self.training:
            n = (mf.sum() * x.size(2)).clamp(min=1.0)
            s1, s2 = (xf * mf).sum((0, 2, 3)), ((xf ** 2) * mf).sum((0, 2, 3))
            mean = s1 / n
            var = (s2 / n - mean ** 2).clamp(min=0.0)
            a = self.collect
            a["n"] = a.get("n", 0.0) + float(n)
            a["s1"] = a.get("s1", 0.0) + s1.double().cpu().numpy()
            a["s2"] = a.get("s2", 0.0) + s2.double().cpu().numpy()
        elif self.training:
            n = (mf.sum() * x.size(2)).clamp(min=1.0)
            mean = (xf * mf).sum((0, 2, 3)) / n
            var = (((xf - mean.view(1, -1, 1, 1)) ** 2) * mf).sum((0, 2, 3)) / n
            with torch.no_grad():
                unb = var * n / (n - 1.0).clamp(min=1.0)
                self.running_mean.lerp_(mean.detach(), self.momentum)
                self.running_var.lerp_(unb.detach(), self.momentum)
        else:
            mean, var = self.running_mean, self.running_var
        y = (xf - mean.view(1, -1, 1, 1)) * torch.rsqrt(var.view(1, -1, 1, 1) + self.eps)
        y = y * self.weight.view(1, -1, 1, 1) + self.bias.view(1, -1, 1, 1)
        return (y * mf).to(x.dtype)


class ConvBNAct(nn.Module):
    def __init__(self, cin: int, cout: int, k: int = 3, s: int = 1):
        super().__init__()
        self.conv = nn.Conv2d(cin, cout, k, s, k // 2, bias=False)
        self.bn = MaskedBatchNorm2d(cout)

    def forward(self, x, m):
        return F.relu(self.bn(self.conv(x), m))


class BasicBlock(nn.Module):
    def __init__(self, cin: int, cout: int, stride=(1, 1)):
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.b1 = MaskedBatchNorm2d(cout)
        self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.b2 = MaskedBatchNorm2d(cout)
        self.has_short = not (tuple(stride) == (1, 1) and cin == cout)
        if self.has_short:
            self.sc = nn.Conv2d(cin, cout, 1, stride, bias=False)
            self.sb = MaskedBatchNorm2d(cout)

    def forward(self, x, m):
        y = F.relu(self.b1(self.c1(x), m))
        y = self.b2(self.c2(y), m)
        s = self.sb(self.sc(x), m) if self.has_short else x
        return F.relu(y + s) * m.to(y.dtype)


class CNNBackbone(nn.Module):
    """ResNet-style. Width is downsampled 4x (T = ceil(W/4) frames), height 16x.  Returns a LOW-level map (stroke level, H/4) and a
    HIGH-level map (component level, H/16).  Every layer sees a column mask, so a line's output does not depend on the batch padding."""

    def __init__(self, ch=(32, 64, 128, 256)):
        super().__init__()
        self.stem = ConvBNAct(1, ch[0])
        self.l1 = nn.ModuleList([BasicBlock(ch[0], ch[1], (2, 2)), BasicBlock(ch[1], ch[1])])
        self.l2 = nn.ModuleList([BasicBlock(ch[1], ch[2], (2, 2)), BasicBlock(ch[2], ch[2])])
        self.l3 = nn.ModuleList([BasicBlock(ch[2], ch[3], (2, 1)), BasicBlock(ch[3], ch[3])])
        self.l4 = nn.ModuleList([BasicBlock(ch[3], ch[3], (2, 1)), BasicBlock(ch[3], ch[3])])

    def forward(self, x, widths):
        W = x.size(-1)

        def mk(f):
            lim = torch.ceil(widths.float() / f).long()
            return (torch.arange(-(-W // f), device=x.device)[None] < lim[:, None])[:, None, None, :]

        m1, m2, m4 = mk(1), mk(2), mk(4)
        x = self.stem(x, m1)
        for blk in self.l1:
            x = blk(x, m2)
        for blk in self.l2:
            x = blk(x, m4)
        low = x
        for blk in list(self.l3) + list(self.l4):
            x = blk(x, m4)
        return low, x


class VectorQuantizerEMA(nn.Module):
    """VQ-VAE codebook with EMA updates, data-dependent init and dead-code restarts.  Input [B,T,D]; `valid` [B,T] marks real frames."""

    def __init__(self, num_codes: int, dim: int, decay: float = 0.99, eps: float = 1e-5, dead_thresh: float = 0.5):
        super().__init__()
        self.K, self.D, self.decay, self.eps, self.dead = num_codes, dim, decay, eps, dead_thresh
        self.register_buffer("embed", torch.randn(num_codes, dim) * 0.1)
        self.register_buffer("embed_avg", self.embed.clone())
        self.register_buffer("cluster_size", torch.ones(num_codes))
        self.register_buffer("initted", torch.zeros(1))
        self._init_py: Optional[bool] = None

    @torch.no_grad()
    def _update(self, flat: torch.Tensor, idx: torch.Tensor) -> None:
        oh = F.one_hot(idx, self.K).to(flat.dtype)
        n, s = oh.sum(0), oh.t() @ flat
        self.cluster_size.mul_(self.decay).add_(n, alpha=1 - self.decay)
        self.embed_avg.mul_(self.decay).add_(s, alpha=1 - self.decay)
        tot = self.cluster_size.sum()
        cs = (self.cluster_size + self.eps) / (tot + self.K * self.eps) * tot
        self.embed.copy_(self.embed_avg / cs.unsqueeze(1))
        dead = self.cluster_size < self.dead
        if flat.size(0) > 0 and bool(dead.any()):
            r = flat[torch.randint(0, flat.size(0), (int(dead.sum()),), device=flat.device)]
            self.embed[dead] = r
            self.embed_avg[dead] = r
            self.cluster_size[dead] = self.dead * 2

    def forward(self, z: torch.Tensor, valid: torch.Tensor):
        with torch.autocast(device_type=z.device.type, enabled=False):
            zf = z.float()
            B, T, D = zf.shape
            flat = zf.reshape(-1, D)
            vflat = valid.reshape(-1)
            if self._init_py is None:
                self._init_py = bool(float(self.initted) > 0)
            if self.training and not self._init_py and int(vflat.sum()) >= self.K:
                with torch.no_grad():
                    fv = flat[vflat].detach()
                    pick = fv[torch.randperm(fv.size(0), device=z.device)[:self.K]]
                    self.embed.copy_(pick)
                    self.embed_avg.copy_(pick)
                    self.initted.fill_(1)
                    self._init_py = True
            d = flat.pow(2).sum(1, keepdim=True) - 2 * flat @ self.embed.t() + self.embed.pow(2).sum(1)[None]
            idx = d.argmin(1)
            q = F.embedding(idx, self.embed).view(B, T, D)
            if self.training:
                self._update(flat[vflat].detach(), idx[vflat])
            vm = valid.unsqueeze(-1).float()
            commit = (((zf - q.detach()) ** 2) * vm).sum() / (vm.sum() * D).clamp(min=1)
            q_st = zf + (q - zf).detach()
            with torch.no_grad():
                p = torch.bincount(idx[vflat], minlength=self.K).float()
                p = p / p.sum().clamp(min=1)
                ppl = torch.exp(-(p * torch.log(p + 1e-10)).sum())
        return q_st.to(z.dtype), q.to(z.dtype), idx.view(B, T), commit, ppl

    def _load_from_state_dict(self, *a, **kw):
        self._init_py = None
        super()._load_from_state_dict(*a, **kw)


class PositionalEncoding(nn.Module):
    def __init__(self, d: int, max_len: int = 4096):
        super().__init__()
        pe = torch.zeros(max_len, d)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
        pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div[: d // 2])
        self.max_len = max_len
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)

    def forward(self, T: int) -> torch.Tensor:
        if T > self.max_len:
            raise ValueError(f"{T} frames exceed the positional-encoding table ({self.max_len}); reduce cfg.data.max_img_width")
        return self.pe[:, :T]


class FiLM(nn.Module):
    """Feature-wise affine modulation from the style embedding (identity at init).  gamma / beta are BOUNDED (|.| <= max_scale, tanh)
    and have NO bias (an unconstrained, biased FiLM learned a constant per-channel offset that carried no style)."""

    def __init__(self, sdim: int, d: int, max_scale: float = 0.5):
        super().__init__()
        self.lin = nn.Linear(sdim, 2 * d, bias=False)
        nn.init.zeros_(self.lin.weight)
        self.max_scale = float(max_scale)

    def forward(self, x, s):
        g, b = (self.max_scale * torch.tanh(self.lin(s))).chunk(2, dim=-1)
        return x * (1 + g.unsqueeze(1)) + b.unsqueeze(1), g, b


class ConvModule(nn.Module):
    def __init__(self, d: int, k: int = 7):
        super().__init__()
        self.ln, self.pw1 = nn.LayerNorm(d), nn.Linear(d, 2 * d)
        self.dw = nn.Conv1d(d, d, k, padding=k // 2, groups=d)
        self.pw2 = nn.Linear(d, d)

    def forward(self, x, valid):
        y = F.glu(self.pw1(self.ln(x)), dim=-1).masked_fill(~valid.unsqueeze(-1), 0.0)
        y = F.gelu(self.dw(y.transpose(1, 2))).transpose(1, 2)
        return x + self.pw2(y)


class EncoderLayer(nn.Module):
    def __init__(self, d, heads, ffn, drop, k):
        super().__init__()
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, heads, dropout=drop, batch_first=True)
        self.conv = ConvModule(d, k)
        self.ff = nn.Sequential(nn.Linear(d, ffn), nn.GELU(), nn.Dropout(drop), nn.Linear(ffn, d))
        self.drop = nn.Dropout(drop)

    def forward(self, x, pad_mask, valid, need_attn=False):
        h = self.n1(x)
        a, w = self.attn(h, h, h, key_padding_mask=pad_mask, need_weights=need_attn, average_attn_weights=False)
        x = x + self.drop(a)
        x = self.conv(x, valid)
        x = x + self.drop(self.ff(self.n2(x)))
        return x, w


class _GradScale(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, a):
        ctx.a = a
        return x.view_as(x)

    @staticmethod
    def backward(ctx, g):
        return g * ctx.a, None


def grad_scale(x: torch.Tensor, a: float) -> torch.Tensor:
    return x if a == 1.0 else _GradScale.apply(x, a)


def masked_mean(x: torch.Tensor, valid: torch.Tensor) -> torch.Tensor:
    w = valid.unsqueeze(-1).to(x.dtype)
    return (x * w).sum(1) / w.sum(1).clamp(min=1)


class StyleEncoder(nn.Module):
    """Style stream. Input: quantisation residuals of both codebook levels (what the discrete content codes cannot express).
    Output: style embedding + descriptor regression + style class + the attention over frames (which frames define the style)."""

    def __init__(self, d_in: int, style_dim: int, n_desc: int, n_cls: int):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(d_in, 256), nn.GELU(), nn.LayerNorm(256))
        self.conv = nn.Conv1d(256, 256, 5, padding=2)
        self.att = nn.Linear(256, 1)
        self.out = nn.Sequential(nn.Linear(256, style_dim), nn.LayerNorm(style_dim))
        self.reg = nn.Sequential(nn.Linear(style_dim, 128), nn.GELU(), nn.Linear(128, n_desc))
        self.cls = nn.Linear(style_dim, n_cls)

    def forward(self, feat, valid):
        h = self.proj(feat).masked_fill(~valid.unsqueeze(-1), 0.0)
        h = F.gelu(self.conv(h.transpose(1, 2))).transpose(1, 2)
        a = self.att(h).squeeze(-1).masked_fill(~valid, -1e4).softmax(-1)          # [B,T]
        s = self.out((h * a.unsqueeze(-1)).sum(1))
        return s, self.reg(s), self.cls(s), a


# ==============================================================================
# the network
# ==============================================================================
class InkLinguaNet(nn.Module):
    def __init__(self, mc: ModelConfig, num_classes: int, num_desc: int, num_clusters: int, img_h: int):
        super().__init__()
        self.mc, self.num_classes = mc, num_classes
        ch, d = mc.channels, mc.d_model
        self.backbone = CNNBackbone(ch)
        self.low_proj = nn.Sequential(nn.Linear(ch[2] * (img_h // 4), mc.low_dim), nn.LayerNorm(mc.low_dim))
        self.high_proj = nn.Sequential(nn.Linear(ch[3] * (img_h // 16), d), nn.LayerNorm(d))
        self.vq_low = VectorQuantizerEMA(mc.codes_low, mc.low_dim, mc.vq_decay, dead_thresh=mc.vq_dead_thresh)
        self.vq_high = VectorQuantizerEMA(mc.codes_high, d, mc.vq_decay, dead_thresh=mc.vq_dead_thresh)
        self.fuse = nn.Sequential(nn.Linear(mc.low_dim + d, d), nn.LayerNorm(d))
        self.aux_head = nn.Linear(d, num_classes)                      # inter-CTC on the quantised content
        self.mask_token = nn.Parameter(torch.randn(d) * 0.02)
        self.pos = PositionalEncoding(d)
        self.lstm = nn.LSTM(d, d // 2, batch_first=True, bidirectional=True) if mc.lstm_prefix else None
        self.lstm_ln = nn.LayerNorm(d) if mc.lstm_prefix else None
        self.film_in = FiLM(mc.style_dim, d, mc.film_max) if mc.film else None
        self.film_out = FiLM(mc.style_dim, d, mc.film_max) if mc.film else None
        self.layers = nn.ModuleList([EncoderLayer(d, mc.heads, mc.ffn, mc.dropout, mc.conv_kernel) for _ in range(mc.enc_layers)])
        self.final_ln = nn.LayerNorm(d)
        self.ctc_head = nn.Linear(d, num_classes)
        self.mask_head = nn.Linear(d, mc.codes_high)
        self.style_enc = StyleEncoder(mc.low_dim + d, mc.style_dim, num_desc, num_clusters)

    def _span_mask(self, valid: torch.Tensor, p: float, span: int) -> torch.Tensor:
        starts = torch.rand(valid.shape, device=valid.device) < (p / max(span, 1))
        m = F.max_pool1d(starts.float().unsqueeze(1), span, 1, span // 2).squeeze(1)[:, :valid.size(1)] > 0
        return m & valid

    def forward(self, x, widths, vq_lambda: float = 1.0, mask_prob: float = 0.0, extras: bool = False, need_attn: bool = False,
                keep_hidden: bool = False) -> Dict[str, Any]:
        """keep_hidden=True also returns the encoder state after every layer (`hidden`: list of [B,T,d]) for the layer-wise probes."""
        low, high = self.backbone(x, widths)
        T = high.size(-1)
        lens = torch.clamp(torch.ceil(widths.float() / 4).long(), min=1, max=T)
        valid = torch.arange(T, device=x.device)[None] < lens[:, None]
        zl = self.low_proj(low.permute(0, 3, 1, 2).flatten(2))
        zh = self.high_proj(high.permute(0, 3, 1, 2).flatten(2))
        ql_st, ql, il, c_l, ppl_l = self.vq_low(zl, valid)
        qh_st, qh, ih, c_h, ppl_h = self.vq_high(zh, valid)

        cl = zl + vq_lambda * (ql_st - zl)                      # content stream: annealed quantisation (continuous -> discrete)
        ch = zh + vq_lambda * (qh_st - zh)
        content = self.fuse(torch.cat([cl, ch], -1))
        aux_lp = F.log_softmax(self.aux_head(content).float(), -1)
        c_pool = masked_mean(content, valid)

        if self.mc.style_input == "residual":                   # style stream: what the codebooks cannot express
            sfeat = torch.cat([zl - ql.detach(), zh - qh.detach()], -1)
        else:
            sfeat = torch.cat([zl, zh], -1)
        style, style_reg, style_cls, style_att = self.style_enc(grad_scale(sfeat, self.mc.style_grad_scale), valid)

        mask = None
        h = content
        if self.training and mask_prob > 0:                     # masked-code modelling (span masking of the content stream)
            mask = self._span_mask(valid, mask_prob, self.mc.mask_span)
            h = torch.where(mask.unsqueeze(-1), self.mask_token.to(content.dtype).expand_as(content), content)
        if self.lstm is not None:
            pk = nn.utils.rnn.pack_padded_sequence(self.lstm_ln(h), lens.cpu(), batch_first=True, enforce_sorted=False)
            r, _ = self.lstm(pk)
            r, _ = nn.utils.rnn.pad_packed_sequence(r, batch_first=True, total_length=T)
            h = h + r.to(h.dtype)
        h = h + self.pos(T).to(h.dtype)
        cond = style.detach() if self.mc.film_detach_style else style
        g_in = g_out = None
        if self.film_in is not None:
            h, g_in, _ = self.film_in(h, cond)
        pad = ~valid
        attns, hidden = [], [h] if keep_hidden else []
        nl = len(self.layers)
        for li, layer in enumerate(self.layers):
            if (self.training and self.mc.layer_drop > 0 and not need_attn           # stochastic depth: the residual path alone
                    and float(torch.rand(())) < self.mc.layer_drop * (li + 1) / nl):
                continue
            h, w = layer(h, pad, valid, need_attn)
            if need_attn:
                attns.append(w)
            if keep_hidden:
                hidden.append(h)
        h = self.final_ln(h)
        if self.film_out is not None:
            h, g_out, _ = self.film_out(h, cond)
        log_probs = F.log_softmax(self.ctc_head(h).float(), -1)

        out: Dict[str, Any] = {"log_probs": log_probs, "out_lens": lens, "aux_log_probs": aux_lp, "style_reg": style_reg.float(),
                               "style_cls": style_cls.float(), "style_emb": style.float(), "content_pool": c_pool.float(),
                               "vq_commit": c_l + c_h, "ppl_low": ppl_l, "ppl_high": ppl_h}
        if mask is not None and bool(mask.any()):
            out["mask_logits"] = self.mask_head(h[mask]).float()
            out["mask_target"] = ih[mask]
        if extras:
            out.update({"idx_low": il, "idx_high": ih, "valid": valid, "attn": attns, "style_att": style_att.float(),
                        "film_gamma": g_in if g_in is not None else g_out})
        if keep_hidden:
            out["hidden"] = hidden
        return out


# ==============================================================================
# losses
# ==============================================================================
def ctc_per_sample(log_probs, out_lens, targets, target_lens) -> torch.Tensor:
    loss = F.ctc_loss(log_probs.transpose(0, 1), targets, out_lens, target_lens, blank=0, reduction="none", zero_infinity=True)
    return loss / target_lens.clamp(min=1).float()


def cross_cov_loss(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    if a.size(0) < 4:
        return a.new_zeros(())
    a = (a - a.mean(0)) / (a.std(0) + 1e-5)
    b = (b - b.mean(0)) / (b.std(0) + 1e-5)
    return ((a.t() @ b / (a.size(0) - 1)) ** 2).mean()


def frame_consistency(student_lp: torch.Tensor, teacher_lp: torch.Tensor, lens: torch.Tensor, rows: Optional[torch.Tensor] = None) -> torch.Tensor:
    """KL(teacher || student) of the per-frame label posteriors, averaged over real frames (of the selected `rows`).  The teacher (EMA)
    reads the weak view, the student the strong view of the same geometry, so frame t describes the same pixels in both (CR-CTC-style
    view consistency, applied to EVERY line -- the unlabeled lines teach invariances without any label)."""
    T = student_lp.size(1)
    valid = (torch.arange(T, device=student_lp.device)[None] < lens[:, None]).float()
    if rows is not None:
        valid = valid * rows[:, None].float()
    kl = (teacher_lp.exp() * (teacher_lp - student_lp)).sum(-1)
    return (kl * valid).sum() / valid.sum().clamp(min=1.0)


def fragment_distillation(student_lp: torch.Tensor, teacher_lp: torch.Tensor, lens: torch.Tensor, trust: torch.Tensor,
                          rows: torch.Tensor, tau: float, anchor: float) -> torch.Tensor:
    """[FRAG-RC] soft self-distillation restricted to trusted fragments.  On trusted frames the target is the SHARPENED teacher
    posterior softmax(log p / tau) (tau < 1: the student becomes more decisive exactly where the teacher is reliable); on the other
    frames of the same lines the target is the UNsharpened teacher posterior with weight `anchor`.  Averaged over all frames of the
    selected rows, so the gradient scales with the trusted share.  rows: [B] bool; trust: [B, T] bool."""
    T = student_lp.size(1)
    valid = (torch.arange(T, device=student_lp.device)[None] < lens[:, None]) & rows[:, None]
    t = teacher_lp.detach()
    sharp = F.log_softmax(t / max(tau, 1e-3), dim=-1)
    kl_s = (sharp.exp() * (sharp - student_lp)).sum(-1)
    kl_a = (t.exp() * (t - student_lp)).sum(-1)
    w = torch.where(trust, torch.ones_like(kl_s), torch.full_like(kl_s, float(anchor))) * valid.float()
    kl = torch.where(trust, kl_s, kl_a)
    return (kl * w).sum() / valid.float().sum().clamp(min=1.0)


class ClosedFormLeakage:
    """[CFL] Closed-form leakage penalty on the content stream, with a transcription-explained floor.  A queue holds recent (content
    embedding, standardised style target, measured-mask) triples; every step a ridge regression is fitted on the queue in closed form
    and applied to the CURRENT batch; the encoder minimises  mean_j relu( E[yhat_j^2] - floor_j - slack )  over the active descriptors
    (targets are z-scores, so E[yhat^2] estimates the explained variance).  floor_j = R2 the TRANSCRIPTION alone achieves."""

    def __init__(self, dim: int, n_desc: int, size: int, ridge: float, slack: float, active: np.ndarray, floor: np.ndarray, device: torch.device):
        self.size, self.ridge, self.slack = int(size), float(ridge), float(slack)
        self.active = [int(j) for j in np.flatnonzero(np.asarray(active, bool))]
        self.floor = torch.as_tensor(np.asarray(floor, np.float32), device=device)
        self.C = torch.zeros(self.size, dim, device=device)
        self.Y = torch.zeros(self.size, n_desc, device=device)
        self.M = torch.zeros(self.size, n_desc, device=device)
        self.n, self.ptr = 0, 0
        self.last: Dict[str, float] = {}

    @torch.no_grad()
    def _push(self, c: torch.Tensor, y: torch.Tensor, m: torch.Tensor) -> None:
        for k in range(c.size(0)):
            self.C[self.ptr], self.Y[self.ptr], self.M[self.ptr] = c[k], y[k], m[k]
            self.ptr = (self.ptr + 1) % self.size
            self.n = min(self.n + 1, self.size)

    def __call__(self, c: torch.Tensor, y: torch.Tensor, m: torch.Tensor) -> torch.Tensor:
        c, y, m = c.float(), y.detach().float(), m.detach().float()
        zero = c.new_zeros(())
        if not self.active or self.n < max(64, self.size // 4):
            self._push(c.detach(), y, m)
            return zero
        with torch.no_grad():
            Cq, Yq, Mq = self.C[:self.n], self.Y[:self.n], self.M[:self.n]
            mu, sd = Cq.mean(0), Cq.std(0) + 1e-4
            A = (Cq - mu) / sd
            W = torch.zeros(A.size(1), Yq.size(1), device=c.device)
            eye = torch.eye(A.size(1), device=c.device)
            for j in self.active:
                mj = Mq[:, j]
                nj = mj.sum()
                if float(nj) < 32:
                    continue
                Aj = A * mj[:, None]
                W[:, j] = torch.linalg.solve(Aj.t() @ A / nj + self.ridge * eye, Aj.t() @ Yq[:, j] / nj)
        pred = ((c - mu) / sd) @ W
        leak = (pred ** 2 * m).sum(0) / m.sum(0).clamp(min=1.0)
        act = torch.zeros_like(leak, dtype=torch.bool)
        act[self.active] = True
        pen = torch.relu(leak - self.floor - self.slack)[act].mean()
        self.last = {"leak_r2_" + str(j): float(leak[j].detach()) for j in self.active}
        self._push(c.detach(), y, m)
        return pen


def compute_losses(out: Dict[str, Any], batch: Dict[str, Any], lc: LossConfig, style_t: torch.Tensor, style_mask: torch.Tensor,
                   cluster_ok: torch.Tensor, training: bool, aux_scale: float = 1.0, vq_scale: float = 1.0,
                   teacher_lp: Optional[torch.Tensor] = None, leak: Optional[ClosedFormLeakage] = None,
                   frag: Optional[Dict[str, Any]] = None) -> Dict[str, torch.Tensor]:
    """All loss terms.  Recognition losses are weighted per sample (labeled = 1, pseudo-label = risk weight, none = 0).  Style losses use
    the (augmentation-transformed) targets `style_t` [B,5] on the entries flagged in `style_mask`.  `teacher_lp`: EMA-teacher posteriors
    (weak view) -> consistency loss; `frag`: trust masks of the unlabeled rows -> fragment distillation."""
    dev = out["log_probs"].device
    w = batch["weight"].to(dev).float()
    tgt, tl = batch["target"].to(dev), batch["target_len"].to(dev)
    L: Dict[str, torch.Tensor] = {}
    ctc = ctc_per_sample(out["log_probs"], out["out_lens"], tgt, tl)
    L["ctc"] = (ctc * w).sum() / w.sum().clamp(min=1.0)
    aux = ctc_per_sample(out["aux_log_probs"], out["out_lens"], tgt, tl)
    L["aux_ctc"] = (aux * w).sum() / w.sum().clamp(min=1.0)
    L["_ctc_ps"] = ctc.detach()
    if training and lc.ctc_smooth > 0:
        # [CTC-LS] label smoothing on EMITTING frames only: -mean_{c != blank} log p_t(c) where the arg-max is a character; the blank frames
        # stay sharp (a plain entropy bonus would also flatten the blanks and break the alignment)
        lpf = out["log_probs"]
        Tn = lpf.size(1)
        vm = (torch.arange(Tn, device=lpf.device)[None] < out["out_lens"][:, None]) & (lpf.argmax(-1) != 0)
        ls = (-lpf[..., 1:].mean(-1) * vm).sum(1) / vm.sum(1).clamp(min=1)
        L["ctc_ls"] = (ls * w).sum() / w.sum().clamp(min=1.0)
    else:
        L["ctc_ls"] = out["log_probs"].new_zeros(())
    sm = style_mask.float()
    cl = batch["cluster"].to(dev)
    L["style_reg"] = (F.smooth_l1_loss(out["style_reg"], style_t, reduction="none") * sm).sum() / sm.sum().clamp(min=1.0)
    cmask = cluster_ok.float() * (cl >= 0).float()
    cls_ps = F.cross_entropy(out["style_cls"], cl, ignore_index=-100, reduction="none", label_smoothing=lc.style_cls_smoothing)
    L["style_cls"] = (cls_ps * cmask).sum() / cmask.sum().clamp(min=1.0)
    if training:
        zero = out["log_probs"].new_zeros(())
        L["leak"] = leak(out["content_pool"], style_t, sm) if (leak is not None and lc.leak > 0) else zero
        use_frag = frag is not None and teacher_lp is not None and lc.frag > 0
        L["frag"] = (fragment_distillation(out["log_probs"], teacher_lp, out["out_lens"], frag["trust"], frag["rows"], frag["tau"], frag["anchor"])
                     if use_frag else zero)
        L["cons"] = (frame_consistency(out["log_probs"], teacher_lp, out["out_lens"], (~frag["rows"]) if use_frag else None)
                     if (teacher_lp is not None and lc.consistency > 0) else zero)
        L["decor"] = cross_cov_loss(out["style_emb"], out["content_pool"])
        L["vq"] = out["vq_commit"]
        L["mask"] = F.cross_entropy(out["mask_logits"], out["mask_target"]) if "mask_logits" in out else zero
        L["total"] = (L["ctc"] + lc.ctc_smooth * L["ctc_ls"] + vq_scale * lc.vq_commit * L["vq"]
                      + aux_scale * (lc.aux_ctc * L["aux_ctc"] + lc.style_reg * L["style_reg"] + lc.style_cls * L["style_cls"]
                                     + lc.leak * L["leak"] + lc.consistency * L["cons"] + lc.frag * L["frag"]
                                     + lc.decorrelate * L["decor"] + lc.masked_code * L["mask"]))
    else:   # validation "task loss": comparable between train-subset and val
        L["total"] = L["ctc"] + lc.style_reg * L["style_reg"] + lc.style_cls * L["style_cls"]
    return L


# ==============================================================================
# EMA teacher
# ==============================================================================
class ModelEMA:
    """Exponential moving average of the weights AND of the float buffers (VQ codebooks, BN statistics)."""

    def __init__(self, model: nn.Module, decay: float):
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)
        self.decay = decay

    @torch.no_grad()
    def update(self, model: nn.Module, step: int) -> None:
        d = min(self.decay, (1 + step) / (10 + step))
        for pe, pm in zip(self.module.parameters(), model.parameters()):
            pe.mul_(d).add_(pm.detach(), alpha=1 - d)
        for (n, be), (_, bm) in zip(self.module.named_buffers(), model.named_buffers()):
            if be.dtype.is_floating_point and not n.endswith("initted"):
                be.mul_(d).add_(bm.detach(), alpha=1 - d)
            else:
                be.copy_(bm)


# ==============================================================================
# [SES] geometry <-> style relations and the GPU augmentation
# ==============================================================================
def transform_slant_deg(slant_deg: torch.Tensor, shear: torch.Tensor, rot: torch.Tensor, vscale: torch.Tensor,
                        inv_hscale: Optional[torch.Tensor] = None) -> torch.Tensor:
    """Closed-form effect of the sampling map  x_in = a * x_out + (shear - rot) * y_out,  y_in = rot * x_out + vscale * y_out  on the
    slant (deg, >0 = leaning right):  tan(slant') = (vscale * tan(slant) + shear - rot) / (a + rot * tan(slant))."""
    a = 1.0 if inv_hscale is None else inv_hscale
    t = torch.tan(torch.deg2rad(slant_deg))
    t2 = (vscale * t + (shear - rot)) / (a + rot * t)
    return torch.rad2deg(torch.atan(t2))


def body_ratio_t(x: torch.Tensor, thr: float = 0.45) -> torch.Tensor:
    """Body ratio of [B,1,H,W] images, measured exactly like style.body_ratio_from_rows; 0 for blank images."""
    H = x.size(2)
    rows = (x > thr).float().sum(3)
    s = F.conv1d(rows, torch.ones(1, 1, 3, device=x.device), padding=1)[:, 0]
    mx = s.amax(1, keepdim=True)
    return ((s >= 0.5 * mx).float().sum(1) / H) * (mx[:, 0] > 0).float()


class GPUAugment:
    """Batch augmentation on [B,1,H,W] ink images (ink = 1).  [SES] Style-Equivariant Supervision: geometric augmentations change the
    style descriptors in KNOWN ways, so the targets are TRANSFORMED instead of discarded (slant: closed form; body ratio: re-measured;
    stroke / density / gap: masked for altered lines).  Two views are returned: STRONG (geometry + morphology + photometric + blur +
    noise) for the student and WEAK (the same geometry only) for the EMA teacher -- frame t covers the same pixels in both."""

    def __init__(self, tc: TrainConfig, style_mean: np.ndarray, style_std: np.ndarray):
        self.tc = tc
        self.mean = torch.as_tensor(style_mean, dtype=torch.float32)
        self.std = torch.as_tensor(style_std, dtype=torch.float32)

    def __call__(self, x: torch.Tensor, style_z: torch.Tensor, style_ok: torch.Tensor, widths: torch.Tensor,
                 min_width: Optional[torch.Tensor] = None
                 ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        tc, B, dev = self.tc, x.size(0), x.device
        H, Wc = x.shape[-2:]
        mean, std = self.mean.to(dev), self.std.to(dev)
        rnd = lambda lo, hi: torch.empty(B, 1, 1, 1, device=dev).uniform_(lo, hi)  # noqa: E731
        st = style_z.to(dev).float().clone()
        sm = style_ok.to(dev).float().clone()
        wd = widths.to(dev).clone()
        cok = torch.ones(B, dtype=torch.bool, device=dev)

        def colmask(w):
            return (torch.arange(Wc, device=dev)[None] < w[:, None]).float()[:, None, None, :]

        if tc.strong_aug:
            body0 = body_ratio_t(x)
            geo = torch.rand(B, device=dev) < tc.aug_geom_prob
            hs = torch.rand(B, device=dev) < tc.aug_hscale_prob
            ela = torch.rand(B, device=dev) < tc.aug_elastic_prob
            shear = torch.empty(B, device=dev).uniform_(-tc.aug_shear_max, tc.aug_shear_max) * geo
            rot = torch.empty(B, device=dev).uniform_(-1, 1) * (tc.aug_tilt_px / (Wc / 2.0)) * geo
            vsc = 1.0 + (torch.empty(B, device=dev).uniform_(*tc.aug_vscale) - 1.0) * geo
            s_h = torch.empty(B, device=dev).uniform_(*tc.aug_hscale)        # horizontal stretch bounded by the canvas and by CTC
            w_f = wd.float().clamp(min=1.0)
            hi_ = Wc / w_f
            lo_ = (min_width.to(dev).float() / w_f) if min_width is not None else torch.zeros_like(w_f)
            s_h = torch.minimum(torch.maximum(s_h, lo_), hi_)
            s_h = torch.where((lo_ > hi_) | ~hs, torch.ones_like(s_h), s_h)
            wd_new = torch.clamp(torch.ceil(w_f * s_h), max=float(Wc)).long()
            s_eff = wd_new.float() / w_f
            hs = hs & (wd_new != wd)
            a_inv = 1.0 / s_eff
            ar = Wc / H
            theta = torch.zeros(B, 2, 3, device=dev)
            theta[:, 0, 0], theta[:, 0, 1], theta[:, 0, 2] = a_inv, (shear - rot) / ar, a_inv - 1.0   # left edge stays put
            theta[:, 1, 0], theta[:, 1, 1] = rot * ar, vsc
            grid = F.affine_grid(theta, list(x.shape), align_corners=False)
            if bool(ela.any()):
                flow = F.interpolate(torch.randn(B, 2, 3, 12, device=dev), size=(H, Wc), mode="bilinear", align_corners=True)
                dx = flow[:, 0] * tc.aug_elastic_px * (2.0 / Wc)
                dy = flow[:, 1] * 0.5 * tc.aug_elastic_px * (2.0 / H)
                grid = grid + torch.stack([dx, dy], -1) * ela.view(B, 1, 1, 1).float()
            if bool((geo | hs | ela).any()):
                x = F.grid_sample(x, grid, padding_mode="zeros", align_corners=False)
            wd = torch.where(hs, wd_new, wd)
            moved = geo | hs
            slant = st[:, 0] * std[0] + mean[0]
            new = torch.where(moved, transform_slant_deg(slant, shear, rot, vsc, a_inv), slant)
            inside = new.abs() < (SLANT_RANGE_DEG - 1.0)
            st[:, 0] = ((new.clamp(-SLANT_RANGE_DEG, SLANT_RANGE_DEG) - mean[0]) / std[0]).clamp(-3.0, 3.0)
            x_weak = x * colmask(wd)
            m = torch.rand(B, device=dev)
            dil = m < tc.aug_morph_prob / 2
            ero = (m >= tc.aug_morph_prob / 2) & (m < tc.aug_morph_prob)
            morph = dil | ero
            if bool(morph.any()):
                d_, e_ = F.max_pool2d(x, 3, 1, 1), -F.max_pool2d(-x, 3, 1, 1)
                x = torch.where(dil.view(B, 1, 1, 1), d_, torch.where(ero.view(B, 1, 1, 1), e_, x))
            altered = moved | ela | morph
            if tc.style_equivariant:
                st[:, 4] = (st[:, 4] + (body_ratio_t(x) - body0) / std[4]).clamp(-3.0, 3.0)
                sm[:, 0] = sm[:, 0] * (inside | ~moved).float()
                sm[:, 1:4] = sm[:, 1:4] * (~altered).float().unsqueeze(1)
            else:
                sm[:, 0] = sm[:, 0] * (~moved).float()
                sm[:, 1:] = sm[:, 1:] * (~altered).float().unsqueeze(1)
            cok = ~altered
        else:
            x_weak = x
        x = (x * rnd(0.75, 1.25) + rnd(-0.05, 0.05)).clamp(0, 1)
        x = x.clamp(1e-4, 1) ** rnd(0.75, 1.3)
        if tc.strong_aug:
            k = torch.rand(B, device=dev) < tc.aug_blur_prob
            if bool(k.any()):
                bl = F.avg_pool2d(F.pad(x, (1, 1, 1, 1), mode="replicate"), 3, 1)
                x = torch.where(k.view(B, 1, 1, 1), bl, x)
            x = x + torch.randn_like(x) * rnd(0.0, 0.05) * (x.amax((1, 2, 3), keepdim=True) > 0)
        col = colmask(wd)
        return (x * col).clamp(0, 1), (x_weak * col).clamp(0, 1), wd, st, sm, cok

In [ ]:
%%writefile inklingua/riskcontrol.py
"""
Empirical risk control shared by the legacy line-level pseudo-labels, FRAG-RC and the zero-shot adapter: ranking quality (AUROC),
isotonic (PAVA) risk curves, the first-violation threshold rule, the character-level rule with a line-clustered standard error,
ridge-logistic regression, and the transfer calibration of [XL-RC] (importance weights, weighted curves, anchor audit).

Everything here is a finite-sample HEURISTIC in the spirit of conformal risk control / Learn-then-Test (Angelopoulos et al.) and of
Selective Conformal Risk Control (arXiv 2512.12844), not a guarantee: the thresholds are data-dependent and the calibration sets are small.
"""
from __future__ import annotations

import math
from typing import Any, Dict, List, Optional, Tuple

import numpy as np


# ---------------------------------------------------------------- ranking quality
def _ranks(x: np.ndarray) -> np.ndarray:
    o = np.argsort(x, kind="mergesort")
    xs, r, i = x[o], np.empty(len(x)), 0
    while i < len(x):
        j = i
        while j + 1 < len(x) and xs[j + 1] == xs[i]:
            j += 1
        r[o[i:j + 1]] = 0.5 * (i + j) + 1.0
        i = j + 1
    return r


def auroc(score: np.ndarray, bad: np.ndarray) -> float:
    """P(score of a bad item > score of a good item); 0.5 = chance.  `score` is a risk (high = suspicious)."""
    score, bad = np.asarray(score, np.float64), np.asarray(bad, bool)
    n1, n0 = int(bad.sum()), int((~bad).sum())
    if n1 == 0 or n0 == 0:
        return float("nan")
    r = _ranks(score)
    return float((r[bad].sum() - n1 * (n1 + 1) / 2.0) / (n1 * n0))


# ---------------------------------------------------------------- isotonic regression and risk curves
def pava_increasing(y: np.ndarray, w: np.ndarray) -> np.ndarray:
    """Weighted isotonic (non-decreasing) regression, pool-adjacent-violators."""
    vals: List[float] = []
    wts: List[float] = []
    cnt: List[int] = []
    for yi, wi in zip(np.asarray(y, np.float64), np.asarray(w, np.float64)):
        vals.append(float(yi)), wts.append(float(max(wi, 1e-12))), cnt.append(1)
        while len(vals) > 1 and vals[-2] > vals[-1]:
            ww = wts[-2] + wts[-1]
            v = (vals[-2] * wts[-2] + vals[-1] * wts[-1]) / ww
            c = cnt[-2] + cnt[-1]
            vals.pop(), wts.pop(), cnt.pop()
            vals[-1], wts[-1], cnt[-1] = v, ww, c
    return np.repeat(vals, cnt) if vals else np.zeros(0)


def risk_curve(score: np.ndarray, ed: np.ndarray, ref_len: np.ndarray, smooth: bool = True) -> Dict[str, np.ndarray]:
    """Cumulative corpus CER of the calibration lines taken in order of increasing predicted risk; `smooth` = after an isotonic fit of
    the local error rate along the ranking (a better score must not mean MORE errors); `se` = s.e. of the ratio estimator."""
    o = np.argsort(score, kind="stable")
    e, l = np.asarray(ed, np.float64)[o], np.asarray(ref_len, np.float64)[o]
    k = np.arange(1, len(e) + 1, dtype=np.float64)
    ce, cl = np.cumsum(e), np.cumsum(l)
    R = ce / np.maximum(cl, 1.0)
    ss = np.cumsum(e ** 2) - 2.0 * R * np.cumsum(e * l) + R ** 2 * np.cumsum(l ** 2)
    se = np.sqrt(np.maximum(ss, 0.0) * k / np.maximum(k - 1.0, 1.0)) / np.maximum(cl, 1.0)
    if smooth and len(e) > 1:
        rate = pava_increasing(e / np.maximum(l, 1.0), l)
        Rs = np.cumsum(rate * l) / np.maximum(cl, 1.0)
    else:
        Rs = R
    return {"order": o, "raw": R, "smooth": Rs, "se": se}


def risk_control_threshold(score: np.ndarray, ed: np.ndarray, ref_len: np.ndarray, alpha: float, z: float, min_n: int,
                           smooth: bool = True) -> Tuple[float, int, float]:
    """Largest set of calibration lines, in order of increasing predicted risk, whose corpus CER plus z standard errors stays <= alpha.
    Returns (theta, k, cer_at_theta); k = 0 -> theta = -inf.  smooth=True: isotonic curve and the rule stops at the FIRST k (>= min_n) at
    which the bound is violated (no winner's curse)."""
    c = risk_curve(score, ed, ref_len, smooth)
    n = len(c["raw"])
    if n < min_n or n == 0:
        return float("-inf"), 0, float("nan")
    bound = c["smooth"] + z * c["se"]
    k = np.arange(1, n + 1)
    if smooth:
        bad = np.flatnonzero((bound > alpha) & (k >= min_n))
        kk = int(bad[0]) if bad.size else n
        if kk < min_n:
            return float("-inf"), 0, float("nan")
    else:
        ok = np.flatnonzero((c["raw"] + z * c["se"] <= alpha) & (k >= min_n))
        if ok.size == 0:
            return float("-inf"), 0, float("nan")
        kk = int(ok[-1]) + 1
    return float(np.asarray(score)[c["order"]][kk - 1]), kk, float(c["raw"][kk - 1])


def theta_at_count(score: np.ndarray, k: int) -> float:
    """Risk threshold that admits the k lowest-risk calibration lines (ties admit more)."""
    if k <= 0:
        return float("-inf")
    return float(np.sort(np.asarray(score))[min(k, len(score)) - 1])


# ---------------------------------------------------------------- character-level rule (FRAG-RC)
def char_risk_threshold(risk: np.ndarray, wrong: np.ndarray, cluster: np.ndarray, alpha: float, z: float, min_n: int,
                        n_grid: int = 200) -> Tuple[float, int, float]:
    """Empirical risk control of ONE Mondrian cell at the CHARACTER level.  Characters are taken in order of increasing predicted risk;
    the local error rate is made monotone (PAVA); the bound is  smoothed rate + z x s.e.  with a CLUSTER-robust s.e. (clusters = lines:
    the errors of one line are correlated); the threshold is the last grid point BEFORE THE FIRST violation of  bound <= alpha.
    Returns (theta, k, raw error rate of the k characters); k = 0 -> theta = -inf."""
    n = len(risk)
    if n < max(min_n, 2):
        return float("-inf"), 0, float("nan")
    o = np.argsort(risk, kind="stable")
    r, y = np.asarray(risk)[o], np.asarray(wrong, np.float64)[o]
    _, gi = np.unique(np.asarray(cluster)[o], return_inverse=True)
    rate = pava_increasing(y, np.ones(n))
    cum_s = np.cumsum(rate) / np.arange(1, n + 1)
    ks = np.unique(np.linspace(min_n, n, n_grid).astype(int))
    ecl, ncl = np.zeros(gi.max() + 1), np.zeros(gi.max() + 1)
    prev, ok_k = 0, 0
    for k in ks:
        np.add.at(ecl, gi[prev:k], y[prev:k])
        np.add.at(ncl, gi[prev:k], 1.0)
        prev = k
        R = float(ecl.sum() / k)
        act = ncl > 0
        G = int(act.sum())
        var = (float(((ecl[act] - R * ncl[act]) ** 2).sum()) * G / (G - 1) / k ** 2) if G > 1 else R * (1 - R) / k
        if cum_s[k - 1] + z * math.sqrt(max(var, 0.0)) > alpha:
            break
        ok_k = int(k)
    if ok_k == 0:
        return float("-inf"), 0, float("nan")
    return float(r[ok_k - 1]), ok_k, float(y[:ok_k].mean())


# ---------------------------------------------------------------- logistic regression (IRLS)
def sigmoid(z: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30.0, 30.0)))


def fit_logistic(Z: np.ndarray, y: np.ndarray, l2: float = 1.0, iters: int = 30) -> np.ndarray:
    """Ridge-penalised logistic regression by IRLS (intercept unpenalised); Z is standardised.  Returns [intercept, weights]."""
    n, d = Z.shape
    A = np.c_[np.ones(n), Z]
    b, R = np.zeros(d + 1), l2 * np.eye(d + 1)
    R[0, 0] = 0.0
    for _ in range(iters):
        p = sigmoid(A @ b)
        W = np.maximum(p * (1 - p), 1e-6)
        step = np.linalg.solve((A * W[:, None]).T @ A + R, A.T @ (p - y) + R @ b)
        b -= step
        if np.abs(step).max() < 1e-8:
            break
    return b


# ---------------------------------------------------------------- [XL-RC] transfer calibration
def domain_shift(Xs: np.ndarray, Xt: np.ndarray, l2: float = 3.0, clip: float = 10.0, folds: int = 3, seed: int = 0
                 ) -> Tuple[np.ndarray, float, float]:
    """Importance weights w = p_target(x) / p_source(x) of the SOURCE calibration lines (mean 1, clipped) from a cross-fitted domain
    classifier, the AUROC of that classifier (0.5 = no detectable shift) and the effective sample size of the weights.  Valid under
    COVARIATE shift only -- the anchor audit checks that assumption."""
    ns, nt = len(Xs), len(Xt)
    X = np.r_[Xs, Xt]
    Z = (X - X.mean(0)) / (X.std(0) + 1e-6)
    y = np.r_[np.zeros(ns), np.ones(nt)]
    fold = np.random.RandomState(seed).permutation(ns + nt) % folds
    p = np.zeros(ns + nt)
    for f in range(folds):
        b = fit_logistic(Z[fold != f], y[fold != f], l2)
        p[fold == f] = sigmoid(np.c_[np.ones((fold == f).sum()), Z[fold == f]] @ b)
    p = np.clip(p, 1e-3, 1 - 1e-3)
    w = (p[:ns] / (1 - p[:ns])) / (nt / ns)
    w = np.minimum(w, clip * w.mean())
    w = w / w.mean()
    return w, auroc(p, y > 0), float(w.sum() ** 2 / np.sum(w ** 2))


def weighted_risk_curve(score: np.ndarray, ed: np.ndarray, ref_len: np.ndarray, w: np.ndarray) -> Dict[str, np.ndarray]:
    """risk_curve with importance weights: cumulative weighted CER, its isotonic version, the sandwich s.e. and the effective sample size."""
    o = np.argsort(score, kind="stable")
    e, l, ww = np.asarray(ed, np.float64)[o], np.asarray(ref_len, np.float64)[o], np.asarray(w, np.float64)[o]
    k = np.arange(1, len(e) + 1.0)
    den = np.maximum(np.cumsum(ww * l), 1e-9)
    R = np.cumsum(ww * e) / den
    rate = pava_increasing(e / np.maximum(l, 1.0), ww * l)
    Rs = np.cumsum(rate * ww * l) / den
    ss = np.cumsum(ww ** 2 * e ** 2) - 2 * R * np.cumsum(ww ** 2 * e * l) + R ** 2 * np.cumsum(ww ** 2 * l ** 2)
    se = np.sqrt(np.maximum(ss, 0.0) * k / np.maximum(k - 1.0, 1.0)) / den
    ess = np.cumsum(ww) ** 2 / np.maximum(np.cumsum(ww ** 2), 1e-12)
    return {"order": o, "raw": R, "smooth": Rs, "se": se, "ess": ess, "rate": rate}


def _transfer_theta(score_s, ed_s, rl_s, w, alpha, z, min_n, min_ess, c):
    cv = weighted_risk_curve(score_s, ed_s, rl_s, w)
    n = len(cv["raw"])
    if n < min_n:
        return float("-inf"), 0, cv
    k = np.arange(1, n + 1)
    bound = c * (cv["smooth"] + z * cv["se"])
    bad = np.flatnonzero(((bound > alpha) | (cv["ess"] < min_ess)) & (k >= min_n))
    kk = int(bad[0]) if bad.size else n
    if kk < min_n:
        return float("-inf"), 0, cv
    return float(np.asarray(score_s)[cv["order"]][kk - 1]), kk, cv


def anchor_audit(score_s: np.ndarray, cv: Dict[str, np.ndarray], score_a: np.ndarray, ed_a: np.ndarray, rl_a: np.ndarray,
                 prior_edits: float = 20.0, lo: float = 0.5, hi: float = 4.0) -> Tuple[float, float, float]:
    """Observed / predicted number of edits on the LABELED target lines (anchors), the prediction being the source error rate at the
    anchors' risk scores.  Poisson-Gamma shrinkage with `prior_edits` pseudo-edits.  > 1: the target is harder than the transferred
    calibration says (concept shift)."""
    ss = np.asarray(score_s)[cv["order"]]
    j = np.clip(np.searchsorted(ss, np.asarray(score_a), side="right") - 1, 0, len(ss) - 1)
    pred = float((cv["rate"][j] * np.asarray(rl_a, np.float64)).sum())
    obs = float(np.sum(ed_a))
    return float(np.clip((obs + prior_edits) / (pred + prior_edits), lo, hi)), pred, obs


def weighted_ks(a: np.ndarray, b: np.ndarray, wa: Optional[np.ndarray] = None) -> float:
    """Two-sample Kolmogorov-Smirnov distance (weights on the first sample)."""
    a, b = np.asarray(a, np.float64), np.asarray(b, np.float64)
    wa = np.ones(len(a)) if wa is None else np.asarray(wa, np.float64)
    pts = np.sort(np.r_[a, b])
    oa = np.argsort(a)
    ca = np.cumsum(wa[oa]) / wa.sum()
    ia = np.searchsorted(a[oa], pts, side="right")
    fa = np.where(ia > 0, ca[np.maximum(ia - 1, 0)], 0.0)
    fb = np.searchsorted(np.sort(b), pts, side="right") / len(b)
    return float(np.abs(fa - fb).max())


def transfer_risk_control(X_s: np.ndarray, score_s: np.ndarray, ed_s: np.ndarray, rl_s: np.ndarray, X_t: np.ndarray, score_t: np.ndarray,
                          alpha: float, z: float, min_n: int = 15, min_ess: float = 10.0, *, score_a: Optional[np.ndarray] = None,
                          ed_a: Optional[np.ndarray] = None, rl_a: Optional[np.ndarray] = None, zero_anchor_inflation: float = 1.5,
                          prior_edits: float = 20.0, clip: float = 10.0, l2: float = 3.0, seed: int = 0) -> Dict[str, Any]:
    """[XL-RC] Which TARGET-language lines may be pseudo-labelled when only SOURCE-language lines carry labels?  (1) covariate-shift weights
    of the source lines toward the target; (2) the weighted isotonic source curve, first violation of c*(CER + z*se) <= alpha or of the
    effective sample size; (3) the factor c from the labeled anchors (audit of the covariate-shift assumption; detects concept shift) or,
    without anchors, c = zero_anchor_inflation (a heuristic)."""
    X_s, X_t = np.asarray(X_s, np.float64), np.asarray(X_t, np.float64)
    if len(X_t) >= 10 and len(X_s) >= 20:
        w, dom_auc, ess_all = domain_shift(X_s, X_t, l2, clip, seed=seed)
    else:
        w, dom_auc, ess_all = np.ones(len(X_s)), float("nan"), float(len(X_s))
    th1, k1, cv = _transfer_theta(score_s, ed_s, rl_s, w, alpha, z, min_n, min_ess, 1.0)
    na = 0 if score_a is None else len(score_a)
    pred = obs = float("nan")
    if na > 0:
        c, pred, obs = anchor_audit(score_s, cv, score_a, ed_a, rl_a, prior_edits, lo=0.7 if na >= 60 else 1.0)
        mode = "transfer+anchors"
    else:
        c, mode = float(zero_anchor_inflation), "transfer-uncalibrated"
    th, k, cv = _transfer_theta(score_s, ed_s, rl_s, w, alpha, z, min_n, min_ess, c)
    pred_cer = float(c * cv["smooth"][k - 1]) if k > 0 else float("nan")
    return {"theta": th, "k": k, "factor": float(c), "mode": mode, "n_anchors": na, "anchor_observed_edits": obs,
            "anchor_predicted_edits": pred, "domain_auroc": dom_auc, "ess_all": ess_all, "ess_at_k": float(cv["ess"][k - 1]) if k > 0 else 0.0,
            "ks_before": weighted_ks(score_s, score_t), "ks_after": weighted_ks(score_s, score_t, w),
            "predicted_cer_selected": pred_cer, "coverage_target": float(np.mean(np.asarray(score_t) <= th)) if k > 0 else 0.0}

In [ ]:
%%writefile inklingua/llm.py
"""The pretrained LLM: uncertainty-aware translation, LM scoring for the constrained N-best rescoring, calibrated [?] marks."""
from __future__ import annotations

import dataclasses
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Sequence, Tuple

import difflib
import numpy as np
import torch

from .config import LLMConfig
from .riskcontrol import auroc
from .text import chrf, detokenize
from .utils import LOG

LANG_NAMES = {"en": "English", "fr": "French", "es": "Spanish", "de": "German", "it": "Italian", "pt": "Portuguese", "ru": "Russian",
              "zh": "Chinese", "ar": "Arabic", "hi": "Hindi", "ja": "Japanese", "nl": "Dutch", "yo": "Yoruba", "ha": "Hausa", "ln": "Lingala",
              "sw": "Swahili", "wo": "Wolof", "bm": "Bambara", "ff": "Fula", "am": "Amharic", "ig": "Igbo", "zu": "Zulu", "tr": "Turkish",
              "pl": "Polish", "ro": "Romanian", "cs": "Czech", "sv": "Swedish", "vi": "Vietnamese", "id": "Indonesian", "fa": "Persian",
              "ur": "Urdu", "ko": "Korean", "el": "Greek", "uk": "Ukrainian", "ca": "Catalan", "la": "Latin", "nb": "Norwegian", "no": "Norwegian",
              "da": "Danish", "et": "Estonian", "he": "Hebrew", "th": "Thai", "bn": "Bengali"}


def lang_name(code: str) -> str:
    return LANG_NAMES.get(code, code)


FEWSHOT = {
    "fr": ("Le livre est sur la table près de la fenêtre.", "Nous nous retrouverons à la gare demain matin."),
    "es": ("El libro está sobre la mesa junto a la ventana.", "Nos encontraremos en la estación mañana por la mañana."),
    "de": ("Das Buch liegt auf dem Tisch neben dem Fenster.", "Wir treffen uns morgen früh am Bahnhof."),
    "it": ("Il libro è sul tavolo vicino alla finestra.", "Ci incontreremo alla stazione domani mattina."),
    "pt": ("O livro está sobre a mesa perto da janela.", "Encontrar-nos-emos na estação amanhã de manhã."),
    "hi": ("किताब खिड़की के पास मेज़ पर है।", "हम कल सुबह स्टेशन पर मिलेंगे।"),
}


@dataclass
class Hypothesis:
    """One recognition hypothesis (LOGICAL order, detokenised by the caller when the corpus needs it) = the 'pseudo-label
    information' handed to the LLM."""
    text: str
    char_conf: List[float] = field(default_factory=list)
    logprob: float = 0.0
    post: float = 1.0


def _word_conf(vals: Sequence[float], agg: str) -> float:
    return float(np.min(vals)) if agg == "min" else float(np.mean(vals))


def mark_uncertain(text: str, char_conf: Sequence[float], thresh: float, agg: str = "mean") -> str:
    """Append [?] to words whose character confidence (mean or minimum over the word) is below `thresh`."""
    if len(char_conf) != len(text):
        return text
    words, cur, curc = [], [], []

    def flush():
        if cur:
            words.append("".join(cur) + ("[?]" if _word_conf(curc, agg) < thresh else ""))
            cur.clear()
            curc.clear()

    for ch, c in zip(text, char_conf):
        if ch == " ":
            flush()
        else:
            cur.append(ch)
            curc.append(c)
    flush()
    return " ".join(words)


def word_marking_data(pred: str, gt: str, char_conf: Sequence[float], agg: str) -> Tuple[List[float], List[bool]]:
    """(confidence, is_wrong) of every predicted word of a line."""
    if len(char_conf) != len(pred):
        return [], []
    scores, cur = [], []
    for ch, c in zip(pred, char_conf):
        if ch == " ":
            if cur:
                scores.append(_word_conf(cur, agg))
                cur = []
        else:
            cur.append(c)
    if cur:
        scores.append(_word_conf(cur, agg))
    pw, gw = pred.split(), gt.split()
    right = [False] * len(pw)
    for op, i1, i2, j1, j2 in difflib.SequenceMatcher(None, gw, pw, autojunk=False).get_opcodes():
        if op == "equal":
            for j in range(j1, j2):
                right[j] = True
    return (scores, [not x for x in right]) if len(scores) == len(pw) else ([], [])


def calibrate_word_mark(preds: Sequence[str], gts: Sequence[str], confs: Sequence[Sequence[float]], precision: float = 0.5,
                        min_share: float = 0.02) -> Dict[str, Any]:
    """[CAL-MARK] threshold / aggregation of the [?] marks: on held-out words the score (mean or min character confidence, whichever
    ranks the wrong words better) is thresholded at the LARGEST value for which the flagged words are wrong with probability >= `precision`."""
    best: Optional[Dict[str, Any]] = None
    for agg in ("mean", "min"):
        sc, wr = [], []
        for p, g, c in zip(preds, gts, confs):
            a, b = word_marking_data(p, g, c, agg)
            sc += a
            wr += b
        s_, w_ = np.asarray(sc, np.float64), np.asarray(wr, bool)
        if len(s_) < 50 or w_.sum() < 20 or (~w_).sum() < 20:
            continue
        au = auroc(1.0 - s_, w_)
        if best is None or au > best["auroc"]:
            best = {"agg": agg, "auroc": au, "s": s_, "w": w_}
    if best is None:
        return {"threshold": None, "reason": "too few validation words"}
    s_, w_ = best["s"], best["w"]
    thr, stats = None, {}
    for t in np.unique(np.quantile(s_, np.linspace(0.01, 0.9, 90))):
        m = s_ < t
        if m.mean() < min_share:
            continue
        prec = float(w_[m].mean())
        if prec >= precision:
            thr, stats = float(t), {"precision": prec, "recall": float(w_[m].sum() / w_.sum()), "share_marked": float(m.mean())}
    out = {"threshold": thr, "agg": best["agg"], "auroc": float(best["auroc"]), "word_error_rate": float(w_.mean()), "n_words": int(len(s_)),
           "target_precision": precision}
    out.update(stats)
    return out


def mbr_select(cands: List[str], weights: Sequence[float]) -> int:
    """Minimum-Bayes-risk choice with chrF utility, weighted by the recogniser posterior."""
    if len(cands) == 1:
        return 0
    w = np.asarray(weights, np.float64)
    w = w / max(w.sum(), 1e-9)
    util = [sum(w[j] * chrf(ci, cj) for j, cj in enumerate(cands)) for ci in cands]
    return int(np.argmax(util))


def fuse_scores(ctc_lp: np.ndarray, lm_lp: np.ndarray, ntok: np.ndarray, beta: float, rho: float = 0.0) -> np.ndarray:
    """Shallow fusion over the SAME N-best list:  score = log P_ctc(h) + beta * (log P_lm(h) + rho * #tokens(h)).  Only hypotheses of
    the recogniser can win, so the LLM cannot invent text."""
    return np.asarray(ctc_lp, np.float64) + beta * (np.asarray(lm_lp, np.float64) + rho * np.asarray(ntok, np.float64))


class LLMEngine:
    """Lazy wrapper around a pretrained model.  backend 'causal': instruction-tuned decoder LLM (prompted; also gives LM scores);
    'seq2seq': Marian / NLLB-style NMT model (N-best translation + MBR); 'dummy': no model, for offline plumbing tests."""

    def __init__(self, cfg: LLMConfig, device: torch.device):
        self.cfg, self.device = cfg, device
        self.tok = self.model = None
        self._failed = False

    def available(self) -> bool:
        if self.cfg.backend == "dummy":
            return True
        if self.model is None and not self._failed:
            self._load()
        return self.model is not None

    def can_score(self) -> bool:
        return self.cfg.backend == "causal" and self.available()

    def _load(self) -> None:
        try:
            from transformers import AutoModelForCausalLM, AutoModelForSeq2SeqLM, AutoTokenizer
            c = self.cfg
            if c.backend == "causal":
                name, cls = c.model_name, AutoModelForCausalLM
            else:
                name = c.seq2seq_models.get(f"{c.source_language}-{c.target_language}") or (
                    c.seq2seq_models[c.target_language] if c.source_language == "en" else None)
                if name is None:
                    raise KeyError(f"no seq2seq model configured for {c.source_language}->{c.target_language}: add the key "
                                   f"'{c.source_language}-{c.target_language}' to cfg.llm.seq2seq_models, or use backend='causal'")
                cls = AutoModelForSeq2SeqLM
            dtype = torch.float16 if self.device.type == "cuda" else torch.float32
            LOG.info("Loading pretrained %s model '%s' (auto-download on first use) ...", c.backend, name)
            self.tok = AutoTokenizer.from_pretrained(name)
            try:
                model = cls.from_pretrained(name, dtype=dtype)
            except TypeError:
                model = cls.from_pretrained(name, torch_dtype=dtype)
            self.model = model.to(self.device).eval()
            if self.tok.pad_token is None:
                self.tok.pad_token = self.tok.eos_token
        except Exception as e:
            self._failed = True
            LOG.warning("LLM could not be loaded (%s). Translation / LLM rescoring will be skipped.", e)

    @torch.no_grad()
    def score_texts(self, texts: List[str]) -> Tuple[np.ndarray, np.ndarray]:
        """Sum of token log-probs and #tokens of every text under the causal LM (cached, batched by length)."""
        assert self.can_score()
        cache = self.__dict__.setdefault("_lm_cache", {})
        todo = sorted({t for t in texts if t not in cache}, key=len)
        prefix = f"Transcription of a line of {lang_name(self.cfg.source_language)} handwriting:\n"
        pre = self.tok(prefix, add_special_tokens=False).input_ids
        pad = self.tok.pad_token_id if self.tok.pad_token_id is not None else 0
        for i in range(0, len(todo), self.cfg.score_batch_size):
            chunk = todo[i:i + self.cfg.score_batch_size]
            seqs = [pre + self.tok(t, add_special_tokens=False).input_ids for t in chunk]
            L = max(len(s) for s in seqs)
            ids = torch.full((len(seqs), L), pad, dtype=torch.long)
            att = torch.zeros((len(seqs), L), dtype=torch.long)
            for k, s in enumerate(seqs):
                ids[k, :len(s)], att[k, :len(s)] = torch.tensor(s), 1
            ids, att = ids.to(self.device), att.to(self.device)
            logits = self.model(input_ids=ids, attention_mask=att).logits.float()
            tok_lp = logits[:, :-1].gather(-1, ids[:, 1:].unsqueeze(-1)).squeeze(-1) - logits[:, :-1].logsumexp(-1)
            for k, s in enumerate(seqs):
                a, b = len(pre) - 1, len(s) - 1
                cache[chunk[k]] = (float(tok_lp[k, a:b].sum()), b - a)
        return (np.array([cache[t][0] for t in texts], np.float64), np.array([cache[t][1] for t in texts], np.float64))

    def for_target(self, lang: str) -> "LLMEngine":
        if lang == self.cfg.target_language:
            return self
        sib = self.__dict__.setdefault("_siblings", {})
        if lang not in sib:
            sib[lang] = LLMEngine(dataclasses.replace(self.cfg, target_language=lang), self.device)
        return sib[lang]

    def _messages(self, hyps: List[Hypothesis], uncertain: bool, c: Optional[LLMConfig] = None, detok: bool = False
                  ) -> Tuple[List[Dict[str, str]], str]:
        c = c or self.cfg
        lang, src = lang_name(c.target_language), lang_name(c.source_language)
        top = hyps[0]
        marked = mark_uncertain(top.text, top.char_conf, c.word_conf_thresh, c.word_conf_agg) if uncertain else top.text
        fix = detokenize if detok else (lambda s: s)
        marked, clean_top = fix(marked), fix(top.text)
        alts: List[str] = []
        if uncertain:
            for h in hyps[1:]:
                a = fix(h.text)
                if a != clean_top and a not in alts:
                    alts.append(a)
        user = f"Transcription: {marked}\n"
        if alts:
            user += "Other possible readings of the same line:\n" + "\n".join(f"- {a}" for a in alts) + "\n"
        user += f"Translate into {lang}. Output only the translation."
        sys_msg = ("You are an expert translator. The input was produced by automatic handwriting recognition and may contain recognition errors."
                   + (" Words followed by [?] are uncertain; use the alternative readings and the context to decide what was most likely written, "
                      "silently correct obvious errors, and never output [?]." if uncertain else "")
                   + f" Translate the intended {src} sentence into {lang}.")
        msgs = [{"role": "system", "content": sys_msg}]
        if c.target_language in FEWSHOT and c.source_language == "en":
            t1, t2 = FEWSHOT[c.target_language]
            if uncertain:
                ex1 = ("Transcription: The book is on the tabel[?] near the window.\nOther possible readings of the same line:\n"
                       f"- The book is on the table near the window.\nTranslate into {lang}. Output only the translation.")
            else:
                ex1 = f"Transcription: The book is on the table near the window.\nTranslate into {lang}. Output only the translation."
            ex2 = f"Transcription: We will meet at the station tomorrow morning.\nTranslate into {lang}. Output only the translation."
            msgs += [{"role": "user", "content": ex1}, {"role": "assistant", "content": t1},
                     {"role": "user", "content": ex2}, {"role": "assistant", "content": t2}]
        msgs.append({"role": "user", "content": user})
        return msgs, marked

    @staticmethod
    def _clean(out: str) -> str:
        out = out.strip().replace("[?]", "")
        lines = [l.strip() for l in out.split("\n") if l.strip()]
        out = lines[0] if lines else ""
        return out.strip().strip('"“”').strip()

    @torch.no_grad()
    def translate(self, hyps: List[Hypothesis], uncertain: Optional[bool] = None, target_language: Optional[str] = None,
                  detok: bool = False) -> Dict[str, Any]:
        """hyps[0] is the best hypothesis (logical order).  uncertain=False -> plain top-1 translation.  detok=True re-attaches IAM
        tokenisation before prompting (only for corpora whose spec says so)."""
        c = self.cfg
        if target_language is not None and target_language != c.target_language:
            if c.backend == "seq2seq":
                return self.for_target(target_language).translate(hyps, uncertain, detok=detok)
            c = dataclasses.replace(c, target_language=target_language)
        uncertain = c.use_uncertainty_prompt if uncertain is None else uncertain
        if not self.available():
            return {"translation": "", "marked_text": "", "candidates": []}
        fix = detokenize if detok else (lambda s: s)
        if c.backend == "dummy":
            return {"translation": f"[{c.target_language}] {fix(hyps[0].text)}", "marked_text": fix(hyps[0].text), "candidates": []}
        if c.backend == "causal":
            msgs, marked = self._messages(hyps, uncertain, c, detok)
            prompt = self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            inp = self.tok(prompt, return_tensors="pt").to(self.device)
            out = self.model.generate(**inp, max_new_tokens=c.max_new_tokens, do_sample=False, pad_token_id=self.tok.pad_token_id)
            txt = self._clean(self.tok.decode(out[0, inp["input_ids"].shape[1]:], skip_special_tokens=True))
            return {"translation": txt, "marked_text": marked, "candidates": [txt]}
        use = hyps[:c.nbest] if uncertain else hyps[:1]
        srcs = [fix(h.text) for h in use]
        inp = self.tok(srcs, return_tensors="pt", padding=True, truncation=True, max_length=128).to(self.device)
        out = self.model.generate(**inp, num_beams=4, max_new_tokens=c.max_new_tokens)
        cands = [t.strip() for t in self.tok.batch_decode(out, skip_special_tokens=True)]
        best = mbr_select(cands, [h.post for h in use])
        return {"translation": cands[best], "marked_text": srcs[0], "candidates": cands}

In [ ]:
%%writefile inklingua/pseudo.py
"""
Legacy (v3-v9) LINE-level pseudo-labelling, kept as the `legacy_pl` baseline of the suite: perturbation views, the label-free
reliability features of a reading, the cross-fitted reliability model with risk-controlled thresholds ([RC-PL v2], [SHADOW],
[GUARD-REF]) and the pseudo-label bank with its gates ([PCG]).  FRAG-RC (inklingua/frag.py) replaces it by default.
"""
from __future__ import annotations

import math
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn.functional as F

from .config import PseudoConfig
from .decoding import seq_confidence
from .riskcontrol import auroc, risk_control_threshold, risk_curve, theta_at_count
from .text import cer, ctc_feasible, edit_distance


# ---------------------------------------------------------------- perturbation views
def eval_noise(x: torch.Tensor, seed: int = 0) -> torch.Tensor:
    """Deterministic noise + blur perturbation (noise-consistency metric, [PCG] view 'gauss noise')."""
    g = torch.Generator().manual_seed(seed)
    n = torch.randn(x.shape, generator=g).to(x.device) * 0.06
    x = (x + n).clamp(0, 1)
    return F.avg_pool2d(F.pad(x, (1, 1, 1, 1), mode="replicate"), 3, 1)


def shear_batch(x: torch.Tensor, s: float) -> torch.Tensor:
    """Horizontal shear of s pixels per pixel of height (s > 0: the writing leans further to the right)."""
    B, _, H, W = x.shape
    theta = torch.zeros(B, 2, 3, device=x.device)
    theta[:, 0, 0], theta[:, 1, 1], theta[:, 0, 1] = 1.0, 1.0, s * H / W
    return F.grid_sample(x, F.affine_grid(theta, list(x.shape), align_corners=False), align_corners=False)


PERTURBATIONS = {
    "clean": lambda x: x,
    "gauss noise": lambda x: eval_noise(x, 1),
    "blur": lambda x: F.avg_pool2d(F.pad(x, (2, 2, 2, 2), mode="replicate"), 5, 1),
    "low contrast": lambda x: x * 0.45,
    "thicker ink": lambda x: F.max_pool2d(x, 3, 1, 1),
    "thinner ink": lambda x: -F.max_pool2d(-x, 3, 1, 1),
    "shear +0.08": lambda x: shear_batch(x, 0.08),
    "shear -0.08": lambda x: shear_batch(x, -0.08),
    "shear +0.15": lambda x: shear_batch(x, 0.15),
    "shear -0.15": lambda x: shear_batch(x, -0.15),
}


def apply_perturbation(fn, x: torch.Tensor, widths: torch.Tensor) -> torch.Tensor:
    """Apply a perturbation and keep the padded columns of the batch at exactly zero."""
    col = (torch.arange(x.size(-1), device=x.device)[None] < widths.to(x.device)[:, None]).to(x.dtype)
    return fn(x) * col[:, None, None, :]


# ---------------------------------------------------------------- reliability of a reading
RELIABILITY_FEATURES = ["log mean char uncertainty", "log max char uncertainty", "share of chars below 0.9", "log mean runner-up prob.",
                        "log near-miss blank frames", "log length", "log frames per char", "log sequence uncertainty",
                        "sqrt mean view disagreement", "sqrt max view disagreement"]


def reliability_views(pc: PseudoConfig) -> Tuple[str, ...]:
    """Views the teacher reads for every line: the [PCG] gate views first, then the extra reliability views."""
    if not pc.consistency and not pc.rel_views:
        return ()
    v = list(pc.cons_views if pc.consistency else ())
    for x in pc.rel_views:
        if x not in v:
            v.append(x)
    return tuple(v)


def reliability_features(lp: np.ndarray, ids: Sequence[int], cf: Sequence[float], frames: Sequence[int], cons: Sequence[float]) -> np.ndarray:
    """Label-free description of one reading from the recogniser's own frame posteriors lp [T,V] and the CERs between the clean reading
    and perturbed readings (`cons`): where a peaky CTC model still leaks doubt."""
    T, K = lp.shape[0], len(ids)
    c = np.asarray(cf, np.float64) if K else np.ones(1)
    u = 1.0 - c
    if K:
        P = np.exp(lp[list(frames)])
        p2 = -np.sort(-P, axis=1)[:, 1]
    else:
        p2 = np.zeros(1)
    p = np.exp(lp)
    near = int(((p.argmax(1) == 0) & (p[:, 0] < 0.95)).sum())
    cons = np.asarray(list(cons), np.float64) if len(cons) else np.zeros(1)
    return np.array([np.log(u.mean() + 1e-4), np.log(u.max() + 1e-4), float((c < 0.9).mean()), np.log(p2.mean() + 1e-5), np.log1p(near),
                     np.log1p(K), np.log((T + 1.0) / (K + 1.0)), np.log(1.0 - seq_confidence(cf) + 1e-4) if K else 0.0,
                     float(np.sqrt(cons.mean())), float(np.sqrt(cons.max()))], np.float64)


class ReliabilityModel:
    """Ridge regression  reading features -> sqrt(line CER).  The output is a RISK (small = reliable); only its RANKING matters."""

    def __init__(self, lam: float = 0.1):
        self.lam = lam
        self.mu = self.sd = self.w = None
        self.b = 0.0

    def fit(self, X: np.ndarray, cer_line: np.ndarray) -> "ReliabilityModel":
        y = np.sqrt(np.clip(np.asarray(cer_line, np.float64), 0.0, 1.0))
        self.mu, self.sd = X.mean(0), X.std(0) + 1e-6
        A = (X - self.mu) / self.sd
        self.b = float(y.mean())
        self.w = np.linalg.solve(A.T @ A + self.lam * len(A) * np.eye(A.shape[1]), A.T @ (y - self.b))
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        return ((X - self.mu) / self.sd) @ self.w + self.b


class SelectionRule:
    """Reliability model(s) + thresholds fitted on the calibration lines (K-fold cross-fitting, coverage guard)."""

    def __init__(self):
        self.models: List[ReliabilityModel] = []
        self.theta: Optional[float] = None
        self.theta_llm: Optional[float] = None
        self.oof: np.ndarray = np.zeros(0)
        self.info: Dict[str, float] = {}

    def predict(self, X: np.ndarray) -> np.ndarray:
        return np.mean([m.predict(X) for m in self.models], axis=0)

    @classmethod
    def fit(cls, X: np.ndarray, ed: np.ndarray, ref_len: np.ndarray, pc: PseudoConfig, seed: int = 0,
            prev_cov: Optional[float] = None, ref_cov: Optional[float] = None) -> "SelectionRule":
        r = cls()
        n = len(X)
        K = int(max(2, min(pc.risk_folds, n // 8)))
        cer_l = ed / np.maximum(ref_len, 1.0)
        fold = np.random.RandomState(seed).permutation(n) % K
        oof = np.zeros(n)
        for f in range(K):
            m = ReliabilityModel().fit(X[fold != f], cer_l[fold != f])
            oof[fold == f] = m.predict(X[fold == f])
            r.models.append(m)
        theta, k, c_at = risk_control_threshold(oof, ed, ref_len, pc.target_cer, pc.risk_z, pc.min_selected_calib, pc.risk_smoothing)
        theta_l, k_l, _ = risk_control_threshold(oof, ed, ref_len, pc.llm_target_cer, pc.risk_z, pc.min_selected_calib, pc.risk_smoothing)
        k_raw, guard = k, 0.0
        if pc.cov_guard and prev_cov is not None and np.isfinite(prev_cov):
            hi = int(math.floor((prev_cov + pc.max_cov_growth) * n))
            base = prev_cov if (ref_cov is None or not np.isfinite(ref_cov)) else max(prev_cov, ref_cov)
            lo = int(math.ceil(pc.min_cov_keep * base * n))
            k = int(np.clip(k, min(lo, hi), hi))
            if k > k_raw:                       # the shrink floor raised k: never above guard_cer_factor x target in calibration CER
                raw_curve = risk_curve(oof, ed, ref_len, False)["raw"]
                okk = np.flatnonzero(raw_curve[:k] <= pc.guard_cer_factor * pc.target_cer)
                k = max(k_raw, int(okk[-1]) + 1 if okk.size else 0)
            if k != k_raw:
                guard = 1.0
                theta = theta_at_count(oof, k)
                c_at = float(risk_curve(oof, ed, ref_len, False)["raw"][max(k, 1) - 1]) if k > 0 else float("nan")
        if np.isfinite(theta) and np.isfinite(theta_l):
            theta_l = max(theta_l, theta)
        elif np.isfinite(theta):
            theta_l = theta
        r.theta, r.theta_llm, r.oof = theta, theta_l, oof
        r.info = {"calib_n": float(n), "calib_cer": float(ed.sum() / max(ref_len.sum(), 1.0)), "calib_selected": float(k),
                  "calib_selected_cer": c_at, "calib_cov_raw": k_raw / max(n, 1), "calib_guard_binding": guard,
                  "calib_auroc_error_detection": auroc(oof, ed > 0), "calib_auroc_seq_conf": auroc(X[:, 7], ed > 0),
                  "calib_coverage": k / max(n, 1)}
        return r


class PseudoLabelBank:
    """Book-keeping for the unlabeled lines.  `hidden_reference` (the held-back ground truth) is read ONLY by `diagnostics()`."""

    def __init__(self, n: int, unlabeled: np.ndarray, hidden_reference: List[str], frames: np.ndarray, calib: Optional[np.ndarray] = None):
        self.n, self.unl, self.hidden = n, np.asarray(unlabeled, bool), list(hidden_reference)
        self.cal = np.zeros(n, bool) if calib is None else (np.asarray(calib, bool) & self.unl)
        self.frames = np.asarray(frames, np.int64)
        self.text = [""] * n
        self.conf = np.zeros(n, np.float32)
        self.stab = np.zeros(n, np.int32)
        self.has = np.zeros(n, bool)
        self.cons = np.full(n, np.nan, np.float32)
        self.risk = np.full(n, np.nan, np.float32)
        self.loss_ema = np.full(n, np.nan, np.float32)
        self.llm_text: List[Optional[str]] = [None] * n
        self.llm_ok = np.zeros(n, bool)
        self.sel = np.zeros(n, bool)
        self.weight = np.zeros(n, np.float32)
        self.train_text = [""] * n
        self.source = np.zeros(n, np.int8)            # 0 none, 1 recogniser, 2 LLM-rescued
        self.rounds = 0
        self._pcg_rejected = np.zeros(n, bool)
        self._pcg_accepted = np.zeros(n, bool)

    def begin_round(self) -> None:
        self.llm_text, self.llm_ok = [None] * self.n, np.zeros(self.n, bool)
        self.rounds += 1

    def update_predictions(self, idxs: Sequence[int], texts: Sequence[str], confs: Sequence[float], cons: Optional[Sequence[float]] = None,
                           risk: Optional[Sequence[float]] = None) -> None:
        for k, (i, t, c) in enumerate(zip(idxs, texts, confs)):
            if self.has[i] and self.text[i] == t:
                self.stab[i] += 1
            else:
                self.stab[i] = 0
                self.loss_ema[i] = np.nan
            self.text[i], self.conf[i], self.has[i] = t, c, True
            if cons is not None:
                self.cons[i] = cons[k]
            if risk is not None:
                self.risk[i] = risk[k]

    def update_loss(self, idxs: Sequence[int], losses: Sequence[float], alpha: float = 0.5) -> None:
        for i, l in zip(idxs, losses):
            self.loss_ema[i] = l if np.isnan(self.loss_ema[i]) else (1 - alpha) * self.loss_ema[i] + alpha * l

    def _usable(self, i: int, text: str, min_chars: int) -> bool:
        return len(text.strip()) >= min_chars and ctc_feasible(text, int(self.frames[i]))

    def select(self, pc: PseudoConfig, theta: Optional[float], theta_llm: Optional[float] = None) -> None:
        """theta = largest predicted risk that is accepted; theta_llm = the looser level up to which an LLM-rescued reading may still be
        accepted; theta None = no calibration lines -> fallback to the sequence confidence.  A line must also be stable across refreshes
        and pass the perturbation-consistency gate."""
        ok = self.unl & self.has
        usable = np.array([self._usable(i, self.text[i], pc.min_chars) if ok[i] else False for i in range(self.n)])
        gate = (np.isfinite(self.cons) & (self.cons <= pc.tau_cons)) if pc.consistency else np.ones(self.n, bool)
        if theta is None:
            reliable = self.conf >= pc.fallback_tau_conf
            rel_risk = np.clip((1.0 - self.conf) / max(1.0 - pc.fallback_tau_conf, 1e-6), 0, 1)
        else:
            reliable = np.isfinite(self.risk) & (self.risk <= theta)
            rel_risk = np.clip(np.nan_to_num(self.risk, nan=1.0) / max(theta, 1e-6), 0, 1) if np.isfinite(theta) else np.ones(self.n)
        conf_ok = ok & usable & (self.stab >= pc.min_stability) & reliable
        cand = conf_ok & gate
        self._pcg_rejected, self._pcg_accepted = conf_ok & ~gate, cand.copy()
        if pc.lops_drop > 0 and cand.sum() > 10:
            score = np.where(np.isnan(self.loss_ema), -np.log(np.clip(self.conf, 1e-6, 1)), self.loss_ema)
            ids = np.flatnonzero(cand)
            k = int(round(pc.lops_drop * len(ids)))
            if k > 0:
                cand[ids[np.argsort(-score[ids])[:k]]] = False
        llm_usable = np.array([bool(self.llm_ok[i]) and self._usable(i, self.llm_text[i] or "", pc.min_chars) for i in range(self.n)])
        llm_floor = np.zeros(self.n, bool) if theta_llm is None else (np.isfinite(self.risk) & (self.risk <= theta_llm))
        llm = ok & llm_usable & ~cand & gate & (self.stab >= pc.min_stability) & llm_floor
        self.sel = cand | llm
        w = np.clip(1.0 - 0.5 * rel_risk, pc.min_weight, 1.0)
        self.weight = np.where(llm, np.clip(w * pc.llm_weight_scale, pc.min_weight * 0.5, 1.0), w).astype(np.float32)
        self.weight[~self.sel] = 0.0
        self.source = np.where(cand, 1, np.where(llm, 2, 0)).astype(np.int8)
        self.train_text = [self.text[i] if cand[i] else (self.llm_text[i] or "") if llm[i] else "" for i in range(self.n)]

    def _corpus_cer(self, idx: Sequence[int], texts: Sequence[str]) -> float:
        e = sum(edit_distance(self.hidden[i], t) for i, t in zip(idx, texts))
        return float(e / max(sum(len(self.hidden[i]) for i in idx), 1))

    def _group_diag(self, ids: np.ndarray, prefix: str) -> Dict[str, float]:
        d: Dict[str, float] = {}
        if len(ids) == 0:
            return d
        d[prefix + "_all_cer"] = self._corpus_cer(ids, [self.text[i] for i in ids])
        d[prefix + "_stable_frac"] = float(np.mean(self.stab[ids] >= 1))
        d[prefix + "_coverage"] = float(self.sel[ids].mean())
        s = ids[self.sel[ids]]
        d[prefix + "_selected"] = float(len(s))
        if len(s):
            d[prefix + "_sel_cer"] = self._corpus_cer(s, [self.train_text[i] for i in s])
            d[prefix + "_sel_cer_line_mean"] = float(np.mean([cer(self.hidden[i], self.train_text[i]) for i in s]))
            d[prefix + "_sel_exact"] = float(np.mean([self.hidden[i] == self.train_text[i] for i in s]))
            for tag, code in (("rc", 1), ("llm", 2)):
                sub = s[self.source[s] == code]
                if len(sub):
                    d[f"{prefix}_{tag}_sel_cer"] = self._corpus_cer(sub, [self.train_text[i] for i in sub])
        return d

    def diagnostics(self) -> Dict[str, float]:
        """Pseudo-label quality against the hidden GT.  pl_* = the TRUE unlabeled lines; shadow_* = the calibration lines."""
        ok_all = self.unl & self.has
        ok = np.flatnonzero(ok_all & ~self.cal)
        if len(ok) == 0:
            return {}
        d = self._group_diag(ok, "pl")
        d["pl_pcg_refused_n"] = float(self._pcg_rejected[ok].sum())
        d["pl_llm_selected"] = float((self.source[ok] == 2).sum())
        d.update(self._group_diag(np.flatnonzero(ok_all & self.cal), "shadow"))
        if np.isfinite(self.cons[ok]).any():
            d["pl_cons_mean"] = float(np.nanmean(self.cons[ok]))
        if np.isfinite(self.risk[ok]).any():
            cer_all = np.array([cer(self.hidden[i], self.text[i]) for i in ok])
            m = np.isfinite(self.risk[ok])
            d["pl_auroc_error_detection"] = auroc(self.risk[ok][m], cer_all[m] > 0)
            conf_i = np.array([1.0 - self.conf[i] for i in ok])
            d["pl_auroc_seq_conf"] = auroc(conf_i[m], cer_all[m] > 0)
        rej, acc = np.flatnonzero(self._pcg_rejected & ~self.cal), np.flatnonzero(self._pcg_accepted & ~self.cal)
        d["pl_pcg_rejected"] = float(len(rej))
        if len(rej):
            d["pl_pcg_rejected_cer"] = self._corpus_cer(rej, [self.text[i] for i in rej])
        if len(acc):
            d["pl_pcg_accepted_cer"] = self._corpus_cer(acc, [self.text[i] for i in acc])
        return d

    def state_dict(self) -> Dict[str, Any]:
        return {k: getattr(self, k) for k in ("text", "conf", "stab", "has", "cons", "risk", "loss_ema", "sel", "weight", "train_text", "source", "rounds")}

    def load_state_dict(self, sd: Dict[str, Any]) -> None:
        for k, v in sd.items():
            setattr(self, k, v)

In [ ]:
%%writefile inklingua/frag.py
"""
[FRAG-RC] Fragment-level, style-stratified, risk-controlled self-distillation (replaces line-level pseudo-labels).

Why (v9 IAM run): the line-level rule kept ~10 % of the unlabeled lines, namely those the teacher already read (CER 1.2 % vs 6.4 %
overall); the thin / tightly written hands that carry most of the errors never got in; and the rule was a knife edge because it was
calibrated on ~190 LINES.  FRAG-RC changes the unit of trust:
  * a CHARACTER, not a line: ~8,000 calibration characters instead of 194 lines -> a stable threshold;
  * a FRAGMENT of a line: the reliable stretches of a hard line still teach, the rest is left alone;
  * a Mondrian cell (corpus x hardness stratum; thin stroke / tight gaps = hard) instead of one global threshold: every cell gets its own
    threshold at the same character-error target, so hard hands and small corpora are represented instead of silently excluded;
  * soft frame targets (sharpened EMA-teacher posteriors) with an anchor on the untrusted frames, not a hard label sequence.
Evidence for a character comes from sources that fail differently: the teacher's posterior, the inter-CTC head that reads the quantised
content stream, and a second (strongly augmented) view of the same pixels read by the same teacher.

Closest precedents (all different in the unit or the guarantee): Seq-UPS, SoftCTC, Star Temporal Classification, conformal risk control
/ Mondrian conformal prediction, Selective Conformal Risk Control (2512.12844), frame-level self-distillation for CTC in speech
(2406.07909), CR-CTC.  No 2025-2026 paper found (2026-11-04) selects characters of a CTC reading with a risk-controlled stratified rule.
"""
from __future__ import annotations

from typing import Any, Dict, List, Sequence, Tuple

import numpy as np

from .riskcontrol import char_risk_threshold, fit_logistic, sigmoid
from .text import align_chars

FRAG_FEATURES = ["logit peak prob", "margin to runner-up", "inter-CTC agreement", "second-view agreement",
                 "min logit peak in the 3-char window", "log run length", "logit min blank prob in the adjacent gaps"]


def _logit_clip(p, eps: float = 1e-4):
    p = np.clip(p, eps, 1.0 - eps)
    return np.log(p / (1.0 - p))


def char_evidence(lp_w: np.ndarray, lp_s: np.ndarray, lp_aux: np.ndarray) -> Tuple[List[int], np.ndarray, np.ndarray, np.ndarray]:
    """Per EMITTED character of the weak-view reading: (ids, first frame, end frame (exclusive), features [K, len(FRAG_FEATURES)]).
    lp_w / lp_s / lp_aux: [T, V] log-posteriors of the same line (valid frames only): main head on the weak view, main head on the strong
    view, inter-CTC head on the weak view.  Label-free."""
    T = lp_w.shape[0]
    nf = len(FRAG_FEATURES)
    if T == 0:
        return [], np.zeros(0, np.int64), np.zeros(0, np.int64), np.zeros((0, nf))
    pw = np.exp(lp_w)
    idx, mp = pw.argmax(1), pw.max(1)
    starts = np.flatnonzero(np.r_[True, idx[1:] != idx[:-1]])
    ends = np.r_[starts[1:], T]
    runs = [k for k in range(len(starts)) if idx[starts[k]] != 0]
    ia, is_ = np.exp(lp_aux).argmax(1), np.exp(lp_s).argmax(1)
    pb = pw[:, 0]
    ids: List[int] = []
    st, en, rows = [], [], []
    for n_, k in enumerate(runs):
        s, e = int(starts[k]), int(ends[k])
        c = int(idx[s])
        f_peak = int(s + mp[s:e].argmax())
        peak = float(mp[f_peak])
        srt = np.sort(pw[f_peak])[::-1]
        run2 = float(srt[1]) if len(srt) > 1 else 0.0
        a_prev = int(ends[runs[n_ - 1]]) if n_ > 0 else 0
        a_next = int(starts[runs[n_ + 1]]) if n_ + 1 < len(runs) else T
        gaps = [pb[a_prev:s], pb[e:a_next]]
        mb = min([float(g.min()) for g in gaps if g.size] or [1.0])
        lo, hi = max(0, s - 1), min(T, e + 1)
        rows.append([_logit_clip(peak), peak - run2, float((ia[s:e] == c).mean()), float((is_[lo:hi] == c).any()), 0.0,
                     float(np.log(max(e - s, 1))), _logit_clip(mb)])
        ids.append(c), st.append(s), en.append(e)
    Fm = np.array(rows, np.float64).reshape(-1, nf)
    if len(Fm):
        lg = Fm[:, 0]
        pad = np.r_[lg[:1], lg, lg[-1:]]
        Fm[:, 4] = np.minimum(np.minimum(pad[:-2], pad[1:-1]), pad[2:])
    return ids, np.array(st, np.int64), np.array(en, np.int64), Fm


def char_wrong_labels(ref: str, hyp: str) -> np.ndarray:
    """1 for every character of the reading that is wrong OR sits next to an omission (a reference character the reading lacks): a
    fragment bordering a missing character is not safe to learn from.  Same length as `hyp`."""
    ops = align_chars(ref, hyp)
    y = np.zeros(len(hyp), np.int8)
    jj = [o[2] for o in ops]
    for k, (op, _i, j) in enumerate(ops):
        if op in ("sub", "ins"):
            y[j] = 1
        elif op == "del":
            for q in range(k - 1, -1, -1):
                if jj[q] is not None:
                    y[jj[q]] = 1
                    break
            for q in range(k + 1, len(ops)):
                if jj[q] is not None:
                    y[jj[q]] = 1
                    break
    return y


class FragModel:
    """Ridge-logistic model  character evidence -> P(character is wrong or borders an omission).  Plain arrays only (checkpoint-safe)."""

    def __init__(self, mu: np.ndarray, sd: np.ndarray, b: np.ndarray):
        self.mu, self.sd, self.b = np.asarray(mu, np.float64), np.asarray(sd, np.float64), np.asarray(b, np.float64)

    @staticmethod
    def fit(X: np.ndarray, y: np.ndarray, l2: float = 3.0) -> "FragModel":
        mu, sd = X.mean(0), X.std(0) + 1e-6
        return FragModel(mu, sd, fit_logistic((X - mu) / sd, np.asarray(y, np.float64), l2))

    def risk(self, X: np.ndarray) -> np.ndarray:
        if len(X) == 0:
            return np.zeros(0)
        return sigmoid(self.b[0] + ((X - self.mu) / self.sd) @ self.b[1:])

    def state(self) -> Dict[str, Any]:
        return {"mu": self.mu, "sd": self.sd, "b": self.b}

    @staticmethod
    def from_state(d: Dict[str, Any]) -> "FragModel":
        return FragModel(d["mu"], d["sd"], d["b"])


def fragment_mask(T: int, st: np.ndarray, en: np.ndarray, risky: np.ndarray, guard: int = 1) -> np.ndarray:
    """Frames of the TRUSTED fragments of one line: the frames of every reliable character, the blank frames between two consecutive
    reliable characters, and the margins before the first / after the last character when it is reliable.  Frames within `guard` frames
    of an untrusted character are never trusted.  A line without any emitted character is not trusted at all."""
    K = len(st)
    trust = np.zeros(T, bool)
    if K == 0:
        return trust
    rel = ~np.asarray(risky, bool)
    for k in range(K):
        if rel[k]:
            trust[st[k]:en[k]] = True
    for k in range(K - 1):
        if rel[k] and rel[k + 1]:
            trust[en[k]:st[k + 1]] = True
    if rel[0]:
        trust[:st[0]] = True
    if rel[-1]:
        trust[en[-1]:] = True
    bad = np.zeros(T, bool)
    for k in range(K):
        if not rel[k]:
            bad[st[k]:en[k]] = True
    if guard > 0 and bad.any():
        bad = np.convolve(bad.astype(np.float64), np.ones(2 * guard + 1), mode="same") > 0
    return trust & ~bad


def cell_theta(theta: Sequence[float], cell: int) -> float:
    """Threshold of a Mondrian cell; indices beyond the rule's cells are clipped (a checkpoint applied to data with more cells)."""
    if len(theta) == 0:
        return float("-inf")
    return float(theta[int(np.clip(cell, 0, len(theta) - 1))])


def frag_trust_batch(lp_w: np.ndarray, aux_w: np.ndarray, lp_s: np.ndarray, lens: np.ndarray, strat: np.ndarray, rows: np.ndarray,
                     model: FragModel, theta: Sequence[float], guard: int) -> Tuple[np.ndarray, int, int]:
    """Trust masks [B, T] of a batch ([B, T, V] numpy log-posteriors; only the lines flagged in `rows` are processed).  Also returns the
    number of emitted characters seen and the number that were trusted."""
    B, T = lp_w.shape[:2]
    trust = np.zeros((B, T), bool)
    n_ch = n_tr = 0
    for b in range(B):
        if not rows[b]:
            continue
        L = int(lens[b])
        ids, st, en, Fm = char_evidence(lp_w[b, :L], lp_s[b, :L], aux_w[b, :L])
        if not len(ids):
            continue
        risky = ~(model.risk(Fm) <= cell_theta(theta, int(strat[b])))
        trust[b, :L] = fragment_mask(L, st, en, risky, guard)
        n_ch += len(ids)
        n_tr += int((~risky).sum())
    return trust, n_ch, n_tr


def fit_trust_rule(X: np.ndarray, y: np.ndarray, line: np.ndarray, cell: np.ndarray, n_cells: int, target: float, z: float, min_chars: int,
                   folds: int, seed: int) -> Tuple[FragModel, List[float], np.ndarray, Dict[str, float]]:
    """Cross-fitted (by line) reliability model + one threshold per Mondrian cell.  Returns (model, theta per cell, out-of-fold risks, info)."""
    rng = np.random.RandomState(seed)
    lines = np.unique(line)
    fold = dict(zip(lines.tolist(), (rng.permutation(len(lines)) % max(folds, 2)).tolist()))
    fo = np.array([fold[int(i)] for i in line])
    oof = np.zeros(len(y))
    for f_ in range(max(folds, 2)):
        te = fo == f_
        if te.all() or not te.any():
            oof[te] = 0.5
            continue
        oof[te] = FragModel.fit(X[~te], y[~te]).risk(X[te])
    model = FragModel.fit(X, y)
    theta, info = [], {}
    for s in range(n_cells):
        m_ = cell == s
        th, k, err = char_risk_threshold(oof[m_], y[m_], line[m_], target, z, min_chars)
        theta.append(th)
        info[f"frag_theta_s{s}"] = th if np.isfinite(th) else float("nan")
        info[f"frag_cal_cov_s{s}"] = float(k / max(int(m_.sum()), 1))
        info[f"frag_cal_err_sel_s{s}"] = err
        info[f"frag_cal_err_all_s{s}"] = float(y[m_].mean()) if m_.any() else float("nan")
        info[f"frag_cal_chars_s{s}"] = float(m_.sum())
    return model, theta, oof, info

In [ ]:
%%writefile inklingua/alphabet.py
"""
Function-preserving growth of the CTC alphabet ([ALPHA-SPLIT]) and language-prior resolution of tied variants ([LEX-TIE]).

Used by (i) `mode="finetune"` / `TrainConfig.init_from` -- a checkpoint trained on one corpus is continued on another whose labeled
lines contain new characters -- and (ii) the zero-shot adapter (inklingua/adapt.py).  New units are APPENDED as tied copies of the
visually closest known unit: the group of k units gets bias - log k, so the group's logsumexp equals the old logit and the model's
behaviour on the old alphabet is unchanged (tested to ~1e-5).
"""
from __future__ import annotations

import math
import unicodedata
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn

from .text import CharCodec, grapheme_clusters

# Visually similar letters of other orthographies (a curated heuristic; extend it with AdaptConfig.visual_base).  Accented forms need no
# entry: Unicode decomposition already gives their base letter.
_VISUAL_BASE = {"ɛ": "e", "ɔ": "o", "ɓ": "b", "ɗ": "d", "ŋ": "n", "ƙ": "k", "ɲ": "n", "ə": "e", "ʃ": "s", "ʋ": "v", "ƴ": "y", "æ": "a", "œ": "o",
                "ø": "o", "ı": "i", "ł": "l", "đ": "d", "ð": "d", "ƒ": "f", "Ɛ": "E", "Ɔ": "O", "Ɓ": "B", "Ɗ": "D", "Ŋ": "N", "Ƙ": "K", "Ə": "E",
                "Ø": "O", "Ł": "L", "Đ": "D", "Æ": "A", "Œ": "O"}


def visual_base_of(ch: str, known: Sequence[str], extra: Optional[Dict[str, str]] = None) -> Optional[str]:
    """The KNOWN unit a new unit looks like (its base letter), or None.  Decomposition first (é -> e, ẹ́ -> e, ɛ̀ -> ɛ -> e), then the table."""
    known = set(known)
    if ch in known:
        return None
    table = _VISUAL_BASE if not extra else {**_VISUAL_BASE, **extra}
    if ch in table and table[ch] in known:
        return table[ch]
    base = "".join(c for c in unicodedata.normalize("NFD", ch) if not unicodedata.combining(c))
    if len(base) == 1 and base != ch:
        if base in known:
            return base
        if base in table and table[base] in known:
            return table[base]
    return None


class CharGroups:
    """Output units that share one visual shape.  `rep[i]` = the unit i is a copy of (itself for ordinary units).  New units are
    APPENDED so every old index keeps its meaning."""

    def __init__(self, old_units: Sequence[str], new_units: Sequence[str], extra: Optional[Dict[str, str]] = None):
        old_units, new_units = list(old_units), list(new_units)
        assert new_units[:len(old_units)] == old_units, "new alphabet must extend the old one (old indices must not move)"
        self.old_units, self.units = old_units, new_units
        self.n_old, self.V = len(old_units) + 1, len(new_units) + 1
        self.rep = np.arange(self.V)
        self.orphans: List[int] = []
        known = set(old_units)
        for i in range(self.n_old, self.V):
            b = visual_base_of(new_units[i - 1], known, extra)
            if b is None:
                self.orphans.append(i)
            else:
                self.rep[i] = old_units.index(b) + 1
        self.members: Dict[int, List[int]] = {}
        for i in range(1, self.V):
            self.members.setdefault(int(self.rep[i]), []).append(i)
        self.M = np.zeros((self.V, self.V))
        self.M[np.arange(self.V), self.rep] = 1.0

    def variants(self, i: int) -> List[int]:
        return self.members.get(int(self.rep[int(i)]), [int(i)])

    def collapse(self, lp: np.ndarray) -> np.ndarray:
        """Frame log-posteriors [..., V] with the mass of every group moved onto its base letter."""
        p = np.exp(np.asarray(lp, np.float64)) @ self.M
        return np.log(np.maximum(p, 1e-30)).astype(np.float32)

    def collapse_t(self, lp: torch.Tensor) -> torch.Tensor:
        M = torch.as_tensor(self.M, dtype=lp.dtype, device=lp.device)
        return torch.log(torch.clamp(lp.exp() @ M, min=1e-30))

    def base_ids(self, ids: Sequence[int]) -> List[int]:
        return [int(self.rep[int(i)]) for i in ids]

    @property
    def n_variant_groups(self) -> int:
        return sum(1 for m in self.members.values() if len(m) > 1)


def extend_codec(codec: CharCodec, new_text: str, extra: Optional[Dict[str, str]] = None) -> Tuple[CharCodec, CharGroups]:
    """Append the grapheme clusters of `new_text` that the alphabet lacks (never reorders: old indices keep their meaning)."""
    old = list(codec.units)
    add: List[str] = []
    for u in grapheme_clusters(new_text):
        if not u.isspace() and not unicodedata.combining(u[0]) and u not in old and u not in add:
            add.append(u)
    new = CharCodec(old + add)
    return new, CharGroups(old, new.units, extra)


def extend_codec_units(old_units: Sequence[str], wanted_units: Sequence[str], extra: Optional[Dict[str, str]] = None) -> Tuple[CharCodec, CharGroups]:
    """Same as extend_codec for a list of wanted units (the alphabet of a new corpus); spaces are kept if wanted."""
    old = list(old_units)
    add = [u for u in wanted_units if u not in old]
    new = CharCodec(old + sorted(set(add), key=add.index))
    return new, CharGroups(old, new.units, extra)


@torch.no_grad()
def extend_ctc_heads(model: nn.Module, groups: CharGroups, orphan_bias: float = -9.0) -> nn.Module:
    """[ALPHA-SPLIT] grow the CTC head and the inter-CTC head WITHOUT changing what the model says about the old alphabet.  Every new
    variant is a copy of its base letter's row, the group of k units gets bias - log k; orphans (no visual base) start with zero
    weights and a very negative bias."""
    V0, V = groups.n_old, groups.V
    for name in ("ctc_head", "aux_head"):
        old: nn.Linear = getattr(model, name)
        new = nn.Linear(old.in_features, V).to(old.weight.device, old.weight.dtype)
        new.weight[:V0], new.bias[:V0] = old.weight, old.bias
        for i in range(V0, V):
            r = int(groups.rep[i])
            if r != i:
                new.weight[i], new.bias[i] = old.weight[r], old.bias[r]
            else:
                new.weight[i], new.bias[i] = 0.0, float(orphan_bias)
        for mem in groups.members.values():
            if len(mem) > 1:
                new.bias[mem] -= math.log(len(mem))
        setattr(model, name, new)
    model.num_classes = V
    return model


# ---------------------------------------------------------------- [LEX-TIE]
def resolve_variants(ids: Sequence[int], frames: Sequence[int], lp: np.ndarray, groups: CharGroups, scorer=None, *, beam: int = 4,
                     vis_w: float = 1.0, lm_w: float = 1.0, rho: float = 0.0, decided_p: float = 0.98, max_options: int = 6,
                     max_combo: int = 48) -> Tuple[str, Dict[str, float]]:
    """The reading `ids` (base letters, from the COLLAPSED posteriors) fixes WHICH letter was written; this decides, for every letter that
    has variants (e / é / è / ê ...), WHICH variant:  score = vis_w * sum log p_vis(variant | group, frame) + lm_w * (log p_LM(line) + rho *
    #tokens).  A letter whose best variant has posterior >= decided_p is settled by vision alone.  Ambiguous letters are resolved word by
    word (joint search inside a word) with a beam over words."""
    units = groups.units
    base = [units[int(i) - 1] for i in ids]
    amb: List[Tuple[int, List[str], np.ndarray]] = []
    n_var, n_dec = 0, 0
    for j, (i, t) in enumerate(zip(ids, frames)):
        mem = groups.variants(int(i))
        if len(mem) == 1:
            continue
        n_var += 1
        v = np.asarray(lp[int(t), mem], np.float64)
        v = v - (v.max() + math.log(float(np.exp(v - v.max()).sum())))
        o = np.argsort(-v, kind="stable")
        if float(np.exp(v[o[0]])) >= decided_p:
            base[j] = units[mem[int(o[0])] - 1]
            n_dec += 1
            continue
        keep = o[:max_options]
        amb.append((j, [units[mem[int(k)] - 1] for k in keep], v[keep]))
    info = {"n_variant_pos": float(n_var), "n_decided": float(n_dec), "n_ambiguous": float(len(amb)), "n_lm_texts": 0.0, "margin": float("inf")}
    if not amb:
        return "".join(base), info
    word_of, w = [], 0
    for u in base:
        word_of.append(w)
        if u.isspace():
            w += 1
    groups_w: List[List[Tuple[int, List[str], np.ndarray]]] = []
    for item in amb:
        if groups_w and word_of[groups_w[-1][0][0]] == word_of[item[0]]:
            groups_w[-1].append(item)
        else:
            groups_w.append([item])
    states: List[Tuple[Dict[int, str], float]] = [({}, 0.0)]
    scores: List[float] = [0.0]
    for gw in groups_w:
        keep_n = [len(o) for _, o, _ in gw]
        while math.prod(keep_n) > max_combo and max(keep_n) > 1:
            keep_n[int(np.argmax(keep_n))] -= 1
        combos = [([], 0.0)]
        for (j, opts, vis), kn in zip(gw, keep_n):
            combos = [(c + [o], cv + vis_w * float(vv)) for c, cv in combos for o, vv in zip(opts[:kn], vis[:kn])]
        cands = [({**st, **{jj: o for (jj, _, _), o in zip(gw, c)}}, sv + cv) for st, sv in states for c, cv in combos]
        texts = []
        for st, _ in cands:
            b2 = list(base)
            for jj, o in st.items():
                b2[jj] = o
            texts.append("".join(b2))
        if scorer is not None:
            lm, nt = scorer(texts)
            tot = [cv + lm_w * (float(a_) + rho * float(n)) for (_, cv), a_, n in zip(cands, lm, nt)]
            info["n_lm_texts"] += float(len(texts))
        else:
            tot = [cv for _, cv in cands]
        order = np.argsort(-np.asarray(tot), kind="stable")[:max(1, beam)]
        states, scores = [cands[int(k)] for k in order], [float(tot[int(k)]) for k in order]
    info["margin"] = float(scores[0] - scores[1]) if len(scores) > 1 else float("inf")
    for jj, c in states[0][0].items():
        base[jj] = c
    return "".join(base), info

In [ ]:
%%writefile inklingua/protocol.py
"""[PROTOCOL] which split is used for what, and a log of every look at an evaluation split."""
from __future__ import annotations

import hashlib
import json
import os
import time
from typing import Any, Dict

import numpy as np

from .utils import json_safe


def write_protocol(cfg, data) -> Dict[str, Any]:
    """protocol/protocol.json.  The TEST split decides nothing and is looked at once per checkpoint (every look is logged); the
    VALIDATION split is cut into two disjoint halves of whole form blocks -- `val_sel` picks the checkpoint / stops training, `val_gate`
    is the only data on which any post-hoc option, temperature, threshold or LLM weight is tuned; hidden train labels are read only by
    diagnostics."""
    d = os.path.join(cfg.run_dir, "protocol")
    os.makedirs(d, exist_ok=True)
    sha = lambda a: hashlib.sha1(np.asarray(a, np.int64).tobytes()).hexdigest()  # noqa: E731
    p = {
        "roles": {
            "train: labeled lines": "supervised CTC loss (text used)",
            "train: calibration lines (labeled, text held out)": "text used ONLY to measure the character error of the trust rule (FRAG-RC); "
                                                                 "their IMAGES join the unlabeled pool (shadow calibration)",
            "train: unlabeled lines": "images only; their hidden text is read by DIAGNOSTICS only (never by a loss, a selection or a threshold)",
            "validation: val_sel": "checkpoint selection (EMA vs student, best epoch) and early stopping",
            "validation: val_gate": "every post-hoc decision: view ensemble, style AdaBN, writer self-distillation, space bonus, LLM weights, "
                                    "temperature, [?] marks.  Never used to select the checkpoint.",
            "test": "reported once per checkpoint; nothing is tuned, selected or calibrated on it"},
        "val_sel_lines": int(len(data.val_sel_idx)), "val_gate_lines": int(len(data.val_gate_idx)),
        "val_sel_sha1": sha(data.val_sel_idx), "val_gate_sha1": sha(data.val_gate_idx),
        "val_split_enabled": bool(0.0 < cfg.data.val_gate_fraction < 1.0), "val_block_lines": int(cfg.data.val_block),
        "labeled_fraction": cfg.data.labeled_fraction, "calibration_fraction": cfg.data.calib_fraction,
        "frag_rc": bool(cfg.train.frag.enabled), "legacy_pseudo_labels": bool(cfg.train.pseudo.enabled),
        "corpora": data.domain_names, "n_strata": int(data.n_strata),
        "transductive_options": ["style AdaBN (reads unlabeled evaluation lines)", "writer self-distillation WSD (reads unlabeled evaluation lines)"],
        "external_knowledge": "the LLM was trained on web text that may contain the corpus a benchmark is drawn from: report LLM-rescored numbers "
                              "as 'with an external language model'",
        "seed": cfg.run.seed, "provenance": json_safe(data.provenance)}
    with open(os.path.join(d, "protocol.json"), "w", encoding="utf-8") as f:
        json.dump(json_safe(p), f, indent=2, ensure_ascii=False)
    return p


def log_test_look(cfg, ckpt: Dict[str, Any], split: str) -> int:
    """Append one line to protocol/test_looks.jsonl for every evaluation run; returns how many looks at `split` there have been."""
    d = os.path.join(cfg.run_dir, "protocol")
    os.makedirs(d, exist_ok=True)
    path = os.path.join(d, "test_looks.jsonl")
    h = None
    try:
        with open(ckpt.get("path", ""), "rb") as f:
            h = hashlib.sha1(f.read(1 << 22)).hexdigest()
    except Exception:
        pass
    with open(path, "a") as f:
        f.write(json.dumps({"time": time.strftime("%Y-%m-%d %H:%M:%S"), "split": split, "checkpoint": ckpt.get("path"),
                            "checkpoint_sha1_first4MB": h, "epoch": ckpt.get("epoch")}) + "\n")
    with open(path) as f:
        return sum(1 for line in f if json.loads(line).get("split") == split)

In [ ]:
%%writefile inklingua/curves.py
"""Training curves (train vs valid, FRAG-RC trust, legacy pseudo-labels, inertia, leakage, VQ)."""
from __future__ import annotations

import math
import os
from typing import Dict, List, Sequence, Tuple

import matplotlib
import numpy as np

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

from .style import STYLE_NAMES


def _plot_series(ax, H: Dict[str, List[float]], series: Sequence[Tuple[str, str]], title: str, ylabel: str = "", logy: bool = False) -> None:
    ep = np.asarray(H.get("epoch", []), float) + 1
    drawn = False
    for key, label in series:
        if key in H:
            y = np.asarray(H[key], float)
            m = np.isfinite(y)
            if m.any():
                n = min(len(ep), len(y))
                m = m[:n]
                ax.plot(ep[:n][m], y[:n][m], marker="o", ms=3, lw=1.6, label=label)
                drawn = True
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("epoch")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
    if logy and drawn:
        ax.set_yscale("log")
    if drawn:
        ax.legend(fontsize=7)


def plot_training_curves(H: Dict[str, List[float]], out_dir: str, individual: bool = True) -> None:
    os.makedirs(out_dir, exist_ok=True)
    leak_series = [(f"leak_r2_{n}", f"R2 of the optimal probe: {n}") for n in STYLE_NAMES]
    cells = sorted({k.split("frag_cal_cov_s")[1] for k in H if k.startswith("frag_cal_cov_s")}, key=lambda s: int(s))
    panels = [
        ("loss", "Loss: train vs valid", "loss",
         [("train_task_total", "train task loss (running, augmented)"), ("train_total", "train full objective (all terms)"),
          ("train_eval_total", "train subset (eval mode)"), ("val_total", "valid"), ("val_ema_total", "valid (EMA)")], False),
        ("ctc", "CTC loss (per char): train vs valid", "CTC",
         [("train_ctc", "train running"), ("train_eval_ctc", "train subset (eval)"), ("val_ctc", "valid")], True),
        ("cer", "CER: train vs valid (val_sel selects the checkpoint, val_gate never does)", "CER",
         [("train_eval_cer", "train subset"), ("val_sel_cer", "val_sel (student)"), ("val_ema_sel_cer", "val_sel (EMA teacher)"),
          ("val_gate_cer", "val_gate (student)"), ("val_cer", "all validation (student)")], True),
        ("wer", "WER: train vs valid", "WER",
         [("train_eval_wer", "train subset"), ("val_wer", "valid (student)"), ("val_ema_wer", "valid (EMA teacher)")], True),
        ("style", "Connected task - style MAE (std. units, measured entries only)", "MAE / loss",
         [("train_eval_style_mae", "MAE train subset"), ("val_style_mae", "MAE valid"), ("train_style_reg", "reg loss (running, SES targets)")], False),
        ("style_acc", "Connected task - style-cluster accuracy", "accuracy", [("train_eval_style_acc", "train subset"), ("val_style_acc", "valid")], False),
        ("frag_trust", "[FRAG-RC] trusted share in the training batches + fragment loss", "value",
         [("train_frag_char_trust", "trusted share of emitted characters"), ("train_frag_trust", "trusted share of frames"),
          ("train_frag", "fragment distillation loss")], False),
        ("frag_cov", "[FRAG-RC] coverage of the trust rule per cell (calibration chars) and on unlabeled lines", "coverage",
         [(f"frag_cal_cov_s{s}", f"calibration, cell {s}") for s in cells] + [("frag_unl_cov", "unlabeled sample (all cells)")], False),
        ("frag_err", "[FRAG-RC] realised error of the TRUSTED characters (unlabeled sample, hidden text; diagnostic)", "error rate",
         [(f"frag_unl_err_sel_s{s}", f"trusted chars, cell {s}") for s in cells] + [("frag_unl_err_sel", "trusted chars, all cells"),
                                                                                    ("frag_unl_err_all", "ALL chars of the sample"),
                                                                                    ("frag_cal_char_err", "all calibration chars")], False),
        ("pseudo", "Legacy pseudo-labels (checked against hidden GT, diagnostics only)", "value",
         [("pl_coverage", "coverage of unlabeled"), ("pl_sel_exact", "selected: exact-match rate"), ("pl_sel_cer", "selected: CER vs hidden GT"),
          ("pl_all_cer", "all unlabeled: CER"), ("pl_stable_frac", "stable fraction")], False),
        ("pl_rank", "Does the reliability score rank the errors? (AUROC)", "AUROC",
         [("pl_auroc_error_detection", "learned line risk (unlabeled)"), ("pl_auroc_seq_conf", "CTC sequence confidence (unlabeled)"),
          ("rc_calib_auroc_error_detection", "learned line risk (calibration)"), ("frag_auroc_risk", "FRAG-RC character risk (calibration)"),
          ("frag_auroc_conf", "character peak probability (calibration)")], False),
        ("cov", "Legacy pseudo-labels: accepted coverage", "coverage",
         [("rc_calib_cov_raw", "calibration lines: raw rule"), ("rc_calib_coverage", "calibration lines: after guard"),
          ("shadow_coverage", "shadow lines"), ("pl_coverage", "unlabeled lines")], False),
        ("inertia", "Model inertia (Hsu et al.)", "value",
         [("val_stability", "prediction stability vs prev. epoch"), ("val_nfr", "negative-flip rate"),
          ("val_noise_cons", "noise consistency (1-CER clean/noisy)"), ("val_noise_gap", "noise CER gap")], False),
        ("vq", "VQ codebook perplexity", "perplexity", [("train_ppl_low", "stroke codebook"), ("train_ppl_high", "component codebook")], False),
        ("lr", "Learning rate / schedules", "value", [("lr_rel", "lr / peak lr"), ("vq_lambda", "quantisation strength"), ("aux_scale", "aux-task curriculum")], False),
        ("leak", "Content / style separation and consistency losses", "loss",
         [("train_leak", "[CFL] leakage penalty"), ("train_cons", "teacher-student frame consistency"), ("train_decor", "cross-covariance")], False),
        ("leak_r2", "[CFL] style readable from the content stream (R2 of the optimal linear probe)", "R2", leak_series, False),
    ]
    ncol = 3
    nrow = int(math.ceil(len(panels) / ncol))
    fig_all, axes = plt.subplots(nrow, ncol, figsize=(17, 4.0 * nrow))
    for (name, title, yl, series, logy), ax in zip(panels, axes.ravel()):
        _plot_series(ax, H, series, title, yl, logy)
        if individual:
            f, a = plt.subplots(figsize=(6.5, 4.2))
            _plot_series(a, H, series, title, yl, logy)
            f.tight_layout()
            f.savefig(os.path.join(out_dir, f"curve_{name}.png"), dpi=130)
            plt.close(f)
    for ax in axes.ravel()[len(panels):]:
        ax.axis("off")
    fig_all.tight_layout()
    fig_all.savefig(os.path.join(out_dir, "curves_all.png"), dpi=110)
    plt.close(fig_all)

In [ ]:
%%writefile inklingua/trainer.py
"""
The training loop: curriculum schedules, the EMA teacher, FRAG-RC calibration refreshes, the legacy line-level pseudo-labels, checkpoint
selection on val_sel, early stopping, resume (warm restart instead of an lr jump) and warm starts from another checkpoint.
"""
from __future__ import annotations

import glob
import json
import math
import os
import random
import time
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn

from .alphabet import CharGroups, extend_ctc_heads
from .config import Config
from .curves import plot_training_curves
from .data import HandwritingDataModule, HandwritingDataset
from .decoding import greedy_decode, seq_confidence
from .frag import FRAG_FEATURES, FragModel, cell_theta, char_evidence, char_wrong_labels, fit_trust_rule, frag_trust_batch
from .llm import LLMEngine, fuse_scores
from .model import ClosedFormLeakage, GPUAugment, InkLinguaNet, ModelEMA, compute_losses
from .pseudo import PERTURBATIONS, RELIABILITY_FEATURES, PseudoLabelBank, SelectionRule, apply_perturbation, eval_noise, reliability_features, reliability_views
from .riskcontrol import auroc
from .style import NUM_DESC, STYLE_NAMES
from .text import cer, detokenize, edit_distance, grapheme_clusters
from .utils import LOG, Meter, atomic_save, json_safe, to_device, torch_load

try:
    from tqdm.auto import tqdm
except Exception:  # pragma: no cover
    def tqdm(x, **kw):  # type: ignore
        return x


def warm_start(model: InkLinguaNet, path: str, codec, strict_alphabet: bool = False) -> Dict[str, Any]:
    """Load the weights of an InkLingua checkpoint (best.pt / last.pt / adapted.pt) into `model`, whose output layer has already been built
    for `codec`.  If the checkpoint's alphabet is a PREFIX of the new one (the data module built the codec with base_units), the new units are
    appended as tied copies ([ALPHA-SPLIT]); otherwise the alphabets must be identical."""
    st = torch_load(path)
    weights = st["weights"] if "weights" in st else st["ema"]
    old_units = list(st.get("units") or grapheme_clusters(st["chars"]))
    new_units = list(codec.units)
    info = {"path": path, "epoch": st.get("epoch"), "old_alphabet": len(old_units), "new_alphabet": len(new_units), "appended_units": 0}
    if old_units == new_units:
        model.load_state_dict(weights)
        return info
    if strict_alphabet or new_units[:len(old_units)] != old_units:
        raise RuntimeError(f"warm start: the checkpoint alphabet ({len(old_units)} units) differs from the current one ({len(new_units)}); "
                           "build the data module with base_units=<checkpoint units> (mode 'finetune' does this) or set init_strict_alphabet=False "
                           "with an extending alphabet")
    V0 = len(old_units) + 1
    # build the network at the OLD size, load, then grow the heads function-preservingly
    for name in ("ctc_head", "aux_head"):
        lin = getattr(model, name)
        setattr(model, name, nn.Linear(lin.in_features, V0).to(lin.weight.device))
    model.num_classes = V0
    model.load_state_dict(weights)
    groups = CharGroups(old_units, new_units)
    extend_ctc_heads(model, groups)
    info.update({"appended_units": len(new_units) - len(old_units), "variant_groups": groups.n_variant_groups, "orphans": len(groups.orphans),
                 "orphan_units": "".join(new_units[i - 1] for i in groups.orphans)})
    return info


class Trainer:
    def __init__(self, cfg: Config, data: HandwritingDataModule, model: InkLinguaNet, llm: LLMEngine, device: torch.device):
        self.cfg, self.data, self.llm, self.device = cfg, data, llm, device
        self.tc, self.pc, self.fc = cfg.train, cfg.train.pseudo, cfg.train.frag
        for v in self.pc.cons_views:
            if v not in PERTURBATIONS:
                raise ValueError(f"unknown consistency view '{v}'; choose from {list(PERTURBATIONS)}")
        if self.pc.consistency and not self.pc.cons_views:
            raise ValueError("PseudoConfig.consistency=True needs at least one view in cons_views")
        self.model = model.to(device)
        self.init_info: Optional[Dict[str, Any]] = None
        if self.tc.init_from:
            self.init_info = warm_start(self.model, self.tc.init_from, data.codec, self.tc.init_strict_alphabet)
            LOG.info("warm start from %s (epoch %s): alphabet %d -> %d units (+%d appended as tied copies)", self.tc.init_from, self.init_info["epoch"],
                     self.init_info["old_alphabet"], self.init_info["new_alphabet"], self.init_info["appended_units"])
        self.ema = ModelEMA(self.model, self.tc.ema_decay)
        self.aug = GPUAugment(self.tc, data.style_mean, data.style_std)
        ds = data.train_ds
        self.shadow = bool(self.pc.shadow_calibration)
        self.bank = PseudoLabelBank(len(ds), ~ds.labeled & (~ds.calib | self.shadow), ds.oracle, np.ceil(ds.width / 4).astype(np.int64),
                                    calib=ds.calib if self.shadow else None)
        self.prev_cov: Optional[float] = None
        self.cov_hist: List[float] = []
        self.leak = ClosedFormLeakage(cfg.model.d_model, NUM_DESC, self.tc.leak_queue, self.tc.leak_ridge, self.tc.leak_slack, data.leak_active,
                                      np.clip(data.text_baseline_r2, 0.0, 1.0), device)
        self.use_amp = bool(self.tc.amp and device.type == "cuda")
        self.amp_dtype = torch.float32
        if self.use_amp:   # bf16 only where it is native (compute capability >= 8); fp16 + GradScaler elsewhere
            self.amp_dtype = torch.bfloat16 if torch.cuda.get_device_capability(device)[0] >= 8 else torch.float16
        use_scaler = self.use_amp and self.amp_dtype == torch.float16
        try:
            self.scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)
        except Exception:
            self.scaler = torch.cuda.amp.GradScaler(enabled=use_scaler)
        decay, no_decay = [], []
        for n, p in self.model.named_parameters():
            (decay if p.ndim >= 2 else no_decay).append(p)
        peak = self.tc.lr * (self.tc.init_lr_scale if self.tc.init_from else 1.0)
        self.opt = torch.optim.AdamW([{"params": decay, "weight_decay": self.tc.weight_decay}, {"params": no_decay, "weight_decay": 0.0}],
                                     lr=peak, betas=(0.9, 0.98))
        self.steps_per_epoch = int(math.ceil(len(ds) / cfg.data.batch_size))
        self.total_steps = self.tc.epochs * self.steps_per_epoch
        self.cycles: List[Dict[str, float]] = [{"start": 0, "end": self.total_steps, "peak": peak, "warm": self.tc.warmup_steps, "lo": 0.0}]
        self.global_step, self.start_epoch = 0, 0
        self.history: Dict[str, List[float]] = {"epoch": []}
        self.best_cer, self.best_from, self.best_epoch = float("inf"), "student", 0
        self.ref_cer, self.since_best, self.stopped_at = float("inf"), 0, None
        self.prev_val_preds: Optional[List[str]] = None
        self.prev_val_cer: Optional[np.ndarray] = None
        self.skipped = 0
        # ---- FRAG-RC state: reliability model + one threshold per cell (re-fitted every `refresh_every` epochs)
        self.frag_state: Optional[Dict[str, Any]] = None
        self.frag_model: Optional[FragModel] = None
        self.stratum_w = np.ones(max(1, data.n_strata))
        if self.fc.enabled and self.pc.enabled:
            LOG.warning('FRAG-RC and the legacy line-level pseudo-labels are both enabled: both write into the loss (use Config.ablation("legacy_pl") '
                        'for the legacy baseline alone)')
        if self.fc.enabled and len(data.calib_idx) == 0:
            LOG.warning('FRAG-RC needs held-out calibration lines (cfg.data.calib_fraction > 0): without them the trust rule is never fitted '
                        'and only the consistency loss acts on the unlabeled lines')

    # ------------------------------------------------------------------ schedules
    def _lr(self, step: int) -> float:
        tc, cyc = self.tc, self.cycles[-1]
        for c in self.cycles:
            if step < c["end"]:
                cyc = c
                break
        t = step - cyc["start"]
        if t < cyc["warm"]:
            return float(cyc["lo"] + (cyc["peak"] - cyc["lo"]) * (t + 1) / max(cyc["warm"], 1))
        p = (t - cyc["warm"]) / max(1.0, cyc["end"] - cyc["start"] - cyc["warm"])
        return float(cyc["peak"] * (tc.min_lr_ratio + (1 - tc.min_lr_ratio) * 0.5 * (1 + math.cos(math.pi * min(1.0, p)))))

    def _extend_schedule(self, old_total: int) -> None:
        """cfg.train.epochs changed between the checkpoint and this run: shorter -> the last cycle is cut; longer -> a NEW cycle is appended
        (warm restart to restart_lr_scale * lr after a finished cycle, continuation from the current lr after an interrupted one)."""
        tc, new_total = self.tc, self.total_steps
        last = self.cycles[-1]
        if new_total == last["end"]:
            return
        if new_total < last["end"]:
            last["end"] = max(new_total, self.global_step + 1)
            LOG.warning("cfg.train.epochs decreased: the last lr cycle now ends at step %d", last["end"])
            return
        if tc.extend_policy == "error":
            raise RuntimeError("cfg.train.epochs was increased on resume and cfg.train.extend_policy='error'")
        lr_now = self._lr(self.global_step)
        finished = self.global_step >= last["end"]
        if not finished:
            last["end"] = self.global_step
        peak = min(tc.restart_lr_scale * tc.lr, tc.lr) if finished else lr_now
        warm = tc.restart_warmup_steps if finished else 0
        self.cycles.append({"start": self.global_step, "end": new_total, "peak": float(peak), "warm": warm, "lo": float(lr_now)})
        LOG.warning("cfg.train.epochs changed (%d -> %d steps): new %s lr cycle from step %d, peak lr %.2e (lr now %.2e)", old_total, new_total,
                    "warm-restart" if finished else "continuation", self.global_step, peak, lr_now)

    def _ramp(self, step: int, start_ep: float, ramp_ep: float) -> float:
        spe = self.steps_per_epoch
        return float(np.clip((step - start_ep * spe) / max(1.0, ramp_ep * spe), 0.0, 1.0))

    def _aux_scale(self, step: int) -> float:
        return self._ramp(step, self.tc.aux_start_epoch, self.tc.aux_ramp_epochs)

    def _vq_lambda(self, step: int) -> float:
        return self._ramp(step, self.tc.vq_start_epoch, self.tc.vq_anneal_epochs)

    @property
    def best_path(self) -> str:
        return os.path.join(self.cfg.ckpt_dir, "best.pt")

    # ------------------------------------------------------------------ one epoch
    def _epoch_pool(self, epoch: int) -> np.ndarray:
        """Lines visited in one epoch: every line that has a recognition target (labeled + selected pseudo-labels) plus a random subset of the
        remaining unlabeled / calibration lines (images only: style / masking / VQ / consistency / FRAG-RC).  Labeled lines of harder cells are
        repeated ([STRATA]); with several corpora, `domain_balance` repeats lines of the smaller ones."""
        ds, tc, dc = self.data.train_ds, self.tc, self.cfg.data
        rng = np.random.RandomState(self.cfg.run.seed * 1000 + epoch)
        act = np.flatnonzero(ds.tgt_weight > 0)
        rest = np.flatnonzero(ds.tgt_weight <= 0)
        fc = self.fc
        ratio = fc.unlabeled_ratio if fc.enabled else tc.unlabeled_aux_ratio
        n_aux = len(rest) if len(act) == 0 else min(len(rest), int(round(ratio * len(act))))
        aux = rng.choice(rest, n_aux, replace=False) if n_aux > 0 else np.zeros(0, np.int64)
        lab = np.flatnonzero(ds.labeled & (ds.tgt_weight > 0))
        extra = np.zeros(len(lab))
        if fc.enabled and fc.stratum_emphasis > 0 and self.frag_state is not None and len(self.stratum_w) == self.data.n_strata:
            extra += self.stratum_w[np.clip(ds.stratum[lab], 0, len(self.stratum_w) - 1)] - 1.0
        n_dom = len(self.data.specs)
        if dc.domain_balance > 0 and n_dom > 1 and len(lab):
            cnt = np.bincount(ds.domain[lab], minlength=n_dom).astype(np.float64)
            wd = (cnt.max() / np.maximum(cnt, 1.0)) ** float(np.clip(dc.domain_balance, 0.0, 1.0))
            extra += wd[ds.domain[lab]] - 1.0
        if extra.any():
            k = np.floor(extra).astype(np.int64) + (rng.rand(len(lab)) < (extra - np.floor(extra))).astype(np.int64)
            act = np.concatenate([act, np.repeat(lab, np.maximum(k, 0))]).astype(np.int64)
        pool = np.concatenate([act, aux]).astype(np.int64)
        need = self.steps_per_epoch * self.cfg.data.batch_size
        reps = int(math.ceil(need / max(len(pool), 1)))
        return np.concatenate([rng.permutation(pool) for _ in range(reps)])[:need]

    def _train_epoch(self, epoch: int) -> Dict[str, float]:
        self.model.train()
        loader = self.data.train_loader(self._epoch_pool(epoch))
        loader.batch_sampler.set_epoch(epoch)
        meter, dev = Meter(), self.device
        keys = ("total", "ctc", "aux_ctc", "style_reg", "style_cls", "leak", "cons", "decor", "vq", "mask", "frag")
        bar = tqdm(loader, desc=f"epoch {epoch + 1}/{self.tc.epochs}", leave=False)
        teacher = self.ema.module.eval()
        for batch in bar:
            batch = to_device(batch, dev)
            lr = self._lr(self.global_step)
            for g in self.opt.param_groups:
                g["lr"] = lr
            x_s, x_w, wd, st_t, st_m, cl_ok = self.aug(batch["image"], batch["style"], batch["style_ok"], batch["width"], batch["min_width"])
            aux, vql = self._aux_scale(self.global_step), self._vq_lambda(self.global_step)
            t_lp, t_out, frag = None, None, None
            use_frag = self.fc.enabled and self.frag_model is not None and self.tc.loss.frag > 0
            if (self.tc.loss.consistency > 0 or use_frag) and aux > 0:
                with torch.no_grad(), torch.autocast(device_type=dev.type, dtype=self.amp_dtype, enabled=self.use_amp):
                    t_out = teacher(x_w, wd)
                t_lp = t_out["log_probs"].float()
            if use_frag and t_out is not None:
                frag = self._frag_batch(t_out, x_s, wd, (~batch["labeled"]) & (batch["weight"] <= 0), batch["stratum"])
            with torch.autocast(device_type=dev.type, dtype=self.amp_dtype, enabled=self.use_amp):
                out = self.model(x_s, wd, vq_lambda=vql, mask_prob=self.cfg.model.mask_prob * aux)
            L = compute_losses(out, batch, self.tc.loss, st_t, st_m, cl_ok, True, aux_scale=aux, vq_scale=vql, teacher_lp=t_lp, leak=self.leak, frag=frag)
            if not torch.isfinite(L["total"]):
                self.skipped += 1
                self.opt.zero_grad(set_to_none=True)
                continue
            self.opt.zero_grad(set_to_none=True)
            self.scaler.scale(L["total"]).backward()
            self.scaler.unscale_(self.opt)
            gn = torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.tc.grad_clip)
            self.scaler.step(self.opt)
            self.scaler.update()
            self.ema.update(self.model, self.global_step)
            self.global_step += 1

            B = batch["image"].size(0)
            vals = torch.stack([L[k].detach().float() for k in keys] + [gn.detach().float(), out["ppl_low"].float(), out["ppl_high"].float()]).cpu().tolist()
            for k, v in zip(keys, vals[:len(keys)]):
                meter.add(k, v, B)
            lc_ = self.tc.loss
            meter.add("task_total", vals[1] + lc_.style_reg * vals[3] + lc_.style_cls * vals[4], B)
            meter.add("grad_norm", vals[len(keys)], 1)
            meter.add("ppl_low", vals[len(keys) + 1], 1)
            meter.add("ppl_high", vals[len(keys) + 2], 1)
            for k, v in self.leak.last.items():
                meter.add(k, v, 1)
            if frag is not None:
                meter.add("frag_trust", frag["trusted_frames"] / max(frag["frames"], 1), 1)
                meter.add("frag_char_trust", frag["n_trusted_chars"] / max(frag["n_chars"], 1), 1)
            ps, lab = L["_ctc_ps"], batch["labeled"]
            if bool(lab.any()):
                meter.add("ctc_labeled", float(ps[lab].mean()), int(lab.sum()))
            pl = (~lab) & (batch["weight"] > 0)
            if bool(pl.any()):
                meter.add("ctc_pseudo", float(ps[pl].mean()), int(pl.sum()))
                self.bank.update_loss(batch["index"][pl].cpu().tolist(), ps[pl].cpu().tolist())
            bar.set_postfix(loss=f"{meter.mean('total'):.3f}", ctc=f"{meter.mean('ctc'):.3f}", lr=f"{lr:.1e}")
        return meter.means()

    # ------------------------------------------------------------------ evaluation
    @torch.no_grad()
    def evaluate(self, model: nn.Module, ds: HandwritingDataset, indices: Optional[Sequence[int]] = None, noise: bool = False) -> Dict[str, Any]:
        model.eval()
        loader = self.data.eval_loader(ds, indices)
        n = len(ds)
        preds: List[Optional[str]] = [None] * n
        conf = np.full(n, np.nan)
        ed_ps, ref_ps = np.full(n, np.nan), np.full(n, np.nan)
        meter, ed_w, ref_w = Meter(), 0, 0
        s_abs, s_cnt = np.zeros(NUM_DESC), np.zeros(NUM_DESC)
        dev, codec = self.device, self.data.codec
        for batch in loader:
            batch = to_device(batch, dev)
            x = apply_perturbation(eval_noise, batch["image"], batch["width"]) if noise else batch["image"]
            with torch.autocast(device_type=dev.type, dtype=self.amp_dtype, enabled=self.use_amp):
                out = model(x, batch["width"])
            B = x.size(0)
            sm = batch["style_ok"].float()
            L = compute_losses(out, batch, self.tc.loss, batch["style"], sm, torch.ones(B, dtype=torch.bool, device=dev), False)
            for k in ("total", "ctc", "style_reg", "style_cls"):
                meter.add(k, float(L[k]), B)
            err = ((out["style_reg"] - batch["style"]).abs() * sm).cpu().numpy()
            s_abs += err.sum(0)
            s_cnt += sm.cpu().numpy().sum(0)
            valid_cl = batch["cluster"] >= 0
            if bool(valid_cl.any()):
                meter.add("style_acc", float((out["style_cls"].argmax(1) == batch["cluster"])[valid_cl].float().mean()), int(valid_cl.sum()))
            lp, lens = out["log_probs"].cpu().numpy(), out["out_lens"].cpu().tolist()
            for b in range(B):
                ids, cf, _ = greedy_decode(lp[b, :lens[b]])
                text, i = codec.decode_ids(ids), int(batch["index"][b])
                ref = batch["oracle"][b]
                preds[i], conf[i] = text, seq_confidence(cf)
                ed_ps[i], ref_ps[i] = edit_distance(ref, text), len(ref)
                ed_w, ref_w = ed_w + edit_distance(ref.split(), text.split()), ref_w + len(ref.split())
        res = dict(meter.means())
        done = np.isfinite(ed_ps)
        res.update({"cer": float(ed_ps[done].sum() / max(ref_ps[done].sum(), 1)), "wer": ed_w / max(ref_w, 1),
                    "style_mae": float(s_abs.sum() / max(s_cnt.sum(), 1.0)), "style_mae_desc": s_abs / np.maximum(s_cnt, 1.0),
                    "preds": preds, "conf": conf, "ed_ps": ed_ps, "ref_ps": ref_ps, "cer_ps": ed_ps / np.maximum(ref_ps, 1)})
        return res

    # ------------------------------------------------------------------ legacy pseudo-labelling (line level)
    @torch.no_grad()
    def _teacher_read(self, idxs: np.ndarray, keep_lp: bool = False, desc: str = "teacher") -> Dict[str, Any]:
        """The EMA teacher reads the given train lines (clean view + the perturbed views) and describes every reading with the label-free
        reliability features."""
        pc, dev, codec = self.pc, self.device, self.data.codec
        teacher = self.ema.module.eval()
        views = reliability_views(pc)
        n_gate = len(pc.cons_views) if pc.consistency else 0
        r: Dict[str, Any] = {"idx": [], "text": [], "conf": [], "cons": [], "X": [], "lp": {}}
        for batch in tqdm(self.data.eval_loader(self.data.train_ds, idxs), desc=desc, leave=False):
            batch = to_device(batch, dev)
            with torch.autocast(device_type=dev.type, dtype=self.amp_dtype, enabled=self.use_amp):
                out = teacher(batch["image"], batch["width"])
            lp, lens = out["log_probs"].cpu().numpy(), out["out_lens"].cpu().tolist()
            B = lp.shape[0]
            reads = []
            for b in range(B):
                ids, cf, fr = greedy_decode(lp[b, :lens[b]])
                reads.append((ids, cf, fr, codec.decode_ids(ids)))
            cons_v = np.zeros((B, len(views)))
            for vi, v in enumerate(views):
                xv = apply_perturbation(PERTURBATIONS[v], batch["image"], batch["width"])
                with torch.autocast(device_type=dev.type, dtype=self.amp_dtype, enabled=self.use_amp):
                    ov = teacher(xv, batch["width"])
                lpv, lnv = ov["log_probs"].cpu().numpy(), ov["out_lens"].cpu().tolist()
                for b in range(B):
                    cons_v[b, vi] = cer(reads[b][3], codec.decode_ids(greedy_decode(lpv[b, :lnv[b]])[0]))
            for b in range(B):
                ids, cf, fr, text = reads[b]
                r["idx"].append(int(batch["index"][b]))
                r["text"].append(text)
                r["conf"].append(seq_confidence(cf))
                r["cons"].append(float(cons_v[b, :n_gate].mean()) if n_gate else float("nan"))
                r["X"].append(reliability_features(lp[b, :lens[b]], ids, cf, fr, cons_v[b]))
                if keep_lp:
                    r["lp"][int(batch["index"][b])] = lp[b, :lens[b]].copy()
        r["X"] = np.array(r["X"]) if r["X"] else np.zeros((0, len(RELIABILITY_FEATURES)))
        return r

    def _save_source_calibration(self, X: np.ndarray, ed: np.ndarray, rl: np.ndarray, epoch: int) -> None:
        """checkpoints/source_calibration.npz: the SOURCE side of the transfer-calibrated risk control of the zero-shot adapter."""
        try:
            os.makedirs(self.cfg.ckpt_dir, exist_ok=True)
            np.savez(os.path.join(self.cfg.ckpt_dir, "source_calibration.npz"), X=np.asarray(X, np.float64), ed=np.asarray(ed, np.float64),
                     rl=np.asarray(rl, np.float64), epoch=int(epoch), chars=self.data.codec.chars, features=np.array(RELIABILITY_FEATURES))
        except Exception as e:
            LOG.warning("could not write source_calibration.npz (%s)", e)

    def _ref_cov(self) -> Optional[float]:
        return float(np.median(self.cov_hist)) if self.cov_hist else None

    @torch.no_grad()
    def refresh_pseudo_labels(self, epoch: int) -> Dict[str, float]:
        """[RC-PL v2] risk-controlled line-level pseudo-labels with SHADOW calibration (the v9 method, kept as a baseline)."""
        pc, bank, ds = self.pc, self.bank, self.data.train_ds
        bank.begin_round()
        use_llm = bool(pc.llm_refine and self.llm.can_score())
        rng = np.random.RandomState(self.cfg.run.seed + 17 * epoch)
        rule: Optional[SelectionRule] = None
        info: Dict[str, float] = {}
        cal_idx = self.data.calib_idx
        rc = None
        if len(cal_idx) >= pc.min_calib:
            rc = self._teacher_read(cal_idx, keep_lp=use_llm and self.shadow, desc="calibration lines")
            ref = [ds.reference(i) for i in rc["idx"]]
            ed = np.array([edit_distance(a, b) for a, b in zip(ref, rc["text"])], np.float64)
            rl = np.array([len(a) for a in ref], np.float64)
            rule = SelectionRule.fit(rc["X"], ed, rl, pc, seed=self.cfg.run.seed + epoch, prev_cov=self.prev_cov, ref_cov=self._ref_cov())
            if pc.save_source_calibration:
                self._save_source_calibration(rc["X"], ed, rl, epoch)
            info = {"rc_" + k: v for k, v in rule.info.items()}
            info["rc_theta"] = float(rule.theta if rule.theta is not None else float("nan"))
            info["rc_theta_llm"] = float(rule.theta_llm if rule.theta_llm is not None else float("nan"))
            self.prev_cov = float(rule.info["calib_coverage"])
            if rule.info["calib_cov_raw"] * rule.info["calib_n"] >= pc.min_selected_calib:
                self.cov_hist = (self.cov_hist + [self.prev_cov])[-int(pc.cov_ref_window):]
        else:
            LOG.warning("only %d calibration lines (< min_calib=%d): FALLBACK to the sequence-confidence threshold %.2f -- expect noisy pseudo-labels",
                        len(cal_idx), pc.min_calib, pc.fallback_tau_conf)
        theta = None if rule is None else (rule.theta if rule.theta is not None else float("-inf"))
        theta_llm = None if rule is None else (rule.theta_llm if rule.theta_llm is not None else float("-inf"))
        use_llm = use_llm and rule is not None and np.isfinite(theta_llm)
        reservoir: List[Tuple[int, np.ndarray]] = []
        n_border = 0
        un = self._teacher_read(self.data.unlabeled_idx, keep_lp=use_llm, desc="pseudo-labelling")
        risk = rule.predict(un["X"]) if rule is not None else np.full(len(un["idx"]), np.nan)
        bank.update_predictions(un["idx"], un["text"], un["conf"], un["cons"], risk)
        pools = [(un, risk)]
        if self.shadow and rc is not None and rule is not None:
            bank.update_predictions(rc["idx"], rc["text"], rc["conf"], rc["cons"], rule.oof)
            pools.append((rc, rule.oof))
        if use_llm:
            for src, rk in pools:
                for k, i in enumerate(src["idx"]):
                    if i not in src["lp"]:
                        continue
                    gate = (not pc.consistency) or (np.isfinite(src["cons"][k]) and src["cons"][k] <= pc.tau_cons)
                    if gate and theta < rk[k] <= theta_llm:
                        n_border += 1
                        if len(reservoir) < pc.llm_budget:
                            reservoir.append((i, src["lp"][i]))
                        else:
                            j = rng.randint(0, n_border)
                            if j < pc.llm_budget:
                                reservoir[j] = (i, src["lp"][i])
        if use_llm and reservoir:
            from .decoding import ctc_prefix_beam_search
            self._llm_rescue([(i, ctc_prefix_beam_search(l, pc.llm_beam)) for i, l in reservoir], n_border)
        bank.select(pc, theta, theta_llm)
        ds.set_pseudo(bank.train_text, bank.weight, include_calib=self.shadow)
        d = bank.diagnostics()
        d.update(info)
        d["pl_fallback"] = float(rule is None)
        LOG.info("pseudo-labels (%s): selected=%d (LLM-rescued %d) coverage=%.2f | selected CER vs hidden GT=%.4f exact=%.3f | all-unlabeled CER=%.4f",
                 "risk-controlled" if rule is not None else "FALLBACK", int(d.get("pl_selected", 0)), int(d.get("pl_llm_selected", 0)),
                 d.get("pl_coverage", 0), d.get("pl_sel_cer", float("nan")), d.get("pl_sel_exact", float("nan")), d.get("pl_all_cer", float("nan")))
        return d

    def _llm_rescue(self, border: List[Tuple[int, list]], n_border_total: int) -> None:
        pc, codec, bank, ds = self.pc, self.data.codec, self.bank, self.data.train_ds
        raw, det = [], []
        for i, nb in border:
            for ids, _lp in nb:
                t = codec.decode_ids(ids)
                raw.append(t), det.append(detokenize(ds.display(i, t)) if ds.detok(i) else ds.display(i, t))
        lm_lp, n_tok = self.llm.score_texts(det)
        pos, changed, agreed = 0, 0, 0
        for i, nb in border:
            K = len(nb)
            fused = fuse_scores(np.array([lp for _, lp in nb]), lm_lp[pos:pos + K], n_tok[pos:pos + K], pc.llm_beta, pc.llm_rho)
            best = int(np.argmax(fused))
            lm_best, ntk_best = lm_lp[pos + best], max(n_tok[pos + best], 1)
            top_t, best_t = raw[pos], raw[pos + best]
            pos += K
            if lm_best / ntk_best < pc.llm_min_tok_lp or not best_t:
                continue
            if best == 0:
                bank.llm_ok[i], bank.llm_text[i] = True, top_t
                agreed += 1
            elif edit_distance(best_t, top_t) / max(len(top_t), 1) <= pc.llm_max_edit_ratio:
                bank.llm_ok[i], bank.llm_text[i] = True, best_t
                changed += 1
        LOG.info("LLM rescoring: %d of %d borderline lines -> %d confirmed, %d corrected", len(border), n_border_total, agreed, changed)

    # ------------------------------------------------------------------ [FRAG-RC]
    def _frag_set_state(self, st: Optional[Dict[str, Any]]) -> None:
        self.frag_state = st
        self.frag_model = FragModel.from_state(st["model"]) if st else None

    def _teacher_pair(self, x_w: torch.Tensor, x_s: torch.Tensor, wd: torch.Tensor):
        dev, teacher = self.device, self.ema.module.eval()
        with torch.no_grad(), torch.autocast(device_type=dev.type, dtype=self.amp_dtype, enabled=self.use_amp):
            return teacher(x_w, wd), teacher(x_s, wd)

    @torch.no_grad()
    def _frag_collect(self, idxs: np.ndarray, ref_fn, passes: int, seed: int) -> Dict[str, np.ndarray]:
        """Evidence (+ correctness against `ref_fn(line)`) of every emitted character of the given train lines, as the training loop will see
        them: teacher on a weak and a strong augmentation of the same pixels."""
        ds, dev, codec = self.data.train_ds, self.device, self.data.codec
        X, Y, LN, ST = [], [], [], []
        for p in range(passes):
            with torch.random.fork_rng(devices=[dev] if dev.type == "cuda" else []):
                torch.manual_seed(seed + 101 * p)
                for batch in self.data.eval_loader(ds, idxs):
                    batch = to_device(batch, dev)
                    x_s, x_w, wd, _a, _b, _c = self.aug(batch["image"], batch["style"], batch["style_ok"], batch["width"], batch["min_width"])
                    o_w, o_s = self._teacher_pair(x_w, x_s, wd)
                    lp_w, aux_w = o_w["log_probs"].float().cpu().numpy(), o_w["aux_log_probs"].float().cpu().numpy()
                    lp_s, lens = o_s["log_probs"].float().cpu().numpy(), o_w["out_lens"].cpu().numpy()
                    for b in range(lp_w.shape[0]):
                        i, L = int(batch["index"][b]), int(lens[b])
                        ids, st, en, Fm = char_evidence(lp_w[b, :L], lp_s[b, :L], aux_w[b, :L])
                        if not len(ids):
                            continue
                        y = char_wrong_labels(ref_fn(i), codec.decode_ids(ids))
                        X.append(Fm), Y.append(y), LN.append(np.full(len(ids), i)), ST.append(np.full(len(ids), int(ds.stratum[i])))
        if not X:
            return {"X": np.zeros((0, len(FRAG_FEATURES))), "y": np.zeros(0), "line": np.zeros(0, np.int64), "strat": np.zeros(0, np.int64)}
        return {"X": np.concatenate(X), "y": np.concatenate(Y).astype(np.float64), "line": np.concatenate(LN), "strat": np.concatenate(ST)}

    def refresh_frag(self, epoch: int) -> Dict[str, float]:
        """Re-calibrate the character-level trust rule on the held-out calibration lines: evidence of every emitted character (true text used
        ONLY to mark characters wrong) -> cross-fitted model -> one threshold per cell -> diagnostics on a sample of UNLABELED lines."""
        fc, ds, S = self.fc, self.data.train_ds, self.data.n_strata
        cal_idx = self.data.calib_idx
        info: Dict[str, float] = {}
        if len(cal_idx) < 20:
            LOG.warning("[FRAG-RC] only %d calibration lines: no trust rule (set cfg.data.calib_fraction > 0)", len(cal_idx))
            return info
        cal = self._frag_collect(cal_idx, ds.reference, fc.calib_passes, self.cfg.run.seed + epoch)
        X, y, ln, stt = cal["X"], cal["y"], cal["line"], cal["strat"]
        if len(y) < 2 * fc.min_chars:
            LOG.warning("[FRAG-RC] %d calibration characters (< %d): no trust rule yet", len(y), 2 * fc.min_chars)
            return info
        model, theta, oof, info = fit_trust_rule(X, y, ln, stt, S, fc.target_err, fc.risk_z, fc.min_chars, fc.folds, self.cfg.run.seed + 7 * epoch)
        self._frag_set_state({"model": model.state(), "theta": [float(t) for t in theta], "alpha": fc.target_err, "epoch": int(epoch), "n_strata": S,
                              "features": list(FRAG_FEATURES)})
        info["frag_auroc_risk"] = auroc(oof, y.astype(bool))
        info["frag_auroc_conf"] = auroc(-X[:, 0], y.astype(bool))
        info["frag_cal_chars"] = float(len(y))
        info["frag_cal_char_err"] = float(y.mean())
        errs = np.array([info[f"frag_cal_err_all_s{s}"] for s in range(S)])
        if np.isfinite(errs).all() and errs.mean() > 0 and fc.stratum_emphasis > 0:
            self.stratum_w = np.clip((errs / errs.mean()) ** fc.stratum_emphasis, 1.0, 3.0)
        un = self.data.unlabeled_idx
        if len(un):                                            # diagnostics on unlabeled lines (hidden text = diagnostics only)
            rng = np.random.RandomState(self.cfg.run.seed + 7 * epoch)
            sub = np.sort(rng.choice(un, min(fc.diag_lines, len(un)), replace=False))
            u = self._frag_collect(sub, lambda i: ds.oracle[i], 1, self.cfg.run.seed + 5 * epoch)
            if len(u["y"]):
                rk = model.risk(u["X"])
                keep = rk <= np.array([cell_theta(theta, c) for c in u["strat"]])
                info["frag_unl_cov"] = float(keep.mean())
                info["frag_unl_err_sel"] = float(u["y"][keep].mean()) if keep.any() else float("nan")
                info["frag_unl_err_all"] = float(u["y"].mean())
                for s in range(S):
                    ms = (u["strat"] == s) & keep
                    info[f"frag_unl_err_sel_s{s}"] = float(u["y"][ms].mean()) if ms.any() else float("nan")
                    info[f"frag_unl_cov_s{s}"] = float(keep[u["strat"] == s].mean()) if (u["strat"] == s).any() else float("nan")
        LOG.info("[FRAG-RC] epoch %d | %d calibration chars (error %.3f), AUROC risk %.3f vs peak prob %.3f | per cell: coverage %s, error of trusted "
                 "chars %s (target %.3f) | UNLABELED sample (hidden text): trusted %.2f of chars at error %.4f (all chars %.4f) | emphasis %s", epoch, len(y),
                 y.mean(), info["frag_auroc_risk"], info["frag_auroc_conf"], [round(info[f"frag_cal_cov_s{s}"], 2) for s in range(S)],
                 [round(info[f"frag_cal_err_sel_s{s}"], 4) for s in range(S)], fc.target_err, info.get("frag_unl_cov", float("nan")),
                 info.get("frag_unl_err_sel", float("nan")), info.get("frag_unl_err_all", float("nan")), np.round(self.stratum_w, 2).tolist())
        return info

    def _frag_batch(self, t_out: Dict[str, Any], x_s: torch.Tensor, wd: torch.Tensor, rows: torch.Tensor, stratum: torch.Tensor) -> Optional[Dict[str, Any]]:
        """Trust masks of the unlabeled rows of a batch (teacher: weak view from `t_out`, strong view read here, inter-CTC head)."""
        dev, fc = self.device, self.fc
        ridx = torch.nonzero(rows).flatten()
        if ridx.numel() == 0 or self.frag_model is None:
            return None
        teacher = self.ema.module.eval()
        with torch.no_grad(), torch.autocast(device_type=dev.type, dtype=self.amp_dtype, enabled=self.use_amp):
            s_out = teacher(x_s[ridx], wd[ridx])
        lp_w, aux_w = t_out["log_probs"].float().cpu().numpy(), t_out["aux_log_probs"].float().cpu().numpy()
        lp_s = np.zeros_like(lp_w)
        ls = s_out["log_probs"].float().cpu().numpy()
        lp_s[ridx.cpu().numpy(), :ls.shape[1]] = ls[:, :lp_s.shape[1]]
        lens = t_out["out_lens"].cpu().numpy()
        trust, n_ch, n_tr = frag_trust_batch(lp_w, aux_w, lp_s, lens, stratum.cpu().numpy(), rows.cpu().numpy(), self.frag_model,
                                             self.frag_state["theta"], fc.guard_frames)
        valid_rows = int(sum(lens[b] for b in ridx.cpu().numpy()))
        return {"trust": torch.from_numpy(trust).to(dev), "rows": rows, "tau": fc.sharpen, "anchor": fc.anchor_weight, "n_chars": n_ch,
                "n_trusted_chars": n_tr, "trusted_frames": int(trust.sum()), "frames": valid_rows}

    # ------------------------------------------------------------------ checkpoints
    def _compat(self) -> Dict[str, Any]:
        c = self.cfg
        return json_safe({"units": self.data.codec.units, "model": c.model.__dict__, "img_h": c.data.img_height, "n_clusters": self.data.n_clusters,
                          "labeled_fraction": c.data.labeled_fraction, "calib_fraction": c.data.calib_fraction,
                          "shadow_calibration": c.train.pseudo.shadow_calibration, "seed": c.run.seed, "n_train": len(self.data.train_ds),
                          "batch_size": c.data.batch_size, "frag_enabled": c.train.frag.enabled, "n_strata": self.data.n_strata,
                          "val_gate_fraction": c.data.val_gate_fraction, "val_block": c.data.val_block, "mixline_prob": c.train.mixline_prob,
                          "labeled_lines_sha1": self.data.provenance.get("labeled_lines_sha1"),
                          "calibration_lines_sha1": self.data.provenance.get("calibration_lines_sha1"), "source": c.data.source,
                          "sources": list(c.data.sources), "init_from": c.train.init_from})

    def _data_meta(self) -> Dict[str, Any]:
        """Everything inference needs about the data that produced the weights."""
        return {"chars": self.data.codec.chars, "units": list(self.data.codec.units), "style_mean": self.data.style_mean, "style_std": self.data.style_std,
                "n_clusters": self.data.n_clusters, "n_strata": int(self.data.n_strata),
                "strata_cuts": {str(k): np.asarray(v).tolist() for k, v in self.data.strata_cuts.items()}, "domains": self.data.domain_names}

    def _state(self, completed_epochs: int) -> Dict[str, Any]:
        rng = {"torch": torch.get_rng_state(), "numpy": np.random.get_state(), "py": random.getstate()}
        if torch.cuda.is_available():
            rng["cuda"] = torch.cuda.get_rng_state_all()
        return {"epoch": completed_epochs, "global_step": self.global_step, "model": self.model.state_dict(), "ema": self.ema.module.state_dict(),
                "opt": self.opt.state_dict(), "scaler": self.scaler.state_dict(), "history": self.history, "best_cer": self.best_cer,
                "best_from": self.best_from, "best_epoch": self.best_epoch, "ref_cer": self.ref_cer, "since_best": self.since_best,
                "stopped_at": self.stopped_at, "cycles": self.cycles, "frag_state": self.frag_state, "stratum_w": np.asarray(self.stratum_w),
                "bank": self.bank.state_dict(), "prev_cov": self.prev_cov, "cov_hist": list(self.cov_hist), "prev_val_preds": self.prev_val_preds,
                "prev_val_cer": self.prev_val_cer, "cfg": self.cfg.to_dict(), "compat": self._compat(), "rng": rng, **self._data_meta()}

    def save_checkpoint(self, completed_epochs: int) -> None:
        d = self.cfg.ckpt_dir
        st = self._state(completed_epochs)
        atomic_save(st, os.path.join(d, "last.pt"))
        if completed_epochs % self.tc.save_every == 0:
            atomic_save(st, os.path.join(d, f"epoch_{completed_epochs:03d}.pt"))
            old = sorted(glob.glob(os.path.join(d, "epoch_*.pt")))
            for p in old[:-self.tc.keep_last] if self.tc.keep_last > 0 else []:
                os.remove(p)

    def _save_best(self, completed_epochs: int, which: str, cer_val: float) -> None:
        src = self.ema.module if which == "ema" else self.model
        atomic_save({"weights": src.state_dict(), "from": which, "epoch": completed_epochs, "val_cer": cer_val, "cfg": self.cfg.to_dict(),
                     "frag_state": self.frag_state, "compat": self._compat(), **self._data_meta()}, self.best_path)

    def find_latest_checkpoint(self) -> Optional[str]:
        d = self.cfg.ckpt_dir
        cands = [os.path.join(d, "last.pt")] + sorted(glob.glob(os.path.join(d, "epoch_*.pt")), reverse=True)
        return next((p for p in cands if os.path.exists(p)), None)

    def _try_resume(self) -> int:
        """Resume from the newest readable checkpoint.  A checkpoint made with a different model / data / seed / labeled split is REFUSED."""
        if not self.tc.resume:
            return 0
        d, mine = self.cfg.ckpt_dir, self._compat()
        cands = [os.path.join(d, "last.pt")] + sorted(glob.glob(os.path.join(d, "epoch_*.pt")), reverse=True)
        for p in cands:
            if not os.path.exists(p):
                continue
            try:
                st = torch_load(p, map_location="cpu")
            except Exception as e:
                LOG.warning("checkpoint %s is unreadable (%s) - trying an older one", p, e)
                continue
            theirs = st.get("compat")
            diff = [k for k in mine if (theirs or {}).get(k) != mine[k]] if theirs else ["<no compatibility record>"]
            if diff:
                raise RuntimeError(f"checkpoint {p} was produced by a different setup (differs in: {diff}).  Delete {d}, change cfg.run.name or set "
                                   f"cfg.train.resume=False.")
            self.model.load_state_dict(st["model"])
            self.ema.module.load_state_dict(st["ema"])
            self.opt.load_state_dict(st["opt"])
            self.scaler.load_state_dict(st["scaler"])
            self.history, self.best_cer, self.best_from = st["history"], st["best_cer"], st["best_from"]
            self.best_epoch, self.ref_cer = st.get("best_epoch", 0), st.get("ref_cer", float("inf"))
            self.since_best, self.stopped_at = st.get("since_best", 0), st.get("stopped_at")
            self.global_step, self.prev_val_preds = st["global_step"], st.get("prev_val_preds")
            self.prev_val_cer = st.get("prev_val_cer")
            self.bank.load_state_dict(st["bank"])
            self._frag_set_state(st.get("frag_state"))
            if st.get("stratum_w") is not None:
                self.stratum_w = np.asarray(st["stratum_w"], dtype=np.float64)
            self.prev_cov = st.get("prev_cov")
            self.cov_hist = list(st.get("cov_hist", []))
            if self.bank.sel.any():
                self.data.train_ds.set_pseudo(self.bank.train_text, self.bank.weight, self.shadow)
            torch.set_rng_state(st["rng"]["torch"].cpu())
            np.random.set_state(st["rng"]["numpy"])
            random.setstate(st["rng"]["py"])
            if "cuda" in st["rng"] and torch.cuda.is_available():
                try:
                    torch.cuda.set_rng_state_all(st["rng"]["cuda"])
                except Exception:
                    pass
            if st.get("cycles"):
                self.cycles = st["cycles"]
            old_total = int(self.cycles[-1]["end"])
            if old_total != self.total_steps:
                self._extend_schedule(old_total)
                if self.stopped_at is not None:
                    self.stopped_at, self.since_best = None, 0
            LOG.info("RESUMED from %s (completed epochs: %d, global step %d, best val_sel CER %.4f at epoch %d, lr now %.2e)", p, st["epoch"],
                     self.global_step, self.best_cer, self.best_epoch, self._lr(self.global_step))
            return int(st["epoch"])
        return 0

    # ------------------------------------------------------------------ main loop
    def _record(self, epoch: int, rec: Dict[str, float]) -> None:
        H = self.history
        H["epoch"].append(epoch)
        n = len(H["epoch"])
        for k, v in rec.items():
            if k not in H:
                H[k] = [float("nan")] * (n - 1)
            H[k].append(float(v))
        for k in H:
            if len(H[k]) < n:
                H[k].append(float("nan"))

    @staticmethod
    def _sub_cer(e: Dict[str, Any], ix) -> float:
        return float(np.nansum(e["ed_ps"][ix]) / max(np.nansum(e["ref_ps"][ix]), 1))

    def fit(self) -> None:
        cfg, tc, pc, data = self.cfg, self.tc, self.pc, self.data
        self.start_epoch = self._try_resume()
        if self.stopped_at is not None:
            LOG.info("Training had stopped early at epoch %d (no val_sel-CER gain for %d epochs); raise cfg.train.epochs to continue.", self.stopped_at, tc.patience)
            return
        if self.start_epoch >= tc.epochs:
            LOG.info("Training already finished (%d epochs).", self.start_epoch)
            return
        LOG.info("Training: %d epochs x %d steps | params %.2fM | device %s | amp=%s (%s) | labeled %d / calibration %d / unlabeled %d | corpora %s",
                 tc.epochs, self.steps_per_epoch, sum(p.numel() for p in self.model.parameters()) / 1e6, self.device, self.use_amp,
                 str(self.amp_dtype).replace("torch.", ""), len(data.labeled_idx), len(data.calib_idx), len(data.unlabeled_idx), data.domain_names)
        n_unl = len(data.unlabeled_idx)
        nan = float("nan")
        for epoch in range(self.start_epoch, tc.epochs):
            t0, rec = time.time(), {}
            if pc.enabled and n_unl > 0 and epoch >= pc.warmup_epochs and (epoch - pc.warmup_epochs) % pc.refresh_every == 0:
                rec.update(self.refresh_pseudo_labels(epoch))
            if self.fc.enabled and n_unl > 0 and len(data.calib_idx) and epoch >= self.fc.warmup_epochs and (epoch - self.fc.warmup_epochs) % self.fc.refresh_every == 0:
                rec.update(self.refresh_frag(epoch))
            tr = self._train_epoch(epoch)
            if "total" not in tr:
                LOG.warning("epoch %d: every step was skipped for a non-finite loss (%d skipped so far)", epoch + 1, self.skipped)
            ev = self.evaluate(self.model, data.val_ds)
            ev_ema = self.evaluate(self.ema.module, data.val_ds)
            sel_i, gate_i = data.val_sel_idx, data.val_gate_idx
            sel_st, sel_em = self._sub_cer(ev, sel_i), self._sub_cer(ev_ema, sel_i)
            ev_tr = self.evaluate(self.model, data.train_ds, data.train_eval_idx)
            if self.prev_val_preds is not None:               # inertia metrics (Hsu et al.)
                rec["val_stability"] = float(np.mean([a == b for a, b in zip(ev["preds"], self.prev_val_preds)]))
                rec["val_nfr"] = float(np.nanmean(ev["cer_ps"] > self.prev_val_cer + 1e-9))
            self.prev_val_preds, self.prev_val_cer = list(ev["preds"]), ev["cer_ps"].copy()
            sub = list(range(min(tc.noise_eval_samples, len(data.val_ds))))
            ev_n = self.evaluate(self.model, data.val_ds, sub, noise=True)
            rec["val_noise_cons"] = 1.0 - float(np.mean([cer(ev["preds"][i], ev_n["preds"][i]) for i in sub]))
            rec["val_noise_gap"] = ev_n["cer"] - self._sub_cer(ev, sub)
            g = tr.get
            rec.update({"lr": self._lr(self.global_step), "lr_rel": self._lr(self.global_step) / self.tc.lr, "vq_lambda": self._vq_lambda(self.global_step),
                        "aux_scale": self._aux_scale(self.global_step), "val_sel_cer": sel_st, "val_ema_sel_cer": sel_em,
                        "val_gate_cer": self._sub_cer(ev, gate_i), "val_ema_gate_cer": self._sub_cer(ev_ema, gate_i),
                        "train_frag": g("frag", nan), "train_frag_trust": g("frag_trust", nan), "train_frag_char_trust": g("frag_char_trust", nan),
                        "train_total": g("total", nan), "train_task_total": g("task_total", nan), "train_ctc": g("ctc", nan),
                        "train_style_reg": g("style_reg", nan), "train_ppl_low": g("ppl_low", nan), "train_ppl_high": g("ppl_high", nan),
                        "train_leak": g("leak", nan), "train_cons": g("cons", nan), "train_decor": g("decor", nan),
                        "train_ctc_labeled": g("ctc_labeled", nan), "train_ctc_pseudo": g("ctc_pseudo", nan), "train_mask": g("mask", nan),
                        "grad_norm": g("grad_norm", nan), "val_total": ev["total"], "val_ctc": ev["ctc"], "val_cer": ev["cer"], "val_wer": ev["wer"],
                        "val_style_mae": ev["style_mae"], "val_style_acc": ev.get("style_acc", nan), "val_ema_total": ev_ema["total"],
                        "val_ema_cer": ev_ema["cer"], "val_ema_wer": ev_ema["wer"], "train_eval_total": ev_tr["total"], "train_eval_ctc": ev_tr["ctc"],
                        "train_eval_cer": ev_tr["cer"], "train_eval_wer": ev_tr["wer"], "train_eval_style_mae": ev_tr["style_mae"],
                        "train_eval_style_acc": ev_tr.get("style_acc", nan)})
            if len(data.specs) > 1:                           # per-corpus validation CER (same checkpoint; informational)
                for d_, name in enumerate(data.domain_names):
                    m_ = np.flatnonzero(data.val_ds.domain == d_)
                    if len(m_):
                        rec[f"val_cer_{name}"] = self._sub_cer(ev, m_)
            for j, n_ in enumerate(STYLE_NAMES):
                rec[f"val_style_mae_{n_}"] = float(ev["style_mae_desc"][j])
                if f"leak_r2_{j}" in tr:
                    rec[f"leak_r2_{n_}"] = tr[f"leak_r2_{j}"]
            self._record(epoch, rec)

            which = "ema" if sel_em <= sel_st else "student"
            cur = min(sel_em, sel_st)
            if cur < self.best_cer:
                self.best_cer, self.best_from, self.best_epoch = cur, which, epoch + 1
                self._save_best(epoch + 1, which, cur)
            if cur < self.ref_cer - tc.min_delta:
                self.ref_cer, self.since_best = cur, 0
            else:
                self.since_best += 1
            stop = tc.patience > 0 and (epoch + 1) >= tc.min_epochs and self.since_best >= tc.patience
            if stop:
                self.stopped_at = epoch + 1
            self.save_checkpoint(epoch + 1)
            with open(os.path.join(cfg.curve_dir, "history.json"), "w") as f:
                json.dump(json_safe(self.history), f)
            last = epoch + 1 == tc.epochs or stop
            plot_training_curves(self.history, cfg.curve_dir, individual=last or (epoch + 1) % tc.save_every == 0)
            LOG.info("epoch %3d/%d | %.0fs | train loss %.3f | val CER %.4f (EMA %.4f) [sel %.4f / gate %.4f] WER %.4f | train-subset CER %.4f | style MAE %.3f "
                     "acc %.3f | VQ ppl %.0f/%.0f | lr %.1e | best(val_sel) %.4f (%s, ep %d)%s", epoch + 1, tc.epochs, time.time() - t0, g("total", nan),
                     ev["cer"], ev_ema["cer"], sel_st, rec["val_gate_cer"], ev["wer"], ev_tr["cer"], ev["style_mae"], ev.get("style_acc", nan),
                     g("ppl_low", nan), g("ppl_high", nan), self._lr(self.global_step), self.best_cer, self.best_from, self.best_epoch,
                     f" | skipped steps {self.skipped}" if self.skipped else "")
            if stop:
                LOG.info("EARLY STOP at epoch %d: no val_sel-CER gain >= %.4f for %d epochs (best %.4f at epoch %d).", epoch + 1, tc.min_delta, tc.patience,
                         self.best_cer, self.best_epoch)
                break
        LOG.info("Training done. Best val_sel CER %.4f (%s, epoch %d). Curves: %s", self.best_cer, self.best_from, self.best_epoch, cfg.curve_dir)

In [ ]:
%%writefile inklingua/render.py
"""Closed-loop style-matched rendering of the translation ([CL-RENDER]): the rendering is MEASURED with the same descriptor code that
produced the style targets and the five controls (font size, stroke, word gap, letter spacing, shear) are corrected from the residuals."""
from __future__ import annotations

import math
import os
from typing import Any, Dict, List, Optional, Tuple

import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont

from .data import font_paths, load_font
from .style import compute_style_descriptors

_FONT_PICK_CACHE: Dict[str, Tuple[Optional[str], bool]] = {}
_SCRIPT_FONT_HINTS = ("notosans", "noto", "cjk", "droid", "freesans", "lohit", "mangal", "wqy", "unifont", "arial", "dejavu", "liberation")
_DESC_SCALE = np.array([5.0, 0.01, 0.03, 0.03, 0.05])          # one unit of mismatch: slant deg, stroke / H, density, gap / H, body ratio
_DESC_DEFAULT = (0.0, 0.05, 0.1, 0.12, 0.5)


def _glyph_bitmap(font, ch: str) -> np.ndarray:
    im = Image.new("L", (96, 96), 0)
    ImageDraw.Draw(im).text((8, 8), ch, font=font, fill=255)
    return np.asarray(im)


def _font_covers(font, text: str) -> bool:
    notdef = _glyph_bitmap(font, "\U0010ffff")
    for ch in set(text):
        if ch.isspace():
            continue
        g = _glyph_bitmap(font, ch)
        if g.max() == 0 or np.array_equal(g, notdef):
            return False
    return True


def pick_font_path(text: str) -> Tuple[Optional[str], bool]:
    """A font that covers all characters of `text` (bundled DejaVu first, then installed system fonts).  Returns (path, covered)."""
    key = "".join(sorted(set(text)))
    if key in _FONT_PICK_CACHE:
        return _FONT_PICK_CACHE[key]
    cands = list(font_paths())
    try:
        from matplotlib import font_manager
        sysf = sorted(font_manager.findSystemFonts(fontext="ttf") + font_manager.findSystemFonts(fontext="otf"),
                      key=lambda p: next((i for i, h in enumerate(_SCRIPT_FONT_HINTS) if h in os.path.basename(p).lower()), len(_SCRIPT_FONT_HINTS)))
        cands += [p for p in sysf if p not in cands][:120]
    except Exception:
        pass
    res: Tuple[Optional[str], bool] = (cands[0] if cands else None, False)
    for p in cands:
        try:
            if _font_covers(ImageFont.truetype(p, 32), text):
                res = (p, True)
                break
        except Exception:
            continue
    _FONT_PICK_CACHE[key] = res
    return res


def _mean_stroke_px(mask: np.ndarray) -> float:
    m = (mask > 127).astype(np.uint8)
    if m.sum() < 10:
        return 0.0
    dt = cv2.distanceTransform(m, cv2.DIST_L2, 3)
    ridge = (dt > 0) & (dt >= cv2.dilate(dt, np.ones((3, 3), np.uint8)) - 1e-6)
    return float(2.0 * dt[ridge].mean()) if ridge.any() else 0.0


def _text_ls(d, xy, s: str, font) -> None:
    """Draw `s` with its baseline at xy[1] (anchor 'ls'); bitmap fallback fonts cannot anchor and are placed by their top."""
    try:
        d.text(xy, s, font=font, fill=255, anchor="ls")
    except Exception:
        d.text((xy[0], xy[1] - int(0.8 * getattr(font, "size", 16))), s, font=font, fill=255)


def _baseline_y(font, Hs: int) -> int:
    """Row of the common baseline: the x-height band is centred on the canvas (every glyph on ONE baseline, so full stops stay down and
    descenders stay visible)."""
    try:
        xh = -font.getbbox("x", anchor="ls")[1]
    except Exception:
        xh = int(0.5 * getattr(font, "size", 32))
    return int(Hs // 2 + xh // 2)


def render_styled_text_open_loop(text: str, desc: np.ndarray, H: int = 64) -> Tuple[np.ndarray, bool]:
    """The v3 renderer (sets the controls once from the predicted descriptors and never looks at the result); kept as the comparison."""
    d_ = np.asarray(desc, np.float64)
    slant, sw, _dens, gap, body = [float(v) if np.isfinite(v) else dflt for v, dflt in zip(d_, _DESC_DEFAULT)]
    path, covered = pick_font_path(text)
    SC = 2
    Hs = H * SC
    size = int(np.clip(body * Hs / 0.38, 24, 140))
    font = load_font(path, size)
    gap_px = int(max(0.28 * size, gap * Hs))
    tmp = ImageDraw.Draw(Image.new("L", (8, 8)))
    words = text.split(" ") or [""]
    boxes = [tmp.textbbox((0, 0), w, font=font) for w in words]
    Wtot = sum(b[2] - b[0] for b in boxes) + gap_px * (len(words) - 1) + 4 * size
    canvas = Image.new("L", (int(Wtot), Hs), 0)
    d, x = ImageDraw.Draw(canvas), 2 * size
    base = _baseline_y(font, Hs)
    for w, b in zip(words, boxes):
        _text_ls(d, (x - b[0], base), w, font)
        x += (b[2] - b[0]) + gap_px
    arr = np.asarray(canvas, np.uint8)
    delta = sw * Hs - _mean_stroke_px(arr)
    k = int(min(round(abs(delta)), 6 if delta > 0 else 3))
    if k >= 1:
        ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k + 1, k + 1))
        arr = cv2.dilate(arr, ker) if delta > 0 else cv2.erode(arr, ker)
    canvas = Image.fromarray(arr)
    t = math.tan(math.radians(slant))
    canvas = canvas.transform(canvas.size, Image.AFFINE, (1, t, -t * Hs / 2, 0, 1, 0), resample=Image.BILINEAR)
    arr = np.asarray(canvas, np.float32) / 255.0
    cols = np.flatnonzero(arr.max(0) > 0.1)
    if cols.size:
        arr = arr[:, max(0, cols[0] - 8):cols[-1] + 9]
    return cv2.resize(arr, (max(8, arr.shape[1] // SC), H), interpolation=cv2.INTER_AREA), covered


def _draw_line(text: str, path: Optional[str], size: int, track: float, gap_px: int, dil: int, slant: float, Hs: int) -> np.ndarray:
    font = load_font(path, size)
    words = text.split(" ") or [""]
    widths = [sum(font.getlength(ch) + track for ch in w) for w in words]
    Wtot = int(sum(widths) + gap_px * (len(words) - 1) + 4 * size)
    canvas = Image.new("L", (max(Wtot, 16), Hs), 0)
    d, x = ImageDraw.Draw(canvas), 2.0 * size
    base = _baseline_y(font, Hs)
    for w, wd in zip(words, widths):
        for ch in w:
            if not ch.isspace():
                _text_ls(d, (x, base), ch, font)
            x += font.getlength(ch) + track
        x += gap_px
    arr = np.asarray(canvas, np.uint8)
    if dil != 0:
        ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (abs(dil) + 1, abs(dil) + 1))
        arr = cv2.dilate(arr, ker) if dil > 0 else cv2.erode(arr, ker)
    t = math.tan(math.radians(slant))
    img = Image.fromarray(arr).transform(Image.fromarray(arr).size, Image.AFFINE, (1, t, -t * Hs / 2, 0, 1, 0), resample=Image.BILINEAR)
    out = np.asarray(img, np.float32) / 255.0
    cols = np.flatnonzero(out.max(0) > 0.1)
    return out[:, max(0, cols[0] - 8 * (Hs // 64)):cols[-1] + 9 * (Hs // 64)] if cols.size else out


def style_mismatch(measured: np.ndarray, target: np.ndarray) -> float:
    """Mean scaled absolute error over the descriptors that are finite in both vectors (inf if none)."""
    m, t = np.asarray(measured, np.float64), np.asarray(target, np.float64)
    ok = np.isfinite(m) & np.isfinite(t)
    return float((np.abs(m - t)[ok] / _DESC_SCALE[ok]).mean()) if ok.any() else float("inf")


def render_styled_text_info(text: str, desc: np.ndarray, H: int = 64, iters: int = 6) -> Dict[str, Any]:
    """CLOSED-LOOP rendering: measure -> correct (font size <- body ratio, dilation <- stroke, word gap <- gap ratio, letter spacing <- density,
    shear <- slant) -> re-measure, for `iters` rounds; the iterate with the smallest mismatch is returned (never worse than round 0)."""
    tgt = np.array([float(v) if np.isfinite(v) else np.nan for v in np.asarray(desc, np.float64)])
    use = np.where(np.isfinite(tgt), tgt, np.array(_DESC_DEFAULT))
    path, covered = pick_font_path(text)
    SC = 2
    Hs = H * SC
    size = int(np.clip(use[4] * Hs / 0.38, 24, 140))
    p = {"size": float(size), "track": 0.0, "gap": float(max(0.28 * size, use[3] * Hs)), "dil": 0, "slant": float(use[0])}
    best: Optional[Dict[str, Any]] = None
    hist: List[float] = []

    def render(p_: Dict[str, float]) -> Tuple[np.ndarray, np.ndarray]:
        arr = _draw_line(text, path, int(round(p_["size"])), p_["track"], int(round(p_["gap"])), int(p_["dil"]), p_["slant"], Hs)
        img = cv2.resize(arr, (max(8, arr.shape[1] // SC), H), interpolation=cv2.INTER_AREA)
        return img, compute_style_descriptors(np.pad(img, ((0, 0), (8, 8))))

    for r in range(max(int(iters), 0) + 1):
        img, m = render(p)
        err = style_mismatch(m, tgt)
        hist.append(err)
        if best is None or err < best["mismatch"]:
            best = {"image": img, "measured": m, "mismatch": err, "params": dict(p)}
        if r == iters:
            break
        if np.isfinite(m[4]) and np.isfinite(tgt[4]) and m[4] > 0:
            p["size"] = float(np.clip(p["size"] * (tgt[4] / m[4]) ** 0.8, 24, 140))
        if np.isfinite(m[1]) and np.isfinite(tgt[1]):
            p["dil"] = int(np.clip(p["dil"] + round((tgt[1] - m[1]) * Hs), -3, 6))
        if np.isfinite(m[3]) and np.isfinite(tgt[3]) and m[3] > 0:
            p["gap"] = float(np.clip(p["gap"] * (tgt[3] / m[3]) ** 0.9, 0.2 * p["size"], 3.0 * p["size"]))
        if np.isfinite(m[2]) and np.isfinite(tgt[2]) and m[2] > 0 and tgt[2] > 0:
            p["track"] = float(np.clip(p["track"] + 0.25 * p["size"] * math.log(m[2] / tgt[2]), -0.1 * p["size"], 0.6 * p["size"]))
        if np.isfinite(m[0]) and np.isfinite(tgt[0]):
            p["slant"] = float(np.clip(p["slant"] + (tgt[0] - m[0]), -45.0, 45.0))
    assert best is not None
    return {"image": best["image"], "covered": covered, "measured": best["measured"], "target": tgt, "mismatch": best["mismatch"],
            "mismatch_round0": hist[0], "rounds": hist, "params": best["params"]}


def render_styled_text(text: str, desc: np.ndarray, H: int = 64, iters: int = 6) -> Tuple[np.ndarray, bool]:
    info = render_styled_text_info(text, desc, H, iters)
    return info["image"], info["covered"]

In [ ]:
%%writefile inklingua/explain.py
"""
Explanation helpers used by the Inferencer: per-sample panels (input, CTC posterior, character confidence, saliency, integrated gradients,
occlusion, per-character attribution, attention rollout, VQ codes, style attention), calibration / selective-prediction statistics,
codebook statistics and the batch-norm helpers of the test-time adaptation.
"""
from __future__ import annotations

import os
from typing import Any, Dict, List, Tuple

import cv2
import matplotlib
import numpy as np
import torch
import torch.nn as nn

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

from .model import MaskedBatchNorm2d

SPACE = "␣"
ALIGN_CAP = 1000          # max #lines used for the O(n*m) character alignments (confusions, calibration)


def chlabel(c: str) -> str:
    return SPACE if c == " " else c


# ---------------------------------------------------------------- calibration / selective prediction
def adaptive_calibration(conf: np.ndarray, correct: np.ndarray, bins: int = 15) -> Dict[str, Any]:
    """Equal-MASS bins (a CTC model puts >90 % of its characters into the top fixed-width bin).  ECE = sum_b (n_b / N) |acc_b - conf_b|."""
    conf, correct = np.asarray(conf, np.float64), np.asarray(correct, np.float64)
    if len(conf) < bins:
        return {"ece": float("nan"), "nll": float("nan"), "brier": float("nan"), "mean_conf": float("nan"), "accuracy": float("nan"), "points": [], "n": int(len(conf))}
    o = np.argsort(conf, kind="stable")
    c, a = conf[o], correct[o]
    pts, ece = [], 0.0
    for p in np.array_split(np.arange(len(c)), bins):
        pts.append((float(c[p].mean()), float(a[p].mean()), len(p)))
        ece += len(p) / len(c) * abs(pts[-1][1] - pts[-1][0])
    cc = np.clip(c, 1e-6, 1 - 1e-6)
    return {"ece": float(ece), "nll": float(-(a * np.log(cc) + (1 - a) * np.log(1 - cc)).mean()), "brier": float(((c - a) ** 2).mean()),
            "mean_conf": float(c.mean()), "accuracy": float(a.mean()), "points": pts, "n": int(len(c))}


def risk_coverage(score: np.ndarray, ed: np.ndarray, ref_len: np.ndarray) -> Dict[str, Any]:
    """Selective prediction: keep the lines with the smallest `score`; CER of the kept lines vs the fraction kept.  AURC = area (lower is better)."""
    n = len(score)
    o = np.argsort(score, kind="stable")
    cov = np.arange(1, n + 1) / n
    risk = np.cumsum(np.asarray(ed, np.float64)[o]) / np.maximum(np.cumsum(np.asarray(ref_len, np.float64)[o]), 1.0)
    aurc = float(((risk[1:] + risk[:-1]) * 0.5 * np.diff(cov)).sum()) if n > 1 else float(risk[-1])
    return {"coverage": cov, "risk": risk, "aurc": aurc}


def coverage_at_risk(rc: Dict[str, Any], target: float) -> float:
    ok = np.flatnonzero(rc["risk"] <= target)
    return float(rc["coverage"][ok[-1]]) if ok.size else 0.0


def risk_at_coverage(rc: Dict[str, Any], cov: float) -> float:
    k = int(np.clip(np.searchsorted(rc["coverage"], cov), 0, len(rc["risk"]) - 1))
    return float(rc["risk"][k])


# ---------------------------------------------------------------- codebook statistics
def _entropy(p: np.ndarray) -> float:
    p = np.asarray(p, np.float64)
    p = p[p > 0]
    return float(-(p * np.log(p)).sum())


def _purity_nmi(c: np.ndarray) -> Tuple[float, float]:
    n = c.sum()
    pur = float(c.max(1).sum() / n)
    pk, pc = c.sum(1) / n, c.sum(0) / n
    mi = _entropy(pc) + _entropy(pk) - _entropy(c / n)
    return pur, float(2 * mi / max(_entropy(pk) + _entropy(pc), 1e-12))


def code_char_stats(cc: np.ndarray, min_count: int = 20, n_null: int = 20, seed: int = 0) -> Dict[str, float]:
    """How well does a discrete code predict the character emitted on its frame?  Purity / NMI of the codes with >= min_count emissions,
    compared with a multinomial null that keeps the emission counts; *_gain = (x - null) / (1 - null)."""
    cc = np.asarray(cc, np.float64)
    tot = cc.sum(1)
    keep = tot >= min_count
    nan = float("nan")
    out = {"component_code_min_count": float(min_count), "component_codes_with_support": float(keep.sum()),
           "component_code_emission_share_with_support": float(tot[keep].sum() / max(tot.sum(), 1))}
    if keep.sum() < 2 or tot[keep].sum() <= 0:
        out.update(component_code_purity=nan, component_code_purity_chance=nan, component_code_purity_gain=nan, component_code_nmi=nan,
                   component_code_nmi_chance=nan, component_code_nmi_gain=nan)
        return out
    sub = cc[keep]
    pur, nmi = _purity_nmi(sub)
    marg, rng = sub.sum(0) / sub.sum(), np.random.RandomState(seed)
    nulls = [_purity_nmi(np.stack([rng.multinomial(int(t), marg) for t in sub.sum(1)]).astype(np.float64)) for _ in range(n_null)]
    pur0, nmi0 = np.mean(nulls, 0)
    out.update(component_code_purity=pur, component_code_purity_chance=float(pur0), component_code_purity_gain=float((pur - pur0) / max(1 - pur0, 1e-9)),
               component_code_nmi=nmi, component_code_nmi_chance=float(nmi0), component_code_nmi_gain=float((nmi - nmi0) / max(1 - nmi0, 1e-9)))
    return out


def pca2(X: np.ndarray) -> np.ndarray:
    Xc = X - X.mean(0)
    _, _, Vt = np.linalg.svd(Xc, full_matrices=False)
    return Xc @ Vt[:2].T


def contrast_rollout(rows: np.ndarray) -> Tuple[np.ndarray, float]:
    """Sink-removed attention rollout: subtract the component shared by all emissions (attention sinks) to keep what is specific to each
    character.  rows: (K, T), each row sums to 1.  Returns (positive residual, mean mass on the 3 most shared frames)."""
    rows = np.asarray(rows, np.float64)
    if rows.ndim != 2 or rows.shape[0] < 2:
        return rows.copy(), float("nan")
    shared = rows.mean(0)
    top = np.argsort(-shared)[:3]
    return np.maximum(rows - shared[None], 0.0), float(rows[:, top].sum(1).mean())


# ---------------------------------------------------------------- figures
def save_fig(fig, path: str, dpi: int = 120) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    fig.tight_layout()
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)


def _show_img(ax, ctx, alpha: float = 1.0) -> None:
    ax.imshow(1.0 - ctx["img"], cmap="gray", vmin=0, vmax=1, extent=[0, ctx["W"], ctx["H"], 0], aspect="auto", alpha=alpha)


def _smooth_norm(m: np.ndarray, sigma: float = 2.0, signed: bool = False) -> np.ndarray:
    m = cv2.GaussianBlur(m.astype(np.float32), (0, 0), sigma)
    s = np.percentile(np.abs(m), 99) + 1e-9
    return np.clip(m / s, -1, 1) if signed else np.clip(m / s, 0, 1)


def viz_input(ax, ctx):
    _show_img(ax, ctx)
    sp, st = ctx["style_pred"], ctx["style_true"]
    tag = f"[{ctx['tag']}] " if ctx.get("tag") else ""
    ax.set_title(f"{tag}GT  : {ctx['gt_display']}\nPRED: {ctx['pred_display']}   (CER {ctx['cer']:.3f}, conf {ctx['conf']:.2f})\n"
                 f"style pred/measured  slant {sp[0]:+.0f}/{st[0]:+.0f} deg | stroke {sp[1]:.3f}/{st[1]:.3f} | gap {sp[3]:.2f}/{st[3]:.2f} | "
                 f"body {sp[4]:.2f}/{st[4]:.2f}   (nan = not measurable)", fontsize=8, loc="left")
    ax.set_yticks([])


def viz_posterior(ax, ctx):
    P, units = ctx["probs"], ctx["units"]
    keep = [v for v in range(P.shape[1]) if P[:, v].max() >= 0.05]
    keep = [0] + sorted([v for v in keep if v != 0], key=lambda v: int(P[:, v].argmax()))
    ax.imshow(P[:, keep].T, aspect="auto", cmap="viridis", vmin=0, vmax=1, extent=[0, ctx["T"] * 4, len(keep), 0])
    ax.set_yticks(np.arange(len(keep)) + 0.5)
    ax.set_yticklabels(["blank"] + [chlabel(units[v - 1]) for v in keep[1:]], fontsize=6)
    ax.set_title("CTC posterior  p(char | frame)", fontsize=8, loc="left")


def viz_charconf(ax, ctx):
    H = ctx["H"]
    _show_img(ax, ctx, alpha=0.35)
    ops = {o[2]: o[0] for o in ctx["ops"] if o[2] is not None}
    cmap = plt.get_cmap("RdYlGn")
    for k, (f, c) in enumerate(zip(ctx["frames"], ctx["char_conf"])):
        bad = ops.get(k, "match") in ("sub", "ins")
        ax.bar(f * 4 + 2, -c * H, width=4, bottom=H, color=cmap(c), edgecolor="red" if bad else "none", linewidth=1.2, alpha=0.9)
        ax.text(f * 4 + 2, -0.03 * H, chlabel(ctx["pred_units"][k]), ha="center", va="bottom", fontsize=6, color="red" if bad else "black")
    ax.set_ylim(H, -0.35 * H)
    ax.set_xlim(0, ctx["W"])
    ax.set_yticks([])
    ax.set_title("per-character confidence (bar height/colour); red = wrong vs ground truth", fontsize=8, loc="left")


def _overlay(ax, ctx, m, title, cmap="inferno", signed=False):
    _show_img(ax, ctx, alpha=0.9)
    if signed:
        rgba = plt.get_cmap("seismic")((m + 1.0) / 2.0)
        rgba[..., 3] = np.clip(np.abs(m), 0, 1) ** 0.8 * 0.9
    else:
        rgba = plt.get_cmap(cmap)(np.clip(m, 0, 1))
        rgba[..., 3] = np.clip(m, 0, 1) ** 0.8 * 0.9
    ax.imshow(rgba, extent=[0, ctx["W"], ctx["H"], 0], aspect="auto")
    ax.set_yticks([])
    ax.set_title(title, fontsize=8, loc="left")


def viz_gradxinput(ax, ctx):
    _overlay(ax, ctx, _smooth_norm(ctx["gradxin"], 1.5), "Gradient x Input saliency (CTC loss of the prediction)")


def viz_ig(ax, ctx):
    _overlay(ax, ctx, _smooth_norm(ctx["ig"], 1.5, signed=True), "Integrated Gradients (red = supports prediction, blue = contradicts)", signed=True)


def viz_occlusion(ax, ctx):
    _show_img(ax, ctx, alpha=0.35)
    ax2 = ax.twinx()
    ax2.plot(ctx["occ_x"], ctx["occ_d"], color="crimson", lw=1.6)
    ax2.fill_between(ctx["occ_x"], 0, ctx["occ_d"], color="crimson", alpha=0.2)
    ax2.set_ylabel("loss increase", fontsize=7)
    ax.set_yticks([])
    ax.set_xlim(0, ctx["W"])
    ax.set_title("occlusion sensitivity (sliding window)", fontsize=8, loc="left")


def viz_char_attr(ax, ctx):
    A = ctx["char_attr"]
    if A.size == 0:
        ax.axis("off")
        return
    ax.imshow(A, aspect="auto", cmap="magma", extent=[0, ctx["W"], A.shape[0], 0], vmin=0, vmax=1)
    ax.set_yticks(np.arange(A.shape[0]) + 0.5)
    ax.set_yticklabels([chlabel(c) for c in ctx["pred_units"][:A.shape[0]]], fontsize=6)
    ax.set_title("per-character attribution: which columns drive each predicted character (Grad x Input)", fontsize=8, loc="left")


def viz_attention(ax, ctx):
    R = ctx["attn_roll"]
    if R is None or len(ctx["frames"]) == 0:
        ax.axis("off")
        return
    rows = np.stack([R[f] / max(R[f].sum(), 1e-9) for f in ctx["frames"]])
    res, sink = contrast_rollout(rows)
    for r, f in enumerate(ctx["frames"][:res.shape[0]]):
        res[r, f] = 0.0
    res = res / (res.max(1, keepdims=True) + 1e-12)
    ax.imshow(res, aspect="auto", cmap="hot", extent=[0, ctx["T"] * 4, res.shape[0], 0], vmin=0, vmax=1)
    ax.scatter([4 * f + 2 for f in ctx["frames"][:res.shape[0]]], np.arange(res.shape[0]) + 0.5, s=10, c="cyan", marker="o", linewidths=0)
    ax.set_yticks(np.arange(res.shape[0]) + 0.5)
    ax.set_yticklabels([chlabel(c) for c in ctx["pred_units"][:res.shape[0]]], fontsize=6)
    tail = "" if not np.isfinite(sink) else f"; sinks held {100 * sink:.0f}% of the raw mass"
    ax.set_title("attention rollout: context of each emission (cyan = emitting frame, shared sink removed" + tail + ")", fontsize=8, loc="left")


def viz_vq(ax, ctx):
    cm = plt.get_cmap("tab20")
    for r, (name, idx) in enumerate([("stroke codes", ctx["idx_low"]), ("component codes", ctx["idx_high"])]):
        ax.imshow(cm(idx % 20)[None], extent=[0, ctx["T"] * 4, 2 - r, 1 - r], aspect="auto")
        if ctx["T"] <= 110:
            for t, v in enumerate(idx):
                ax.text(t * 4 + 2, 1.5 - r, str(int(v)), ha="center", va="center", fontsize=4.5)
    ax.set_ylim(0, 2)
    ax.set_yticks([1.5, 0.5])
    ax.set_yticklabels(["stroke codes", "component codes"], fontsize=7)
    ax.set_xlim(0, ctx["T"] * 4)
    ax.set_title("discrete VQ codes per frame (colour = code id mod 20)", fontsize=8, loc="left")


def viz_style_attention(ax, ctx):
    """Which frames the style encoder attends to when it summarises the writer's style (uniform = 1/T)."""
    a = ctx.get("style_att")
    if a is None or len(a) == 0:
        ax.axis("off")
        return
    _show_img(ax, ctx, alpha=0.35)
    ax2 = ax.twinx()
    x = np.arange(len(a)) * 4 + 2
    ax2.plot(x, a * len(a), color="purple", lw=1.4)
    ax2.axhline(1.0, color="gray", ls=":", lw=1)
    ax2.set_ylabel("attention x T (1 = uniform)", fontsize=7)
    ax.set_yticks([])
    ax.set_xlim(0, ctx["W"])
    ax.set_title("style attention: which frames define the predicted style (dotted = uniform)", fontsize=8, loc="left")


SAMPLE_VIZ = [("input", viz_input, 1.2), ("ctc_posterior", viz_posterior, 2.0), ("char_confidence", viz_charconf, 1.4),
              ("grad_x_input", viz_gradxinput, 1.2), ("integrated_gradients", viz_ig, 1.2), ("occlusion", viz_occlusion, 1.2),
              ("char_attribution", viz_char_attr, 2.4), ("attention_rollout", viz_attention, 2.4), ("vq_codes", viz_vq, 1.0),
              ("style_attention", viz_style_attention, 1.2)]


# ---------------------------------------------------------------- batch-norm helpers (test-time adaptation)
def bn_layers(model: nn.Module) -> List[Tuple[str, MaskedBatchNorm2d]]:
    return [(n, m) for n, m in model.named_modules() if isinstance(m, MaskedBatchNorm2d)]


def snapshot_bn(model: nn.Module) -> Dict[str, Tuple[np.ndarray, np.ndarray]]:
    return {n: (m.running_mean.detach().cpu().numpy().astype(np.float64), m.running_var.detach().cpu().numpy().astype(np.float64)) for n, m in bn_layers(model)}


def load_bn(model: nn.Module, state: Dict[str, Tuple[np.ndarray, np.ndarray]]) -> None:
    with torch.no_grad():
        for n, m in bn_layers(model):
            mu, var = state[n]
            m.running_mean.copy_(torch.as_tensor(mu, dtype=m.running_mean.dtype, device=m.running_mean.device))
            m.running_var.copy_(torch.as_tensor(var, dtype=m.running_var.dtype, device=m.running_var.device))


def adapt_bn_state(src: Dict[str, Tuple[np.ndarray, np.ndarray]], acc: Dict[str, Dict[str, Any]], n_lines: int, tau: float) -> Dict[str, Tuple[np.ndarray, np.ndarray]]:
    """Source statistics shrunk toward the statistics of a group of unlabeled lines (AdaBN with lam = n / (n + tau)); the adapted variance
    is the variance of the two-component mixture."""
    lam = n_lines / (n_lines + tau) if (n_lines + tau) > 0 else 0.0
    out: Dict[str, Tuple[np.ndarray, np.ndarray]] = {}
    for n, (mu_s, v_s) in src.items():
        a = acc.get(n) or {}
        if lam <= 0.0 or a.get("n", 0.0) <= 0.0:
            out[n] = (mu_s, v_s)
            continue
        mu_c = a["s1"] / a["n"]
        v_c = np.maximum(a["s2"] / a["n"] - mu_c ** 2, 0.0)
        mu = lam * mu_c + (1.0 - lam) * mu_s
        out[n] = (mu, lam * (v_c + (mu_c - mu) ** 2) + (1.0 - lam) * (v_s + (mu_s - mu) ** 2))
    return out

In [ ]:
%%writefile inklingua/posthoc.py
"""
Post-hoc inference options, all label-free at test time, each ADOPTED only if the Bonferroni-adjusted 95 % form-clustered paired-bootstrap
interval of its gain on the `val_gate` lines excludes zero (the test lines only report):
  (0) [VIEW-ENS] posterior averaging over mild re-renderings of the same line,
  (1) [STYLE-ADABN] batch-norm statistics per handwriting-style cluster (transductive),
  (1b) [WSD] writer self-distillation: a few label-free fine-tuning steps per style cluster, guided by the FRAG-RC trust masks (transductive),
  (2) a one-parameter word-gap (space) bonus in the greedy decoder;
plus the structure of the errors (character classes, forms, style bins).
"""
from __future__ import annotations

import copy
import time
from collections import Counter, defaultdict
from typing import Any, Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

from .data import Collator
from .decoding import decode_with_space_bonus
from .explain import adapt_bn_state, bn_layers, load_bn, snapshot_bn
from .frag import frag_trust_batch
from .model import fragment_distillation
from .pseudo import PERTURBATIONS, apply_perturbation
from .style import STYLE_NAMES, assign_nearest, kmeans_np
from .text import CHAR_CLASSES, align_chars, char_class, edit_distance, paired_bootstrap_delta
from .utils import LOG, to_device


class PosthocMixin:
    # ------------------------------------------------------------------ frame posteriors with optional per-group BN statistics
    @torch.no_grad()
    def _lp_pass(self, ds, indices: Sequence[int], assign: Optional[Dict[int, int]] = None,
                 states: Optional[Dict[int, Dict[str, Tuple[np.ndarray, np.ndarray]]]] = None, view: Optional[str] = None) -> Dict[int, np.ndarray]:
        """Frame log-posteriors (valid frames only) of the given lines; with `assign` / `states` every line is read by the model that carries
        the batch-norm statistics of its group; the source statistics are always restored."""
        out: Dict[int, np.ndarray] = {}
        groups: Dict[int, List[int]] = {-1: [int(i) for i in indices]}
        if assign is not None:
            groups = defaultdict(list)
            for i in indices:
                groups[assign[int(i)]].append(int(i))
        src = snapshot_bn(self.model)
        try:
            for g, idxs in groups.items():
                if assign is not None and states is not None:
                    load_bn(self.model, states.get(g, src))
                for batch in self.data.eval_loader(ds, idxs):
                    batch = to_device(batch, self.device)
                    x = batch["image"] if view is None else apply_perturbation(PERTURBATIONS[view], batch["image"], batch["width"])
                    o = self.model(x, batch["width"])
                    lp, lens = o["log_probs"].cpu().numpy(), o["out_lens"].cpu().tolist()
                    for b in range(lp.shape[0]):
                        out[int(batch["index"][b])] = lp[b, :lens[b]].copy()
        finally:
            load_bn(self.model, src)
        return out

    def _lp_views(self, indices: Sequence[int], ds, views: Sequence[str], cache: Optional[Dict[str, Dict[int, np.ndarray]]] = None) -> Dict[int, np.ndarray]:
        """[VIEW-ENS] frame log-posteriors of the clean line averaged (in PROBABILITY space) with those of re-renderings of the same pixels."""
        def get(v: Optional[str]) -> Dict[int, np.ndarray]:
            if cache is None:
                return self._lp_pass(ds, indices, view=v)
            k = "clean" if v is None else v
            if k not in cache:
                cache[k] = self._lp_pass(ds, indices, view=v)
            return cache[k]
        acc = {i: lp.copy() for i, lp in get(None).items()}
        if not views:
            return acc
        for v in views:
            for i, lp in get(v).items():
                acc[i] = np.logaddexp(acc[i], lp)
        n = float(len(views) + 1)
        return {i: (lp - np.log(n)).astype(np.float32) for i, lp in acc.items()}

    def _tta_assign(self, ds, ids: Sequence[int], k: int) -> Dict[int, int]:
        """Group lines by handwriting style: k-means on the MEASURED descriptors (label-free)."""
        X = ds.style[np.asarray(list(ids), np.int64)].astype(np.float64)
        if len(X) < 2 * max(k, 1):
            return {int(i): 0 for i in ids}
        a = assign_nearest(X, kmeans_np(X, k, seed=0))
        return {int(i): int(c) for i, c in zip(ids, a)}

    @torch.no_grad()
    def _bn_collect(self, ds, assign: Dict[int, int]) -> Dict[int, Tuple[Dict[str, Dict[str, Any]], int]]:
        groups: Dict[int, List[int]] = defaultdict(list)
        for i, g in assign.items():
            groups[g].append(i)
        res: Dict[int, Tuple[Dict[str, Dict[str, Any]], int]] = {}
        layers = bn_layers(self.model)
        for g, idxs in groups.items():
            for _, m in layers:
                m.collect = {}
            try:
                for batch in self.data.eval_loader(ds, idxs):
                    batch = to_device(batch, self.device)
                    self.model(batch["image"], batch["width"])
                res[g] = ({n: m.collect for n, m in layers}, len(idxs))
            finally:
                for _, m in layers:
                    m.collect = None
        return res

    def _edits(self, lps: Dict[int, np.ndarray], ids: Sequence[int], refs: Dict[int, str], space_id: Optional[int], delta: float
               ) -> Tuple[np.ndarray, np.ndarray, List[str]]:
        codec = self.data.codec
        ed, rl, preds = [], [], []
        for i in ids:
            t = codec.decode_ids(decode_with_space_bonus(lps[int(i)].astype(np.float32), space_id, delta))
            preds.append(t)
            ed.append(edit_distance(refs[int(i)], t)), rl.append(len(refs[int(i)]))
        return np.array(ed, np.float64), np.maximum(np.array(rl, np.float64), 1.0), preds

    def _posthoc_block(self, ds, R: Dict[str, Any]) -> Dict[str, Any]:
        """Order: view ensemble, style-cluster AdaBN (not combined with the ensemble), writer self-distillation (replaces the previous readings
        if adopted), then the space bonus on top of whatever was adopted.  The final readings go to R["pred_posthoc"]."""
        ic, dm, codec = self.ic, self.data, self.data.codec
        if not ic.posthoc:
            return {}
        val_ids = [int(i) for i in dm.val_gate_idx]
        if len(val_ids) > ic.selective_val_lines:
            val_ids = self._random_subset(val_ids, ic.selective_val_lines, 6)
        test_ids = [int(i) for i in R["idx"]]
        blk = max(1, int(ic.bootstrap_block))
        gv, gt = np.asarray(val_ids) // blk, np.asarray(test_ids) // blk
        refs_v = {i: dm.val_ds.oracle[i] for i in val_ids}
        refs_t = {i: ds.oracle[i] for i in test_ids}
        space_id = codec.c2i.get(" ")
        res: Dict[str, Any] = {"validation_lines": len(val_ids), "test_lines": len(test_ids), "val_role": "val_gate",
                               "rule": "an option is adopted only if the Bonferroni-adjusted (over the grid searched) 95% form-clustered paired-bootstrap "
                                       "interval of its CER gain on the val_gate lines excludes zero; test lines only report"}
        lpv, lpt = self._lp_pass(dm.val_ds, val_ids), self._lp_pass(ds, test_ids)
        ev0, rv0, _ = self._edits(lpv, val_ids, refs_v, space_id, 0.0)
        et0, rt0, pt0 = self._edits(lpt, test_ids, refs_t, space_id, 0.0)
        res["val_CER_base"], res["test_CER_base"] = float(ev0.sum() / rv0.sum()), float(et0.sum() / rt0.sum())
        lpv_c, lpt_c, cur_pred = lpv, lpt, pt0
        ev_cur, et_cur = ev0, et0
        adopted: Dict[str, Any] = {"view_ensemble": [], "tta": False, "wsd": None, "space_bonus": 0.0}
        self._wsd_lp = {}
        self.run_extra = {"test_ids": test_ids, "edits_greedy": et0.copy(), "pred_greedy": list(pt0), "ref_len": rt0.copy()}

        # ---- (0) view ensemble ------------------------------------------------------------------------------------------------
        self._ens_views = ()
        if ic.view_ensemble and ic.view_sets:
            cache_v: Dict[str, Dict[int, np.ndarray]] = {"clean": lpv}
            cands: Dict[str, Any] = {}
            best_e: Tuple[Any, ...] = (float("inf"), None, None, None)
            for vs in ic.view_sets:
                lp_e = self._lp_views(val_ids, dm.val_ds, vs, cache_v)
                e_e, r_e, _ = self._edits(lp_e, val_ids, refs_v, space_id, 0.0)
                cands["+".join(vs)] = float(e_e.sum() / r_e.sum())
                if cands["+".join(vs)] < best_e[0]:
                    best_e = (cands["+".join(vs)], vs, e_e, lp_e)
            cache_v.clear()
            gate_e = paired_bootstrap_delta(best_e[2], ev0, rv0, ic.bootstrap, 7, groups=gv, n_tests=len(ic.view_sets))
            use_e = bool(gate_e["hi_adj"] < 0)
            lp_te = self._lp_views(test_ids, ds, best_e[1], {"clean": lpt})
            e_te, r_te, p_te = self._edits(lp_te, test_ids, refs_t, space_id, 0.0)
            res["view_ensemble"] = {"val_CER_by_view_set": cands, "chosen": list(best_e[1]), "val_delta_vs_base": gate_e, "adopted": use_e,
                                    "test_CER": float(e_te.sum() / r_te.sum()), "test_delta_vs_base": paired_bootstrap_delta(e_te, et0, rt0, ic.bootstrap, 8, groups=gt),
                                    "note": "clean line + re-renderings of the same pixels; frame posteriors averaged in probability space; label-free"}
            LOG.info("[VIEW-ENS] views %s: val CER %.4f -> %.4f (adjusted hi %.4f) -> %s | test CER %.4f -> %.4f", "+".join(best_e[1]), res["val_CER_base"],
                     best_e[0], gate_e["hi_adj"], "ADOPTED" if use_e else "not adopted", res["test_CER_base"], res["view_ensemble"]["test_CER"])
            if use_e:
                self._ens_views = tuple(best_e[1])
                adopted["view_ensemble"] = list(best_e[1])
                lpv_c, lpt_c, cur_pred, ev_cur, et_cur = best_e[3], lp_te, p_te, best_e[2], e_te
            del lp_te

        # ---- (1) style-cluster AdaBN ---------------------------------------------------------------------------------------------
        if ic.tta and bn_layers(self.model) and not self._ens_views:
            src = snapshot_bn(self.model)
            av, at = self._tta_assign(dm.val_ds, val_ids, ic.tta_clusters), self._tta_assign(ds, test_ids, ic.tta_clusters)
            accv, acct = self._bn_collect(dm.val_ds, av), self._bn_collect(ds, at)
            grid: Dict[str, Any] = {}
            best: Tuple[Any, ...] = (float("inf"), None, None, None)
            for tau in ic.tta_shrink:
                states = {g: adapt_bn_state(src, a, n, tau) if n >= ic.tta_min_lines else src for g, (a, n) in accv.items()}
                lp_a = self._lp_pass(dm.val_ds, val_ids, av, states)
                e_a, r_a, _ = self._edits(lp_a, val_ids, refs_v, space_id, 0.0)
                grid[str(tau)] = float(e_a.sum() / r_a.sum())
                if grid[str(tau)] < best[0]:
                    best = (grid[str(tau)], tau, e_a, lp_a)
            tau_b, e_b, lp_b = best[1], best[2], best[3]
            gate = paired_bootstrap_delta(e_b, ev0, rv0, ic.bootstrap, 1, groups=gv, n_tests=len(ic.tta_shrink))
            states_t = {g: adapt_bn_state(src, a, n, tau_b) if n >= ic.tta_min_lines else src for g, (a, n) in acct.items()}
            lp_t = self._lp_pass(ds, test_ids, at, states_t)
            e_t, r_t, p_t = self._edits(lp_t, test_ids, refs_t, space_id, 0.0)
            use = bool(gate["hi_adj"] < 0)
            res["tta"] = {"clusters": ic.tta_clusters, "val_CER_by_shrinkage_tau": grid, "chosen_tau": tau_b, "val_delta_vs_base": gate, "adopted": use,
                          "test_CER": float(e_t.sum() / r_t.sum()), "test_delta_vs_base": paired_bootstrap_delta(e_t, et0, rt0, ic.bootstrap, 2, groups=gt),
                          "test_lines_per_cluster": {str(g): n for g, (_, n) in acct.items()},
                          "note": "transductive and label-free: batch-norm statistics of each style cluster from the unlabeled test lines (AdaBN, shrunk)"}
            LOG.info("TTA (style-cluster AdaBN, tau=%s): val CER %.4f -> %.4f -> %s | test CER %.4f -> %.4f", tau_b, res["val_CER_base"], best[0],
                     "ADOPTED" if use else "not adopted", res["test_CER_base"], res["tta"]["test_CER"])
            if use:
                adopted["tta"] = True
                lpv_c, lpt_c, cur_pred, ev_cur, et_cur = lp_b, lp_t, p_t, e_b, e_t

        # ---- (1b) writer self-distillation ---------------------------------------------------------------------------------------
        if ic.wsd and getattr(self, "frag_model", None) is not None and getattr(self, "frag_state", None) is not None and ic.wsd_steps:
            k_t = max(2, int(ic.wsd_clusters))
            k_v = int(max(2, min(k_t, round(len(val_ids) / max(len(test_ids) / k_t, 1.0)))))
            av, at = self._tta_assign(dm.val_ds, val_ids, k_v), self._tta_assign(ds, test_ids, k_t)
            t_wsd = time.time()
            reads_v, info_v = self._wsd_reads(dm.val_ds, val_ids, av, ic.wsd_steps, lpv, seed=11)
            reads_t, info_t = self._wsd_reads(ds, test_ids, at, ic.wsd_steps, lpt, seed=13)
            cand, best_w = {}, (float("inf"), None, None)
            for s_ in sorted(int(s) for s in ic.wsd_steps):
                e_w, r_w, _ = self._edits(reads_v[s_], val_ids, refs_v, space_id, 0.0)
                cand[str(s_)] = float(e_w.sum() / r_w.sum())
                if cand[str(s_)] < best_w[0]:
                    best_w = (cand[str(s_)], s_, e_w)
            s_b = int(best_w[1])
            gate_w = paired_bootstrap_delta(best_w[2], ev_cur, rv0, ic.bootstrap, 21, groups=gv, n_tests=len(cand))
            use_w = bool(gate_w["hi_adj"] < 0)
            e_tw, r_tw, p_tw = self._edits(reads_t[s_b], test_ids, refs_t, space_id, 0.0)
            rep_w = paired_bootstrap_delta(e_tw, et_cur, rt0, ic.bootstrap, 22, groups=gt)

            def by_cluster(assign, ids, e_new, e_old, r_):
                out_ = {}
                for g_ in sorted(set(assign.values())):
                    m_ = np.array([assign[int(i)] == g_ for i in ids])
                    out_[str(g_)] = {"lines": int(m_.sum()), "CER_before": float(e_old[m_].sum() / max(r_[m_].sum(), 1)),
                                     "CER_after": float(e_new[m_].sum() / max(r_[m_].sum(), 1))}
                return out_

            res["wsd"] = {"val_CER_by_steps": cand, "chosen_steps": s_b, "val_delta_vs_before": gate_w, "adopted": use_w, "test_CER": float(e_tw.sum() / r_tw.sum()),
                          "test_delta_vs_before": rep_w, "clusters": {"val_gate": k_v, "test": k_t},
                          "val_by_cluster": by_cluster(av, val_ids, best_w[2], ev_cur, rv0), "test_by_cluster": by_cluster(at, test_ids, e_tw, et_cur, rt0),
                          "val_group_info": {str(g): v for g, v in info_v.items()}, "test_group_info": {str(g): v for g, v in info_t.items()},
                          "seconds": float(time.time() - t_wsd),
                          "note": "transductive and label-free: each cluster's copy of the model moves for a few steps on the unlabeled images of that cluster "
                                  "only, guided by the FRAG-RC trust masks; val_gate lines are adapted the same way before the gain is measured"}
            LOG.info("[WSD] steps %s: val_gate CER %.4f -> %.4f (adjusted hi %.4f) -> %s | test CER %.4f -> %.4f (%.0fs)", s_b, float(ev_cur.sum() / rv0.sum()),
                     best_w[0], gate_w["hi_adj"], "ADOPTED" if use_w else "not adopted", float(et_cur.sum() / rt0.sum()), res["wsd"]["test_CER"], res["wsd"]["seconds"])
            self.run_extra.update({"wsd_test_pred": list(p_tw), "wsd_test_edits": e_tw.copy(), "wsd_test_cluster": [int(at[int(i)]) for i in test_ids]})
            if use_w:
                adopted["wsd"] = s_b
                self._ens_views = ()
                self._wsd_lp = {id(dm.val_ds): reads_v[s_b], id(ds): reads_t[s_b]}
                lpv_c, lpt_c, cur_pred, ev_cur, et_cur = reads_v[s_b], reads_t[s_b], p_tw, best_w[2], e_tw
            del reads_v, reads_t
        elif ic.wsd:
            res["wsd"] = {"adopted": False, "skipped": "no FRAG-RC rule in the checkpoint (train with cfg.train.frag.enabled=True and calibration lines)"}

        # ---- (2) space bonus ------------------------------------------------------------------------------------------------------
        if space_id is not None and ic.space_bonus_grid:
            grid2: Dict[str, float] = {}
            best2 = (float("inf"), 0.0, ev_cur)
            for dlt in ic.space_bonus_grid:
                e_d, r_d, _ = self._edits(lpv_c, val_ids, refs_v, space_id, float(dlt))
                grid2[str(dlt)] = float(e_d.sum() / r_d.sum())
                if grid2[str(dlt)] < best2[0]:
                    best2 = (grid2[str(dlt)], float(dlt), e_d)
            gate2 = paired_bootstrap_delta(best2[2], ev_cur, rv0, ic.bootstrap, 3, groups=gv, n_tests=len(ic.space_bonus_grid))
            e_s, r_s, p_s = self._edits(lpt_c, test_ids, refs_t, space_id, best2[1])
            use2 = bool(best2[1] != 0.0 and gate2["hi_adj"] < 0)
            res["space_bonus"] = {"val_CER_by_bonus": grid2, "chosen": best2[1], "val_delta_vs_before": gate2, "adopted": use2,
                                  "test_CER": float(e_s.sum() / r_s.sum()), "test_delta_vs_before": paired_bootstrap_delta(e_s, et_cur, rt0, ic.bootstrap, 4, groups=gt),
                                  "test_space_count": {"reference": int(sum(refs_t[i].count(" ") for i in test_ids)), "before": int(sum(p.count(" ") for p in cur_pred)),
                                                       "with_bonus": int(sum(p.count(" ") for p in p_s))}}
            LOG.info("space bonus: val CER %.4f -> %.4f at +%.1f nats -> %s | test CER %.4f -> %.4f", float(ev_cur.sum() / rv0.sum()), best2[0], best2[1],
                     "ADOPTED" if use2 else "not adopted", float(et_cur.sum() / rt0.sum()), float(e_s.sum() / r_s.sum()))
            if use2:
                adopted["space_bonus"] = best2[1]
                cur_pred, et_cur = p_s, e_s
        res["adopted"] = adopted
        self.run_extra.update({"final_pred": list(cur_pred), "final_edits": et_cur.copy()})
        res["test_CER_final"] = float(et_cur.sum() / rt0.sum())
        res["test_delta_final_vs_base"] = paired_bootstrap_delta(et_cur, et0, rt0, ic.bootstrap, 5, groups=gt)
        R["pred_posthoc"] = {i: p for i, p in zip(test_ids, cur_pred)}
        return res

    # ------------------------------------------------------------------ [WSD]
    @torch.no_grad()
    def _read_with(self, model: nn.Module, ds, lines: Sequence[int]) -> Dict[int, np.ndarray]:
        res: Dict[int, np.ndarray] = {}
        model.eval()
        for batch in self.data.eval_loader(ds, [int(i) for i in lines]):
            batch = to_device(batch, self.device)
            o = model(batch["image"], batch["width"])
            lp, lens = o["log_probs"].cpu().numpy(), o["out_lens"].cpu().tolist()
            for b in range(lp.shape[0]):
                res[int(batch["index"][b])] = lp[b, :lens[b]].copy()
        return res

    def _wsd_reads(self, ds, ids: Sequence[int], assign: Dict[int, int], steps_list: Sequence[int], base_lp: Dict[int, np.ndarray], seed: int = 0
                   ) -> Tuple[Dict[int, Dict[int, np.ndarray]], Dict[int, Dict[str, float]]]:
        """For every style cluster a COPY of the model is fine-tuned for a few steps on the cluster's own UNLABELED images: a strong-view student
        is distilled from the frozen model's clean-view posteriors, the FRAG-RC trust masks decide which frames are sharpened.  Only the CTC head,
        the final LayerNorm, the output FiLM and the last encoder layer move.  Returns {steps: {line: log-posteriors}} and per-group diagnostics."""
        ic, dev, codec, fs = self.ic, self.device, self.data.codec, self.frag_state
        steps_list = sorted(int(s) for s in steps_list)
        out: Dict[int, Dict[int, np.ndarray]] = {s: dict(base_lp) for s in steps_list}
        info: Dict[int, Dict[str, float]] = {}
        groups: Dict[int, List[int]] = defaultdict(list)
        for i in ids:
            groups[assign[int(i)]].append(int(i))
        views = ("gauss noise", "blur", "thicker ink", "thinner ink", "shear +0.08", "shear -0.08", "low contrast")
        collate = Collator(codec)
        self.model.eval()
        for g, lines in sorted(groups.items()):
            if len(lines) < ic.wsd_min_lines:
                info[g] = {"n": len(lines), "adapted": 0.0}
                continue
            rng = np.random.RandomState(seed + 31 * int(g) + 7)
            stu = copy.deepcopy(self.model).eval()
            for p in stu.parameters():
                p.requires_grad_(False)
            params = []
            for m_ in [stu.ctc_head, stu.final_ln, stu.layers[-1]] + ([stu.film_out] if stu.film_out is not None else []):
                for p in m_.parameters():
                    p.requires_grad_(True)
                    params.append(p)
            opt = torch.optim.AdamW(params, lr=ic.wsd_lr, weight_decay=0.0)
            order, ptr, tf = list(rng.permutation(lines)), 0, []
            for step in range(1, steps_list[-1] + 1):
                if ptr + ic.wsd_batch > len(order):
                    order, ptr = list(rng.permutation(lines)), 0
                sel, ptr = order[ptr:ptr + ic.wsd_batch], ptr + ic.wsd_batch
                items = []
                for i in sel:
                    it = ds[int(i)]
                    it["text"], it["weight"], it["oracle"] = "", 0.0, ""          # the label never enters the batch
                    items.append(it)
                batch = to_device(collate(items), dev)
                x, w = batch["image"], batch["width"]
                v1, v2 = rng.choice(len(views), 2, replace=False)
                with torch.no_grad():
                    o_w = self.model(x, w)
                    o_2 = self.model(apply_perturbation(PERTURBATIONS[views[v2]], x, w), w)
                    B = x.size(0)
                    trust, _nc, _nt = frag_trust_batch(o_w["log_probs"].float().cpu().numpy(), o_w["aux_log_probs"].float().cpu().numpy(),
                                                       o_2["log_probs"].float().cpu().numpy(), o_w["out_lens"].cpu().numpy(), batch["stratum"].cpu().numpy(),
                                                       np.ones(B, bool), self.frag_model, fs["theta"], self.cfg.train.frag.guard_frames)
                o_s = stu(apply_perturbation(PERTURBATIONS[views[v1]], x, w), w)
                loss = fragment_distillation(o_s["log_probs"], o_w["log_probs"].float(), o_w["out_lens"], torch.from_numpy(trust).to(dev),
                                             torch.ones(B, dtype=torch.bool, device=dev), ic.wsd_tau, ic.wsd_anchor)
                opt.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                opt.step()
                tf.append(float(trust.sum() / max(float(sum(int(l) for l in o_w["out_lens"].cpu().numpy())), 1.0)))
                if step in steps_list:
                    for i, lp in self._read_with(stu, ds, lines).items():
                        out[step][i] = lp
            info[g] = {"n": len(lines), "adapted": 1.0, "trusted_frame_share": float(np.mean(tf)) if tf else float("nan")}
            del stu, opt
        return out, info

    # ------------------------------------------------------------------ structure of the errors
    def _error_structure_block(self, R: Dict[str, Any]) -> Dict[str, Any]:
        """Where do the errors come from?  Character classes (Unicode categories), form-level clustering and the measured style descriptors."""
        n = len(R["gt"])
        pos = self._random_subset(range(n), self.ic.error_analysis_lines, 5)
        ref_cnt, err_cnt, ins_cnt = Counter(), Counter(), Counter()
        sub_n = case_only = 0
        sp = {"deleted": 0, "inserted": 0, "substituted_away": 0}
        n_edits = 0
        for k in pos:
            gt, pr = R["gt"][k], R["pred"][k]
            for op, ri, hi in align_chars(gt, pr):
                if ri is not None:
                    ref_cnt[char_class(gt[ri])] += 1
                if op == "match":
                    continue
                n_edits += 1
                if op == "sub":
                    err_cnt[char_class(gt[ri])] += 1
                    sub_n += 1
                    case_only += int(gt[ri].lower() == pr[hi].lower() and gt[ri] != pr[hi])
                    sp["substituted_away"] += int(gt[ri] == " ")
                elif op == "del":
                    err_cnt[char_class(gt[ri])] += 1
                    sp["deleted"] += int(gt[ri] == " ")
                else:
                    ins_cnt[char_class(pr[hi])] += 1
                    sp["inserted"] += int(pr[hi] == " ")
        tot_ref = max(sum(ref_cnt.values()), 1)
        M: Dict[str, Any] = {"lines_analysed": len(pos),
                             "by_character_class": {c: {"share_of_reference_chars": ref_cnt[c] / tot_ref,
                                                        "error_rate_on_reference_chars": (err_cnt[c] / ref_cnt[c]) if ref_cnt[c] else float("nan"),
                                                        "inserted_chars": int(ins_cnt[c])} for c in CHAR_CLASSES},
                             "space_edits": {**sp, "share_of_all_edits": float(sum(sp.values()) / max(n_edits, 1))},
                             "case_only_share_of_substitutions": float(case_only / max(sub_n, 1))}
        order = np.argsort(np.asarray(R["idx"]))
        e_o, l_o = np.asarray(R["ed"], np.float64)[order], np.maximum(np.asarray(R["reflen"], np.float64)[order], 1.0)
        c_o = e_o / l_o
        blk = np.arange(len(c_o)) // 20
        bc = np.array([e_o[blk == b].sum() / l_o[blk == b].sum() for b in np.unique(blk)])
        blk30 = np.arange(len(c_o)) // 30
        ub = np.unique(blk30)
        grand = c_o.mean()
        ss_b = sum(((blk30 == b).sum()) * (c_o[blk30 == b].mean() - grand) ** 2 for b in ub)
        ss_t = float(((c_o - grand) ** 2).sum())
        M["form_structure"] = {
            "block_size": 20, "block_CER_quantiles_10_25_50_75_90_99": [float(v) for v in np.percentile(bc, [10, 25, 50, 75, 90, 99])],
            "worst_block_first_line_index": int(np.asarray(R["idx"])[order][int(np.argmax(bc)) * 20]), "worst_block_CER": float(bc.max()),
            "lag1_autocorrelation_of_line_CER": float(np.corrcoef(c_o[:-1], c_o[1:])[0, 1]) if len(c_o) > 2 and c_o.std() > 0 else float("nan"),
            "between_block_variance_share_30_line_blocks": float(ss_b / ss_t) if ss_t > 0 else float("nan"),
            "same_share_expected_without_structure": float((len(ub) - 1) / max(len(c_o) - 1, 1)),
            "note": "consecutive evaluation lines are stored form by form: a share far above the 'expected' value means errors cluster by form / writer"}
        sty = np.asarray(R["style_raw"], np.float64)
        ed, rl = np.asarray(R["ed"], np.float64), np.maximum(np.asarray(R["reflen"], np.float64), 1.0)
        bins: Dict[str, Any] = {}
        for j, name in enumerate(STYLE_NAMES):
            v = sty[:, j]
            ok = np.isfinite(v)
            if ok.sum() < 50:
                continue
            q = np.quantile(v[ok], np.linspace(0, 1, 6))
            rows = []
            for b in range(5):
                m = ok & (v >= q[b]) & ((v <= q[b + 1]) if b == 4 else (v < q[b + 1]))
                rows.append({"range": [float(q[b]), float(q[b + 1])], "n": int(m.sum()), "CER": float(ed[m].sum() / rl[m].sum()) if m.any() else float("nan")})
            rk_v, rk_c = np.argsort(np.argsort(v[ok])), np.argsort(np.argsort((ed / rl)[ok]))
            rho = float(np.corrcoef(rk_v, rk_c)[0, 1]) if rk_v.std() > 0 and rk_c.std() > 0 else float("nan")
            bins[name] = {"quintiles": rows, "spearman_with_line_CER": rho}
        M["cer_by_style_quintile"] = bins
        fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))
        cls_rate = [M["by_character_class"][c]["error_rate_on_reference_chars"] * 100 for c in CHAR_CLASSES]
        ax[0].bar(CHAR_CLASSES, np.nan_to_num(cls_rate), color="indianred")
        for x_, c in enumerate(CHAR_CLASSES):
            ax[0].text(x_, np.nan_to_num(cls_rate[x_]), f"{100 * M['by_character_class'][c]['share_of_reference_chars']:.1f}% of chars", ha="center", va="bottom", fontsize=7)
        ax[0].set_ylabel("error rate on reference characters (%)"), ax[0].set_title("which characters fail?", fontsize=9)
        ax[1].plot(bc * 100, lw=1.2), ax[1].axhline(100 * float(e_o.sum() / l_o.sum()), color="k", ls="--", lw=1)
        ax[1].set_xlabel("block of 20 consecutive lines (form order)"), ax[1].set_ylabel("block CER (%)")
        ax[1].set_title(f"errors cluster by form / writer (lag-1 corr {M['form_structure']['lag1_autocorrelation_of_line_CER']:.2f})", fontsize=9)
        for name, b in bins.items():
            ax[2].plot(range(1, 6), [r["CER"] * 100 for r in b["quintiles"]], "o-", label=f"{name} (rho {b['spearman_with_line_CER']:+.2f})")
        ax[2].set_xlabel("quintile of the measured descriptor (1 = lowest)"), ax[2].set_ylabel("CER (%)"), ax[2].legend(fontsize=7)
        ax[2].set_title("CER by handwriting style", fontsize=9)
        self._fig("03_error_structure/error_structure.png", "Where the errors come from: character classes, forms, handwriting style", fig)
        return M

In [ ]:
%%writefile inklingua/interpret.py
"""
The interpretability suite (v11).  Every analysis answers one question about the trained model, works on the evaluation split without any
extra label beyond the references it already has, and writes the numbers behind its figure (CSV / JSON) next to the figure:

  layer probes              where in the network does the character identity (and the writer's style) become linearly readable?
  uncertainty decomposition  which signal predicts a wrong line: the model's confidence, the posterior entropy, MC-dropout disagreement,
                             view disagreement or the FRAG-RC character risk?
  character counterfactuals  erase the ink of one character: which other characters of the reading change?  (dependency structure)
  style neighbours           what does the style space group?  nearest lines in the style embedding and their measured descriptors
  alignments                 the CTC alignment of every line (character, frames, pixels, confidence) as jsonl for downstream work
  long-line stress test      CER on concatenated pairs of test lines vs the same lines read alone (length generalisation)
  group tables               CER with form-clustered intervals per corpus, per hardness cell and per style cluster
"""
from __future__ import annotations

import csv
import json
import os
from collections import defaultdict
from typing import Any, Dict, List, Optional, Sequence

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

from .decoding import greedy_decode, greedy_runs, seq_confidence
from .frag import char_evidence
from .pseudo import PERTURBATIONS, apply_perturbation
from .riskcontrol import auroc
from .style import STYLE_NAMES, assign_nearest, kmeans_np
from .text import align_chars, bootstrap_cer_ci, cer, edit_distance
from .utils import LOG, json_safe, to_device


def _write_csv(path: str, header: Sequence[str], rows: Sequence[Sequence[Any]]) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(header)
        for r in rows:
            w.writerow(["" if (isinstance(v, float) and not np.isfinite(v)) or v is None else v for v in r])


def _ridge_classifier_cv(X: np.ndarray, y: np.ndarray, folds: int = 2, lam: float = 1.0, seed: int = 0) -> float:
    """Cross-validated accuracy of a closed-form least-squares classifier (ridge to one-hot targets): a cheap linear probe."""
    n = len(y)
    if n < 20:
        return float("nan")
    classes, yi = np.unique(y, return_inverse=True)
    if len(classes) < 2:
        return float("nan")
    fold = np.random.RandomState(seed).permutation(n) % folds
    Xs = (X - X.mean(0)) / (X.std(0) + 1e-6)
    correct = 0
    for f in range(folds):
        tr, te = fold != f, fold == f
        if te.sum() == 0 or tr.sum() < 5:
            continue
        A = np.c_[Xs[tr], np.ones(tr.sum())]
        Y = np.eye(len(classes))[yi[tr]]
        W = np.linalg.solve(A.T @ A + lam * np.eye(A.shape[1]), A.T @ Y)
        pred = (np.c_[Xs[te], np.ones(te.sum())] @ W).argmax(1)
        correct += int((pred == yi[te]).sum())
    return float(correct / n)


def _ridge_r2_cv(X: np.ndarray, y: np.ndarray, folds: int = 2, lam: float = 1.0, seed: int = 0) -> float:
    n = len(y)
    if n < 20 or y.std() < 1e-9:
        return float("nan")
    fold = np.random.RandomState(seed).permutation(n) % folds
    Xs = (X - X.mean(0)) / (X.std(0) + 1e-6)
    sse, sst = 0.0, float(((y - y.mean()) ** 2).sum())
    for f in range(folds):
        tr, te = fold != f, fold == f
        A = np.c_[Xs[tr], np.ones(tr.sum())]
        W = np.linalg.solve(A.T @ A + lam * np.eye(A.shape[1]), A.T @ (y[tr] - y[tr].mean()))
        sse += float(((y[te] - (np.c_[Xs[te], np.ones(te.sum())] @ W + y[tr].mean())) ** 2).sum())
    return float(np.clip(1.0 - sse / max(sst, 1e-9), 0.0, 1.0))


class InterpretMixin:
    # ------------------------------------------------------------------ layer probes
    @torch.no_grad()
    def _layer_probes(self, ds, ids: Sequence[int], folder: str) -> Dict[str, Any]:
        """Hidden states after every encoder layer at the EMITTING frames (label = the emitted unit) -> linear probe accuracy per layer;
        mean-pooled hidden states -> linear R2 of the measured slant per layer.  Shows where content and style are represented."""
        self.model.eval()
        H_by_layer: Dict[int, List[np.ndarray]] = defaultdict(list)
        pooled: Dict[int, List[np.ndarray]] = defaultdict(list)
        labels: List[int] = []
        slant: List[float] = []
        for batch in self.data.eval_loader(ds, [int(i) for i in ids]):
            batch = to_device(batch, self.device)
            o = self.model(batch["image"], batch["width"], keep_hidden=True)
            lp, lens = o["log_probs"].cpu().numpy(), o["out_lens"].cpu().tolist()
            hid = [h.float().cpu().numpy() for h in o["hidden"]]
            for b in range(lp.shape[0]):
                L = lens[b]
                am = lp[b, :L].argmax(1)
                emit = np.flatnonzero(am != 0)
                if len(emit) == 0:
                    continue
                labels += am[emit].tolist()
                for li, h in enumerate(hid):
                    H_by_layer[li].append(h[b, emit])
                    pooled[li].append(h[b, :L].mean(0))
                slant.append(float(ds.style_raw[int(batch["index"][b]), 0]))
        if not labels:
            return {}
        y = np.array(labels)
        keep = np.isin(y, [c for c, n in zip(*np.unique(y, return_counts=True)) if n >= 5])
        sl = np.array(slant)
        ok = np.isfinite(sl)
        rows = []
        for li in sorted(H_by_layer):
            X = np.concatenate(H_by_layer[li])
            acc = _ridge_classifier_cv(X[keep], y[keep], seed=li)
            Xp = np.stack(pooled[li])
            r2 = _ridge_r2_cv(Xp[ok], sl[ok], seed=li) if ok.sum() >= 20 else float("nan")
            rows.append({"layer": li, "name": "encoder input (content + FiLM)" if li == 0 else f"after layer {li}", "char_probe_accuracy": acc, "slant_probe_r2": r2,
                         "n_frames": int(keep.sum())})
        maj = float(np.bincount(np.unique(y[keep], return_inverse=True)[1]).max() / max(keep.sum(), 1))
        out = {"layers": rows, "majority_class_accuracy": maj, "n_lines": len(slant), "note": "2-fold ridge probes; accuracy far above the majority class = linearly "
               "readable character identity; slant R2 from the mean-pooled state = how much writer style the content stream still carries at that depth"}
        _write_csv(os.path.join(folder, "layer_probes.csv"), ["layer", "name", "char_probe_accuracy", "slant_probe_r2", "n_frames"],
                   [[r["layer"], r["name"], r["char_probe_accuracy"], r["slant_probe_r2"], r["n_frames"]] for r in rows])
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot([r["layer"] for r in rows], [r["char_probe_accuracy"] for r in rows], "o-", label="character identity (probe accuracy)")
        ax.axhline(maj, color="gray", ls=":", label="majority class")
        ax2 = ax.twinx()
        ax2.plot([r["layer"] for r in rows], [r["slant_probe_r2"] for r in rows], "s--", color="purple", label="slant (probe R2)")
        ax.set_xlabel("encoder depth (0 = input of the Transformer)"), ax.set_ylabel("probe accuracy"), ax2.set_ylabel("R2", color="purple")
        ax.legend(loc="center right", fontsize=8), ax2.legend(loc="lower right", fontsize=8)
        ax.set_title("where does the reading form?  linear probes per layer", fontsize=9)
        self._fig("10_explanations/layer_probes.png", "Layer-wise linear probes: character identity and slant", fig)
        return out

    # ------------------------------------------------------------------ uncertainty decomposition
    def _uncertainty_decomposition(self, ds, R: Dict[str, Any], folder: str) -> Dict[str, Any]:
        """Per line: 1 - sequence confidence, mean posterior entropy at the emitting frames, MC-dropout disagreement, view disagreement and
        (when the checkpoint carries a trust rule) the mean FRAG-RC character risk.  AUROC of each for 'line has an error', their rank correlations
        and the CER of the lines each signal would keep at 80 % coverage."""
        ic = self.ic
        pos = {int(i): k for k, i in enumerate(R["idx"])}
        ids = self._random_subset([int(i) for i in R["idx"]], ic.uncertainty_lines, 31)
        sig: Dict[str, Dict[int, float]] = defaultdict(dict)
        self.model.eval()
        drop_mods = [m for m in self.model.modules() if isinstance(m, nn.Dropout)]
        K = int(ic.mc_dropout_samples)
        for batch in self.data.eval_loader(ds, ids):
            batch = to_device(batch, self.device)
            x, w = batch["image"], batch["width"]
            with torch.no_grad():
                o = self.model(x, w)
                o2 = self.model(apply_perturbation(PERTURBATIONS["shear +0.08"], x, w), w)
            lp, lens = o["log_probs"].cpu().numpy(), o["out_lens"].cpu().tolist()
            lp2, aux = o2["log_probs"].cpu().numpy(), o["aux_log_probs"].cpu().numpy()
            reads = []
            for b in range(lp.shape[0]):
                i, L = int(batch["index"][b]), lens[b]
                ids_, cf, fr = greedy_decode(lp[b, :L])
                text = self.data.codec.decode_ids(ids_)
                reads.append(text)
                p = np.exp(lp[b, :L])
                am = p.argmax(1)
                emit = am != 0
                ent = -(p * np.log(np.clip(p, 1e-9, 1))).sum(1)
                sig["1 - sequence confidence"][i] = 1.0 - seq_confidence(cf)
                sig["posterior entropy (emitting frames)"][i] = float(ent[emit].mean()) if emit.any() else float(ent.mean())
                sig["view disagreement (shear)"][i] = cer(text, self.data.codec.decode_ids(greedy_decode(lp2[b, :L])[0]))
                if getattr(self, "frag_model", None) is not None:
                    _, _, _, Fm = char_evidence(lp[b, :L], lp2[b, :L], aux[b, :L])
                    sig["FRAG-RC mean character risk"][i] = float(self.frag_model.risk(Fm).mean()) if len(Fm) else 1.0
            if K > 0 and drop_mods:
                for m in drop_mods:
                    m.train()
                try:
                    dis = np.zeros(lp.shape[0])
                    with torch.no_grad():
                        for k in range(K):
                            torch.manual_seed(1000 + k)
                            ok = self.model(x, w)
                            lpk, lk = ok["log_probs"].cpu().numpy(), ok["out_lens"].cpu().tolist()
                            for b in range(lp.shape[0]):
                                dis[b] += cer(reads[b], self.data.codec.decode_ids(greedy_decode(lpk[b, :lk[b]])[0])) / K
                finally:
                    for m in drop_mods:
                        m.eval()
                for b in range(lp.shape[0]):
                    sig[f"MC-dropout disagreement ({K} passes)"][int(batch["index"][b])] = float(dis[b])
        names = list(sig)
        S = np.array([[sig[n][i] for n in names] for i in ids], np.float64)
        bad = np.array([R["ed"][pos[i]] > 0 for i in ids])
        ed = np.array([R["ed"][pos[i]] for i in ids], np.float64)
        rl = np.maximum(np.array([R["reflen"][pos[i]] for i in ids], np.float64), 1.0)
        res: Dict[str, Any] = {"n_lines": len(ids), "line_error_rate": float(bad.mean()), "signals": {}}
        for j, n in enumerate(names):
            o = np.argsort(S[:, j], kind="stable")
            k80 = max(1, int(round(0.8 * len(ids))))
            res["signals"][n] = {"auroc_line_has_error": auroc(S[:, j], bad), "CER_of_kept_lines_at_80pct_coverage": float(ed[o[:k80]].sum() / rl[o[:k80]].sum())}
        if len(names) > 1:
            rk = np.argsort(np.argsort(S, axis=0), axis=0).astype(np.float64)
            C = np.corrcoef(rk.T)
            res["spearman_between_signals"] = {names[a]: {names[b]: float(C[a, b]) for b in range(len(names))} for a in range(len(names))}
        _write_csv(os.path.join(folder, "uncertainty_signals.csv"), ["line_index", "has_error", "edits", "ref_len"] + names,
                   [[i, int(bad[k]), int(ed[k]), int(rl[k])] + S[k].tolist() for k, i in enumerate(ids)])
        fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
        aur = [res["signals"][n]["auroc_line_has_error"] for n in names]
        ax[0].barh(range(len(names)), aur, color="steelblue")
        ax[0].set_yticks(range(len(names))), ax[0].set_yticklabels(names, fontsize=8), ax[0].set_xlabel("AUROC: line has an error"), ax[0].axvline(0.5, color="k", ls=":")
        ax[0].set_title("which uncertainty signal predicts a wrong line?", fontsize=9)
        for j, n in enumerate(names):
            o = np.argsort(S[:, j], kind="stable")
            cov = np.arange(1, len(ids) + 1) / len(ids)
            risk = np.cumsum(ed[o]) / np.maximum(np.cumsum(rl[o]), 1)
            ax[1].plot(cov, risk * 100, label=n)
        ax[1].set_xlabel("coverage (most certain lines first)"), ax[1].set_ylabel("CER of kept lines (%)"), ax[1].legend(fontsize=7)
        ax[1].set_title("selective prediction by each signal", fontsize=9)
        self._fig("02_uncertainty/uncertainty_decomposition.png", "Uncertainty decomposition: confidence vs entropy vs MC-dropout vs views vs FRAG-RC risk", fig)
        return res

    # ------------------------------------------------------------------ character counterfactuals
    @torch.no_grad()
    def _counterfactuals(self, ds, ids: Sequence[int], folder: str) -> List[Dict[str, Any]]:
        """Erase the ink of one emitted character (its CTC frames x 4 px) and read again: which characters of the reading change?  A character
        that changes when ANOTHER one is erased depends on context (good for words, bad for isolated glyphs); the diagonal is the character's
        own dependence on its ink."""
        codec = self.data.codec
        out = []
        self.model.eval()
        for i in ids:
            ink = ds.get_ink(int(i))
            H, W0 = ink.shape
            x = torch.from_numpy(np.pad(ink, ((0, 0), (0, (-W0) % 4))))[None, None].to(self.device)
            w = torch.tensor([W0], device=self.device)
            o = self.model(x, w)
            T = int(o["out_lens"][0])
            lp = o["log_probs"][0, :T].cpu().numpy()
            ids_, st, en = greedy_runs(lp)
            if not len(ids_) or len(ids_) > 60:
                continue
            base = codec.decode_ids(ids_)
            xs = x.repeat(len(ids_), 1, 1, 1)
            for k in range(len(ids_)):
                xs[k, :, :, int(st[k]) * 4:int(en[k]) * 4] = 0.0
            D = np.zeros((len(ids_), len(ids_)), np.int8)
            new_texts = []
            for c in range(0, len(ids_), 32):
                oc = self.model(xs[c:c + 32], w.repeat(min(32, len(ids_) - c)))
                lpc, lc = oc["log_probs"].cpu().numpy(), oc["out_lens"].cpu().tolist()
                for q in range(lpc.shape[0]):
                    k = c + q
                    t2 = codec.decode_ids(greedy_decode(lpc[q, :lc[q]])[0])
                    new_texts.append(t2)
                    for op, ri, hi in align_chars(base, t2):
                        if op != "match" and ri is not None:
                            D[k, ri] = 1
            n_ctx = int((D.sum(1) - np.diag(D)).sum())
            out.append({"line": int(i), "reading": ds.display(int(i), base), "gt": ds.display(int(i)), "chars": [codec.i2c.get(c, "?") for c in ids_],
                        "self_dependence_share": float(np.diag(D).mean()), "context_changes_per_erasure": float(n_ctx / max(len(ids_), 1)),
                        "readings_after_erasure": [ds.display(int(i), t) for t in new_texts], "dependency": D.tolist()})
            fig, ax = plt.subplots(2, 1, figsize=(max(6, min(16, W0 / 80)), 2.2 + 0.18 * len(ids_)), gridspec_kw={"height_ratios": [1, max(1.5, 0.12 * len(ids_))]})
            ax[0].imshow(1 - ink, cmap="gray", aspect="auto"), ax[0].set_yticks([]), ax[0].set_xticks([])
            ax[0].set_title(f"line {i}: reading '{out[-1]['reading']}'", fontsize=8, loc="left")
            ax[1].imshow(D, cmap="Reds", aspect="auto", vmin=0, vmax=1)
            labs = [c if c != " " else "␣" for c in out[-1]["chars"]]
            ax[1].set_xticks(range(len(labs))), ax[1].set_xticklabels(labs, fontsize=6), ax[1].set_yticks(range(len(labs))), ax[1].set_yticklabels(labs, fontsize=6)
            ax[1].set_xlabel("character of the reading that CHANGED"), ax[1].set_ylabel("character whose ink was ERASED")
            self._fig(f"10_explanations/counterfactuals/line_{int(i):05d}.png", f"Character counterfactuals - line {i}", fig)
        with open(os.path.join(folder, "counterfactuals.json"), "w", encoding="utf-8") as f:
            json.dump(json_safe(out), f, indent=1, ensure_ascii=False)
        return out

    # ------------------------------------------------------------------ style neighbours
    def _style_neighbours(self, ds, R: Dict[str, Any], folder: str) -> Dict[str, Any]:
        """Nearest lines in the style embedding (cosine) for a few random query lines, with their measured descriptors; plus the mean absolute
        descriptor difference of embedding neighbours vs random pairs (a style space that groups similar hands has a ratio well below 1)."""
        E = np.asarray(R["style_emb"], np.float64)
        n = len(E)
        if n < 20:
            return {}
        En = E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-9)
        raw = np.asarray(R["style_raw"], np.float64)
        rng = np.random.RandomState(17)
        q_pos = rng.choice(n, min(self.ic.neighbour_queries, n), replace=False)
        rows, ratios = [], {}
        nb_diff, rd_diff = [], []
        for q in range(n):
            sims = En @ En[q]
            sims[q] = -np.inf
            nb = np.argsort(-sims)[:5]
            r = rng.randint(0, n, 5)
            nb_diff.append(np.nanmean(np.abs(raw[nb] - raw[q]), 0))
            rd_diff.append(np.nanmean(np.abs(raw[r] - raw[q]), 0))
        with np.errstate(all="ignore"):
            nbm, rdm = np.nanmean(np.stack(nb_diff), 0), np.nanmean(np.stack(rd_diff), 0)
            ratios = {s: float(a / b) if b > 0 else float("nan") for s, a, b in zip(STYLE_NAMES, nbm, rdm)}
        fig, axes = plt.subplots(len(q_pos), 6, figsize=(18, 1.3 * len(q_pos)))
        axes = np.atleast_2d(axes)
        for r_, q in enumerate(q_pos):
            sims = En @ En[q]
            sims[q] = -np.inf
            nb = np.argsort(-sims)[:5]
            for c_, j in enumerate([q] + nb.tolist()):
                i = int(R["idx"][j])
                ax = axes[r_, c_]
                ax.imshow(1 - ds.get_ink(i), cmap="gray", aspect="auto"), ax.set_xticks([]), ax.set_yticks([])
                ax.set_title(("QUERY " if c_ == 0 else f"sim {sims[j]:.2f} ") + f"line {i} | slant {raw[j, 0]:+.0f} stroke {raw[j, 1]:.3f}", fontsize=6)
                rows.append([int(R["idx"][q]), c_, i, float(sims[j]) if c_ else 1.0] + raw[j].tolist())
        self._fig("05_style/style_neighbours.png", "Nearest neighbours in the style embedding (query = left column)", fig)
        _write_csv(os.path.join(folder, "style_neighbours.csv"), ["query_line", "rank", "line_index", "cosine"] + STYLE_NAMES, rows)
        return {"neighbour_vs_random_descriptor_gap_ratio": ratios, "note": "< 1 means embedding neighbours have more similar measured style than random pairs"}

    # ------------------------------------------------------------------ alignments
    @torch.no_grad()
    def _export_alignments(self, ds, ids: Sequence[int], path: str) -> int:
        """01_recognition/alignments.jsonl: the CTC alignment of every line (character, first / end frame, pixel span, peak confidence)."""
        codec = self.data.codec
        n = 0
        with open(path, "w", encoding="utf-8") as f:
            for batch in self.data.eval_loader(ds, [int(i) for i in ids]):
                batch = to_device(batch, self.device)
                o = self.model(batch["image"], batch["width"])
                lp, lens = o["log_probs"].cpu().numpy(), o["out_lens"].cpu().tolist()
                for b in range(lp.shape[0]):
                    i, L = int(batch["index"][b]), lens[b]
                    ids_, st, en = greedy_runs(lp[b, :L])
                    p = np.exp(lp[b, :L])
                    chars = [{"ch": codec.i2c.get(c, "?"), "frame0": int(s), "frame1": int(e), "px0": int(s * 4), "px1": int(e * 4),
                              "conf": float(p[s:e, c].max())} for c, s, e in zip(ids_, st, en)]
                    f.write(json.dumps(json_safe({"line_index": i, "corpus": ds.domain_names[int(ds.domain[i])], "rtl": bool(ds.rtl[i]),
                                                  "reading_visual_order": codec.decode_ids(ids_), "reading": ds.display(i, codec.decode_ids(ids_)),
                                                  "frames": int(L), "chars": chars}), ensure_ascii=False) + "\n")
                    n += 1
        return n

    # ------------------------------------------------------------------ long-line stress test
    @torch.no_grad()
    def _longline_stress(self, ds, R: Dict[str, Any], folder: str) -> Dict[str, Any]:
        """Pairs of test lines of the same corpus are concatenated (12 px gap) and read as one line; the CER of the concatenation is compared with
        the CER of the two lines read alone (same references).  A large gap = the model does not generalise to longer lines."""
        n_pairs = int(self.ic.longline_pairs)
        if n_pairs <= 0 or len(R["idx"]) < 4:
            return {}
        codec = self.data.codec
        rng = np.random.RandomState(23)
        idx = np.asarray(R["idx"])
        pos = {int(i): k for k, i in enumerate(idx)}
        sep = " " if " " in codec.c2i else ""
        pairs = []
        by_dom: Dict[int, List[int]] = defaultdict(list)
        for i in idx:
            by_dom[int(ds.domain[int(i)])].append(int(i))
        tries = 0
        while len(pairs) < n_pairs and tries < 20 * n_pairs:
            tries += 1
            d = int(rng.choice(list(by_dom)))
            if len(by_dom[d]) < 2:
                continue
            a, b = rng.choice(by_dom[d], 2, replace=False)
            if ds.width[a] + 12 + ds.width[b] <= self.cfg.data.max_img_width:
                pairs.append((int(a), int(b)))
        if not pairs:
            return {}
        ed_cat, ed_sep, rl, lens = [], [], [], []
        self.model.eval()
        for c in range(0, len(pairs), 8):
            chunk = pairs[c:c + 8]
            imgs, widths = [], []
            for a, b in chunk:
                ia, ib = ds.get_ink(a), ds.get_ink(b)
                g = np.zeros((ia.shape[0], 12), np.float32)
                imgs.append(np.concatenate([ia, g, ib], 1))
                widths.append(imgs[-1].shape[1])
            Wm = int(np.ceil(max(widths) / 4) * 4)
            x = torch.zeros(len(chunk), 1, imgs[0].shape[0], Wm)
            for k, im in enumerate(imgs):
                x[k, 0, :, :im.shape[1]] = torch.from_numpy(im)
            o = self.model(x.to(self.device), torch.tensor(widths, device=self.device))
            lp, ln = o["log_probs"].cpu().numpy(), o["out_lens"].cpu().tolist()
            for k, (a, b) in enumerate(chunk):
                ref = ds.oracle[a] + sep + ds.oracle[b]
                hyp = codec.decode_ids(greedy_decode(lp[k, :ln[k]])[0])
                ed_cat.append(edit_distance(ref, hyp))
                ed_sep.append(R["ed"][pos[a]] + R["ed"][pos[b]])
                rl.append(len(ref))
                lens.append(len(ref))
        ed_cat, ed_sep, rl = np.array(ed_cat, np.float64), np.array(ed_sep, np.float64), np.maximum(np.array(rl, np.float64), 1)
        res = {"pairs": len(pairs), "CER_concatenated": float(ed_cat.sum() / rl.sum()), "CER_same_lines_read_alone": float(ed_sep.sum() / rl.sum()),
               "mean_length_chars": float(np.mean(lens)), "note": "a concatenation separates the two lines by 12 px (MixLine-style); the reference is 'A B'"}
        res["generalisation_gap"] = res["CER_concatenated"] - res["CER_same_lines_read_alone"]
        with open(os.path.join(folder, "longline_stress.json"), "w") as f:
            json.dump(json_safe(res), f, indent=2)
        fig, ax = plt.subplots(figsize=(5, 3.6))
        ax.bar(["read alone (sum)", "concatenated"], [res["CER_same_lines_read_alone"] * 100, res["CER_concatenated"] * 100], color=["seagreen", "indianred"])
        ax.set_ylabel("CER (%)"), ax.set_title(f"long-line stress test ({len(pairs)} pairs, mean {res['mean_length_chars']:.0f} chars)", fontsize=9)
        self._fig("07_robustness/longline_stress.png", "Long-line stress test: concatenated pairs vs the same lines read alone", fig)
        return res

    # ------------------------------------------------------------------ group tables
    def _group_tables(self, ds, R: Dict[str, Any], folder: str) -> Dict[str, Any]:
        """CER (+ form-clustered 95 % CI) per corpus, per hardness cell and per style cluster, for the greedy and the final readings."""
        idx = np.asarray(R["idx"])
        ed, rl = np.asarray(R["ed"], np.float64), np.maximum(np.asarray(R["reflen"], np.float64), 1.0)
        edf = np.asarray(R.get("ed_final", R["ed"]), np.float64)
        blk = max(1, int(self.ic.bootstrap_block))
        grp = idx // blk
        X = ds.style[idx].astype(np.float64)
        k = min(8, max(2, len(idx) // 50)) if len(idx) >= 100 else 1
        clus = assign_nearest(X, kmeans_np(X, k, seed=0)) if k > 1 else np.zeros(len(idx), int)
        groupings = {"corpus": np.array([ds.domain_names[int(ds.domain[i])] for i in idx]), "hardness_cell": ds.stratum[idx].astype(str),
                     "style_cluster": clus.astype(str)}
        rows, out = [], {}
        for gname, g in groupings.items():
            out[gname] = {}
            for v in sorted(set(g.tolist())):
                m = g == v
                if m.sum() < 3:
                    continue
                lo, hi = bootstrap_cer_ci(ed[m], rl[m], min(self.ic.bootstrap, 500), 3, groups=grp[m])
                rec = {"lines": int(m.sum()), "CER": float(ed[m].sum() / rl[m].sum()), "CER_ci95": [lo, hi], "CER_final": float(edf[m].sum() / rl[m].sum())}
                out[gname][v] = rec
                rows.append([gname, v, rec["lines"], rec["CER"], lo, hi, rec["CER_final"]])
        _write_csv(os.path.join(folder, "groups.csv"), ["grouping", "group", "lines", "CER", "ci_lo", "ci_hi", "CER_final"], rows)
        return out

    def run_interpretability(self, ds, R: Dict[str, Any]) -> Dict[str, Any]:
        """Run every analysis, each guarded so that one failure never costs the evaluation."""
        ic, out = self.ic, {}
        if not ic.interpret:
            return out
        all_ids = [int(i) for i in R["idx"]]
        jobs = [
            ("layer_probes", lambda: self._layer_probes(ds, self._random_subset(all_ids, ic.probe_lines, 41), self.dirs["10_explanations"])),
            ("uncertainty_decomposition", lambda: self._uncertainty_decomposition(ds, R, self.dirs["02_uncertainty"])),
            ("counterfactuals", lambda: {"lines": [c["line"] for c in self._counterfactuals(ds, [i for i, _ in self._pick(R, ic.counterfactual_lines)],
                                                                                        self.dirs["10_explanations"])]}),
            ("style_neighbours", lambda: self._style_neighbours(ds, R, self.dirs["05_style"])),
            ("longline_stress", lambda: self._longline_stress(ds, R, self.dirs["07_robustness"])),
            ("groups", lambda: self._group_tables(ds, R, self.dirs["03_error_structure"])),
        ]
        if ic.export_alignments:
            jobs.append(("alignments_exported", lambda: self._export_alignments(ds, all_ids, os.path.join(self.dirs["01_recognition"], "alignments.jsonl"))))
        for name, fn in jobs:
            try:
                out[name] = fn()
            except Exception:
                LOG.exception("interpretability analysis '%s' failed (the rest of the evaluation is unaffected)", name)
        return out

In [ ]:
%%writefile inklingua/inferencer.py
"""
Evaluation: metrics with form-clustered intervals, the structured output folders (one question per folder), calibration, error structure,
codebooks, the style task, post-hoc options, robustness, the trust rule, LLM rescoring, per-sample explanations, translation and the run
report.  Layout of <run>/inference/<split>/ (docs/INFERENCE_OUTPUTS.md):

    00_summary/           metrics.json, report.html (flat table of every number), v11_extras.json, training_curves.png
    01_recognition/       predictions.csv, worst_lines.txt, alignments.jsonl, cer_distribution.png, top_confusions.png
    02_uncertainty/       calibration.png, word_marks.json, uncertainty_decomposition.png + uncertainty_signals.csv
    03_error_structure/   error_structure.png, groups.csv (per corpus / hardness cell / style cluster)
    04_posthoc/           posthoc.json (view ensemble, AdaBN, writer self-distillation, space bonus; adopted or not, and why)
    05_style/             style regression, confusion, embeddings, disentanglement probe, FiLM, style_neighbours.png/.csv
    06_codebooks/         usage, code -> character, PCA
    07_robustness/        perturbation profile, longline_stress.png/.json
    08_self_distillation/ pseudo_label_reliability.png, the FRAG-RC trust audit on the evaluation split
    09_language_model/    llm_rescoring_tuning.png, llm_rescoring.json
    10_explanations/      samples/<tag>/ (dashboards + single panels), layer_probes.png/.csv, counterfactuals/
    11_translation/       translations.json, translation_ablation.png, panels/
    12_report/            index.html + the numbers behind every panel (inklingua/report.py)
"""
from __future__ import annotations

import csv
import html
import json
import os
import shutil
import time
from collections import Counter
from typing import Any, Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch

from .config import Config, ModelConfig, dc_from_dict
from .data import HandwritingDataModule
from .decoding import ctc_prefix_beam_search, greedy_decode, log_softmax_np, seq_confidence
from .explain import (ALIGN_CAP, SAMPLE_VIZ, adaptive_calibration, chlabel, code_char_stats, contrast_rollout, coverage_at_risk, pca2,
                      risk_at_coverage, risk_coverage, save_fig)
from .frag import FragModel, cell_theta, char_evidence, char_wrong_labels
from .interpret import InterpretMixin
from .llm import Hypothesis, LLMEngine, calibrate_word_mark, fuse_scores
from .model import InkLinguaNet, ctc_per_sample
from .posthoc import PosthocMixin
from .protocol import log_test_look
from .pseudo import PERTURBATIONS, RELIABILITY_FEATURES, ReliabilityModel, SelectionRule, apply_perturbation, reliability_features, reliability_views
from .render import compute_style_descriptors, render_styled_text_info, render_styled_text_open_loop, style_mismatch
from .riskcontrol import auroc, risk_control_threshold
from .style import NUM_DESC, STYLE_NAMES
from .text import (CharCodec, align_chars, bootstrap_cer_ci, cer, chrf, detokenize, edit_distance, grapheme_clusters, is_linguistic, paired_bootstrap_delta,
                   paired_mean_ci, ridge_cv_r2, text_baseline_r2)
from .utils import LOG, json_safe, to_device, torch_load

try:
    from tqdm.auto import tqdm
except Exception:  # pragma: no cover
    def tqdm(x, **kw):  # type: ignore
        return x

FOLDERS = ("00_summary", "01_recognition", "02_uncertainty", "03_error_structure", "04_posthoc", "05_style", "06_codebooks", "07_robustness",
           "08_self_distillation", "09_language_model", "10_explanations", "11_translation", "12_report")


class Inferencer(PosthocMixin, InterpretMixin):
    def __init__(self, cfg: Config, data: HandwritingDataModule, llm: LLMEngine, device: torch.device):
        self.cfg, self.data, self.llm, self.device, self.ic = cfg, data, llm, device, cfg.infer
        self.model: Optional[InkLinguaNet] = None
        self.ckpt_info: Dict[str, Any] = {}
        self.out = cfg.infer_dir
        self.dirs = {f: os.path.join(self.out, f) for f in FOLDERS}
        self.fig_list: List[Tuple[str, str]] = []
        self.frag_state: Optional[Dict[str, Any]] = None
        self.frag_model: Optional[FragModel] = None
        self._wsd_lp: Dict[int, Dict[int, np.ndarray]] = {}
        self._ens_views: Tuple[str, ...] = ()
        self.run_extra: Dict[str, Any] = {}

    # ------------------------------------------------------------------ loading
    def _resolve(self, name: str) -> str:
        d = self.cfg.ckpt_dir
        if name == "best":
            p = os.path.join(d, "best.pt")
            return p if os.path.exists(p) else os.path.join(d, "last.pt")
        if name == "last":
            return os.path.join(d, "last.pt")
        return name

    def load(self, path: Optional[str] = None) -> None:
        p = path or self._resolve(self.ic.checkpoint)
        if not os.path.exists(p):
            raise FileNotFoundError(f"No checkpoint at {p}. Train first (cfg.run.do_train=True).")
        st = torch_load(p)
        weights = st["weights"] if "weights" in st else st["ema"]
        mcfg = dc_from_dict(ModelConfig, st["cfg"]["model"]) if "cfg" in st else self.cfg.model
        units = list(st.get("units") or grapheme_clusters(st.get("chars", self.data.codec.chars)))
        if units != self.data.codec.units:
            raise RuntimeError(f"the checkpoint was trained with a different alphabet ({len(units)} units) than the one derived from the current data "
                               f"configuration ({len(self.data.codec.units)}).  Use the same cfg.data (sources, seed, labeled_fraction, limits) and "
                               "cfg.run.name that produced the checkpoint (mode 'finetune' is the way to continue on other data).")
        ck_h = (st.get("cfg") or {}).get("data", {}).get("img_height")
        if ck_h is not None and ck_h != self.cfg.data.img_height:
            raise RuntimeError(f"checkpoint img_height={ck_h} but cfg.data.img_height={self.cfg.data.img_height}")
        self.model = InkLinguaNet(mcfg, len(units) + 1, NUM_DESC, st.get("n_clusters", self.data.n_clusters), self.cfg.data.img_height).to(self.device)
        self.model.load_state_dict(weights)
        self.model.eval()
        fs = st.get("frag_state")
        self.frag_state = fs
        self.frag_model = FragModel.from_state(fs["model"]) if fs else None
        if fs and fs.get("n_strata") not in (None, self.data.n_strata):
            LOG.warning("the FRAG-RC rule has %s cells but the data has %d: cell indices are clipped", fs.get("n_strata"), self.data.n_strata)
        self.ckpt_info = {"path": p, "epoch": st.get("epoch"), "weights_from": st.get("from", "ema"), "val_sel_cer_at_save": st.get("val_cer"),
                          "frag_rule_epoch": (fs or {}).get("epoch"), "domains": st.get("domains")}
        LOG.info("Loaded %s (epoch %s, %s weights)", p, st.get("epoch"), self.ckpt_info["weights_from"])

    def _fig(self, rel: str, caption: str, fig) -> None:
        save_fig(fig, os.path.join(self.out, rel))
        self.fig_list.append((rel, caption))

    # ------------------------------------------------------------------ passes over a dataset
    @torch.no_grad()
    def _predict(self, ds, indices, perturb=None) -> Tuple[List[str], np.ndarray]:
        preds, conf = [""] * len(ds), np.zeros(len(ds))
        for batch in self.data.eval_loader(ds, indices):
            batch = to_device(batch, self.device)
            x = batch["image"] if perturb is None else apply_perturbation(perturb, batch["image"], batch["width"])
            out = self.model(x, batch["width"])
            lp, lens = out["log_probs"].cpu().numpy(), out["out_lens"].cpu().tolist()
            for b in range(lp.shape[0]):
                ids, cf, _ = greedy_decode(lp[b, :lens[b]])
                i = int(batch["index"][b])
                preds[i], conf[i] = self.data.codec.decode_ids(ids), seq_confidence(cf)
        return preds, conf

    @staticmethod
    def _random_subset(ids: Sequence[int], k: int, seed: int) -> List[int]:
        ids = list(ids)
        if k >= len(ids):
            return sorted(int(i) for i in ids)
        return sorted(int(i) for i in np.random.RandomState(seed).permutation(ids)[:k])

    @torch.no_grad()
    def _collect(self, ds, indices: Optional[Sequence[int]] = None, extras: bool = True, cons: bool = False, keep_lp: Optional[Sequence[int]] = None) -> Dict[str, Any]:
        """One pass: readings (visual order), confidences, reliability features, style predictions, embeddings, codebook usage.  `ds.oracle` is
        read directly (evaluation code); for unlabeled TRAIN lines it is the hidden transcription, used for diagnostics only."""
        m, codec, pc = self.model, self.data.codec, self.cfg.train.pseudo
        Kl, Kh, V = m.vq_low.K, m.vq_high.K, codec.num_classes
        views = reliability_views(pc) if cons else ()
        n_gate = len(pc.cons_views) if (cons and pc.consistency) else 0
        keep = set(int(i) for i in keep_lp) if keep_lp is not None else set()
        R: Dict[str, Any] = {k: [] for k in ("idx", "gt", "pred", "conf", "char_conf", "ed", "reflen", "ed_w", "reflen_w", "style_pred", "style_raw", "cl_pred",
                                             "cl_true", "style_emb", "content", "gamma", "X", "cons")}
        R["lp"] = {}
        R["use_low"], R["use_high"], R["code_char"] = np.zeros(Kl), np.zeros(Kh), np.zeros((Kh, V))
        for batch in tqdm(self.data.eval_loader(ds, indices), desc="inference pass", leave=False):
            batch = to_device(batch, self.device)
            out = m(batch["image"], batch["width"], extras=extras)
            lp, lens = out["log_probs"].cpu().numpy(), out["out_lens"].cpu().tolist()
            B = lp.shape[0]
            cons_v = np.zeros((B, len(views)))
            reads = [greedy_decode(lp[b, :lens[b]]) for b in range(B)]
            for vi, v in enumerate(views):
                xv = apply_perturbation(PERTURBATIONS[v], batch["image"], batch["width"])
                ov = m(xv, batch["width"])
                lpv, lnv = ov["log_probs"].cpu().numpy(), ov["out_lens"].cpu().tolist()
                for b in range(B):
                    cons_v[b, vi] = cer(codec.decode_ids(reads[b][0]), codec.decode_ids(greedy_decode(lpv[b, :lnv[b]])[0]))
            if extras:
                il, ih = out["idx_low"].cpu().numpy(), out["idx_high"].cpu().numpy()
            for b in range(B):
                T = lens[b]
                ids, cf, fr = reads[b]
                text, i_ds = codec.decode_ids(ids), int(batch["index"][b])
                gt = ds.oracle[i_ds]
                R["idx"].append(i_ds)
                R["gt"].append(gt), R["pred"].append(text), R["conf"].append(seq_confidence(cf))
                R["char_conf"].append(cf)
                R["ed"].append(edit_distance(gt, text)), R["reflen"].append(len(gt))
                R["ed_w"].append(edit_distance(gt.split(), text.split())), R["reflen_w"].append(len(gt.split()))
                R["X"].append(reliability_features(lp[b, :T], ids, cf, fr, cons_v[b]))
                R["cons"].append(float(cons_v[b, :n_gate].mean()) if n_gate else float("nan"))
                if i_ds in keep:
                    R["lp"][i_ds] = lp[b, :T].astype(np.float16)
                if extras:
                    R["use_low"] += np.bincount(il[b, :T], minlength=Kl)
                    R["use_high"] += np.bincount(ih[b, :T], minlength=Kh)
                    am = lp[b, :T].argmax(1)
                    for t in np.flatnonzero(am != 0):
                        R["code_char"][ih[b, t], am[t]] += 1
            R["style_pred"].append(out["style_reg"].cpu().numpy())
            R["style_raw"].append(ds.style_raw[batch["index"].cpu().numpy()])
            R["cl_pred"].append(out["style_cls"].argmax(1).cpu().numpy())
            R["cl_true"].append(batch["cluster"].cpu().numpy())
            R["style_emb"].append(out["style_emb"].cpu().numpy())
            R["content"].append(out["content_pool"].cpu().numpy())
            if extras and out.get("film_gamma") is not None:
                R["gamma"].append(out["film_gamma"].float().cpu().numpy())
        for k in ("style_pred", "style_raw", "cl_pred", "cl_true", "style_emb", "content"):
            R[k] = np.concatenate(R[k])
        R["conf"], R["ed"], R["reflen"] = np.array(R["conf"]), np.array(R["ed"]), np.array(R["reflen"])
        R["ed_w"], R["reflen_w"] = np.array(R["ed_w"]), np.array(R["reflen_w"])
        R["X"], R["cons"] = np.array(R["X"]), np.array(R["cons"])
        R["cer_ps"] = R["ed"] / np.maximum(R["reflen"], 1)
        R["gamma"] = np.concatenate(R["gamma"]) if R["gamma"] else np.zeros((0, 1))
        return R

    # ------------------------------------------------------------------ per-sample explanation
    def _explain(self, ds, i: int, full: bool = True, tag: str = "") -> Dict[str, Any]:
        dev, codec, model = self.device, self.data.codec, self.model
        ink = ds.get_ink(i)
        H, W0 = ink.shape
        x = torch.from_numpy(np.pad(ink, ((0, 0), (0, (-W0) % 4))))[None, None].to(dev)
        w = torch.tensor([W0], device=dev)
        with torch.no_grad():
            out = model(x, w, extras=True, need_attn=full)
        T = int(out["out_lens"][0])
        lp = out["log_probs"][0, :T].cpu().numpy()
        ids, cf, fr = greedy_decode(lp)
        pred, gt = codec.decode_ids(ids), ds.oracle[i]
        sp = out["style_reg"][0].cpu().numpy()
        ctx: Dict[str, Any] = {
            "i": i, "tag": tag, "img": ink, "W": W0, "H": H, "T": T, "gt": gt, "pred": pred, "gt_display": ds.display(i), "pred_display": ds.display(i, pred),
            "detok": ds.detok(i), "conf": seq_confidence(cf), "char_conf": cf, "frames": fr, "ids": ids, "probs": np.exp(lp), "units": codec.units,
            "pred_units": [codec.i2c.get(c, "?") for c in ids], "cer": cer(gt, pred), "ops": align_chars(gt, pred),
            "nbest": [(codec.decode_ids(a), l) for a, l in ctc_prefix_beam_search(lp, self.ic.beam)],
            "style_pred": self.data.denorm_style(sp), "style_true": ds.style_raw[i].astype(np.float64),
            "cl_pred": int(out["style_cls"][0].argmax()), "cl_true": int(ds.cluster[i]),
            "idx_low": out["idx_low"][0, :T].cpu().numpy(), "idx_high": out["idx_high"][0, :T].cpu().numpy(),
            "style_att": out["style_att"][0, :T].cpu().numpy(), "attn_roll": None, "gradxin": np.zeros((H, W0)), "ig": np.zeros((H, W0)),
            "occ_x": np.zeros(1), "occ_d": np.zeros(1), "char_attr": np.zeros((0, W0))}
        if not full:
            return ctx
        if out["attn"]:
            Tp = out["attn"][0].shape[-1]
            Rl = torch.eye(Tp)
            for A in out["attn"]:
                A = 0.5 * A[0].float().mean(0).cpu() + 0.5 * torch.eye(Tp)
                Rl = (A / A.sum(-1, keepdim=True)) @ Rl
            ctx["attn_roll"] = Rl.numpy()[:T, :T]
            if ctx["frames"]:
                rows = np.stack([ctx["attn_roll"][f] / max(ctx["attn_roll"][f].sum(), 1e-9) for f in ctx["frames"]])
                far = np.abs(np.arange(T)[None] - np.asarray(ctx["frames"])[:, None]) > 3
                ctx["attn_nonlocal"] = float((rows * far).sum(1).mean())
                ctx["attn_sink"] = contrast_rollout(rows)[1]
        if ids:
            with torch.enable_grad():
                ctx.update(self._grad_maps(x, w, ids, fr, W0))
        return ctx

    def _grad_maps(self, x, w, ids, frames, W0) -> Dict[str, Any]:
        dev, model = self.device, self.model
        K = len(ids)
        tgt, tl = torch.tensor(ids, device=dev), torch.tensor([K], device=dev)
        xg = x.clone().requires_grad_(True)
        out = model(xg, w)
        lp = out["log_probs"]
        loss = ctc_per_sample(lp, out["out_lens"], tgt, tl).sum()
        g, = torch.autograd.grad(loss, xg, retain_graph=True)
        gxi = (g * xg).abs()[0, 0, :, :W0].detach().cpu().numpy()
        rows = []
        for k in range(min(K, 70)):
            gk, = torch.autograd.grad(lp[0, frames[k], ids[k]], xg, retain_graph=True)
            rows.append((gk * xg).abs()[0, 0].sum(0)[:W0].detach().cpu().numpy())
        import cv2
        A = np.stack(rows)
        A = np.stack([cv2.GaussianBlur(r[None].astype(np.float32), (0, 0), 3)[0] for r in A])
        A = A / (A.max(1, keepdims=True) + 1e-9)
        S = self.ic.ig_steps
        xb = (torch.linspace(1.0 / S, 1.0, S, device=dev).view(-1, 1, 1, 1) * x).requires_grad_(True)
        ob = model(xb, w.repeat(S))
        lb = ctc_per_sample(ob["log_probs"], ob["out_lens"], tgt.repeat(S), tl.repeat(S))
        gb, = torch.autograd.grad((-lb).sum(), xb)
        ig = (x[0, 0] * gb.mean(0)[0])[:, :W0].detach().cpu().numpy()
        with torch.no_grad():
            win, stride = self.ic.occl_window, self.ic.occl_stride
            starts = list(range(0, max(1, W0 - win + 1), stride))
            xs = x.repeat(len(starts), 1, 1, 1)
            for n, s0 in enumerate(starts):
                xs[n, :, :, s0:s0 + win] = 0
            base = float(ctc_per_sample(*self._fw(x, w), tgt, tl))
            ds_ = []
            for c in range(0, len(starts), 32):
                xc = xs[c:c + 32]
                lo, ln = self._fw(xc, w.repeat(len(xc)))
                ds_ += (ctc_per_sample(lo, ln, tgt.repeat(len(xc)), tl.repeat(len(xc))) - base).cpu().tolist()
        return {"gradxin": gxi, "char_attr": A, "ig": ig, "occ_x": np.array(starts) + win / 2.0, "occ_d": np.array(ds_)}

    def _fw(self, x, w):
        o = self.model(x, w)
        return o["log_probs"], o["out_lens"]

    def _sample_figs(self, ctx, tag: str) -> None:
        W = ctx["W"]
        wi = float(np.clip(W / 90.0, 8, 17))
        n, ratios = len(SAMPLE_VIZ), [r for _, _, r in SAMPLE_VIZ]
        fig, axes = plt.subplots(n, 1, figsize=(wi, 0.85 * sum(ratios) + 1.0), gridspec_kw={"height_ratios": ratios})
        for ax, (_, fn, _) in zip(axes, SAMPLE_VIZ):
            fn(ax, ctx)
            ax.set_xlim(0, W)
        axes[-1].set_xlabel("pixel column (frame stride = 4 px)")
        self._fig(f"10_explanations/samples/{tag}/dashboard.png", f"Explanation dashboard - {tag}", fig)
        for name, fn, r in SAMPLE_VIZ:
            f, ax = plt.subplots(figsize=(wi, 1.1 * r + 1.0))
            fn(ax, ctx)
            ax.set_xlim(0, W)
            save_fig(f, os.path.join(self.dirs["10_explanations"], "samples", tag, f"{name}.png"))

    # ------------------------------------------------------------------ translation
    def _hyps(self, ds, ctx, nbest: int) -> List[Hypothesis]:
        """Hypotheses in LOGICAL order (RTL lines are reversed back); confidences follow the units."""
        nb = ctx["nbest"]
        lp0 = dict(nb).get(ctx["pred"], max(l for _, l in nb))
        alts = [(t, l) for t, l in nb if t != ctx["pred"]][:max(0, nbest - 1)]
        lps = np.array([lp0] + [l for _, l in alts])
        post = np.exp(lps - lps.max())
        post /= post.sum()
        i = ctx["i"]
        cf = list(ctx["char_conf"])
        if bool(ds.rtl[i]):
            cf = cf[::-1]
        return [Hypothesis(ds.display(i, ctx["pred"]), cf, float(lp0), float(post[0]))] + [
            Hypothesis(ds.display(i, t), [], float(l), float(p)) for (t, l), p in zip(alts, post[1:])]

    def _rescored_reading(self, ds, ctx) -> Optional[str]:
        rs = getattr(self, "_rescore", None)
        if not rs or not rs.get("adopted") or not self.llm.can_score() or not ctx.get("nbest"):
            return None
        nb = ctx["nbest"]
        i = ctx["i"]
        lm, nt = self.llm.score_texts([detokenize(ds.display(i, t)) if ds.detok(i) else ds.display(i, t) for t, _ in nb])
        return nb[int(np.argmax(fuse_scores(np.array([l for _, l in nb]), lm, nt, rs["beta"], rs["rho"])))][0]

    def _translate_block(self, ds, ctxs: List[Dict[str, Any]]) -> Dict[str, Any]:
        """Cascade evaluation (handwriting -> text -> translation).  Headline numbers use ONLY the randomly drawn lines.  Arms: plain = top-1
        reading, plain prompt | uncertain = marked text + N-best, uncertain prompt | rescored = the validated LLM-rescored reading.  The primary
        `translation` column follows cfg.llm.use_uncertainty_prompt.  Every arm is scored against an oracle made with ITS OWN prompt."""
        if not ctxs or not self.llm.available():
            LOG.warning("Translation skipped (no LLM available).")
            return {}
        rows, n_panels = [], 0
        risk_of = getattr(self, "_risk_by_idx", {})
        use_unc = bool(self.cfg.llm.use_uncertainty_prompt)
        for ctx in tqdm(ctxs, desc="translating", leave=False):
            i = ctx["i"]
            detok = ds.detok(i)
            hyps = self._hyps(ds, ctx, self.cfg.llm.nbest)
            unc = self.llm.translate(hyps, uncertain=True, detok=detok)
            plain = self.llm.translate(hyps, uncertain=False, detok=detok)["translation"]
            row = {"index": i, "tag": ctx.get("tag", "random"), "gt": ctx["gt_display"], "pred": ctx["pred_display"], "cer": ctx["cer"],
                   "marked_text": unc["marked_text"], "uncertain_translation": unc["translation"], "plain_translation": plain,
                   "translation": unc["translation"] if use_unc else plain, "prompt_used": "uncertain" if use_unc else "plain",
                   "risk": risk_of.get(int(i), float("nan"))}
            if self.ic.translation_ablation:
                ora = self.llm.translate([Hypothesis(ctx["gt_display"])], uncertain=False, detok=detok)["translation"]
                ora_u = self.llm.translate([Hypothesis(ctx["gt_display"], [1.0] * len(ctx["gt_display"]))], uncertain=True, detok=detok)["translation"]
                row.update({"oracle_translation": ora, "oracle_translation_uncertain_prompt": ora_u, "chrf_plain_vs_oracle": chrf(plain, ora),
                            "chrf_uncertain_vs_oracle": chrf(unc["translation"], ora), "chrf_uncertain_vs_oracle_same_prompt": chrf(unc["translation"], ora_u)})
                rr = self._rescored_reading(ds, ctx)
                if rr is not None:
                    rt = plain if rr == ctx["pred"] else self.llm.translate([Hypothesis(ds.display(i, rr))], uncertain=False, detok=detok)["translation"]
                    row.update({"rescored_reading": ds.display(i, rr), "rescored_translation": rt, "chrf_rescored_vs_oracle": chrf(rt, ora)})
            if self.ic.render_translation and row["translation"] and n_panels < self.ic.render_translation_max:
                self._translation_panel(ctx, row)
                n_panels += 1
            rows.append(row)
        rnd = [r for r in rows if r["tag"] == "random"]
        summ: Dict[str, Any] = {"n": len(rows), "n_random": len(rnd), "target_language": self.cfg.llm.target_language, "source_language": self.cfg.llm.source_language,
                                "primary_prompt": "uncertain" if use_unc else "plain", "rows": rows}
        if self.ic.translation_ablation and rnd:
            blk = max(1, int(self.ic.bootstrap_block))
            gr = [int(r['index']) // blk for r in rnd]
            pl = [r["chrf_plain_vs_oracle"] for r in rnd]
            un = [r["chrf_uncertain_vs_oracle"] for r in rnd]
            un_m = [r["chrf_uncertain_vs_oracle_same_prompt"] for r in rnd]
            summ.update({"chrf_plain_vs_oracle": float(np.mean(pl)), "chrf_uncertain_vs_oracle_plain_prompt_oracle": float(np.mean(un)),
                         "chrf_uncertain_vs_oracle_same_prompt": float(np.mean(un_m)),
                         "chrf_uncertain_minus_plain_prompt_confounded": paired_mean_ci(un, pl, self.ic.bootstrap, 0, gr),
                         "chrf_uncertain_minus_plain_same_prompt_oracles": paired_mean_ci(un_m, pl, self.ic.bootstrap, 0, gr)})
            ok_ = [r for r in rnd if r["cer"] == 0]
            if ok_:
                summ["prompt_effect_on_error_free_lines"] = {
                    "n": len(ok_), "chrf_uncertain_prompt_vs_plain_prompt": float(np.mean([chrf(r["uncertain_translation"], r["oracle_translation"]) for r in ok_])),
                    "note": "100 = the two prompts give the same translation of a correctly read line; the shortfall is prompt variance, not HTR error"}
            err = [r for r in rnd if r["cer"] > 0]
            summ["n_random_lines_with_errors"] = len(err)
            if err:
                e_pl, e_um = [r["chrf_plain_vs_oracle"] for r in err], [r["chrf_uncertain_vs_oracle_same_prompt"] for r in err]
                summ.update({"chrf_plain_on_lines_with_errors": float(np.mean(e_pl)), "chrf_uncertain_same_prompt_on_lines_with_errors": float(np.mean(e_um)),
                             "chrf_uncertain_minus_plain_same_prompt_on_lines_with_errors": paired_mean_ci(e_um, e_pl, self.ic.bootstrap, 0, [int(r['index']) // blk for r in err])})
            rs_rows = [r for r in rnd if "chrf_rescored_vs_oracle" in r]
            if rs_rows:
                summ["rescored_arm"] = {"n": len(rs_rows), "reading_changed_share": float(np.mean([r["rescored_reading"] != r["pred"] for r in rs_rows])),
                                        "chrf_rescored": float(np.mean([r["chrf_rescored_vs_oracle"] for r in rs_rows])),
                                        "chrf_rescored_minus_plain": paired_mean_ci([r["chrf_rescored_vs_oracle"] for r in rs_rows], [r["chrf_plain_vs_oracle"] for r in rs_rows],
                                                                                    self.ic.bootstrap, 0, [int(r['index']) // blk for r in rs_rows])}
            elif getattr(self, "_rescore", None) is not None and not self._rescore.get("adopted"):
                summ["rescored_arm"] = {"skipped": "LLM N-best rescoring showed no significant gain on the val_gate lines, so it is not used"}
            rk = np.array([r["risk"] for r in rnd], np.float64)
            if np.isfinite(rk).all() and len(rnd) >= 10:
                o = np.argsort(rk, kind="stable")
                sel = {}
                for cov in (1.0, 0.8, 0.6, 0.4):
                    k = max(1, int(round(cov * len(rnd))))
                    keep = o[:k]
                    sel[f"{int(cov * 100)}%"] = {"n_answered": int(k), "chrf_plain": float(np.mean([pl[i] for i in keep])),
                                                 "chrf_uncertain_same_prompt": float(np.mean([un_m[i] for i in keep])),
                                                 "mean_cer_of_answered_lines": float(np.mean([rnd[i]["cer"] for i in keep]))}
                summ["selective_translation"] = {"coverage": sel, "ranking": "learned reliability score of the reading (val_gate-fitted)",
                                                 "note": "lines above the coverage cut-off would be answered with '[illegible]' instead of a translation"}
            fid = [r for r in rows if "rendered_style" in r and r.get("font_covers_text", True)]
            if fid:
                ps_ = np.array([r["predicted_style"] for r in fid], np.float64)
                rs_c = np.array([r["rendered_style"] for r in fid], np.float64)
                rs_o = np.array([r["rendered_style_open_loop"] for r in fid], np.float64)
                with np.errstate(all="ignore"):
                    summ["style_transfer_fidelity"] = {
                        "n": len(fid), "descriptors": STYLE_NAMES,
                        "mae_closed_loop": [float(np.nanmean(np.abs(ps_[:, j] - rs_c[:, j]))) for j in range(NUM_DESC)],
                        "mae_open_loop_v3": [float(np.nanmean(np.abs(ps_[:, j] - rs_o[:, j]))) for j in range(NUM_DESC)],
                        "mean_scaled_mismatch_closed_loop": float(np.mean([r["style_mismatch"] for r in fid])),
                        "mean_scaled_mismatch_open_loop_v3": float(np.mean([r["style_mismatch_open_loop"] for r in fid]))}
            fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.2))
            labs, a, b = [f"random lines (n={len(rnd)})"], [summ["chrf_plain_vs_oracle"]], [summ["chrf_uncertain_vs_oracle_same_prompt"]]
            if err:
                labs.append(f"... with HTR errors (n={len(err)})")
                a.append(summ["chrf_plain_on_lines_with_errors"])
                b.append(summ["chrf_uncertain_same_prompt_on_lines_with_errors"])
            xx = np.arange(len(labs))
            ax[0].bar(xx - 0.18, a, 0.36, label="plain top-1 (oracle: plain prompt)")
            ax[0].bar(xx + 0.18, b, 0.36, label="uncertainty-aware (oracle: same prompt)")
            ax[0].set_xticks(xx), ax[0].set_xticklabels(labs, fontsize=8), ax[0].set_ylabel("chrF vs translation of the TRUE text")
            d_ = summ["chrf_uncertain_minus_plain_same_prompt_oracles"]
            ax[0].set_title(f"Cascade loss, prompt-matched oracles\nuncertain - plain = {d_['delta']:+.2f} chrF [{d_['lo']:+.2f}, {d_['hi']:+.2f}]", fontsize=8)
            ax[0].legend(fontsize=7), ax[0].set_ylim(0, 105)
            if "selective_translation" in summ:
                cv = summ["selective_translation"]["coverage"]
                ax[1].plot(list(cv), [v["chrf_plain"] for v in cv.values()], "o-", label="plain")
                ax[1].plot(list(cv), [v["chrf_uncertain_same_prompt"] for v in cv.values()], "s-", label="uncertainty-aware")
                ax[1].set_xlabel("share of lines translated (most reliable first); rest -> '[illegible]'"), ax[1].set_ylabel("chrF of the answered lines")
                ax[1].set_title("selective translation", fontsize=8), ax[1].legend(fontsize=7)
            else:
                ax[1].axis("off")
            self._fig("11_translation/translation_ablation.png", "Translation: plain vs uncertainty-aware (prompt-matched) and selective translation", fig)
        return summ

    def _translation_panel(self, ctx, row) -> None:
        info = render_styled_text_info(row["translation"], ctx["style_pred"], ctx["H"], iters=6)
        styled, covered, rs = info["image"], info["covered"], info["measured"]
        op, _ = render_styled_text_open_loop(row["translation"], ctx["style_pred"], ctx["H"])
        ro = compute_style_descriptors(np.pad(op, ((0, 0), (8, 8))))
        tgt = np.asarray(ctx["style_pred"], np.float64)
        row.update({"rendered_style": rs.tolist(), "rendered_style_open_loop": ro.tolist(), "predicted_style": list(map(float, ctx["style_pred"])),
                    "font_covers_text": bool(covered), "style_mismatch": float(style_mismatch(rs, tgt)), "style_mismatch_open_loop": float(style_mismatch(ro, tgt))})
        fig = plt.figure(figsize=(float(np.clip(ctx["W"] / 90, 8, 14)), 5.6))
        gs = fig.add_gridspec(3, 1, height_ratios=[1.0, 1.0, 1.4])
        a0, a1, a2 = fig.add_subplot(gs[0]), fig.add_subplot(gs[1]), fig.add_subplot(gs[2])
        a0.imshow(1 - ctx["img"], cmap="gray", aspect="equal"), a0.set_title("handwritten input", fontsize=8, loc="left")
        a1.imshow(1 - styled, cmap="gray", aspect="equal")
        f_ = lambda v, d=2: "n/a" if not np.isfinite(v) else f"{v:.{d}f}"          # noqa: E731
        a1.set_title(f"translation ({self.cfg.llm.target_language}) rendered with the PREDICTED style [closed-loop]  -  slant {tgt[0]:+.0f}/{f_(rs[0], 0)} deg | "
                     f"stroke {tgt[1]:.3f}/{f_(rs[1], 3)} | ink {tgt[2]:.2f}/{f_(rs[2])} | gap {tgt[3]:.2f}/{f_(rs[3])} | body {tgt[4]:.2f}/{f_(rs[4])}  "
                     f"(predicted/rendered; mismatch {row['style_mismatch']:.2f} vs {row['style_mismatch_open_loop']:.2f} open-loop)"
                     + ("" if covered else "  [font lacks some glyphs]"), fontsize=6.5, loc="left")
        for a in (a0, a1):
            a.set_xticks([]), a.set_yticks([])
        a2.axis("off")
        txt = f"GT        : {row['gt']}\nHTR       : {row['pred']}   (CER {row['cer']:.3f})\nsent to LLM: {row['marked_text']}\ntranslation: {row['translation']}"
        if "oracle_translation" in row:
            txt += f"\nplain top-1 : {row['plain_translation']}   (chrF {row['chrf_plain_vs_oracle']:.1f})\noracle(GT)  : {row['oracle_translation']}\n" \
                   f"uncertain   : chrF {row['chrf_uncertain_vs_oracle']:.1f} vs plain-prompt oracle, {row['chrf_uncertain_vs_oracle_same_prompt']:.1f} vs same-prompt oracle"
        a2.text(0, 1, txt, va="top", ha="left", fontsize=7.5, family="monospace", wrap=True)
        self._fig(f"11_translation/panels/line_{ctx['i']:05d}.png", f"Translation panel - line {ctx['i']} [{row['tag']}]", fig)

    # ------------------------------------------------------------------ recognition errors
    def _align_cache(self, R: Dict[str, Any], sub_ids: Sequence[int]) -> None:
        keep = set(int(i) for i in sub_ids)
        R["ops"] = {k: align_chars(R["gt"][k], R["pred"][k]) for k, i in enumerate(R["idx"]) if i in keep}

    def _recognition_block(self, R: Dict[str, Any], ds) -> Dict[str, Any]:
        M: Dict[str, Any] = {}
        n = len(R["gt"])
        ed, rl = R["ed"].astype(np.float64), np.maximum(R["reflen"], 1).astype(np.float64)
        ling = np.array([is_linguistic(g) for g in R["gt"]])
        M["n_nonlinguistic_lines"] = int((~ling).sum())
        M["CER_linguistic"] = float(ed[ling].sum() / max(rl[ling].sum(), 1))
        ed_ns = np.array([edit_distance(g.replace(" ", ""), p.replace(" ", "")) for g, p in zip(R["gt"], R["pred"])], np.float64)
        rl_ns = np.array([max(len(g.replace(" ", "")), 1) for g in R["gt"]], np.float64)
        M["CER_ignoring_spaces"] = float(ed_ns.sum() / rl_ns.sum())
        o = np.argsort(-ed)
        M["error_concentration"] = {f"share_of_edits_in_worst_{p}pct_lines": float(ed[o[:max(1, int(round(n * p / 100)))]].sum() / max(ed.sum(), 1)) for p in (1, 5, 10)}
        M["lines_with_cer_above_0.5"] = int((R["cer_ps"] > 0.5).sum())
        sub = [k for k in range(n) if k in R["ops"]]
        subs, dels, inss, sp_ops = Counter(), Counter(), Counter(), Counter()
        for k in sub:
            gt, pr = R["gt"][k], R["pred"][k]
            for op, ri, hi in R["ops"][k]:
                if op == "sub":
                    subs[(gt[ri], pr[hi])] += 1
                    sp_ops["sub_with_space"] += int(gt[ri] == " " or pr[hi] == " ")
                elif op == "del":
                    dels[gt[ri]] += 1
                    sp_ops["space_deleted"] += int(gt[ri] == " ")
                elif op == "ins":
                    inss[pr[hi]] += 1
                    sp_ops["space_inserted"] += int(pr[hi] == " ")
        tot_ops = sum(subs.values()) + sum(dels.values()) + sum(inss.values())
        if tot_ops:
            M["space_errors"] = {"n_alignment_lines": len(sub), "share_of_all_char_errors": float((sp_ops["sub_with_space"] + sp_ops["space_deleted"] + sp_ops["space_inserted"]) / tot_ops),
                                 **{k: int(v) for k, v in sp_ops.items()}}
        if len(self.data.specs) > 1:
            blk = max(1, int(self.ic.bootstrap_block))
            M["per_corpus"] = {}
            for d_, name in enumerate(ds.domain_names):
                m = np.array([int(ds.domain[int(i)]) == d_ for i in R["idx"]])
                if m.any():
                    lo, hi = bootstrap_cer_ci(ed[m], rl[m], self.ic.bootstrap, 0, groups=np.asarray(R["idx"])[m] // blk)
                    M["per_corpus"][name] = {"lines": int(m.sum()), "CER": float(ed[m].sum() / rl[m].sum()), "CER_ci95": [lo, hi],
                                             "WER": float(R["ed_w"][m].sum() / max(R["reflen_w"][m].sum(), 1)), "exact_line_accuracy": float((R["cer_ps"][m] == 0).mean())}
        with open(os.path.join(self.dirs["01_recognition"], "worst_lines.txt"), "w", encoding="utf-8") as f:
            f.write("The 40 lines with the most character errors (check them by eye: corpora contain transcription errors and mis-segmented lines).\n"
                    "Lines whose reference has no letter or digit are listed at the end: they are not handwriting.  Right-to-left lines are shown in logical order.\n")
            for j in [j for j in o if ling[j]][:40]:
                i = int(R["idx"][j])
                f.write(f"line {i:5d} [{ds.domain_names[int(ds.domain[i])]}] | edits {int(ed[j]):3d} | CER {R['cer_ps'][j]:.2f} | conf {R['conf'][j]:.3f}\n"
                        f"   GT  : {ds.display(i)}\n   PRED: {ds.display(i, R['pred'][j])}\n")
            for j in [j for j in o if not ling[j]][:10]:
                f.write(f"[non-linguistic reference] line {R['idx'][j]:5d} | edits {int(ed[j]):3d} | GT: {R['gt'][j]!r} | PRED: {R['pred'][j]!r}\n")
        fig, ax = plt.subplots(1, 3, figsize=(16, 4))
        ax[0].hist(R["cer_ps"], bins=40, color="steelblue"), ax[0].set_xlabel("line CER"), ax[0].set_ylabel("#lines")
        ax[0].set_yscale("log"), ax[0].set_title("per-line CER distribution (log count)", fontsize=9)
        ax[1].scatter(R["reflen"], R["cer_ps"], s=6, alpha=0.4), ax[1].set_xlabel("#characters in line"), ax[1].set_ylabel("line CER"), ax[1].set_title("CER vs line length", fontsize=9)
        cs = np.cumsum(np.sort(ed)[::-1]) / max(ed.sum(), 1)
        ax[2].plot(np.arange(1, n + 1) / n * 100, cs * 100), ax[2].set_xlabel("% of lines (worst first)"), ax[2].set_ylabel("% of all character errors"), ax[2].grid(alpha=0.3)
        ax[2].set_title("how concentrated are the errors?", fontsize=9)
        self._fig("01_recognition/cer_distribution.png", "Per-line CER distribution and error concentration", fig)
        fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
        for a, (cnt, ttl) in zip(ax, [(subs, "top substitutions (ref -> hyp)"), (dels, "most deleted chars"), (inss, "most inserted chars")]):
            top = cnt.most_common(15)
            lab = [f"{chlabel(k[0])}->{chlabel(k[1])}" if isinstance(k, tuple) else chlabel(k) for k, _ in top]
            a.barh(range(len(top))[::-1], [v for _, v in top], color="indianred"), a.set_yticks(range(len(top))[::-1])
            a.set_yticklabels(lab, fontsize=8), a.set_title(ttl, fontsize=9)
        self._fig("01_recognition/top_confusions.png", f"Character confusions ({len(sub)} random lines)", fig)
        M["top_confusions"] = [{"ref": k[0], "hyp": k[1], "count": int(v)} for k, v in subs.most_common(30)]
        return M

    # ------------------------------------------------------------------ is the confidence trustworthy?
    def _char_outcomes(self, R: Dict[str, Any], T: float = 1.0) -> Tuple[np.ndarray, np.ndarray]:
        conf, ok = [], []
        for k, i_ds in enumerate(R["idx"]):
            if k not in R["ops"] or i_ds not in R["lp"]:
                continue
            lp = R["lp"][i_ds].astype(np.float32)
            if T != 1.0:
                lp = log_softmax_np(lp / T)
            _, cf, _ = greedy_decode(lp)
            for op, ri, hi in R["ops"][k]:
                if hi is not None and hi < len(cf):
                    conf.append(cf[hi])
                    ok.append(float(op == "match"))
        return np.array(conf), np.array(ok)

    def _calibrate_marking(self, Rval: Dict[str, Any], same_split: bool) -> Dict[str, Any]:
        info = calibrate_word_mark(Rval["pred"], Rval["gt"], Rval["char_conf"], self.ic.mark_precision)
        if info.get("threshold") is not None:
            self.cfg.llm.word_conf_thresh, self.cfg.llm.word_conf_agg = float(info["threshold"]), info["agg"]
        info["used_threshold"], info["used_agg"] = self.cfg.llm.word_conf_thresh, self.cfg.llm.word_conf_agg
        info["fitted_on"] = "val_gate lines" + (" (SAME lines as evaluated: optimistic)" if same_split else "")
        LOG.info("[CAL-MARK] %s: %s-confidence < %.4f -> flagged words are wrong %s of the time (recall %s, %s of the words flagged)", info["fitted_on"],
                 info["used_agg"], info["used_threshold"], f"{info['precision']:.0%}" if "precision" in info else "n/a",
                 f"{info['recall']:.0%}" if "recall" in info else "n/a", f"{info['share_marked']:.0%}" if "share_marked" in info else "n/a")
        with open(os.path.join(self.dirs["02_uncertainty"], "word_marks.json"), "w") as f:
            json.dump(json_safe(info), f, indent=2)
        return info

    def _calibration_block(self, R: Dict[str, Any], Rval: Dict[str, Any], same_split: bool) -> Dict[str, Any]:
        M: Dict[str, Any] = {}
        grid = np.geomspace(0.6, 6.0, 31)
        nll = []
        for T in grid:
            c, a = self._char_outcomes(Rval, float(T))
            nll.append(adaptive_calibration(c, a, 15)["nll"] if len(c) >= 15 else float("nan"))
        if not np.isfinite(nll).any():
            return {}
        T_star = float(grid[int(np.nanargmin(nll))])
        c0, a0 = self._char_outcomes(R, 1.0)
        c1, a1 = self._char_outcomes(R, T_star)
        cal0, cal1 = adaptive_calibration(c0, a0), adaptive_calibration(c1, a1)
        M["temperature"] = {"T": T_star, "fitted_on": "val_gate lines" + (" (SAME lines as evaluated: optimistic)" if same_split else ""),
                            "char_ece_before": cal0["ece"], "char_ece_after": cal1["ece"], "char_nll_before": cal0["nll"], "char_nll_after": cal1["nll"],
                            "mean_conf_before": cal0["mean_conf"], "mean_conf_after": cal1["mean_conf"], "char_accuracy": cal0["accuracy"], "n_chars": cal0["n"]}
        bad = R["ed"] > 0
        sc_conf = 1.0 - R["conf"]
        if same_split:
            fold = np.random.RandomState(0).permutation(len(R["X"])) % 2
            sc_rel = np.zeros(len(R["X"]))
            for f in (0, 1):
                mdl = ReliabilityModel().fit(R["X"][fold != f], R["cer_ps"][fold != f])
                sc_rel[fold == f] = mdl.predict(R["X"][fold == f])
        else:
            sc_rel = ReliabilityModel().fit(Rval["X"], Rval["cer_ps"]).predict(R["X"])
        R["risk_rel"] = sc_rel
        scores = {"CTC sequence confidence": sc_conf, "learned reliability (val-fitted)": sc_rel,
                  "oracle ordering (true CER)": R["cer_ps"] + 1e-9 * np.random.RandomState(0).rand(len(sc_conf))}
        rcs = {k: risk_coverage(v, R["ed"], R["reflen"]) for k, v in scores.items()}
        base = float(R["ed"].sum() / max(R["reflen"].sum(), 1))
        M["selective_prediction"] = {
            "corpus_CER_at_full_coverage": base,
            "auroc_line_has_error": {"CTC sequence confidence": auroc(sc_conf, bad), "learned reliability": auroc(sc_rel, bad)},
            "aurc": {k: v["aurc"] for k, v in rcs.items()},
            "CER_at_80pct_coverage": {k: risk_at_coverage(v, 0.8) for k, v in rcs.items()},
            "CER_at_50pct_coverage": {k: risk_at_coverage(v, 0.5) for k, v in rcs.items()},
            "coverage_at_CER_leq_2pct": {k: coverage_at_risk(v, 0.02) for k, v in rcs.items()},
            "coverage_at_CER_leq_5pct": {k: coverage_at_risk(v, 0.05) for k, v in rcs.items()}}
        sat = (R["conf"] > 0.95)
        M["confidence_saturation"] = {"share_of_lines_with_conf_above_0.95": float(sat.mean()),
                                      "CER_of_those_lines": float(R["ed"][sat].sum() / max(R["reflen"][sat].sum(), 1)) if sat.any() else float("nan"),
                                      "share_of_erroneous_lines_with_conf_above_0.95": float(sat[bad].mean()) if bad.any() else float("nan")}
        fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))
        ax[0].plot([0, 1], [0, 1], "k--", lw=1)
        for cal, lab in ((cal0, f"before (ECE {cal0['ece']:.3f})"), (cal1, f"after T={T_star:.2f} (ECE {cal1['ece']:.3f})")):
            if cal["points"]:
                ax[0].plot([p[0] for p in cal["points"]], [p[1] for p in cal["points"]], "o-", ms=4, label=lab)
        ax[0].set_xlabel("character confidence"), ax[0].set_ylabel("character accuracy"), ax[0].legend(fontsize=8), ax[0].set_title("reliability diagram, equal-mass bins", fontsize=9)
        for k, v in rcs.items():
            ax[1].plot(v["coverage"], v["risk"] * 100, label=f"{k} (AURC {v['aurc'] * 100:.2f})")
        ax[1].axhline(base * 100, color="k", ls=":", lw=1), ax[1].set_xlabel("coverage (fraction of lines answered)"), ax[1].set_ylabel("CER of answered lines (%)")
        ax[1].legend(fontsize=7), ax[1].set_title("selective prediction (risk - coverage)", fontsize=9)
        ok_ = R["cer_ps"] == 0
        ax[2].hist([R["conf"][ok_], R["conf"][~ok_]], bins=25, label=["exact lines", "lines with errors"], stacked=True)
        ax[2].set_xlabel("CTC sequence confidence"), ax[2].set_yscale("log"), ax[2].legend(fontsize=8), ax[2].set_title("confidence vs correctness (log count)", fontsize=9)
        self._fig("02_uncertainty/calibration.png", "Calibration (ECE with temperature) and selective prediction", fig)
        LOG.info("calibration: char ECE %.4f -> %.4f (T=%.2f) | line AUROC CTC conf %.3f vs learned %.3f", cal0["ece"], cal1["ece"], T_star,
                 M["selective_prediction"]["auroc_line_has_error"]["CTC sequence confidence"], M["selective_prediction"]["auroc_line_has_error"]["learned reliability"])
        return M

    # ------------------------------------------------------------------ discrete codebooks
    def _codebook_block(self, R: Dict[str, Any]) -> Dict[str, Any]:
        codec, m, M = self.data.codec, self.model, {}
        fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
        for a, key, name in zip(ax, ("use_low", "use_high"), ("stroke codebook", "component codebook")):
            u = R[key]
            p = u / max(u.sum(), 1)
            ppl = float(np.exp(-(p[p > 0] * np.log(p[p > 0])).sum()))
            M[f"{name.split()[0]}_codebook"] = {"size": int(len(u)), "perplexity": ppl, "dead_fraction": float((u == 0).mean())}
            a.bar(range(len(u)), np.sort(u)[::-1], color="teal"), a.set_yscale("log")
            a.set_title(f"{name}: usage (perplexity {ppl:.1f}/{len(u)}, dead {100 * (u == 0).mean():.1f}%)", fontsize=9), a.set_xlabel("code (sorted)")
        self._fig("06_codebooks/codebook_usage.png", "Codebook utilisation on the evaluation split", fig)
        cc_m = R["code_char"]
        rows = np.argsort(-cc_m.sum(1))[:40]
        cols = np.argsort(-cc_m.sum(0))[:30]
        sub_m = cc_m[np.ix_(rows, cols)]
        sub_m = sub_m / np.maximum(sub_m.sum(1, keepdims=True), 1)
        M.update(code_char_stats(cc_m, min_count=20))
        fig, ax = plt.subplots(figsize=(11, 8))
        im = ax.imshow(sub_m, aspect="auto", cmap="magma")
        ax.set_xticks(range(len(cols))), ax.set_xticklabels([chlabel(codec.i2c.get(int(c), "blank")) for c in cols], fontsize=7)
        ax.set_yticks(range(len(rows))), ax.set_yticklabels([f"code {int(r)}" for r in rows], fontsize=6)
        ax.set_title(f"which character does each component code stand for?  (codes with >= 20 emissions: purity {M['component_code_purity']:.2f}, "
                     f"chance {M['component_code_purity_chance']:.2f}, NMI {M['component_code_nmi']:.2f})", fontsize=8)
        fig.colorbar(im, ax=ax, fraction=0.03)
        self._fig("06_codebooks/code_vs_character.png", "Component code -> emitted character (row-normalised)", fig)
        fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
        for a, vq, key, name in zip(ax, (m.vq_low, m.vq_high), ("use_low", "use_high"), ("stroke", "component")):
            P = pca2(vq.embed.detach().cpu().numpy())
            a.scatter(P[:, 0], P[:, 1], s=8 + 120 * R[key] / max(R[key].max(), 1), alpha=0.6, c=np.log1p(R[key]), cmap="viridis")
            a.set_title(f"{name} codebook (PCA, size ~ usage)", fontsize=9)
        self._fig("06_codebooks/codebook_pca.png", "Codebook geometry", fig)
        return M

    # ------------------------------------------------------------------ connected task: style
    def _style_block(self, R: Dict[str, Any]) -> Dict[str, Any]:
        dm, codec, m, M = self.data, self.data.codec, self.model, {}
        sp, st = dm.denorm_style(R["style_pred"]), R["style_raw"].astype(np.float64)
        okm = np.isfinite(st)
        sd, mu = dm.style_std.astype(np.float64), dm.style_mean.astype(np.float64)
        M["style_measurable_fraction"] = dict(zip(STYLE_NAMES, okm.mean(0).tolist()))
        r2, mae, mae_base, corr = [], [], [], []
        for j in range(NUM_DESC):
            k = okm[:, j]
            if k.sum() < 30:
                r2.append(np.nan), mae.append(np.nan), mae_base.append(np.nan), corr.append(np.nan)
                continue
            y, p = st[k, j], sp[k, j]
            sst = ((y - y.mean()) ** 2).sum()
            r2.append(float(1 - ((y - p) ** 2).sum() / sst) if y.std() / sd[j] >= 0.2 else np.nan)
            mae.append(float(np.abs(y - p).mean())), mae_base.append(float(np.abs(y - mu[j]).mean()))
            corr.append(float(np.corrcoef(y, p)[0, 1]) if p.std() > 1e-9 else np.nan)
        M["style_r2"], M["style_corr"] = dict(zip(STYLE_NAMES, r2)), dict(zip(STYLE_NAMES, corr))
        M["style_mae"], M["style_mae_of_the_train_mean"] = dict(zip(STYLE_NAMES, mae)), dict(zip(STYLE_NAMES, mae_base))
        M["style_r2_text_only_floor_train"] = dict(zip(STYLE_NAMES, dm.text_baseline_r2.tolist()))
        valid = R["cl_true"] >= 0
        M["style_cluster_acc"] = float((R["cl_pred"] == R["cl_true"])[valid].mean()) if valid.any() else float("nan")
        fig, ax = plt.subplots(1, NUM_DESC, figsize=(4 * NUM_DESC, 3.8))
        for j, a in enumerate(ax):
            k = okm[:, j]
            if k.sum() >= 2:
                a.scatter(st[k, j], sp[k, j], s=5, alpha=0.4)
                lo, hi = min(st[k, j].min(), sp[k, j].min()), max(st[k, j].max(), sp[k, j].max())
                a.plot([lo, hi], [lo, hi], "k--", lw=1)
            a.set_title(f"{STYLE_NAMES[j]}  R2={r2[j]:.2f}  r={corr[j]:.2f}\nmeasured on {100 * okm[:, j].mean():.0f}% of lines", fontsize=8)
            a.set_xlabel("target (classical image analysis)"), a.set_ylabel("predicted")
        self._fig("05_style/style_pred_vs_true.png", "Connected task: style-descriptor regression (measured entries only)", fig)
        K = m.style_enc.cls.out_features
        cm = np.zeros((K, K))
        for t, p in zip(R["cl_true"][valid], R["cl_pred"][valid]):
            cm[t, p] += 1
        fig, ax = plt.subplots(figsize=(6, 5))
        ax.imshow(cm / np.maximum(cm.sum(1, keepdims=True), 1), cmap="Blues"), ax.set_xlabel("predicted style cluster"), ax.set_ylabel("true style cluster")
        ax.set_title(f"style-cluster confusion (acc {M['style_cluster_acc']:.2f})", fontsize=9)
        self._fig("05_style/style_confusion.png", "Style-cluster confusion matrix", fig)
        fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
        Ps, Pc = pca2(R["style_emb"]), pca2(R["content"])
        k0 = okm[:, 0]
        sc = ax[0].scatter(Ps[k0, 0], Ps[k0, 1], c=st[k0, 0], s=6, cmap="coolwarm"), ax[0].set_title("style embedding, colour = slant", fontsize=9)
        fig.colorbar(sc[0], ax=ax[0])
        clv = np.asarray(R["cl_true"]).reshape(-1)
        okc = clv >= 0
        ax[1].scatter(Ps[okc, 0], Ps[okc, 1], c=clv[okc], s=6, cmap="tab10", vmin=-0.5, vmax=9.5), ax[1].set_title("style embedding, colour = style cluster", fontsize=9)
        sc = ax[2].scatter(Pc[k0, 0], Pc[k0, 1], c=st[k0, 0], s=6, cmap="coolwarm"), ax[2].set_title("CONTENT embedding, colour = slant (should be mixed)", fontsize=9)
        fig.colorbar(sc[0], ax=ax[2])
        self._fig("05_style/style_vs_content_embedding.png", "Style vs content embedding spaces (PCA)", fig)
        Z = np.nan_to_num((st - mu) / sd, nan=0.0)
        r2s = ridge_cv_r2(R["style_emb"], Z, okm)
        r2c = ridge_cv_r2(R["content"], Z, okm)
        floor = text_baseline_r2(R["gt"], Z, okm, codec.units)
        lowv = okm.sum(0) < 30
        r2s, r2c, floor = (np.where(lowv, np.nan, v) for v in (r2s, r2c, floor))
        M["probe_r2_style_emb"], M["probe_r2_content_emb"] = dict(zip(STYLE_NAMES, r2s.tolist())), dict(zip(STYLE_NAMES, r2c.tolist()))
        M["probe_r2_transcription_only"] = dict(zip(STYLE_NAMES, floor.tolist()))
        exc = np.maximum(r2c - floor, 0.0)
        M["content_leakage_excess_over_transcription"] = dict(zip(STYLE_NAMES, exc.tolist()))
        M["content_leakage_excess_mean"] = float(np.nanmean(exc)) if np.isfinite(exc).any() else float("nan")
        fig, ax = plt.subplots(1, 2, figsize=(13, 4))
        xx = np.arange(NUM_DESC)
        ax[0].bar(xx - 0.27, r2s, 0.27, label="style embedding"), ax[0].bar(xx, r2c, 0.27, label="content embedding")
        ax[0].bar(xx + 0.27, floor, 0.27, label="transcription only (floor)", color="gray")
        ax[0].set_xticks(xx), ax[0].set_xticklabels(STYLE_NAMES, rotation=20, fontsize=8), ax[0].set_ylabel("5-fold ridge R2 (measured entries)"), ax[0].legend(fontsize=8)
        ax[0].set_title("how much style can be read out of each stream?", fontsize=9)

        def corr_to_pcs(E):
            Ec = E - E.mean(0)
            _, _, Vt = np.linalg.svd(Ec, full_matrices=False)
            Zp = Ec @ Vt[:8].T
            Zs = Zp / (Zp.std(0) + 1e-9)
            C = np.full((NUM_DESC, Zs.shape[1]), np.nan)
            for j in range(NUM_DESC):
                k = okm[:, j]
                if k.sum() > 30:
                    y = (st[k, j] - st[k, j].mean()) / (st[k, j].std() + 1e-9)
                    C[j] = (y[:, None] * Zs[k]).mean(0)
            return C

        Cm = np.concatenate([corr_to_pcs(R["style_emb"]), np.full((NUM_DESC, 1), np.nan), corr_to_pcs(R["content"])], 1)
        im = ax[1].imshow(Cm, cmap="seismic", vmin=-1, vmax=1, aspect="auto")
        ax[1].set_yticks(range(NUM_DESC)), ax[1].set_yticklabels(STYLE_NAMES, fontsize=8), ax[1].set_xticks([3.5, 12.5])
        ax[1].set_xticklabels(["style PCs 1-8", "content PCs 1-8"], fontsize=8), ax[1].set_title("correlation: descriptors vs embedding PCs", fontsize=9)
        fig.colorbar(im, ax=ax[1])
        self._fig("05_style/disentanglement_probe.png", "Content / style disentanglement probe", fig)
        G = R["gamma"]
        if len(G) > 5 and G.shape[1] > 1:
            Gc = G - G.mean(0, keepdims=True)
            M["film"] = {"gamma_rms": float(np.sqrt((G ** 2).mean())), "gamma_between_line_std": float(G.std(0).mean()),
                         "share_of_gamma_energy_that_varies_between_lines": float((Gc ** 2).sum() / max((G ** 2).sum(), 1e-12)),
                         "slant_r2_from_gamma": float(ridge_cv_r2(G, Z[:, :1], okm[:, :1])[0])}
            k0i = np.flatnonzero(okm[:, 0])
            if len(k0i) > 5:
                o = k0i[np.argsort(st[k0i, 0])]
                fig, ax = plt.subplots(figsize=(10, 4))
                vmax = max(float(np.abs(Gc).max()), 1e-6)
                im = ax.imshow(Gc[o], aspect="auto", cmap="coolwarm", vmin=-vmax, vmax=vmax)
                ax.set_xlabel("feature channel"), ax.set_ylabel("lines sorted by measured slant"), fig.colorbar(im, ax=ax)
                ax.set_title(f"style-FiLM scale gamma minus its mean over lines  (between-line share of energy {M['film']['share_of_gamma_energy_that_varies_between_lines']:.2f})", fontsize=8)
                self._fig("05_style/film_gamma.png", "Style FiLM modulation (centred)", fig)
        return M

    # ------------------------------------------------------------------ robustness
    def _robustness(self, ds) -> Dict[str, Any]:
        sub = self._random_subset(range(len(ds)), self.ic.robustness_lines, 5)
        clean, _ = self._predict(ds, sub)
        res = {}
        for name, fn in PERTURBATIONS.items():
            pr, _ = self._predict(ds, sub, None if name == "clean" else fn)
            ed = np.array([edit_distance(ds.oracle[i], pr[i]) for i in sub], np.float64)
            rl = np.array([len(ds.oracle[i]) for i in sub], np.float64)
            lo, hi = bootstrap_cer_ci(ed, rl, self.ic.bootstrap, 0, groups=np.asarray(sub) // max(1, int(self.ic.bootstrap_block)))
            res[name] = {"cer": float(ed.sum() / max(rl.sum(), 1)), "cer_ci95": [lo, hi], "n": len(sub), "consistency": 1 - float(np.mean([cer(clean[i], pr[i]) for i in sub]))}
        fig, ax = plt.subplots(1, 2, figsize=(12, 4))
        names = list(res)
        yerr = np.array([[res[k]["cer"] - res[k]["cer_ci95"][0], res[k]["cer_ci95"][1] - res[k]["cer"]] for k in names]).T
        ax[0].bar(names, [res[k]["cer"] for k in names], yerr=np.maximum(yerr, 0), color="indianred", capsize=2), ax[0].set_ylabel("CER vs ground truth")
        ax[1].bar(names, [res[k]["consistency"] for k in names], color="seagreen"), ax[1].set_ylabel("consistency = 1 - CER(clean pred, perturbed pred)")
        for a in ax:
            a.tick_params(axis="x", rotation=35, labelsize=8)
        ax[0].set_title(f"robustness to input perturbations ({len(sub)} random lines, 95% CI)", fontsize=9), ax[1].set_title("prediction consistency (model inertia)", fontsize=9)
        self._fig("07_robustness/robustness_profile.png", "Robustness / consistency under perturbations", fig)
        return res

    # ------------------------------------------------------------------ self-distillation diagnostics (hidden ground truth)
    def _pl_reliability(self) -> Dict[str, Any]:
        """How trustworthy are the lines the legacy rule WOULD select?  Checked against the hidden transcriptions of the unlabeled train lines."""
        ds, dm, pc = self.data.train_ds, self.data, self.cfg.train.pseudo
        unl, cal = dm.unlabeled_idx, dm.calib_idx
        if len(unl) == 0:
            return {}
        idx = self._random_subset(unl, self.ic.pl_diag_samples, 3)
        U = self._collect(ds, idx, extras=False, cons=True)
        res: Dict[str, Any] = {"n_unlabeled_lines": len(idx), "CER_on_unlabeled": float(U["ed"].sum() / max(U["reflen"].sum(), 1))}
        bad = U["ed"] > 0
        curves = {"CTC sequence confidence": risk_coverage(1.0 - U["conf"], U["ed"], U["reflen"]),
                  "oracle ordering": risk_coverage(U["cer_ps"] + 1e-9 * np.random.RandomState(0).rand(len(idx)), U["ed"], U["reflen"])}
        res["auroc_ctc_confidence"] = auroc(1.0 - U["conf"], bad)
        if len(cal) >= pc.min_calib:
            C = self._collect(ds, cal, extras=False, cons=True)
            rule = SelectionRule.fit(C["X"], C["ed"].astype(np.float64), C["reflen"].astype(np.float64), pc, seed=self.cfg.run.seed)
            risk = rule.predict(U["X"])
            curves["learned line risk (calibration-fitted)"] = risk_coverage(risk, U["ed"], U["reflen"])
            res["auroc_learned_risk"] = auroc(risk, bad)
            res["calibration"] = {k: float(v) for k, v in rule.info.items()}
            res["selected_by_rule_at_z"] = {}
            for z_ in (0.0, 1.0, pc.risk_z):
                th_, k_, _c = risk_control_threshold(rule.oof, C["ed"].astype(np.float64), C["reflen"].astype(np.float64), pc.target_cer, z_, pc.min_selected_calib, pc.risk_smoothing)
                s_ = risk <= th_ if (k_ > 0 and np.isfinite(th_)) else np.zeros(len(idx), bool)
                res["selected_by_rule_at_z"][f"z={z_}"] = {"calibration_lines_kept": int(k_), "coverage_unlabeled": float(s_.mean()),
                                                         "CER_vs_hidden_GT": float(U["ed"][s_].sum() / max(U["reflen"][s_].sum(), 1)) if s_.any() else float("nan")}
        else:
            LOG.warning("no / too few calibration lines (%d): the learned line reliability could not be evaluated", len(cal))
        sat = U["conf"] > 0.95
        res["ctc_confidence_saturation"] = {"share_of_lines_above_0.95": float(sat.mean()),
                                            "CER_of_those_lines": float(U["ed"][sat].sum() / max(U["reflen"][sat].sum(), 1)) if sat.any() else float("nan")}
        fig, ax = plt.subplots(1, 2, figsize=(13, 4.4))
        for k, v in curves.items():
            ax[0].plot(v["coverage"], v["risk"] * 100, label=f"{k} (AURC {v['aurc'] * 100:.2f})")
        ax[0].axhline(pc.target_cer * 100, color="k", ls="--", lw=1, label=f"target CER {100 * pc.target_cer:.1f}%")
        ax[0].set_xlabel("fraction of unlabeled lines selected"), ax[0].set_ylabel("CER of the selected lines vs hidden GT (%)")
        ax[0].set_yscale("log"), ax[0].legend(fontsize=7), ax[0].set_title("which signal ranks pseudo-label quality?", fontsize=9)
        ths = np.linspace(0.3, 0.999, 24)
        cov = [float((U["conf"] >= t).mean()) for t in ths]
        cc = [float(U["ed"][U["conf"] >= t].sum() / max(U["reflen"][U["conf"] >= t].sum(), 1)) if (U["conf"] >= t).any() else np.nan for t in ths]
        ax[1].plot(ths, cov, "o-", label="coverage"), ax[1].plot(ths, np.array(cc) * 100, "^-", label="CER of selected (%)")
        ax[1].set_xlabel("CTC sequence-confidence threshold"), ax[1].legend(fontsize=8), ax[1].set_title("saturation: raising the confidence threshold hardly reduces the CER", fontsize=9)
        self._fig("08_self_distillation/pseudo_label_reliability.png", "How trustworthy are the line-level pseudo-labels the model would select?", fig)
        return res

    # ------------------------------------------------------------------ [LLM-N] constrained LLM rescoring
    @torch.no_grad()
    def _nbest_pass(self, ds, indices) -> Dict[int, Dict[str, Any]]:
        res: Dict[int, Dict[str, Any]] = {}
        codec = self.data.codec
        ov = self._wsd_lp.get(id(ds))
        if ov is not None:
            miss = [int(i) for i in indices if int(i) not in ov]
            lps = {int(i): ov[int(i)] for i in indices if int(i) in ov}
            if miss:
                lps.update(self._lp_views(miss, ds, ()))
        else:
            lps = self._lp_views([int(i) for i in indices], ds, getattr(self, "_ens_views", ()))
        for i, lp in tqdm(lps.items(), desc="N-best", leave=False):
            nb = ctc_prefix_beam_search(lp, self.ic.rescore_beam)
            res[int(i)] = {"texts": [codec.decode_ids(ids) for ids, _ in nb], "ctc": [float(l) for _, l in nb], "greedy": codec.decode_ids(greedy_decode(lp)[0]),
                           "ref": ds.oracle[int(i)], "lm_texts": [detokenize(ds.display(int(i), codec.decode_ids(ids))) if ds.detok(int(i)) else ds.display(int(i), codec.decode_ids(ids)) for ids, _ in nb]}
        return res

    def _llm_rescoring_eval(self, ds) -> Dict[str, Any]:
        ic = self.ic
        if not ic.llm_rescoring:
            return {}
        if not self.llm.can_score():
            LOG.warning("LLM rescoring evaluation skipped (needs cfg.llm.backend='causal' and a loadable model).")
            return {}
        rng = np.random.RandomState(0)
        val_perm = rng.permutation(np.asarray(self.data.val_gate_idx))
        n_tune = min(ic.rescore_val_lines, len(val_perm) // 2 if ds is self.data.val_ds else len(val_perm))
        val_idx = sorted(int(i) for i in val_perm[:n_tune])
        pool = [int(i) for i in val_perm[n_tune:]] if ds is self.data.val_ds else list(range(len(ds)))
        ev_idx = sorted(pool if ic.rescore_test_lines is None else [int(i) for i in np.random.RandomState(1).permutation(pool)[:ic.rescore_test_lines]])
        nb_val, nb_ev = self._nbest_pass(self.data.val_ds, val_idx), self._nbest_pass(ds, ev_idx)

        def lm_scores(nb):
            det = [t for v in nb.values() for t in v["lm_texts"]]
            lm, nt = self.llm.score_texts(det)
            pos = 0
            for v in nb.values():
                k = len(v["texts"])
                v["lm"], v["ntok"] = lm[pos:pos + k], nt[pos:pos + k]
                pos += k

        lm_scores(nb_val), lm_scores(nb_ev)

        def choose(v, beta, rho):
            return int(np.argmax(fuse_scores(v["ctc"], v["lm"], v["ntok"], beta, rho)))

        def corpus(nb, beta, rho):
            ed = np.array([edit_distance(v["ref"], v["texts"][choose(v, beta, rho)]) for v in nb.values()], np.float64)
            return ed, np.array([len(v["ref"]) for v in nb.values()], np.float64)

        grid: Dict[str, float] = {}
        best = (float("inf"), 0.0, 0.0)
        betas, rhos = list(ic.rescore_betas), list(ic.rescore_rhos)
        done: set = set()
        extensions = rho_ext = 0
        while True:
            for beta in betas:
                for rho in (rhos if beta > 0 else (0.0,)):
                    if (beta, rho) in done:
                        continue
                    done.add((beta, rho))
                    ed, rl = corpus(nb_val, beta, rho)
                    c = float(ed.sum() / max(rl.sum(), 1))
                    grid[f"beta={beta},rho={rho}"] = c
                    if c < best[0] - 1e-12:
                        best = (c, beta, rho)
            if ic.rescore_extend_grid and best[1] >= max(betas) and extensions < 5:
                betas.append(float(max(betas)) * 2.0)
                extensions += 1
                continue
            if ic.rescore_extend_rho and best[1] > 0 and best[2] >= max(rhos) and rho_ext < 3:
                rhos.append(float(max(rhos)) + 2.0)
                rho_ext += 1
                continue
            break
        _, beta, rho = best
        ed_vb, rl_v = corpus(nb_val, 0.0, 0.0)
        ed_vn, _ = corpus(nb_val, beta, rho)
        blk = max(1, int(ic.bootstrap_block))
        gv, ge = np.asarray(list(nb_val.keys())) // blk, np.asarray(list(nb_ev.keys())) // blk
        val_gate = paired_bootstrap_delta(ed_vn, ed_vb, rl_v, ic.bootstrap, 1, groups=gv, n_tests=max(1, len(grid) - 1))
        adopted = bool(beta > 0 and val_gate["hi_adj"] < 0)
        self._rescore = {"beta": float(beta), "rho": float(rho), "adopted": adopted}
        self._nb_eval = nb_ev
        ed_base, rl = corpus(nb_ev, 0.0, 0.0)
        ed_new, _ = corpus(nb_ev, beta, rho)
        ed_greedy = np.array([edit_distance(v["ref"], v["greedy"]) for v in nb_ev.values()], np.float64)
        ed_orcl = np.array([min(edit_distance(v["ref"], t) for t in v["texts"]) for v in nb_ev.values()], np.float64)
        changed = float(np.mean([choose(v, beta, rho) != 0 for v in nb_ev.values()]))
        tot = max(rl.sum(), 1)
        out = {"n_tune_lines": len(nb_val), "n_eval_lines": len(nb_ev), "beta": beta, "rho": rho, "beta_grid_extended_times": extensions,
               "beta_is_interior": bool(beta < max(betas)), "rho_grid_extended_times": rho_ext, "rho_is_interior": bool(rho < max(rhos) or beta == 0),
               "val_cer_grid": grid, "val_cer_baseline": grid.get("beta=0.0,rho=0.0"), "CER_greedy": float(ed_greedy.sum() / tot),
               "CER_beam_top1": float(ed_base.sum() / tot), "CER_rescored": float(ed_new.sum() / tot), "CER_oracle_nbest": float(ed_orcl.sum() / tot),
               "CER_rescored_ci95": list(bootstrap_cer_ci(ed_new, rl, ic.bootstrap, 0, groups=ge)), "fraction_of_lines_changed": changed,
               "val_paired_delta_vs_beam": val_gate, "adopted_on_validation": adopted,
               "paired_bootstrap_rescored_minus_beam": paired_bootstrap_delta(ed_new, ed_base, rl, ic.bootstrap, 0, groups=ge), "llm": self.cfg.llm.model_name}
        LOG.info("[LLM-N] beta=%.2f rho=%.1f (tuned on %d val_gate lines) | eval CER beam %.4f -> rescored %.4f (oracle N-best %.4f) | %s", beta, rho, len(nb_val),
                 out["CER_beam_top1"], out["CER_rescored"], out["CER_oracle_nbest"], "ADOPTED" if adopted else "not adopted")
        fig, ax = plt.subplots(figsize=(6.5, 4))
        for r_ in sorted({float(k.split("rho=")[1]) for k in grid}):
            xs = sorted({b for b, _ in done if (f"beta={b},rho={r_}" in grid)})
            if len(xs) > 1:
                ax.plot(xs, [grid[f"beta={b},rho={r_}"] for b in xs], "o-", label=f"rho={r_}")
        ax.axhline(grid.get("beta=0.0,rho=0.0", np.nan), color="k", ls="--", label="recogniser only")
        ax.set_xscale("symlog", linthresh=0.1), ax.set_xlabel("beta (LM weight)"), ax.set_ylabel("val_gate CER"), ax.legend(fontsize=8)
        ax.set_title("LLM N-best rescoring: tuning on val_gate lines", fontsize=9)
        self._fig("09_language_model/llm_rescoring_tuning.png", "LLM N-best rescoring - tuning curve (val_gate lines only)", fig)
        with open(os.path.join(self.dirs["09_language_model"], "llm_rescoring.json"), "w") as f:
            json.dump(json_safe(out), f, indent=2)
        return out

    # ------------------------------------------------------------------ gallery picks
    def _pick(self, R, n_total: int) -> List[Tuple[int, str]]:
        """Gallery lines with their tag: the worst lines, the longest exactly-read lines, and random lines."""
        ling = np.array([is_linguistic(g) for g in R["gt"]])
        order = np.array([j for j in np.argsort(R["cer_ps"], kind="stable") if ling[j]], dtype=int)
        k = max(1, n_total // 3)
        worst = [int(R["idx"][j]) for j in order[::-1][:k] if R["cer_ps"][j] > 0]
        longest_ok = [j for j in order if R["cer_ps"][j] == 0]
        best = [int(R["idx"][j]) for j in sorted(longest_ok, key=lambda j: -R["reflen"][j])[:k]]
        taken = set(worst) | set(best)
        ok_idx = {int(i) for i, g in zip(R["idx"], ling) if g}
        rest = [int(i) for i in np.random.RandomState(0).permutation(R["idx"]) if int(i) not in taken and int(i) in ok_idx]
        picked = [(i, "worst") for i in worst] + [(i, "best") for i in best] + [(i, "random") for i in rest]
        return picked[:n_total]

    # ------------------------------------------------------------------ run
    def run(self) -> Dict[str, Any]:
        cfg, ic = self.cfg, self.ic
        if self.model is None:
            self.load()
        for d in self.dirs.values():
            os.makedirs(d, exist_ok=True)
        if ic.split != "test":
            LOG.warning("evaluating on '%s': the checkpoint was SELECTED on val_sel, so this number is optimistic and must not be reported as the result.", ic.split)
        ds = {"test": self.data.test_ds, "val": self.data.val_ds}[ic.split]
        look = log_test_look(cfg, self.ckpt_info, ic.split)
        idx = None if ic.max_eval_samples is None else self._random_subset(range(len(ds)), ic.max_eval_samples, 9)
        t0 = time.time()
        all_idx = list(range(len(ds))) if idx is None else idx
        sub_ids = self._random_subset(all_idx, ALIGN_CAP, 4)
        R = self._collect(ds, idx, extras=True, cons=True, keep_lp=sub_ids)
        n = len(R["gt"])
        same_split = ds is self.data.val_ds
        if same_split and idx is None:
            Rval = R
        else:
            val_sel = self._random_subset([int(i) for i in self.data.val_gate_idx], ic.selective_val_lines, 6)
            Rval = self._collect(self.data.val_ds, val_sel, extras=False, cons=True, keep_lp=self._random_subset(val_sel, ALIGN_CAP, 7))
        self._align_cache(R, sub_ids)
        if Rval is not R:
            self._align_cache(Rval, list(Rval["lp"].keys()))
        blk = max(1, int(ic.bootstrap_block))
        g_ci = np.asarray(R["idx"]) // blk
        cer_ci = bootstrap_cer_ci(R["ed"], R["reflen"], ic.bootstrap, 0, groups=g_ci)
        wer_ci = bootstrap_cer_ci(R["ed_w"], R["reflen_w"], ic.bootstrap, 0, groups=g_ci)
        metrics: Dict[str, Any] = {
            "checkpoint": self.ckpt_info, "split": ic.split, "n_lines": n, "corpora": ds.domain_names,
            "SYNTHETIC_SMOKE_TEST_NOT_FOR_PUBLICATION": bool(self.data.is_synthetic),
            "CER": float(R["ed"].sum() / max(R["reflen"].sum(), 1)), "CER_ci95": list(cer_ci),
            "WER": float(R["ed_w"].sum() / max(R["reflen_w"].sum(), 1)), "WER_ci95": list(wer_ci),
            "mean_line_CER": float(R["cer_ps"].mean()), "exact_line_accuracy": float((R["cer_ps"] == 0).mean()),
            "labeled_fraction": cfg.data.labeled_fraction, "calibration_fraction": cfg.data.calib_fraction,
            "model_selection": "val_sel CER (EMA or student); all post-hoc tuning on val_gate (see protocol/protocol.json)",
            "evaluation_looks_so_far": look, "val_sel_lines": int(len(self.data.val_sel_idx)), "val_gate_lines": int(len(self.data.val_gate_idx)),
            "provenance": self.data.provenance}
        LOG.info("[%s] CER %.4f [%.4f, %.4f] | WER %.4f [%.4f, %.4f] | exact lines %.3f (%d lines)", ic.split, metrics["CER"], cer_ci[0], cer_ci[1], metrics["WER"],
                 wer_ci[0], wer_ci[1], metrics["exact_line_accuracy"], n)
        metrics.update(self._recognition_block(R, ds))
        cal = self._calibration_block(R, Rval, same_split)
        if cal:
            metrics["calibration"] = cal
        metrics["uncertainty_marking"] = self._calibrate_marking(Rval, same_split)
        metrics["error_structure"] = self._error_structure_block(R)
        metrics.update(self._codebook_block(R))
        metrics.update(self._style_block(R))
        LOG.info("global analyses done (%.0fs)", time.time() - t0)
        post = self._posthoc_block(ds, R)
        if post:
            metrics["posthoc"] = post
            metrics["CER_final_with_validated_posthoc"] = post["test_CER_final"]
            with open(os.path.join(self.dirs["04_posthoc"], "posthoc.json"), "w") as f:
                json.dump(json_safe(post), f, indent=2)
            R["ed_final"] = np.asarray(self.run_extra.get("final_edits", R["ed"]))
        metrics["robustness"] = self._robustness(ds)
        metrics["pseudo_label_reliability"] = self._pl_reliability()
        resc = self._llm_rescoring_eval(ds)
        if resc:
            metrics["llm_rescoring"] = resc
            adopted_ = bool(resc.get("adopted_on_validation"))
            metrics["CER_system_final"] = float(resc["CER_rescored"] if adopted_ else resc["CER_beam_top1"])
            metrics["CER_system_final_note"] = ("beam top-1 + validated options: " + ("LLM N-best rescoring ADOPTED on val_gate" if adopted_ else
                                                "LLM rescoring NOT adopted (no significant val_gate gain)") + "; CER above is the greedy reading without it")
        metrics["interpretability"] = self.run_interpretability(ds, R)
        curves = os.path.join(cfg.curve_dir, "curves_all.png")
        if os.path.exists(curves):
            shutil.copy(curves, os.path.join(self.dirs["00_summary"], "training_curves.png"))
            self.fig_list.insert(0, ("00_summary/training_curves.png", "Training curves"))

        # ---- per-sample explanations + translation ----------------------------------------------------------------------------------
        vis = self._pick(R, ic.n_vis)
        ctxs = [self._explain(ds, i, full=True, tag=tag) for i, tag in tqdm(vis, desc="explaining samples", leave=False)]
        nl = [c["attn_nonlocal"] for c in ctxs if "attn_nonlocal" in c]
        if nl:
            metrics["attention_rollout_nonlocal_share"] = float(np.mean(nl))
        sk = [c["attn_sink"] for c in ctxs if np.isfinite(c.get("attn_sink", float("nan")))]
        if sk:
            metrics["attention_sink_mass"] = float(np.mean(sk))
        taken = {c["i"] for c in ctxs}
        need = max(ic.n_translate - len(ctxs), 0)
        gt_of = {int(i): g for i, g in zip(R["idx"], R["gt"])}
        extra = [int(j) for j in np.random.RandomState(2).permutation(R["idx"]) if int(j) not in taken and is_linguistic(gt_of[int(j)])][:need]
        tr_ctxs = ctxs[:ic.n_translate] + [self._explain(ds, j, full=False, tag="random") for j in extra]
        rk_ = R.get("risk_rel")
        self._risk_by_idx = ({int(i): float(v) for i, v in zip(R["idx"], rk_)} if rk_ is not None else {int(i): float(1.0 - c) for i, c in zip(R["idx"], R["conf"])})
        tr = self._translate_block(ds, tr_ctxs)
        for k, ctx in enumerate(ctxs):
            self._sample_figs(ctx, f"sample_{k:02d}_{ctx['tag']}_line{ctx['i']:05d}")
        if tr:
            metrics["translation"] = {k: v for k, v in tr.items() if k != "rows"}
            with open(os.path.join(self.dirs["11_translation"], "translations.json"), "w", encoding="utf-8") as f:
                json.dump(json_safe(tr), f, indent=2, ensure_ascii=False, allow_nan=False)

        # ---- files -----------------------------------------------------------------------------------------------------------------
        with open(os.path.join(self.dirs["01_recognition"], "predictions.csv"), "w", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            pp = R.get("pred_posthoc")
            w.writerow(["line_index", "corpus", "ground_truth", "prediction", "CER", "confidence", "view_disagreement", "edits", "ref_len"]
                       + (["prediction_posthoc", "CER_posthoc"] if pp else []) + [f"measured_{s}" for s in STYLE_NAMES] + [f"pred_{s}" for s in STYLE_NAMES])
            st_, sp_ = R["style_raw"], self.data.denorm_style(R["style_pred"])
            for k in range(n):
                i = int(R["idx"][k])
                ex = ([ds.display(i, pp[i]), f"{cer(R['gt'][k], pp[i]):.4f}"] if pp else [])
                w.writerow([i, ds.domain_names[int(ds.domain[i])], ds.display(i), ds.display(i, R["pred"][k]), f"{R['cer_ps'][k]:.4f}", f"{R['conf'][k]:.4f}",
                            "" if not np.isfinite(R["cons"][k]) else f"{R['cons'][k]:.4f}", int(R["ed"][k]), int(R["reflen"][k])] + ex
                           + ["" if not np.isfinite(v) else f"{v:.4f}" for v in st_[k]] + [f"{v:.4f}" for v in sp_[k]])
        with open(os.path.join(self.dirs["00_summary"], "metrics.json"), "w", encoding="utf-8") as f:
            json.dump(json_safe(metrics), f, indent=2, allow_nan=False, ensure_ascii=False)
        with open(os.path.join(self.dirs["00_summary"], "config_used.json"), "w") as f:
            json.dump(cfg.to_dict(), f, indent=2)
        if self.data.is_synthetic:
            with open(os.path.join(self.out, "SYNTHETIC_SMOKE_TEST_NOT_FOR_PUBLICATION.txt"), "w") as f:
                f.write("These outputs come from generated fake handwriting; the numbers have no scientific meaning.\n")
        if ic.html_report:
            self._html(metrics)
        if ic.report:
            try:
                from .report import build_run_report
                self._write_report_extras(R, ds)
                build_run_report(self.out, cfg.run_dir, curve_dir=cfg.curve_dir)
            except Exception:
                LOG.exception("the run report could not be built (the metrics and figures above are complete)")
        LOG.info("Inference outputs written to %s (%d figures)", self.out, len(self.fig_list) + len(ctxs) * len(SAMPLE_VIZ))
        return metrics

    # ------------------------------------------------------------------ report extras (what only the model can supply)
    @torch.no_grad()
    def _evidence_pass(self, ds, ids: Sequence[int], view: str = "shear +0.08"):
        self.model.eval()
        for batch in self.data.eval_loader(ds, [int(i) for i in ids]):
            batch = to_device(batch, self.device)
            x, w = batch["image"], batch["width"]
            o = self.model(x, w)
            o2 = self.model(apply_perturbation(PERTURBATIONS[view], x, w), w)
            lp, aux = o["log_probs"].float().cpu().numpy(), o["aux_log_probs"].float().cpu().numpy()
            lp2, lens = o2["log_probs"].float().cpu().numpy(), o["out_lens"].cpu().tolist()
            for b in range(lp.shape[0]):
                L = int(lens[b])
                ids_, st, en, Fm = char_evidence(lp[b, :L], lp2[b, :L], aux[b, :L])
                yield int(batch["index"][b]), L, ids_, st, en, Fm, lp[b, :L]

    def _write_report_extras(self, R: Dict[str, Any], ds) -> None:
        """00_summary/v11_extras.json (+ the PNGs of the reading traces).  The trust rule fitted on TRAIN calibration lines is applied, unchanged,
        to the evaluation lines: the report shows whether it transfers.  Ground truth is read here for REPORTING only."""
        codec, fm = self.data.codec, self.frag_model
        order = sorted(int(i) for i in R["idx"])
        pos = {int(i): k for k, i in enumerate(R["idx"])}
        ref_of = {int(i): g for i, g in zip(R["idx"], R["gt"])}
        pred_of = {int(i): p for i, p in zip(R["idx"], R["pred"])}
        cer_of = {int(i): float(c) for i, c in zip(R["idx"], R["cer_ps"])}
        ex = self.run_extra or {}
        X: Dict[str, Any] = {"version": "v11", "strata": [int(ds.stratum[i]) for i in order], "corpus": [ds.domain_names[int(ds.domain[i])] for i in order]}
        if ex.get("wsd_test_pred") is not None:
            tid = ex["test_ids"]
            wp, we, wc = dict(zip(tid, ex["wsd_test_pred"])), dict(zip(tid, ex["wsd_test_edits"].tolist())), dict(zip(tid, ex["wsd_test_cluster"]))
            X["wsd_pred"], X["wsd_edits"], X["wsd_cluster"] = [ds.display(i, wp[i]) for i in order], [int(we[i]) for i in order], [int(wc[i]) for i in order]
        theta = (self.frag_state or {}).get("theta")
        if fm is not None and theta is not None:
            rows_X, rows_y, rows_ln, rows_st, line_risk = [], [], [], [], {}
            for i, L, ids_, st, en, Fm, _lp in tqdm(self._evidence_pass(ds, order), total=len(order), desc="trust audit", leave=False):
                if not len(ids_):
                    line_risk[i] = 1.0
                    continue
                rk = fm.risk(Fm)
                line_risk[i] = float(rk.mean())
                rows_X.append(np.c_[rk, Fm[:, 0]]), rows_y.append(char_wrong_labels(ref_of[i], codec.decode_ids(ids_))), rows_ln.append(np.full(len(ids_), i))
                rows_st.append(np.full(len(ids_), int(ds.stratum[i])))
            X["risk"] = [line_risk.get(i, 1.0) for i in order]
            if rows_X:
                RX, y, stt = np.concatenate(rows_X), np.concatenate(rows_y).astype(bool), np.concatenate(rows_st)
                th = np.array([cell_theta(theta, c) for c in stt])
                tr = RX[:, 0] <= th
                lines_in = np.concatenate(rows_ln)
                sres = []
                for s_ in sorted(set(stt.tolist())):
                    m_ = stt == s_
                    sres.append({"stratum": int(s_), "lines": int(len(set(lines_in[m_].tolist()))), "chars": int(m_.sum()), "trusted_share": float(tr[m_].mean()),
                                 "err_trusted": float(y[m_ & tr].mean()) if (m_ & tr).any() else None, "err_doubtful": float(y[m_ & ~tr].mean()) if (m_ & ~tr).any() else None})
                X["frag_audit"] = {"target": float((self.frag_state or {}).get("alpha", float("nan"))), "rule_epoch": (self.frag_state or {}).get("epoch"), "view": "shear +0.08",
                                   "chars": int(len(y)), "trusted_share": float(tr.mean()), "err_trusted": float(y[tr].mean()) if tr.any() else None,
                                   "err_doubtful": float(y[~tr].mean()) if (~tr).any() else None, "err_all": float(y.mean()), "auroc_risk": auroc(RX[:, 0], y),
                                   "auroc_conf": auroc(-RX[:, 1], y), "strata": sres}
                with open(os.path.join(self.dirs["08_self_distillation"], "frag_trust_audit.json"), "w") as f:
                    json.dump(json_safe(X["frag_audit"]), f, indent=2)
        nb, rs_ = getattr(self, "_nb_eval", None), getattr(self, "_rescore", None)
        if nb and rs_:
            beta, rho = float(rs_["beta"]), float(rs_["rho"])
            ch_rows = []
            for i, v in nb.items():
                fused = fuse_scores(v["ctc"], v["lm"], v["ntok"], beta, rho)
                k = int(np.argmax(fused))
                eds = [edit_distance(v["ref"], t) for t in v["texts"]]
                ch_rows.append((eds[0] - eds[k], int(i), k, eds, fused, v))
            changed = [r for r in ch_rows if r[2] != 0]
            helped = sorted([r for r in changed if r[0] > 0], key=lambda r: -r[0])[:3]
            hurt = sorted([r for r in changed if r[0] < 0], key=lambda r: r[0])[:2]
            ex_list = []
            for tag, r in [("rescoring helped", r) for r in helped] + [("rescoring hurt", r) for r in hurt]:
                gain, i, k, eds, fused, v = r
                keep = sorted(set(range(min(6, len(v["texts"])))) | {k})
                ex_list.append({"line": i, "tag": tag, "gt": ds.display(i), "chosen": keep.index(k), "gain_edits": int(gain),
                                "items": [{"rank": int(j), "text": ds.display(i, v["texts"][j]), "ctc": float(v["ctc"][j]), "lm": float(v["lm"][j]), "fused": float(fused[j]),
                                           "ed": int(eds[j])} for j in keep]})
            X["nbest"] = {"beta": beta, "rho": rho, "adopted": bool(rs_["adopted"]), "llm": self.cfg.llm.model_name, "lines": len(ch_rows),
                          "changed_share": float(len(changed) / max(len(ch_rows), 1)), "helped_lines": int(sum(1 for r in changed if r[0] > 0)),
                          "hurt_lines": int(sum(1 for r in changed if r[0] < 0)), "net_edits_saved": int(sum(r[0] for r in changed)), "examples": ex_list}
        picks = list(self._pick(R, 9))
        if ex.get("wsd_test_edits") is not None:
            d = ex["edits_greedy"] - ex["wsd_test_edits"]
            tid = ex["test_ids"]
            srt = np.argsort(-d, kind="stable")
            picks += [(int(tid[j]), "WSD helped most") for j in srt[:3] if d[j] > 0] + [(int(tid[j]), "WSD hurt most") for j in srt[::-1][:2] if d[j] < 0]
        seen, plan = set(), []
        for i, tag in picks:
            if i not in seen and i in pos:
                seen.add(i), plan.append((i, tag))
        tdir = os.path.join(self.dirs["12_report"], "08_reading_traces")
        os.makedirs(tdir, exist_ok=True)
        traces = []
        from PIL import Image
        wsd_pred = dict(zip(order, X["wsd_pred"])) if X.get("wsd_pred") else {}
        for i, L, ids_, st, en, Fm, lp in self._evidence_pass(ds, [p[0] for p in plan]):
            tag = next(t for j, t in plan if j == i)
            ink = ds.get_ink(i)
            Image.fromarray((255.0 * (1.0 - np.clip(ink, 0, 1))).astype(np.uint8)).save(os.path.join(tdir, f"line_{i:05d}.png"))
            hyp = codec.decode_ids(ids_)
            wrong = char_wrong_labels(ref_of[i], hyp) if len(ids_) else np.zeros(0)
            rk = fm.risk(Fm) if (fm is not None and len(ids_)) else (1.0 - 1.0 / (1.0 + np.exp(-Fm[:, 0])) if len(ids_) else np.zeros(0))
            th_ = cell_theta(theta, int(ds.stratum[i])) if (theta is not None and fm is not None) else 0.5
            chars = []
            for k in range(len(ids_)):
                fpk = int(st[k] + np.exp(lp[st[k]:en[k]]).max(1).argmax())
                p = np.exp(lp[fpk])
                p2 = p.copy()
                p2[0], p2[ids_[k]] = 0.0, 0.0
                a = int(p2.argmax())
                chars.append({"ch": codec.i2c.get(ids_[k], "?"), "x0": float(st[k] / max(L, 1)), "x1": float(en[k] / max(L, 1)), "risk": float(rk[k]), "p": float(p[ids_[k]]),
                              "alt": codec.i2c.get(a, None) if p2[a] > 1e-3 else None, "palt": float(p2[a]), "trusted": bool(rk[k] <= th_), "wrong": bool(wrong[k])})
            traces.append({"line": i, "tag": tag, "cer": cer_of[i], "gt": ds.display(i), "pr": ds.display(i, pred_of[i]), "pw": wsd_pred.get(i), "rtl": bool(ds.rtl[i]),
                           "image": f"08_reading_traces/line_{i:05d}.png", "chars": chars})
        X["traces"] = traces
        with open(os.path.join(self.dirs["00_summary"], "v11_extras.json"), "w", encoding="utf-8") as f:
            json.dump(json_safe(X), f, ensure_ascii=False, allow_nan=False)

    def _html(self, metrics: Dict[str, Any]) -> None:
        def flat(d, p=""):
            for k, v in d.items():
                if k in ("val_cer_grid", "provenance", "rows", "top_confusions", "interpretability"):
                    continue
                if isinstance(v, dict):
                    yield from flat(v, f"{p}{k}.")
                elif isinstance(v, (list, tuple)) and all(isinstance(x, (int, float)) for x in v) and len(v) <= 4:
                    yield f"{p}{k}", ", ".join(f"{x:.4f}" if isinstance(x, float) else str(x) for x in v)
                elif isinstance(v, bool):
                    yield f"{p}{k}", str(v)
                elif isinstance(v, (int, float, str)):
                    yield f"{p}{k}", v
        fm = lambda v: f"{v:.4f}" if isinstance(v, float) else html.escape(str(v))   # noqa: E731
        rows = "".join(f"<tr><td>{html.escape(k)}</td><td>{fm(v)}</td></tr>" for k, v in flat(metrics))
        figs = "".join(f"<h3>{html.escape(c)}</h3><img src='../{p}' style='max-width:100%'>" for p, c in self.fig_list)
        banner = ("<div style='background:#b00020;color:#fff;padding:10px;font-weight:bold'>SYNTHETIC SMOKE TEST - fake handwriting - NOT FOR PUBLICATION</div>"
                  if self.data.is_synthetic else "")
        with open(os.path.join(self.dirs["00_summary"], "report.html"), "w", encoding="utf-8") as f:
            f.write("<html><head><meta charset='utf-8'><title>InkLingua metrics</title></head><body style='font-family:sans-serif;max-width:1300px;margin:auto'>"
                    f"{banner}<h1>InkLingua - every number of the evaluation</h1><p>The interactive report is in <a href='../12_report/index.html'>12_report/index.html</a>; "
                    f"the folders 01-11 hold the numbers behind every figure.</p><table border=1 cellpadding=3>{rows}</table>{figs}</body></html>")

In [ ]:
%%writefile inklingua/adapt.py
"""
Model adaptability to new handwriting:

  * `run_finetune` (mode "finetune"): continue ANY InkLingua checkpoint on a new corpus / corpus mix with LABELS (a registry key, a custom
    DatasetSpec or a local parquet folder).  The alphabet grows function-preservingly ([ALPHA-SPLIT]), every component of the normal training
    loop (FRAG-RC, MixLine, protocol, suite-compatible outputs) applies.  This is the path for "a new dataset with transcriptions".
  * `ZeroShotAdapter` (mode "adapt"): a FOLDER of line images in a language / orthography the model never saw, with few or no labels: alphabet
    growth + [LEX-TIE] language-prior resolution of tied variants + [XL-RC] transfer-calibrated risk control of the pseudo-labels + [TIE-MARGINAL]
    training.  Translation into any language.  This is the path for "handwriting without transcriptions".
  * writer self-distillation (inklingua/posthoc.py) is the transductive, label-free path at inference time.
"""
from __future__ import annotations

import copy
import csv
import json
import os
import time
import unicodedata
from typing import Any, Dict, List, Optional, Sequence, Tuple

import matplotlib
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

from .alphabet import CharGroups, extend_codec, extend_ctc_heads, resolve_variants
from .config import Config, ModelConfig, dc_from_dict
from .data import BucketBatchSampler, HandwritingDataModule, preprocess_line, to_u8
from .decoding import greedy_decode, seq_confidence
from .llm import Hypothesis, LLMEngine
from .model import GPUAugment, InkLinguaNet, MaskedBatchNorm2d, ModelEMA, VectorQuantizerEMA, ctc_per_sample
from .pseudo import PERTURBATIONS, RELIABILITY_FEATURES, SelectionRule, apply_perturbation, reliability_features, reliability_views
from .riskcontrol import transfer_risk_control
from .style import NUM_DESC
from .text import CharCodec, cer, ctc_min_frames, edit_distance, grapheme_clusters, normalize_text, to_logical, to_visual
from .utils import LOG, atomic_save, json_safe, resolve_device, resolve_llm_device, set_seed, setup_logging, torch_load

IMAGE_EXT = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".webp")


# ==============================================================================
# fine-tuning on a labeled corpus
# ==============================================================================
def run_finetune(cfg: Config) -> Optional[Dict[str, Any]]:
    """Warm start from cfg.train.init_from on the corpora of cfg.data.sources.  The model config is taken from the checkpoint (shapes must
    match), the data module builds the alphabet as an EXTENSION of the checkpoint's, the Trainer grows the output layers, then the normal
    train -> evaluate path runs (resumable, its own run directory)."""
    from .main import run_single
    p = cfg.train.init_from
    if not p or not os.path.exists(p):
        raise FileNotFoundError(f"mode 'finetune' needs cfg.train.init_from = path of an InkLingua checkpoint (got {p!r})")
    st = torch_load(p)
    if "cfg" in st:
        cfg.model = dc_from_dict(ModelConfig, st["cfg"]["model"])
        cfg.data.img_height = st["cfg"]["data"].get("img_height", cfg.data.img_height)
    units = list(st.get("units") or grapheme_clusters(st["chars"]))
    LOG.info("[FINETUNE] from %s (epoch %s, %d units) onto %s", p, st.get("epoch"), len(units), cfg.data.sources)
    return run_single(cfg, base_units=units)


# ==============================================================================
# target-language data (a folder of line images)
# ==============================================================================
def list_line_images(folder: str, limit: Optional[int] = None) -> List[str]:
    if not folder or not os.path.isdir(folder):
        raise FileNotFoundError(f"cfg.adapt.lines_dir='{folder}' is not a directory")
    names = sorted(f for f in os.listdir(folder) if f.lower().endswith(IMAGE_EXT))
    if not names:
        raise FileNotFoundError(f"no line images ({', '.join(IMAGE_EXT)}) in {folder}")
    return names[:limit] if limit else names


def read_label_file(path: str, rtl: bool = False) -> Dict[str, str]:
    """UTF-8 'filename<TAB>text' per line -> {filename: visual-order text}, also indexed by file stem.  Text is NFC-normalised; RTL text (logical
    order in the file) is converted to visual order."""
    out: Dict[str, str] = {}
    if not path:
        return out
    with open(path, encoding="utf-8-sig") as f:
        for ln in f:
            ln = ln.rstrip("\r\n")
            if not ln.strip():
                continue
            name, sep, txt = ln.partition("\t")
            if not sep:
                name, _, txt = ln.partition(" ")
            txt = to_visual(unicodedata.normalize("NFC", normalize_text(txt)), rtl)
            name = os.path.basename(name.strip())
            out[name] = txt
            out.setdefault(os.path.splitext(name)[0], txt)
    return out


class LineSet(Dataset):
    """Preprocessed target-language lines (uint8 ink, exactly the preprocessing of training) + the text they are trained on."""

    def __init__(self, folder: str, names: List[str], img_h: int, max_w: int, min_w: int, pad_x: int):
        self.names, self.img_h = names, img_h
        self.ink = []
        for n in names:
            with Image.open(os.path.join(folder, n)) as im:
                self.ink.append(to_u8(preprocess_line(im, img_h, max_w, min_w, pad_x)))
        self.width = np.array([a.shape[1] for a in self.ink])
        self.text = [""] * len(names)
        self.weight = np.zeros(len(names), np.float32)
        self.tied = [False] * len(names)

    def __len__(self) -> int:
        return len(self.names)

    def __getitem__(self, i: int) -> Dict[str, Any]:
        ink = self.ink[i].astype(np.float32) / 255.0
        return {"index": i, "image": torch.from_numpy(ink)[None], "width": ink.shape[1], "text": self.text[i], "weight": float(self.weight[i]), "tied": bool(self.tied[i])}


def collate_lines(items: List[Dict[str, Any]], codec, pad_multiple: int = 4) -> Dict[str, Any]:
    B, H = len(items), items[0]["image"].shape[1]
    Wm = int(np.ceil(max(it["width"] for it in items) / pad_multiple) * pad_multiple)
    imgs = torch.zeros(B, 1, H, Wm)
    tgt, tlen, minw = [], [], []
    for b, it in enumerate(items):
        imgs[b, :, :, :it["width"]] = it["image"]
        ids = codec.encode(it["text"]) if it["weight"] > 0 else []
        tgt += ids
        tlen.append(len(ids))
        minw.append(4 * ctc_min_frames(ids) if ids else 0)
    return {"image": imgs, "width": torch.tensor([it["width"] for it in items]), "target": torch.tensor(tgt, dtype=torch.long),
            "target_len": torch.tensor(tlen, dtype=torch.long), "weight": torch.tensor([it["weight"] if tl > 0 else 0.0 for it, tl in zip(items, tlen)]),
            "tied": torch.tensor([bool(it.get("tied", False)) for it in items]), "min_width": torch.tensor(minw, dtype=torch.long),
            "index": torch.tensor([it["index"] for it in items], dtype=torch.long)}


def resolve_ckpt(cfg: Config, name: str) -> str:
    d = cfg.ckpt_dir
    if name == "best":
        p = os.path.join(d, "best.pt")
        return p if os.path.exists(p) else os.path.join(d, "last.pt")
    if name == "last":
        return os.path.join(d, "last.pt")
    return name


def _set_adapt_mode(model: nn.Module) -> None:
    """Dropout on, but the batch-norm statistics and the VQ codebooks stay those of the source model."""
    model.train()
    for m in model.modules():
        if isinstance(m, (MaskedBatchNorm2d, VectorQuantizerEMA)):
            m.eval()


def load_source_calibration(path: str) -> Optional[Dict[str, Any]]:
    if not path or not os.path.exists(path):
        return None
    z = np.load(path, allow_pickle=False)
    return {"X": z["X"], "ed": z["ed"], "rl": z["rl"], "epoch": int(z["epoch"]), "chars": str(z["chars"])}


# ==============================================================================
# the zero-shot adapter
# ==============================================================================
class ZeroShotAdapter:
    """Adapt a trained checkpoint to the handwriting in `cfg.adapt.lines_dir`, then transcribe and translate it into every target language.
    0. [ALPHA-SPLIT] extend the alphabet; 1. the EMA teacher reads every line, the SOURCE calibration set gives the risk score;
    2. [LEX-TIE] ambiguous variants decided by vision + language model; 3. [XL-RC] threshold transferred to the target (weights + anchor audit);
    4. the student trains on the selected pseudo-labels (+ training anchors); the teacher follows by EMA; repeat.  `eval_file` is only measured."""

    def __init__(self, cfg: Config, device: torch.device, llm: Optional[LLMEngine] = None):
        self.cfg, self.ac, self.device = cfg, cfg.adapt, device
        self.pc = cfg.train.pseudo
        self.llm = llm
        self.log: Dict[str, Any] = {"rounds": []}
        self.tied: set = set()

    def load_source(self) -> None:
        p = resolve_ckpt(self.cfg, self.ac.checkpoint)
        if not os.path.exists(p):
            raise FileNotFoundError(f"no source checkpoint at {p}: train first (cfg.run.do_train) or set cfg.adapt.checkpoint")
        st = torch_load(p)
        weights = st["weights"] if "weights" in st else st["ema"]
        ckc = st.get("cfg") or self.cfg.to_dict()
        mcfg = dc_from_dict(ModelConfig, ckc["model"])
        self.dcfg = ckc["data"]
        self.src_units = list(st.get("units") or grapheme_clusters(st["chars"]))
        model = InkLinguaNet(mcfg, len(self.src_units) + 1, NUM_DESC, st.get("n_clusters", 1), self.dcfg["img_height"])
        model.load_state_dict(weights)
        self.model0, self.src_codec = model, CharCodec(self.src_units)
        self.ckpt_path = p
        LOG.info("[ZERO-SHOT] source model %s: %d units, epoch %s", p, len(self.src_units), st.get("epoch"))

    def load_lines(self) -> None:
        ac = self.ac
        names = list_line_images(ac.lines_dir, ac.max_lines)
        self.labels = read_label_file(ac.anchors_file, ac.rtl)
        self.eval_labels = read_label_file(ac.eval_file, ac.rtl)
        lab = lambda n, d: d.get(n, d.get(os.path.splitext(n)[0]))                     # noqa: E731
        d = self.dcfg
        self.ds = LineSet(ac.lines_dir, names, d["img_height"], d["max_img_width"], d["min_img_width"], d["pad_x"])
        anchors = [i for i, n in enumerate(names) if lab(n, self.labels) is not None]
        ev = [i for i, n in enumerate(names) if lab(n, self.eval_labels) is not None and i not in set(anchors)]
        rs = np.random.RandomState(self.cfg.run.seed)
        anchors = list(rs.permutation(anchors)) if anchors else []
        if ac.anchor_limit is not None:
            anchors = anchors[:ac.anchor_limit]
        self.anchor_ref = {int(i): lab(names[i], self.labels) for i in anchors}
        self.eval_ref = {int(i): lab(names[i], self.eval_labels) for i in ev}
        n_tr = int(round(ac.anchor_train_share * len(anchors))) if len(anchors) >= ac.min_anchors_to_train else 0
        self.anchors_train, self.anchors_cal = [int(i) for i in anchors[:n_tr]], [int(i) for i in anchors[n_tr:]]
        self.target_idx = np.array([i for i in range(len(names)) if i not in self.eval_ref and i not in set(self.anchors_train)], np.int64)
        LOG.info("[ZERO-SHOT] %d target lines | anchors: %d train + %d calibration | evaluation lines (measure only): %d", len(names), len(self.anchors_train),
                 len(self.anchors_cal), len(self.eval_ref))

    def extend_alphabet(self) -> None:
        extra_text = self.ac.charset + "".join(self.anchor_ref.values())
        self.codec, self.groups = extend_codec(self.src_codec, extra_text, self.ac.visual_base)
        self.model = extend_ctc_heads(self.model0, self.groups).to(self.device)
        g = self.groups
        LOG.info("[ALPHA-SPLIT] +%d units (%d variants of %d known letters, %d without visual base: %s)", g.V - g.n_old, g.V - g.n_old - len(g.orphans),
                 g.n_variant_groups, len(g.orphans), "".join(g.units[i - 1] for i in g.orphans) or "-")

    def _loader(self, idxs: Sequence[int], bs: int, shuffle: bool = False, epoch: int = 0) -> DataLoader:
        s = BucketBatchSampler(self.ds.width, bs, shuffle, 50, seed=self.cfg.run.seed + epoch, indices=list(idxs))
        return DataLoader(self.ds, batch_sampler=s, collate_fn=lambda it: collate_lines(it, self.codec), num_workers=0)

    @torch.no_grad()
    def read(self, model: nn.Module, idxs: Sequence[int]) -> Dict[str, Any]:
        model.eval()
        pc, g = self.pc, self.groups
        views = reliability_views(pc)
        n_gate = len(pc.cons_views) if pc.consistency else 0
        r: Dict[str, Any] = {"idx": [], "text": [], "ids": [], "frames": [], "conf": [], "cons": [], "X": [], "lp": {}}
        for batch in self._loader(idxs, self.ac.batch_size):
            x, w = batch["image"].to(self.device), batch["width"].to(self.device)
            o = model(x, w)
            lp, lens = o["log_probs"].cpu().numpy(), o["out_lens"].cpu().tolist()
            B = lp.shape[0]
            reads = []
            for b in range(B):
                lpc = g.collapse(lp[b, :lens[b]])
                ids, cf, fr = greedy_decode(lpc)
                reads.append((lpc, ids, cf, fr, self.codec.decode_ids(ids)))
            cons_v = np.zeros((B, len(views)))
            for vi, v in enumerate(views):
                ov = model(apply_perturbation(PERTURBATIONS[v], x, w), w)
                lpv, lnv = ov["log_probs"].cpu().numpy(), ov["out_lens"].cpu().tolist()
                for b in range(B):
                    cons_v[b, vi] = cer(reads[b][4], self.codec.decode_ids(greedy_decode(g.collapse(lpv[b, :lnv[b]]))[0]))
            for b in range(B):
                lpc, ids, cf, fr, text = reads[b]
                i = int(batch["index"][b])
                r["idx"].append(i), r["text"].append(text), r["ids"].append(ids), r["frames"].append(fr)
                r["conf"].append(seq_confidence(cf))
                r["cons"].append(float(cons_v[b, :n_gate].mean()) if n_gate else float("nan"))
                r["X"].append(reliability_features(lpc, ids, cf, fr, cons_v[b]))
                r["lp"][i] = lp[b, :lens[b]].copy()
        r["X"] = np.array(r["X"]) if r["X"] else np.zeros((0, len(RELIABILITY_FEATURES)))
        return r

    def _scorer(self):
        if self.llm is not None and self.llm.can_score():
            rtl = self.ac.rtl
            base = self.llm.score_texts
            return (lambda texts: base([to_logical(t, True) for t in texts])) if rtl else base
        return None

    def resolve(self, R: Dict[str, Any], rows: Sequence[int], use_lm: bool = True) -> Dict[int, Tuple[str, Dict[str, float]]]:
        ac, sc = self.ac, (self._scorer() if use_lm else None)
        out = {}
        for k in rows:
            i = R["idx"][k]
            out[i] = resolve_variants(R["ids"][k], R["frames"][k], R["lp"][i], self.groups, sc, beam=ac.lex_beam, vis_w=ac.lex_vis_weight, lm_w=ac.lex_lm_weight,
                                      rho=ac.lex_rho, decided_p=ac.lex_decided_prob, max_options=ac.lex_max_options)
        return out

    def measure(self, model: nn.Module, label: str) -> Dict[str, float]:
        if not self.eval_ref:
            return {}
        idxs = sorted(self.eval_ref)
        R = self.read(model, idxs)
        rows = list(range(len(R["idx"])))
        res_lm, res_v = self.resolve(R, rows, True), self.resolve(R, rows, False)
        ref = [self.eval_ref[i] for i in R["idx"]]
        out = {}
        for tag, res in (("lex_tie", res_lm), ("vision_only", res_v)):
            ed = sum(edit_distance(a, res[i][0]) for a, i in zip(ref, R["idx"]))
            out[f"CER_{tag}"] = ed / max(sum(len(a) for a in ref), 1)
        LOG.info("[ZERO-SHOT] %s | eval CER %.2f %% with language-prior tie-breaking, %.2f %% vision only (%d lines)", label, 100 * out["CER_lex_tie"],
                 100 * out["CER_vision_only"], len(ref))
        return out

    def select(self, R: Dict[str, Any], rule: Optional[SelectionRule], src: Optional[Dict[str, Any]], resolved: Dict[int, Tuple[str, Dict]]
               ) -> Tuple[Dict[int, Tuple[str, float]], Dict[str, Any]]:
        ac, pc = self.ac, self.pc
        pos = {i: k for k, i in enumerate(R["idx"])}
        score_all = rule.predict(R["X"]) if rule is not None else np.full(len(R["idx"]), np.nan)
        tgt_rows = [pos[int(i)] for i in self.target_idx if int(i) in pos]
        cal_rows = [pos[i] for i in self.anchors_cal if i in pos and i in resolved]
        info: Dict[str, Any] = {"mode": "none", "k": 0, "theta": float("-inf")}
        if src is None or rule is None:
            LOG.warning("[XL-RC] no source calibration (source_calibration.npz): NO pseudo-labels are selected -- only the training anchors are learned from. "
                        "Mode 'source_calib' rebuilds the file for a checkpoint.")
            return {}, info
        X_s, ed_s, rl_s, score_s = src["X"], src["ed"], src["rl"], rule.oof
        sa = ea = la = None
        if cal_rows:
            sa = score_all[cal_rows]
            ea = np.array([edit_distance(self.anchor_ref[R["idx"][k]], resolved[R["idx"][k]][0]) for k in cal_rows], np.float64)
            la = np.array([len(self.anchor_ref[R["idx"][k]]) for k in cal_rows], np.float64)
        res = transfer_risk_control(X_s, score_s, ed_s, rl_s, R["X"][tgt_rows], score_all[tgt_rows], ac.target_cer, ac.risk_z, ac.min_selected, ac.min_ess,
                                    score_a=sa, ed_a=ea, rl_a=la, zero_anchor_inflation=ac.zero_anchor_inflation, prior_edits=ac.anchor_prior_edits, seed=self.cfg.run.seed)
        info.update(res)
        theta = res["theta"]
        sel: Dict[int, Tuple[str, float]] = {}
        if res["k"] > 0:
            lo = float(np.min(score_all[tgt_rows])) if tgt_rows else 0.0
            pool = list(dict.fromkeys(tgt_rows + cal_rows))
            for k in pool:
                i = R["idx"][k]
                if i not in resolved or not np.isfinite(score_all[k]) or score_all[k] > theta:
                    continue
                if pc.consistency and not (np.isfinite(R["cons"][k]) and R["cons"][k] <= pc.tau_cons):
                    continue
                text = resolved[i][0]
                ids_ = self.codec.encode(text)
                if len(text) < pc.min_chars or ctc_min_frames(ids_) > len(R["lp"][i]):
                    continue
                rel = float(np.clip((score_all[k] - lo) / max(theta - lo, 1e-9), 0.0, 1.0))
                sel[i] = (text, 1.0 - 0.5 * rel)
        info["selected"] = len(sel)
        info["coverage_lines"] = len(sel) / max(len(tgt_rows), 1)
        return sel, info

    def train_round(self, student: nn.Module, ema: ModelEMA, opt, sel: Dict[int, Tuple[str, float]], rnd: int) -> Dict[str, float]:
        """[TIE-MARGINAL] a pseudo-label whose variants the language model could not settle is trained with the loss of the MARGINAL label
        (CTC on group-summed posteriors with the base-letter text): no gradient between the copies, so the model is not taught its own coin flips."""
        ac, ds = self.ac, self.ds
        for i in range(len(ds)):
            ds.text[i], ds.weight[i] = "", 0.0
        for i in self.anchors_train:
            ds.text[i], ds.weight[i] = self.anchor_ref[i], 1.0
        for i, (t, w) in sel.items():
            ds.text[i], ds.weight[i] = t, w
        ds.tied = [(i in sel and i in self.tied) for i in range(len(ds))]
        idx = [i for i in range(len(ds)) if ds.weight[i] > 0]
        if not idx:
            return {"n_train": 0, "loss": float("nan")}
        _set_adapt_mode(student)
        tot, cnt = 0.0, 0
        for ep in range(ac.epochs_per_round):
            for batch in self._loader(idx, ac.batch_size, True, rnd * 100 + ep):
                x, wd = batch["image"].to(self.device), batch["width"].to(self.device)
                B = x.size(0)
                zs = torch.zeros(B, NUM_DESC, device=self.device)
                x_s, _, wd, _, _, _ = self.aug(x, zs, zs, wd, batch["min_width"])
                out = student(x_s, wd)
                tgt, tl = batch["target"].to(self.device), batch["target_len"].to(self.device)
                per = ctc_per_sample(out["log_probs"], out["out_lens"], tgt, tl)
                tied = batch["tied"].to(self.device)
                if bool(tied.any()):
                    tb = torch.tensor(self.groups.base_ids(tgt.tolist()), dtype=torch.long, device=self.device)
                    per_m = ctc_per_sample(self.groups.collapse_t(out["log_probs"]), out["out_lens"], tb, tl)
                    per = torch.where(tied, per_m, per)
                wt = batch["weight"].to(self.device)
                loss = (per * wt).sum() / wt.sum().clamp(min=1e-6)
                if not torch.isfinite(loss):
                    continue
                opt.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_([p for g_ in opt.param_groups for p in g_["params"]], self.cfg.train.grad_clip)
                opt.step()
                self.step += 1
                ema.update(student, self.step)
                tot, cnt = tot + float(loss), cnt + 1
        student.eval()
        return {"n_train": len(idx), "loss": tot / max(cnt, 1)}

    def _optimizer(self, student: nn.Module):
        for p in student.parameters():
            p.requires_grad_(False)
        train = [student.ctc_head, student.final_ln] + ([student.film_out] if student.film_out is not None else [])
        train += list(student.layers)[len(student.layers) - max(0, self.ac.unfreeze_layers):] if self.ac.unfreeze_layers > 0 else []
        params = []
        for m in train:
            for p in m.parameters():
                p.requires_grad_(True)
                params.append(p)
        return torch.optim.AdamW(params, lr=self.ac.lr, weight_decay=1e-2, betas=(0.9, 0.98))

    def run(self) -> Dict[str, Any]:
        ac, cfg = self.ac, self.cfg
        out_dir = ac.out_dir or os.path.join(cfg.run_dir, "adapt")
        os.makedirs(out_dir, exist_ok=True)
        self.load_source()
        self.load_lines()
        self.extend_alphabet()
        src = load_source_calibration(ac.source_calibration or os.path.join(cfg.ckpt_dir, "source_calibration.npz"))
        if src is None:
            LOG.warning("[XL-RC] source_calibration.npz not found (written by training with the legacy pseudo-labels; mode 'source_calib' rebuilds it)")
        rule = SelectionRule.fit(src["X"], src["ed"], src["rl"], self.pc, seed=cfg.run.seed) if src is not None else None
        student = self.model
        ema = ModelEMA(student, ac.ema_decay)
        self.aug = GPUAugment(cfg.train, np.zeros(NUM_DESC, np.float32), np.ones(NUM_DESC, np.float32))
        opt = self._optimizer(student)
        self.step = 0
        metrics: Dict[str, Any] = {"zero_shot_start": self.measure(ema.module, "round 0 (before any training)")}
        all_idx = list(range(len(self.ds)))
        for rnd in range(ac.rounds):
            t0 = time.time()
            read_idx = [i for i in all_idx if i not in self.eval_ref]
            R = self.read(ema.module, read_idx)
            score_all = rule.predict(R["X"]) if rule is not None else np.zeros(len(R["idx"]))
            pos = {i: k for k, i in enumerate(R["idx"])}
            order = [pos[int(i)] for i in self.target_idx if int(i) in pos]
            order = sorted(order, key=lambda k: score_all[k])[:ac.lex_max_lines]
            rows = list(dict.fromkeys(order + [pos[i] for i in self.anchors_cal if i in pos]))
            resolved = self.resolve(R, rows, True)
            sel, info = self.select(R, rule, src, resolved)
            self.tied = {i for i, (_, inf_) in resolved.items() if inf_.get("n_ambiguous", 0) > 0 and inf_.get("margin", float("inf")) < ac.lex_tie_margin}
            info["tied_lines"] = len([i for i in sel if i in self.tied])
            tr = self.train_round(student, ema, opt, sel, rnd)
            rec = {"round": rnd + 1, "seconds": round(time.time() - t0, 1), **{k: (round(v, 4) if isinstance(v, float) else v) for k, v in info.items()}, **tr,
                   "eval": self.measure(ema.module, f"after round {rnd + 1}")}
            self.log["rounds"].append(rec)
            LOG.info("[ZERO-SHOT] round %d/%d | %s | threshold covers %.1f %% of the target lines | factor %.2f (%d anchors) | selected %d lines | domain AUROC %.2f | "
                     "train loss %.3f", rnd + 1, ac.rounds, info.get("mode"), 100 * info.get("coverage_target", 0.0), info.get("factor", float("nan")),
                     info.get("n_anchors", 0), info.get("selected", 0), info.get("domain_auroc", float("nan")), tr.get("loss", float("nan")))
        metrics["rounds"] = self.log["rounds"]
        metrics["zero_shot_end"] = self.measure(ema.module, "final")
        self._finish(ema.module, out_dir, metrics)
        return metrics

    def _finish(self, model: nn.Module, out_dir: str, metrics: Dict[str, Any]) -> None:
        ac, cfg = self.ac, self.cfg
        R = self.read(model, [i for i in range(len(self.ds))])
        src = load_source_calibration(ac.source_calibration or os.path.join(cfg.ckpt_dir, "source_calibration.npz"))
        rule = SelectionRule.fit(src["X"], src["ed"], src["rl"], self.pc, seed=cfg.run.seed) if src is not None else None
        score = rule.predict(R["X"]) if rule is not None else np.full(len(R["idx"]), np.nan)
        rows = sorted(range(len(R["idx"])), key=lambda k: (not np.isfinite(score[k]), score[k]))[:ac.lex_final_lines]
        resolved = self.resolve(R, rows, True)
        rowsd = []
        for k in range(len(R["idx"])):
            i = R["idx"][k]
            text = resolved[i][0] if i in resolved else self.resolve(R, [k], False)[i][0]
            shown = to_logical(text, ac.rtl)
            langs = {}
            if self.llm is not None and k < ac.translate_max:
                hyp = Hypothesis(text=shown, char_conf=[], logprob=0.0, post=1.0)
                for lg in ac.target_languages:
                    try:
                        langs[lg] = self.llm.translate([hyp], uncertain=False, target_language=lg)["translation"]
                    except Exception as e:
                        langs[lg] = f"<translation failed: {e}>"
            ref = self.eval_ref.get(i, self.anchor_ref.get(i, ""))
            rowsd.append({"file": self.ds.names[i], "transcription": shown, "risk": float(score[k]) if np.isfinite(score[k]) else "",
                          "split": "eval" if i in self.eval_ref else ("anchor" if i in self.anchor_ref else "target"), "reference": to_logical(ref, ac.rtl),
                          **{f"translation_{lg}": langs.get(lg, "") for lg in ac.target_languages}})
        with open(os.path.join(out_dir, "adapt_results.csv"), "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=list(rowsd[0].keys()))
            w.writeheader()
            w.writerows(rowsd)
        atomic_save({"weights": model.state_dict(), "chars": self.codec.chars, "units": self.codec.units, "cfg": cfg.to_dict(), "from": "adapt", "n_clusters": 1,
                     "epoch": self.step}, os.path.join(out_dir, "adapted.pt"))
        with open(os.path.join(out_dir, "adapt_metrics.json"), "w", encoding="utf-8") as f:
            json.dump(json_safe({**metrics, "new_units": self.codec.units[len(self.src_units):], "n_lines": len(self.ds), "n_anchors_train": len(self.anchors_train),
                                 "n_anchors_calibration": len(self.anchors_cal), "source_checkpoint": self.ckpt_path, "steps": self.step}), f, indent=2, ensure_ascii=False)
        LOG.info("[ZERO-SHOT] wrote %s", out_dir)


def run_adapt(cfg: Optional[Config] = None) -> Dict[str, Any]:
    from . import __version__
    cfg = cfg or Config()
    cfg.make_dirs()
    setup_logging(cfg)
    set_seed(cfg.run.seed)
    device = resolve_device(cfg.run.device)
    LOG.info("InkLingua v%s | zero-shot adaptation | device %s", __version__, device)
    llm = LLMEngine(cfg.llm, resolve_llm_device(cfg.llm.device, device))
    return ZeroShotAdapter(cfg, device, llm).run()


def run_adapt_curve(cfg: Optional[Config] = None) -> Dict[str, Any]:
    """CER on the evaluation lines as a function of the number of LABELED target lines (cfg.adapt.anchor_counts)."""
    cfg = cfg or Config()
    cfg.make_dirs()
    setup_logging(cfg)
    device = resolve_device(cfg.run.device)
    llm = LLMEngine(cfg.llm, resolve_llm_device(cfg.llm.device, device))
    curve = {}
    for k in cfg.adapt.anchor_counts:
        c = copy.deepcopy(cfg)
        c.adapt.anchor_limit = int(k)
        c.adapt.out_dir = os.path.join(cfg.adapt.out_dir or os.path.join(cfg.run_dir, "adapt"), f"anchors_{k}")
        set_seed(cfg.run.seed)
        m = ZeroShotAdapter(c, device, llm).run()
        curve[int(k)] = {"start": m.get("zero_shot_start"), "end": m.get("zero_shot_end"), "last_round": (m.get("rounds") or [{}])[-1].get("selected")}
    base = cfg.adapt.out_dir or os.path.join(cfg.run_dir, "adapt")
    with open(os.path.join(base, "adapt_curve.json"), "w") as f:
        json.dump(json_safe(curve), f, indent=2)
    try:
        ks = sorted(curve)
        fig, ax = plt.subplots(figsize=(5, 3.4))
        for key, lab in (("CER_lex_tie", "language prior + vision"), ("CER_vision_only", "vision only")):
            ax.plot(ks, [100 * (curve[k]["end"] or {}).get(key, float("nan")) for k in ks], "o-", label=f"adapted, {lab}")
            ax.plot(ks, [100 * (curve[k]["start"] or {}).get(key, float("nan")) for k in ks], "x--", label=f"before, {lab}")
        ax.set_xlabel("labeled target lines (anchors)"), ax.set_ylabel("CER on evaluation lines (%)"), ax.legend(fontsize=6)
        fig.tight_layout()
        fig.savefig(os.path.join(base, "adapt_curve.png"), dpi=150)
        plt.close(fig)
    except Exception as e:
        LOG.warning("curve plot failed: %s", e)
    return curve


def run_source_calib(cfg: Optional[Config] = None) -> str:
    """Rebuild source_calibration.npz for an existing checkpoint (needs the SOURCE data: same cfg.data / seed / run.name)."""
    from .inferencer import Inferencer
    from .trainer import Trainer
    cfg = cfg or Config()
    cfg.make_dirs()
    setup_logging(cfg)
    set_seed(cfg.run.seed)
    device = resolve_device(cfg.run.device)
    data = HandwritingDataModule(cfg).prepare()
    llm = LLMEngine(cfg.llm, device)
    inf = Inferencer(cfg, data, llm, device)
    inf.load(resolve_ckpt(cfg, cfg.adapt.checkpoint))
    tr = Trainer(cfg, data, inf.model, llm, device)
    rc = tr._teacher_read(data.calib_idx, desc="source calibration lines")
    ref = [data.train_ds.reference(i) for i in rc["idx"]]
    ed = np.array([edit_distance(a, b) for a, b in zip(ref, rc["text"])], np.float64)
    rl = np.array([len(a) for a in ref], np.float64)
    tr._save_source_calibration(rc["X"], ed, rl, -1)
    p = os.path.join(cfg.ckpt_dir, "source_calibration.npz")
    LOG.info("source calibration: %d lines, CER %.2f %% -> %s", len(ed), 100 * ed.sum() / max(rl.sum(), 1), p)
    return p

In [ ]:
%%writefile inklingua/report.py
"""
The run report ("Reading Atlas"): one self-contained HTML page + numbered folders of the numbers behind every panel.  Built from the files a run
writes (predictions.csv, metrics.json, curves/history.json, protocol/, v11_extras.json); it also reads the flat layout of v9 / v10 runs.  Panels
whose data are missing are left out.  Nothing in it is simulated.
"""
from __future__ import annotations

import csv
import json
import math
import os
import time
from typing import Any, Dict, List, Optional, Sequence

import numpy as np

from .style import STYLE_NAMES
from .text import bootstrap_cer_ci, edit_distance
from .utils import LOG, json_safe

_REPORT_REFS = [{"label": "HTR-VT, all 6,482 labeled IAM-line training lines, no external data (arXiv 2409.08573; Pattern Recognition 2025)", "cer": 0.047,
                 "note": "different label budget: context only, NOT a like-for-like comparison"}]
_STYLE_LABELS = {"stroke_width": ("ink thickness", "thin -> thick"), "gap_ratio": ("word-gap ratio", "tight -> wide"), "slant_deg": ("slant (deg)", "left -> right"),
                 "ink_density": ("ink density", "light -> dark"), "body_ratio": ("body / line height", "small -> large")}


def _fnum(s: Any) -> float:
    try:
        return float(s)
    except (TypeError, ValueError):
        return float("nan")


def _rd(path: str) -> Optional[Any]:
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None


def _first(*paths: str) -> Optional[str]:
    return next((p for p in paths if os.path.exists(p)), None)


def _lst(a: Any, nd: int = 4) -> List[Any]:
    a = np.asarray(a, np.float64)
    return [None if not np.isfinite(v) else round(float(v), nd) for v in a]


def _spearman(a: np.ndarray, b: np.ndarray) -> float:
    m = np.isfinite(a) & np.isfinite(b)
    if m.sum() < 5:
        return float("nan")
    ra, rb = np.argsort(np.argsort(a[m])).astype(float), np.argsort(np.argsort(b[m])).astype(float)
    ra, rb = ra - ra.mean(), rb - rb.mean()
    d = math.sqrt(float((ra ** 2).sum() * (rb ** 2).sum()))
    return float((ra * rb).sum() / d) if d > 0 else float("nan")


def _write_csv(path: str, header: Sequence[str], rows: Sequence[Sequence[Any]]) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(header)
        for r in rows:
            w.writerow(["" if (isinstance(v, float) and not np.isfinite(v)) or v is None else v for v in r])


def read_predictions_csv(path: str) -> Dict[str, Any]:
    rows = []
    with open(path, newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            rows.append(r)
    rows.sort(key=lambda r: int(r["line_index"]))
    n = len(rows)
    L: Dict[str, Any] = {"idx": np.array([int(r["line_index"]) for r in rows]), "gt": [r["ground_truth"] for r in rows], "pr": [r["prediction"] for r in rows],
                         "corpus": [r.get("corpus", "") for r in rows], "conf": np.array([_fnum(r["confidence"]) for r in rows]),
                         "dis": np.array([_fnum(r.get("view_disagreement")) for r in rows]), "ed": np.array([_fnum(r["edits"]) for r in rows]),
                         "rl": np.maximum(np.array([_fnum(r["ref_len"]) for r in rows]), 1.0)}
    L["cer"] = L["ed"] / L["rl"]
    for s in STYLE_NAMES:
        L[s] = np.array([_fnum(r.get(f"measured_{s}")) for r in rows]) if n else np.zeros(0)
    if n and "prediction_posthoc" in rows[0]:
        L["pf"] = [r["prediction_posthoc"] for r in rows]
        L["edf"] = np.array([round(_fnum(r["CER_posthoc"]) * rl) if r.get("CER_posthoc") not in (None, "") else e for r, rl, e in zip(rows, L["rl"], L["ed"])], np.float64)
    else:
        L["pf"], L["edf"] = list(L["pr"]), L["ed"].copy()
    return L


def _hardness_strata(L: Dict[str, Any], n_str: int = 3) -> np.ndarray:
    def z(v):
        m = np.isfinite(v)
        out = np.zeros(len(v))
        if m.sum() > 2 and v[m].std() > 0:
            out[m] = (v[m] - v[m].mean()) / v[m].std()
        return out
    h = -(z(L["stroke_width"]) + z(L["gap_ratio"]))
    return np.digitize(h, np.quantile(h, np.linspace(0, 1, n_str + 1)[1:-1])).astype(int)


def _bin_table(v: np.ndarray, ed: np.ndarray, rl: np.ndarray, edf: np.ndarray, groups: np.ndarray, edges: Sequence[float], seed: int = 0) -> List[Dict[str, Any]]:
    out = []
    b = np.digitize(v, edges[1:-1], right=True)
    fin = np.isfinite(v)
    for k in range(len(edges) - 1):
        m = fin & (b == k)
        if m.sum() < 5:
            out.append({"lo": float(edges[k]), "hi": float(edges[k + 1]), "n": int(m.sum()), "cer": None, "ci": None, "cer_final": None})
            continue
        lo, hi = bootstrap_cer_ci(ed[m], rl[m], 300, seed + k, groups=groups[m])
        out.append({"lo": float(edges[k]), "hi": float(edges[k + 1]), "n": int(m.sum()), "cer": float(ed[m].sum() / rl[m].sum()), "ci": [float(lo), float(hi)],
                    "cer_final": float(edf[m].sum() / rl[m].sum())})
    return out


def collect_report_data(infer_dir: str, run_dir: Optional[str] = None, curve_dir: Optional[str] = None) -> Dict[str, Any]:
    """All numbers of the report, from files only (v11 layout first, flat v9/v10 layout as a fallback)."""
    mp = _first(os.path.join(infer_dir, "00_summary", "metrics.json"), os.path.join(infer_dir, "metrics.json"))
    pp = _first(os.path.join(infer_dir, "01_recognition", "predictions.csv"), os.path.join(infer_dir, "predictions.csv"))
    xp = _first(os.path.join(infer_dir, "00_summary", "v11_extras.json"), os.path.join(infer_dir, "v10_extras.json"))
    if pp is None:
        raise FileNotFoundError(f"no predictions.csv under {infer_dir}")
    M = (_rd(mp) if mp else None) or {}
    P = read_predictions_csv(pp)
    X = (_rd(xp) if xp else None) or {}
    run_dir = run_dir or os.path.dirname(os.path.dirname(os.path.normpath(infer_dir)))
    H = _rd(os.path.join(curve_dir or os.path.join(run_dir, "curves"), "history.json")) or {}
    prot = _rd(os.path.join(run_dir, "protocol", "protocol.json"))
    n = len(P["idx"])
    blk = 20
    grp = P["idx"] // blk
    synthetic = bool(M.get("SYNTHETIC_SMOKE_TEST_NOT_FOR_PUBLICATION"))
    ph = M.get("posthoc") or {}
    rs = M.get("llm_rescoring") or {}
    cer_all = float(P["ed"].sum() / P["rl"].sum())
    ci = M.get("CER_ci95") or list(bootstrap_cer_ci(P["ed"], P["rl"], 400, 0, groups=grp))
    final_cer = M.get("CER_system_final", M.get("CER_final_with_validated_posthoc", cer_all))
    D: Dict[str, Any] = {"meta": {"title": "InkLingua - Reading Atlas", "run": os.path.basename(os.path.normpath(run_dir)), "synthetic": synthetic,
                                  "split": M.get("split", "test"), "n": n, "generated": time.strftime("%Y-%m-%d %H:%M"),
                                  "version": X.get("version", "v11") if X else ("v11" if prot else "older files (read by the v11 report)"),
                                  "corpora": M.get("corpora") or sorted(set(c for c in P["corpus"] if c))}}
    ck = M.get("checkpoint") or {}
    D["head"] = {"cer": cer_all, "ci": ci, "wer": M.get("WER"), "wer_ci": M.get("WER_ci95"), "exact": M.get("exact_line_accuracy"), "final_cer": final_cer,
                 "final_note": M.get("CER_system_final_note"), "n": n, "ckpt_epoch": ck.get("epoch"), "weights_from": ck.get("weights_from"),
                 "val_at_save": ck.get("val_sel_cer_at_save", ck.get("val_cer_at_save")), "labeled_fraction": M.get("labeled_fraction"),
                 "calibration_fraction": M.get("calibration_fraction"), "looks": M.get("evaluation_looks_so_far"), "val_sel_lines": M.get("val_sel_lines"),
                 "val_gate_lines": M.get("val_gate_lines"), "per_corpus": M.get("per_corpus")}
    led: List[Dict[str, Any]] = [{"id": "greedy", "label": "greedy reading", "cer": cer_all, "ci": ci, "kind": "base", "adopted": True,
                                  "note": "arg-max CTC path of the selected checkpoint; nothing tuned on the test split"}]
    for key, label in (("view_ensemble", "view ensemble"), ("tta", "style-cluster AdaBN"), ("wsd", "writer self-distillation (WSD)"), ("space_bonus", "word-gap bonus")):
        o = ph.get(key)
        if not isinstance(o, dict) or o.get("test_CER") is None:
            continue
        gv = o.get("val_delta_vs_base") or o.get("val_delta_vs_before") or {}
        gt_ = o.get("test_delta_vs_base") or o.get("test_delta_vs_before") or {}
        led.append({"id": key, "label": label, "cer": o["test_CER"], "kind": "option", "adopted": bool(o.get("adopted")), "val_gain": [gv.get("lo"), gv.get("hi")],
                    "val_hi_adj": gv.get("hi_adj"), "test_gain": gt_.get("delta"), "test_gain_ci": [gt_.get("lo"), gt_.get("hi")],
                    "note": ("adopted: Bonferroni-adjusted validation gain excludes 0" if o.get("adopted") else
                             "NOT adopted: the validation gain interval includes 0 - test value shown for information only")})
    if ph.get("test_CER_final") is not None:
        led.append({"id": "posthoc_final", "label": "greedy + validated options", "cer": ph["test_CER_final"], "kind": "milestone", "adopted": True,
                    "note": "only the options adopted on val_gate"})
    if rs:
        led.append({"id": "beam", "label": "CTC beam, top-1", "cer": rs.get("CER_beam_top1"), "kind": "milestone", "adopted": True, "note": "prefix beam search on the same posteriors"})
        led.append({"id": "llm", "label": "LLM N-best rescoring", "cer": rs.get("CER_rescored"), "ci": rs.get("CER_rescored_ci95"), "kind": "option",
                    "adopted": bool(rs.get("adopted_on_validation")), "note": "an external language model may only choose among the recogniser's own hypotheses; "
                    + ("adopted on validation" if rs.get("adopted_on_validation") else "NOT adopted on validation"), "llm": rs.get("llm")})
        led.append({"id": "oracle", "label": "oracle N-best (ceiling)", "cer": rs.get("CER_oracle_nbest"), "kind": "ceiling", "adopted": False,
                    "note": "best hypothesis in the N-best list chosen WITH the reference: the most any rescorer could reach"})
    D["ledger"] = led
    D["refs"] = [] if (synthetic or (D["meta"]["corpora"] and D["meta"]["corpora"] != ["iam"])) else _REPORT_REFS

    strat = np.array(X["strata"], int) if X.get("strata") and len(X["strata"]) == n else _hardness_strata(P)
    clus = np.array(X["wsd_cluster"], int) if X.get("wsd_cluster") and len(X["wsd_cluster"]) == n else None
    pw = X.get("wsd_pred") if X.get("wsd_pred") and len(X["wsd_pred"]) == n else None
    edw = np.array(X["wsd_edits"], np.float64) if X.get("wsd_edits") and len(X["wsd_edits"]) == n else None
    risk = 1.0 - np.nan_to_num(P["conf"], nan=0.5)
    D["lines"] = {"idx": P["idx"].tolist(), "cer": _lst(P["cer"]), "conf": _lst(P["conf"]), "dis": _lst(P["dis"]), "ed": P["ed"].astype(int).tolist(),
                  "rl": P["rl"].astype(int).tolist(), "edf": P["edf"].astype(int).tolist(), "strat": strat.tolist(), "corpus": P["corpus"],
                  **{k: _lst(P[k], 5) for k in STYLE_NAMES}, "gt": P["gt"], "pr": P["pr"], "pf": P["pf"],
                  "edw": None if edw is None else edw.astype(int).tolist(), "pw": pw, "clus": None if clus is None else clus.tolist()}
    starts = np.unique(grp)
    be = np.array([P["ed"][grp == g].sum() for g in starts])
    bl = np.array([P["rl"][grp == g].sum() for g in starts])
    D["blocks"] = {"size": blk, "first": (starts * blk).tolist(), "cer": _lst(be / np.maximum(bl, 1))}
    o = np.lexsort((-P["cer"], -P["ed"]))
    cum = np.cumsum(P["ed"][o]) / max(P["ed"].sum(), 1)
    xs = np.arange(1, n + 1) / n
    pick = np.unique(np.linspace(0, n - 1, min(n, 400)).astype(int))
    shares = {str(p): float(cum[max(int(round(p / 100 * n)) - 1, 0)]) for p in (5, 10, 20, 50)}
    D["lorenz"] = {"x": _lst(xs[pick]), "y": _lst(cum[pick]), "shares": shares}
    ro, oo = np.argsort(risk, kind="stable"), np.argsort(P["cer"], kind="stable")
    ab, cc, co, sh = [], [], [], []
    for a in range(0, 91):
        k = max(int(round((1 - a / 100.0) * n)), 1)
        ab.append(a / 100.0)
        cc.append(float(P["ed"][ro[:k]].sum() / P["rl"][ro[:k]].sum()))
        co.append(float(P["ed"][oo[:k]].sum() / P["rl"][oo[:k]].sum()))
        sh.append(float(1.0 - P["ed"][ro[:k]].sum() / max(P["ed"].sum(), 1)))
    D["abstain"] = {"a": ab, "conf": cc, "oracle": co, "share": sh, "base": cer_all, "aurc_conf": float(np.mean(cc)), "aurc_oracle": float(np.mean(co)),
                    "risk_name": "the model's own doubt (1 - sequence confidence)", "frag": None, "aurc_frag": None}
    if "risk" in X and len(X["risk"]) == n:
        rf = np.argsort(np.array(X["risk"], np.float64), kind="stable")
        cf = [float(P["ed"][rf[:max(int(round((1 - a / 100.0) * n)), 1)]].sum() / P["rl"][rf[:max(int(round((1 - a / 100.0) * n)), 1)]].sum()) for a in range(0, 91)]
        D["abstain"]["frag"], D["abstain"]["aurc_frag"] = cf, float(np.mean(cf))
    bins = []
    for s in STYLE_NAMES:
        v = P[s]
        if np.isfinite(v).sum() < 50:
            continue
        edges = np.quantile(v[np.isfinite(v)], np.linspace(0, 1, 6))
        edges[0], edges[-1] = edges[0] - 1e-9, edges[-1] + 1e-9
        lab, dirn = _STYLE_LABELS.get(s, (s, ""))
        bins.append({"key": s, "label": lab, "direction": dirn, "spearman": _spearman(v, P["cer"]), "rows": _bin_table(v, P["ed"], P["rl"], P["edf"], grp, edges, 11)})
    D["bins"] = bins
    D["lenbins"] = {"key": "length", "label": "reference length (characters)", "direction": "short -> long",
                    "rows": _bin_table(P["rl"].astype(np.float64), P["ed"], P["rl"], P["edf"], grp, [0, 20, 35, 50, 65, 1e9], 21)}
    if H and H.get("epoch"):
        keep = ("epoch", "train_eval_cer", "val_cer", "val_ema_cer", "val_sel_cer", "val_ema_sel_cer", "val_gate_cer", "val_ema_gate_cer", "train_ctc_labeled", "lr",
                "pl_coverage", "rc_calib_cov_raw", "pl_sel_cer", "train_frag", "train_frag_trust", "train_frag_char_trust")
        ser = {k: _lst(H[k], 5) for k in H if k in keep or k.startswith("frag_") or k.startswith("val_cer_")}
        D["train"] = {"series": ser, "best_epoch": ck.get("epoch"), "n_strata": sum(1 for k in H if k.startswith("frag_theta_s"))}
    D["frag"] = X.get("frag_audit")
    D["wsd"] = ph.get("wsd") if isinstance(ph.get("wsd"), dict) and ph["wsd"].get("test_by_cluster") else None
    D["traces"] = X.get("traces") or []
    D["nbest"] = X.get("nbest")
    D["interp"] = M.get("interpretability") or {}
    ins: Dict[str, str] = {}
    fs = (M.get("error_structure") or {}).get("form_structure") or {}
    r5, r20 = shares["5"], shares["20"]
    ins["terrain"] = (f"The 5 % of the lines with the most edits hold {r5 * 100:.0f} % of all character edits, the top 20 % hold {r20 * 100:.0f} %."
                      + (f" Neighbouring lines are alike (lag-1 autocorrelation of line CER {fs['lag1_autocorrelation_of_line_CER']:.2f}): the model fails in stretches - "
                         "a hand, a form - not at random." if fs.get("lag1_autocorrelation_of_line_CER") is not None else ""))
    a20 = D["abstain"]
    ins["abstain"] = (f"Declining the 20 % least reliable lines (by the model's own doubt) leaves a CER of {a20['conf'][20] * 100:.2f} % on the rest (from {cer_all * 100:.2f} %); "
                      f"those lines carry {a20['share'][20] * 100:.0f} % of all edits. An oracle that knew the errors would reach {a20['oracle'][20] * 100:.2f} %.")
    sb = next((b for b in bins if b["key"] == "stroke_width"), None)
    gb = next((b for b in bins if b["key"] == "gap_ratio"), None)
    parts = []
    for b, nm in ((sb, "ink thickness"), (gb, "word-gap ratio")):
        if b and b["rows"][0]["cer"] is not None and b["rows"][-1]["cer"] is not None:
            parts.append(f"{nm}: {b['rows'][0]['cer'] * 100:.1f} % in the lowest quintile vs {b['rows'][-1]['cer'] * 100:.1f} % in the highest")
    lb = D["lenbins"]["rows"]
    if lb[-1]["cer"] is not None:
        parts.append(f"lines of more than 65 characters: {lb[-1]['cer'] * 100:.1f} % ({lb[-1]['n']} lines)")
    ins["bins"] = "CER by measured property - " + "; ".join(parts) + "." if parts else ""
    if "train" in D:
        s_ = D["train"]["series"]
        tr_ = [v for v in s_.get("train_eval_cer", []) if v is not None]
        va_ = [v for v in (s_.get("val_sel_cer") or s_.get("val_cer", [])) if v is not None]
        if tr_ and va_:
            ins["train"] = (f"At the end of training the labeled lines are read at CER {tr_[-1] * 100:.2f} % while held-out validation lines are read at {va_[-1] * 100:.2f} % - "
                            f"a {va_[-1] / max(tr_[-1], 1e-6):.0f}x gap: the labeled set is memorised, so more passes over it cannot help; only new information can.")
    adopted_opts = [e["label"] for e in led if e["kind"] == "option" and e["adopted"]]
    ins["ledger"] = (f"System CER {final_cer * 100:.2f} % vs {cer_all * 100:.2f} % for the plain greedy reading. Options adopted on validation: "
                     + (", ".join(adopted_opts) if adopted_opts else "none") + ". Options whose validation evidence did not exclude zero are shown hollow and are not applied.")
    D["insights"] = ins
    looks = []
    lp_ = os.path.join(run_dir, "protocol", "test_looks.jsonl")
    if os.path.exists(lp_):
        with open(lp_) as f:
            looks = [json.loads(x) for x in f if x.strip()]
    D["integrity"] = {"protocol": prot, "looks": looks, "provenance": M.get("provenance"), "model_selection": M.get("model_selection"), "checkpoint": ck,
                      "posthoc_rule": ph.get("rule")}
    D["caveats"] = [
        "Writer-independence of the test split is not verified here: if train and test share writers, the transductive gains (WSD, AdaBN) are optimistic for unseen writers.",
        "A single run is a single draw: claims about a method need the seed-paired suite (labeled_only / legacy_pl / full over >= 3 seeds), not one run.",
        "The external language model was trained on web text that may contain the corpus the benchmark is drawn from: report LLM-rescored numbers as 'with an external language model'.",
        "The form-clustered intervals treat 20 consecutive lines as one cluster; the corpora have no writer id, so the true cluster structure is unknown.",
        "Hardness strata, style clusters and the FRAG-RC evidence use measured pen properties only - never the transcription.",
        "Right-to-left corpora are shown in logical order; the recogniser reads them in visual order (the metrics are identical)."]
    if synthetic:
        D["caveats"].insert(0, "SYNTHETIC SMOKE TEST: generated fake handwriting; the numbers have no scientific meaning.")
    return D


def export_report_folders(D: Dict[str, Any], out: str) -> None:
    L = D["lines"]
    os.makedirs(out, exist_ok=True)
    _write_csv(os.path.join(out, "01_ledger", "ledger.csv"), ["step", "CER", "ci_lo", "ci_hi", "adopted", "kind", "note"],
               [[e["label"], e.get("cer"), (e.get("ci") or [None, None])[0], (e.get("ci") or [None, None])[1], e.get("adopted"), e.get("kind"), e.get("note")] for e in D["ledger"]])
    with open(os.path.join(out, "01_ledger", "headline.json"), "w") as f:
        json.dump(json_safe({"meta": D["meta"], "head": D["head"], "insight": D["insights"].get("ledger")}), f, indent=2, allow_nan=False)
    keys = ["idx", "corpus", "cer", "conf", "dis", "ed", "rl", "edf", "strat"] + list(STYLE_NAMES) + ["gt", "pr", "pf"] + (["edw", "pw", "clus"] if L.get("edw") else [])
    _write_csv(os.path.join(out, "02_error_terrain", "lines.csv"), keys, [[L[k][i] for k in keys] for i in range(len(L["idx"]))])
    _write_csv(os.path.join(out, "02_error_terrain", "blocks.csv"), ["first_line", "lines", "CER"], [[a, D["blocks"]["size"], c] for a, c in zip(D["blocks"]["first"], D["blocks"]["cer"])])
    _write_csv(os.path.join(out, "03_concentration_abstention", "lorenz.csv"), ["share_of_lines_worst_first", "share_of_edits"], list(zip(D["lorenz"]["x"], D["lorenz"]["y"])))
    A = D["abstain"]
    _write_csv(os.path.join(out, "03_concentration_abstention", "risk_coverage.csv"), ["abstained_share", "CER_of_kept_by_model_doubt", "CER_of_kept_oracle", "share_of_edits_removed"],
               list(zip(A["a"], A["conf"], A["oracle"], A["share"])))
    rows = []
    for b in D["bins"] + [D["lenbins"]]:
        for k, r in enumerate(b["rows"]):
            rows.append([b["label"], k + 1, r["lo"], r["hi"], r["n"], r["cer"], (r["ci"] or [None, None])[0], (r["ci"] or [None, None])[1], r["cer_final"]])
    _write_csv(os.path.join(out, "04_who_is_hard", "style_bins.csv"), ["property", "quintile", "from", "to", "lines", "CER", "ci_lo", "ci_hi", "CER_final"], rows)
    if D.get("train"):
        s = D["train"]["series"]
        ks = list(s.keys())
        _write_csv(os.path.join(out, "05_training_story", "history.csv"), ks, list(zip(*[s[k] for k in ks])))
        fk = [k for k in ks if k.startswith("frag_") or k.startswith("train_frag") or k == "epoch"]
        if len(fk) > 1:
            _write_csv(os.path.join(out, "06_trust_audit", "frag_timeline.csv"), fk, list(zip(*[s[k] for k in fk])))
    if D.get("frag"):
        fa = D["frag"]
        _write_csv(os.path.join(out, "06_trust_audit", "frag_audit.csv"), ["stratum", "lines", "characters", "trusted_share", "error_of_trusted", "error_of_doubtful", "target"],
                   [[s["stratum"], s["lines"], s["chars"], s["trusted_share"], s["err_trusted"], s["err_doubtful"], fa.get("target")] for s in fa["strata"]])
    if D.get("wsd"):
        w = D["wsd"]
        _write_csv(os.path.join(out, "07_writer_adaptation", "wsd_clusters.csv"), ["split", "cluster", "lines", "CER_before", "CER_after"],
                   [[sp, g, c["lines"], c["CER_before"], c["CER_after"]] for sp, key in (("val_gate", "val_by_cluster"), ("test", "test_by_cluster")) for g, c in (w.get(key) or {}).items()])
    if D.get("traces"):
        with open(os.path.join(out, "08_reading_traces", "traces.json"), "w", encoding="utf-8") as f:
            json.dump(json_safe(D["traces"]), f, indent=1, ensure_ascii=False, allow_nan=False)
    I = D["integrity"]
    os.makedirs(os.path.join(out, "09_integrity"), exist_ok=True)
    with open(os.path.join(out, "09_integrity", "protocol.json"), "w") as f:
        json.dump(json_safe(I), f, indent=2, allow_nan=False)
    with open(os.path.join(out, "09_integrity", "test_looks.jsonl"), "w") as f:
        f.write("".join(json.dumps(x) + "\n" for x in I.get("looks") or []))
    with open(os.path.join(out, "09_integrity", "caveats.md"), "w", encoding="utf-8") as f:
        f.write("# What this report does not establish\n\n" + "\n".join(f"- {c}" for c in D["caveats"]) + "\n")


def build_run_report(infer_dir: str, run_dir: Optional[str] = None, out_dir: Optional[str] = None, curve_dir: Optional[str] = None) -> str:
    """Write <infer_dir>/12_report/ (index.html + the numbered folders).  Returns the path of index.html."""
    D = collect_report_data(infer_dir, run_dir, curve_dir)
    out = out_dir or os.path.join(infer_dir, "12_report")
    export_report_folders(D, out)
    data = json.dumps(json_safe(D), ensure_ascii=False, allow_nan=False).replace("</", "<\\/")
    page = _REPORT_HTML.replace("/*__CSS__*/", _REPORT_CSS).replace("/*__JS__*/", _REPORT_JS).replace("/*__DATA__*/null", data)
    path = os.path.join(out, "index.html")
    with open(path, "w", encoding="utf-8") as f:
        f.write(page)
    LOG.info("Run report written: %s", path)
    return path


_REPORT_CSS = r''':root{--paper:#f7f4ee;--card:#fffdf9;--ink:#1d2330;--muted:#6b7280;--line:#ddd6c8;--accent:#b5532f;--good:#1f7a6d;--warn:#c58a1a;--cool:#3b6ea8;--purple:#7a4fa3;--shadow:0 1px 2px rgba(30,25,15,.06),0 8px 24px rgba(30,25,15,.05)}
@media (prefers-color-scheme:dark){:root:not([data-theme="light"]){--paper:#14161a;--card:#1b1e24;--ink:#ebe8e1;--muted:#9aa1ad;--line:#343944;--accent:#e0805a;--good:#46b5a3;--warn:#e0aa42;--cool:#6fa3dc;--purple:#b08ad8;--shadow:0 1px 2px rgba(0,0,0,.4)}}
:root[data-theme="dark"]{--paper:#14161a;--card:#1b1e24;--ink:#ebe8e1;--muted:#9aa1ad;--line:#343944;--accent:#e0805a;--good:#46b5a3;--warn:#e0aa42;--cool:#6fa3dc;--purple:#b08ad8;--shadow:0 1px 2px rgba(0,0,0,.4)}
*{box-sizing:border-box}html{scroll-behavior:smooth}
body{margin:0;background:var(--paper);color:var(--ink);font:15px/1.55 -apple-system,"Segoe UI",Roboto,Helvetica,Arial,sans-serif}
.mono,.txt{font-family:ui-monospace,"SF Mono",Menlo,Consolas,monospace}
h1,h2,h3{font-family:Georgia,"Iowan Old Style","Times New Roman",serif;font-weight:600;letter-spacing:-.01em;margin:0}
.layout{display:grid;grid-template-columns:210px minmax(0,1fr);gap:0;max-width:1480px;margin:0 auto}
nav{position:sticky;top:0;align-self:start;height:100vh;padding:28px 16px 24px 24px;overflow:auto;border-right:1px solid var(--line)}
nav .brand{font-family:Georgia,serif;font-size:18px;font-weight:600;margin-bottom:2px}nav .sub{font-size:12px;color:var(--muted);margin-bottom:22px}
nav a{display:flex;gap:9px;align-items:baseline;padding:5px 0;color:var(--muted);text-decoration:none;font-size:13.5px}nav a b{font-variant-numeric:tabular-nums;color:var(--accent);font-weight:600;width:18px}nav a:hover,nav a.on{color:var(--ink)}
main{padding:30px 40px 90px;min-width:0}
header.top{padding-bottom:26px;border-bottom:1px solid var(--line);margin-bottom:12px}
header.top h1{font-size:34px;margin-bottom:6px}header.top p{margin:0;color:var(--muted);max-width:78ch}
.badges{display:flex;flex-wrap:wrap;gap:8px;margin:16px 0 4px}.badge{font-size:12px;padding:3px 10px;border-radius:99px;border:1px solid var(--line);background:var(--card);color:var(--muted)}
.badge.warn{background:#b00020;color:#fff;border-color:#b00020;font-weight:700}.badge.good{color:var(--good);border-color:var(--good)}
.kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(170px,1fr));gap:14px;margin-top:22px}
.kpi{background:var(--card);border:1px solid var(--line);border-radius:14px;padding:14px 16px;box-shadow:var(--shadow)}.kpi .v{font:600 30px/1.1 Georgia,serif;font-variant-numeric:tabular-nums}.kpi .l{font-size:12px;color:var(--muted);margin-top:4px}.kpi .s{font-size:12px;color:var(--muted);font-variant-numeric:tabular-nums}
section{padding:46px 0 10px;scroll-margin-top:8px}
section>.hd{display:flex;gap:14px;align-items:baseline}section>.hd .n{font:600 14px Georgia,serif;color:var(--accent);border:1px solid var(--accent);border-radius:99px;width:28px;height:28px;display:grid;place-items:center;flex:none}
section h2{font-size:26px}.insight{font-size:16.5px;margin:10px 0 18px;max-width:84ch;border-left:3px solid var(--accent);padding:2px 0 2px 14px}
.card{background:var(--card);border:1px solid var(--line);border-radius:14px;padding:16px 18px;box-shadow:var(--shadow);margin:14px 0}
.howto{font-size:13px;color:var(--muted);max-width:90ch;margin:8px 2px 0}.howto b{color:var(--ink)}
svg.chart{width:100%;height:auto;display:block;overflow:visible}svg text{fill:var(--ink);font:12px -apple-system,"Segoe UI",sans-serif}svg .m{fill:var(--muted)}svg .ax{stroke:var(--line);stroke-width:1}svg .gr{stroke:var(--line);stroke-width:1;stroke-dasharray:2 4}
.row2{display:grid;grid-template-columns:repeat(auto-fit,minmax(380px,1fr));gap:16px}.grid5{display:grid;grid-template-columns:repeat(auto-fit,minmax(300px,1fr));gap:14px}
table{border-collapse:collapse;width:100%;font-size:13px}th,td{padding:6px 8px;border-bottom:1px solid var(--line);text-align:left;vertical-align:top}th{font-weight:600;color:var(--muted);font-size:12px}
td.r,th.r{text-align:right;font-variant-numeric:tabular-nums;white-space:nowrap}
.tip{position:fixed;z-index:50;pointer-events:none;background:var(--ink);color:var(--paper);font-size:12px;padding:8px 10px;border-radius:8px;max-width:420px;display:none;box-shadow:0 6px 24px rgba(0,0,0,.25)}.tip .txt{white-space:pre-wrap;word-break:break-all}
canvas{display:block;max-width:100%}.cv{position:relative}.cv canvas+canvas{position:absolute;left:0;top:0}
button,select,input[type=text]{font:inherit;font-size:13px;color:var(--ink);background:var(--card);border:1px solid var(--line);border-radius:8px;padding:5px 10px}button{cursor:pointer}button:hover{border-color:var(--accent)}
input[type=range]{width:100%;accent-color:var(--accent)}
.ctrl{display:flex;flex-wrap:wrap;gap:10px;align-items:center;margin:8px 0 12px}
b.sub{background:rgba(181,83,47,.22);border-radius:3px;padding:0 1px}b.del{background:rgba(197,138,26,.35);border-radius:3px;padding:0 1px}b.ins{background:rgba(122,79,163,.30);border-radius:3px;padding:0 1px}
.diff{font-size:13px;line-height:1.55;word-break:break-word}.diff .k{display:inline-block;width:46px;color:var(--muted);font:11px sans-serif}
.bar{height:8px;border-radius:4px;background:var(--line);overflow:hidden;min-width:60px}.bar i{display:block;height:100%;background:var(--accent)}
.trace{display:grid;gap:8px}.trace .imgwrap{position:relative;background:#fff;border-radius:8px;overflow:hidden;border:1px solid var(--line)}.trace img{display:block;width:100%;height:auto}
.trace .ch{position:absolute;top:0;bottom:0;border-left:2px solid;opacity:.9}.trace .ch.t{border-color:var(--good);background:rgba(31,122,109,.10)}.trace .ch.d{border-color:var(--warn);background:rgba(197,138,26,.16)}.trace .ch.w{box-shadow:inset 0 -4px 0 #c0392b}
.legend{display:flex;flex-wrap:wrap;gap:14px;font-size:12px;color:var(--muted);margin:6px 0}.legend i{display:inline-block;width:12px;height:12px;border-radius:3px;margin-right:5px;vertical-align:-2px}
.note{font-size:12.5px;color:var(--muted)}.pill{font-size:11px;padding:1px 8px;border-radius:99px;border:1px solid var(--line);color:var(--muted)}.pill.ok{color:var(--good);border-color:var(--good)}.pill.no{color:var(--warn);border-color:var(--warn)}
.cav li{margin:6px 0}
@media (max-width:900px){.layout{grid-template-columns:1fr}nav{display:none}main{padding:20px 16px 60px}}
'''

_REPORT_JS = r'''(function(){
"use strict";
const NS="http://www.w3.org/2000/svg";
function mk(ns,tag,props,kids){
  const e=ns?document.createElementNS(NS,tag):document.createElement(tag);
  for(const k in (props||{})){const v=props[k];
    if(v===null||v===undefined||v===false)continue;
    if(k==="class")e.setAttribute("class",v);
    else if(k==="html")e.innerHTML=v;
    else if(k.slice(0,2)==="on")e.addEventListener(k.slice(2),v);
    else e.setAttribute(k,v===true?"":v);}
  for(const c of kids.flat(Infinity)){if(c===null||c===undefined||c===false)continue;e.append(c.nodeType?c:document.createTextNode(String(c)));}
  return e;}
const h=(tag,props,...kids)=>mk(false,tag,props,kids);
const s=(tag,props,...kids)=>mk(true,tag,props,kids);
const esc=t=>String(t).replace(/&/g,"&amp;").replace(/</g,"&lt;").replace(/>/g,"&gt;");
const fin=v=>v!==null&&v!==undefined&&isFinite(v);
const pct=(x,d=2)=>fin(x)?(x*100).toFixed(d)+" %":"n/a";
const pt=(x,d=2)=>fin(x)?(x>=0?"+":"")+(x*100).toFixed(d)+" pt":"n/a";
const f3=(x,d=3)=>fin(x)?Number(x).toFixed(d):"n/a";
const cv=n=>getComputedStyle(document.documentElement).getPropertyValue(n).trim();
const rgb=hex=>{const v=parseInt(hex.replace("#",""),16);return [v>>16&255,v>>8&255,v&255];};
const mix=(a,b,t)=>{t=Math.max(0,Math.min(1,t));return "rgb("+a.map((x,i)=>Math.round(x+(b[i]-x)*t)).join(",")+")";};
const clamp=(v,a,b)=>Math.max(a,Math.min(b,v));
const mean=a=>{let t=0,n=0;for(const v of a)if(fin(v)){t+=v;n++;}return n?t/n:NaN;};
const quant=(a,q)=>{const b=a.filter(fin).sort((x,y)=>x-y);return b.length?b[Math.min(b.length-1,Math.floor(q*b.length))]:NaN;};
const mounts=[];
const D=window.__D;
const tip=h("div",{class:"tip"});
function showTip(html,ev){tip.innerHTML=html;tip.style.display="block";const w=tip.offsetWidth,hh=tip.offsetHeight;let x=ev.clientX+14,y=ev.clientY+14;if(x+w>innerWidth-8)x=ev.clientX-w-14;if(y+hh>innerHeight-8)y=ev.clientY-hh-14;tip.style.left=x+"px";tip.style.top=y+"px";}
const hideTip=()=>{tip.style.display="none";};
function diff(ref,hyp){
  const a=Array.from(ref),b=Array.from(hyp),n=a.length,m=b.length;
  if(n*m>40000)return [esc(ref),esc(hyp)];
  const dp=[];for(let i=0;i<=n;i++){dp.push(new Uint16Array(m+1));dp[i][0]=i;}
  for(let j=0;j<=m;j++)dp[0][j]=j;
  for(let i=1;i<=n;i++)for(let j=1;j<=m;j++)dp[i][j]=Math.min(dp[i-1][j]+1,dp[i][j-1]+1,dp[i-1][j-1]+(a[i-1]===b[j-1]?0:1));
  let i=n,j=m;const ra=[],rb=[];const vis=c=>c===" "?"␣":esc(c);
  while(i>0||j>0){
    if(i>0&&j>0&&dp[i][j]===dp[i-1][j-1]+(a[i-1]===b[j-1]?0:1)){
      if(a[i-1]===b[j-1]){ra.push(esc(a[i-1]));rb.push(esc(b[j-1]));}
      else{ra.push('<b class="sub">'+vis(a[i-1])+"</b>");rb.push('<b class="sub">'+vis(b[j-1])+"</b>");}
      i--;j--;}
    else if(i>0&&dp[i][j]===dp[i-1][j]+1){ra.push('<b class="del">'+vis(a[i-1])+"</b>");i--;}
    else{rb.push('<b class="ins">'+vis(b[j-1])+"</b>");j--;}}
  return [ra.reverse().join(""),rb.reverse().join("")];}
const diffBlock=(gt,pr,extra)=>{const [r,p]=diff(gt,pr);let html='<div class="diff txt"><div><span class="k">truth</span>'+r+'</div><div><span class="k">reads</span>'+p+"</div>";
  if(extra){const [r2,p2]=diff(gt,extra.text);html+='<div><span class="k">'+esc(extra.label)+"</span>"+p2+"</div>";}return html+"</div>";};
const SEC=[];
function frame(id,title,insight,body,how){
  const n=SEC.length+1;SEC.push([id,title]);
  return h("section",{id},h("div",{class:"hd"},h("span",{class:"n"},n),h("h2",null,title)),
    insight?h("p",{class:"insight"},insight):null,...body,how?h("p",{class:"howto",html:how}):null);}
function lineChart(o){
  const W=o.w||980,Hh=o.h||300,m={l:56,r:18,t:22,b:34},pw=W-m.l-m.r,ph=Hh-m.t-m.b;
  const xs=o.xs,all=[];o.series.forEach(se=>se.vals.forEach(v=>{if(fin(v)&&(!o.ylog||v>0))all.push(v);}));
  if(!all.length||!xs.length)return null;
  let y0=o.ymin!==undefined?o.ymin:(o.ylog?Math.min(...all)*.8:Math.min(0,Math.min(...all))),y1=o.ymax!==undefined?o.ymax:Math.max(...all)*1.06;
  if(o.ylog){y0=Math.max(y0,1e-4);}
  const xa=Math.min(...xs),xb=Math.max(...xs)||1;
  const X=v=>m.l+pw*(v-xa)/Math.max(xb-xa,1e-9),Y=v=>o.ylog?m.t+ph*(1-(Math.log(v)-Math.log(y0))/(Math.log(y1)-Math.log(y0))):m.t+ph*(1-(v-y0)/(y1-y0));
  const svg=s("svg",{viewBox:`0 0 ${W} ${Hh}`,class:"chart"});
  const ticks=[];if(o.ylog){let t=Math.pow(10,Math.floor(Math.log10(y0)));for(;t<=y1*1.01;t*=10){[1,2,5].forEach(k=>{const v=t*k;if(v>=y0&&v<=y1)ticks.push(v);});}}else{for(let k=0;k<=4;k++)ticks.push(y0+(y1-y0)*k/4);}
  ticks.forEach(v=>{svg.append(s("line",{x1:m.l,x2:W-m.r,y1:Y(v),y2:Y(v),class:"gr"}),s("text",{x:m.l-8,y:Y(v)+4,"text-anchor":"end",class:"m"},o.yfmt?o.yfmt(v):v.toFixed(2)));});
  const step=Math.max(1,Math.ceil(xs.length/12));
  xs.forEach((v,i)=>{if(i%step===0)svg.append(s("text",{x:X(v),y:Hh-12,"text-anchor":"middle",class:"m"},v));});
  svg.append(s("line",{x1:m.l,x2:W-m.r,y1:m.t+ph,y2:m.t+ph,class:"ax"}));
  (o.markers||[]).forEach(mk_=>{if(mk_.x===null||mk_.x===undefined)return;svg.append(s("line",{x1:X(mk_.x),x2:X(mk_.x),y1:m.t,y2:m.t+ph,stroke:"var(--ink)","stroke-dasharray":"3 3",opacity:.6}),s("text",{x:X(mk_.x)+4,y:m.t+ph-6,class:"m"},mk_.label));});
  o.series.forEach(se=>{let d="",pen=false,cnt=0;se.vals.forEach((v,i)=>{if(!fin(v)||(o.ylog&&v<=0))return;d+=(pen?"L":"M")+X(xs[i]).toFixed(1)+" "+Y(v).toFixed(1);pen=true;cnt++;});
    if(d)svg.append(s("path",{d,fill:"none",stroke:se.color,"stroke-width":se.w||2.2,"stroke-dasharray":se.dash||null,"stroke-linejoin":"round"}));
    if(cnt&&cnt<se.vals.length*.6)se.vals.forEach((v,i)=>{if(fin(v)&&!(o.ylog&&v<=0))svg.append(s("circle",{cx:X(xs[i]),cy:Y(v),r:3,fill:se.color}));});});
  let lx=m.l;o.series.forEach(se=>{svg.append(s("line",{x1:lx,x2:lx+18,y1:10,y2:10,stroke:se.color,"stroke-width":3,"stroke-dasharray":se.dash||null}),s("text",{x:lx+23,y:14},se.name));lx+=34+se.name.length*6.3;});
  return svg;}
function secLedger(){
  const rows=D.ledger.filter(e=>fin(e.cer));if(!rows.length)return null;
  const W=1000,rowH=50,L=250,R=110,top=30,H=top+rows.length*rowH+34;
  const mx=Math.max(...rows.map(e=>e.ci?e.ci[1]:e.cer),...(D.refs||[]).map(r=>r.cer))*1.1;
  const X=v=>L+(W-L-R)*v/mx;const svg=s("svg",{viewBox:`0 0 ${W} ${H}`,class:"chart"});
  for(let k=0;k<=5;k++){const v=mx*k/5;svg.append(s("line",{x1:X(v),x2:X(v),y1:top-8,y2:H-30,class:"gr"}),s("text",{x:X(v),y:H-12,"text-anchor":"middle",class:"m"},(v*100).toFixed(1)+" %"));}
  const col=e=>e.kind==="ceiling"?"var(--muted)":e.kind==="option"?(e.adopted?"var(--good)":"var(--warn)"):e.kind==="base"?"var(--ink)":"var(--cool)";
  rows.forEach((e,i)=>{const y=top+i*rowH,c=col(e),filled=e.kind!=="ceiling"&&!(e.kind==="option"&&!e.adopted);
    svg.append(s("text",{x:L-14,y:y+17,"text-anchor":"end","font-weight":600},e.label),
      s("text",{x:L-14,y:y+33,"text-anchor":"end",class:"m"},e.kind==="option"?(e.adopted?"adopted":"not adopted"):e.kind==="ceiling"?"reference ceiling":e.kind==="milestone"?"milestone":"baseline"),
      s("rect",{x:L,y:y+6,width:Math.max(2,X(e.cer)-L),height:26,rx:5,fill:filled?c:"none",stroke:c,"stroke-width":2,"stroke-dasharray":filled?null:"5 4",opacity:filled?.88:1,
        onmousemove:ev=>showTip("<b>"+esc(e.label)+"</b><br>CER "+pct(e.cer)+"<br>"+esc(e.note||"")+(e.val_hi_adj!==undefined&&e.val_hi_adj!==null?"<br>validation gain, upper bound (Bonferroni-adjusted): "+pt(e.val_hi_adj):"")+(fin(e.test_gain)?"<br>test change vs. what it was applied to: "+pt(e.test_gain):""),ev),onmouseleave:hideTip}));
    if(e.ci)svg.append(s("line",{x1:X(e.ci[0]),x2:X(e.ci[1]),y1:y+19,y2:y+19,stroke:"var(--ink)","stroke-width":1.6}),s("line",{x1:X(e.ci[0]),x2:X(e.ci[0]),y1:y+13,y2:y+25,stroke:"var(--ink)"}),s("line",{x1:X(e.ci[1]),x2:X(e.ci[1]),y1:y+13,y2:y+25,stroke:"var(--ink)"}));
    svg.append(s("text",{x:X(e.ci?e.ci[1]:e.cer)+10,y:y+24,"font-weight":600},pct(e.cer)));});
  (D.refs||[]).forEach(r=>{svg.append(s("line",{x1:X(r.cer),x2:X(r.cer),y1:top-8,y2:H-30,stroke:"var(--purple)","stroke-width":1.6,"stroke-dasharray":"2 4"}),s("text",{x:X(r.cer)+5,y:top-12,fill:"var(--purple)"},"literature: "+pct(r.cer,1)));});
  const det=h("table",null,h("thead",null,h("tr",null,h("th",null,"step"),h("th",{class:"r"},"CER"),h("th",null,"evidence / what it is"))),
    h("tbody",null,rows.map(e=>h("tr",null,h("td",null,e.label),h("td",{class:"r"},pct(e.cer)),h("td",null,(e.note||"")+(e.val_hi_adj!==undefined&&e.val_hi_adj!==null?"  [validation gain upper bound, adjusted: "+pt(e.val_hi_adj)+"]":"")+(fin(e.test_gain)?"  [test change: "+pt(e.test_gain)+"]":""))))));
  const body=[h("div",{class:"card"},svg),h("div",{class:"card"},det)];
  const pc=D.head.per_corpus;if(pc&&Object.keys(pc).length>1){body.push(h("div",{class:"card"},h("h3",null,"Per corpus (greedy reading)"),h("table",null,h("thead",null,h("tr",null,["corpus","lines","CER","95 % CI","WER","exact lines"].map((t,i)=>h("th",{class:i?"r":""},t)))),
    h("tbody",null,Object.entries(pc).map(([k,v])=>h("tr",null,h("td",null,k),h("td",{class:"r"},v.lines),h("td",{class:"r"},pct(v.CER)),h("td",{class:"r"},v.CER_ci95?pct(v.CER_ci95[0],1)+" - "+pct(v.CER_ci95[1],1):""),h("td",{class:"r"},pct(v.WER,1)),h("td",{class:"r"},pct(v.exact_line_accuracy,1))))))));}
  return frame("ledger","The ledger",D.insights.ledger,body,
    "<b>Solid bars</b> are applied; <b>hollow bars</b> were evaluated but not adopted because their validation evidence (a Bonferroni-adjusted, form-clustered paired bootstrap on <b>val_gate</b>) did not exclude zero - the test split never decides. "+((D.refs||[]).length?"The dotted purple line is a published number for a different label budget, shown for context only.":""));}
function secTerrain(){
  const Ls=D.lines,n=Ls.idx.length;if(n<20)return null;
  const wrap=h("div",{class:"card"}),sel=h("div",{class:"card"},h("p",{class:"note"},"Drag across the terrain to select a stretch of lines, or use the buttons."));
  const tracks=[
    {name:"line CER",v:Ls.cer,kind:"err"},
    {name:"model doubt",v:Ls.conf.map(c=>fin(c)?1-c:null),kind:"doubt"},
    {name:"ink thickness",v:Ls.stroke_width,kind:"div"},
    {name:"word-gap ratio",v:Ls.gap_ratio,kind:"div"},
    {name:"hardness cell",v:Ls.strat,kind:"cat"}];
  const dEd=Ls.ed.map((e,i)=>e-Ls.edf[i]);if(dEd.some(v=>v!==0))tracks.push({name:"edits saved: options",v:dEd,kind:"delta"});
  if(Ls.edw){const dw=Ls.edf.map((e,i)=>e-Ls.edw[i]);if(dw.some(v=>v!==0))tracks.push({name:"edits saved: WSD",v:dw,kind:"delta"});}
  const ML=150,TH=19,GAP=4,BH=64;let state={a:null,b:null,drag:false,hover:-1};
  const base=h("canvas"),over=h("canvas"),box=h("div",{class:"cv"},base,over);wrap.append(box,h("div",{class:"legend"},
    h("span",null,h("i",{style:"background:var(--accent)"}),"hard / thin / tight"),h("span",null,h("i",{style:"background:var(--cool)"}),"easy / thick / wide"),
    h("span",null,h("i",{style:"background:var(--good)"}),"edits saved"),h("span",null,h("i",{style:"background:var(--purple)"}),"edits added")),
    h("div",{class:"ctrl"},h("button",{onclick:()=>pick("worst")},"jump to the worst 20-line block"),h("button",{onclick:()=>pick("best")},"jump to the best block"),h("button",{onclick:()=>{state.a=state.b=null;draw();sel.replaceChildren(h("p",{class:"note"},"Selection cleared."));}},"clear")));
  let W=900,Ht=0;
  function layout(){W=Math.max(640,wrap.clientWidth-36);Ht=tracks.length*(TH+GAP)+BH+30;const dpr=window.devicePixelRatio||1;
    [base,over].forEach(c=>{c.width=W*dpr;c.height=Ht*dpr;c.style.width=W+"px";c.style.height=Ht+"px";c.getContext("2d").setTransform(dpr,0,0,dpr,0,0);});}
  const paper=()=>rgb(cv("--paper")),A=()=>rgb(cv("--accent")),C=()=>rgb(cv("--cool")),G=()=>rgb(cv("--good")),P=()=>rgb(cv("--purple"));
  const stats=tracks.map(t=>{const m=mean(t.v);const sd=Math.sqrt(mean(t.v.map(x=>fin(x)?(x-m)*(x-m):null)))||1;return {m,sd,lo:quant(t.v,.03),hi:quant(t.v,.97)};});
  function colour(t,st,v){if(!fin(v))return null;const p=paper();
    if(t.kind==="err")return mix(p,A(),Math.sqrt(clamp(v/.35,0,1)));
    if(t.kind==="doubt")return mix(p,A(),clamp((v-st.lo)/Math.max(st.hi-st.lo,1e-6),0,1));
    if(t.kind==="div"){const z=clamp((v-st.m)/st.sd/2.2,-1,1);return z<0?mix(p,A(),-z):mix(p,C(),z);}
    if(t.kind==="cat"){return mix(p,A(),clamp(v/Math.max(st.hi,1),0,1)*.9+.05);}
    if(t.kind==="delta"){const z=clamp(v/3,-1,1);return z>0?mix(p,G(),z):z<0?mix(p,P(),-z):mix(p,p,0);}
    return null;}
  function draw(){const ctx=base.getContext("2d"),pw=W-ML-8;ctx.clearRect(0,0,W,Ht);ctx.font="12px -apple-system,Segoe UI,sans-serif";ctx.textBaseline="middle";
    tracks.forEach((t,k)=>{const y=k*(TH+GAP)+2;ctx.fillStyle=cv("--ink");ctx.fillText(t.name,0,y+TH/2);
      for(let c=0;c<pw;c++){const i0=Math.floor(c*n/pw),i1=Math.max(i0+1,Math.floor((c+1)*n/pw));let tot=0,cnt=0;for(let i=i0;i<i1&&i<n;i++){const v=t.v[i];if(fin(v)){tot+=v;cnt++;}}
        if(cnt){const col=colour(t,stats[k],tot/cnt);if(col){ctx.fillStyle=col;ctx.fillRect(ML+c,y,1.25,TH);}}}});
    const y0=tracks.length*(TH+GAP)+6;ctx.fillStyle=cv("--ink");ctx.fillText("20-line block CER",0,y0+BH/2);
    const bw=D.blocks;const mxb=Math.max(.2,quant(bw.cer,.98));
    bw.cer.forEach((v,b)=>{if(!fin(v))return;const x0=ML+pw*bw.first[b]/n,x1=ML+pw*Math.min(n,bw.first[b]+bw.size)/n;const hh=BH*clamp(v/mxb,0,1);ctx.fillStyle=cv("--accent");ctx.globalAlpha=.85;ctx.fillRect(x0,y0+BH-hh,Math.max(1,x1-x0-.5),hh);ctx.globalAlpha=1;});
    ctx.strokeStyle=cv("--line");ctx.beginPath();ctx.moveTo(ML,y0+BH+.5);ctx.lineTo(W-8,y0+BH+.5);ctx.stroke();
    ctx.fillStyle=cv("--muted");ctx.fillText("line 0",ML,y0+BH+14);ctx.textAlign="right";ctx.fillText("line "+Ls.idx[n-1],W-8,y0+BH+14);ctx.textAlign="left";
    const oc=over.getContext("2d");oc.clearRect(0,0,W,Ht);
    if(state.a!==null){const i0=Math.min(state.a,state.b),i1=Math.max(state.a,state.b),x0=ML+pw*i0/n,x1=ML+pw*(i1+1)/n;oc.fillStyle="rgba(59,110,168,.20)";oc.fillRect(x0,0,Math.max(2,x1-x0),Ht-16);oc.strokeStyle=cv("--cool");oc.strokeRect(x0+.5,.5,Math.max(2,x1-x0),Ht-17);}
    if(state.hover>=0){const x=ML+pw*state.hover/n;oc.strokeStyle=cv("--ink");oc.globalAlpha=.55;oc.beginPath();oc.moveTo(x+.5,0);oc.lineTo(x+.5,Ht-16);oc.stroke();oc.globalAlpha=1;}}
  const lineAt=ev=>{const r=base.getBoundingClientRect(),pw=W-ML-8;return clamp(Math.floor((ev.clientX-r.left-ML)/pw*n),0,n-1);};
  const inPlot=ev=>{const r=base.getBoundingClientRect();return ev.clientX-r.left>=ML;};
  over.addEventListener("mousemove",ev=>{if(!inPlot(ev)){hideTip();return;}const i=lineAt(ev);state.hover=i;if(state.drag){state.b=i;}
    const pw=W-ML-8,i0=Math.floor(Math.floor((i/n)*pw)*n/pw),i1=Math.min(n-1,Math.max(i0,Math.floor((Math.floor((i/n)*pw)+1)*n/pw)-1));let wi=i0;for(let q=i0;q<=i1;q++)if(Ls.ed[q]>Ls.ed[wi])wi=q;
    showTip("<b>lines "+Ls.idx[i0]+(i1>i0?"-"+Ls.idx[i1]:"")+"</b>"+(Ls.corpus[i]?" ("+esc(Ls.corpus[i])+")":"")+"<br>CER "+pct(mean(Ls.cer.slice(i0,i1+1)))+" - doubt "+f3(mean(Ls.conf.slice(i0,i1+1).map(c=>fin(c)?1-c:null)))+"<br>stroke "+f3(Ls.stroke_width[i],3)+" - gap "+f3(Ls.gap_ratio[i],3)+'<div class="txt">worst here (line '+Ls.idx[wi]+", CER "+pct(Ls.cer[wi],1)+")<br>"+esc(Ls.gt[wi].slice(0,70))+"<br>"+esc(Ls.pr[wi].slice(0,70))+"</div>",ev);draw();});
  over.addEventListener("mouseleave",()=>{state.hover=-1;hideTip();draw();});
  over.addEventListener("mousedown",ev=>{if(!inPlot(ev))return;state.a=state.b=lineAt(ev);state.drag=true;draw();});
  window.addEventListener("mouseup",()=>{if(state.drag){state.drag=false;summarise();draw();}});
  function pick(which){const bw=D.blocks;let k=0;bw.cer.forEach((v,b)=>{if(fin(v)&&(which==="worst"?v>bw.cer[k]:v<bw.cer[k]))k=b;});state.a=bw.first[k];state.b=Math.min(n-1,bw.first[k]+bw.size-1);summarise();draw();}
  function summarise(){const i0=Math.min(state.a,state.b),i1=Math.max(state.a,state.b);let es=0,ls=0,et=0,lt=0;for(let i=0;i<n;i++){et+=Ls.ed[i];lt+=Ls.rl[i];if(i>=i0&&i<=i1){es+=Ls.ed[i];ls+=Ls.rl[i];}}
    const inside=[];for(let i=i0;i<=i1;i++)inside.push(i);inside.sort((a,b)=>Ls.ed[b]-Ls.ed[a]);
    const zs=(arr)=>{const m=mean(arr),sd=Math.sqrt(mean(arr.map(x=>fin(x)?(x-m)*(x-m):null)))||1;return (mean(arr.slice(i0,i1+1))-m)/sd;};
    sel.replaceChildren(h("div",{class:"row2"},
      h("div",null,h("h3",null,"lines "+Ls.idx[i0]+" - "+Ls.idx[i1]+" ("+(i1-i0+1)+" lines)"),
        h("p",null,"CER here ",h("b",null,pct(es/ls)),", everywhere else ",h("b",null,pct((et-es)/(lt-ls))),". This stretch holds ",h("b",null,(100*es/et).toFixed(1)+" %")," of all edits in ",(100*(i1-i0+1)/n).toFixed(1)+" % of the lines."),
        h("p",{class:"note"},"Mean ink thickness is ",f3(zs(Ls.stroke_width),2)," sd and word-gap ratio ",f3(zs(Ls.gap_ratio),2)," sd from the corpus mean (negative = thinner / tighter).")),
      h("div",null,h("h3",null,"worst lines in the selection"),...inside.slice(0,4).map(i=>h("div",{html:'<span class="note">line '+Ls.idx[i]+" - "+Ls.ed[i]+" edits</span>"+diffBlock(Ls.gt[i],Ls.pr[i])})))));}
  mounts.push(()=>{layout();draw();});window.addEventListener("resize",()=>{layout();draw();});
  return frame("terrain","Error terrain",D.insights.terrain,[wrap,sel],
    "Each column is a few consecutive test lines (corpora are stored form by form, so neighbours usually share a writer). <b>Rows 1-2</b>: where the reading is wrong and where the model itself doubts it. <b>Rows 3-4</b>: pen properties measured from the pixels (never from the text). <b>Bottom</b>: CER per block of 20 lines.");}
function secAbstain(){
  const Lz=D.lorenz,A=D.abstain;if(!Lz||!A)return null;
  const W=480,Hh=340,m={l:52,r:14,t:20,b:40},pw=W-m.l-m.r,ph=Hh-m.t-m.b;
  const lor=s("svg",{viewBox:`0 0 ${W} ${Hh}`,class:"chart"});
  const X=v=>m.l+pw*v,Y=v=>m.t+ph*(1-v);
  [0,.25,.5,.75,1].forEach(v=>{lor.append(s("line",{x1:m.l,x2:W-m.r,y1:Y(v),y2:Y(v),class:"gr"}),s("text",{x:m.l-7,y:Y(v)+4,"text-anchor":"end",class:"m"},Math.round(v*100)+"%"),s("text",{x:X(v),y:Hh-22,"text-anchor":"middle",class:"m"},Math.round(v*100)+"%"));});
  lor.append(s("line",{x1:X(0),y1:Y(0),x2:X(1),y2:Y(1),stroke:"var(--muted)","stroke-dasharray":"4 4"}));
  lor.append(s("path",{d:"M"+Lz.x.map((x,i)=>X(x).toFixed(1)+" "+Y(Lz.y[i]).toFixed(1)).join("L"),fill:"none",stroke:"var(--accent)","stroke-width":2.6}));
  [["5","5 %"],["10","10 %"],["20","20 %"]].forEach(([k,l])=>{const x=+k/100,y=Lz.shares[k];lor.append(s("circle",{cx:X(x),cy:Y(y),r:4.5,fill:"var(--accent)"}),s("text",{x:X(x)+8,y:Y(y)+16,"font-weight":600},"worst "+l+" of lines = "+Math.round(y*100)+" % of edits"));});
  lor.append(s("text",{x:m.l+pw/2,y:Hh-4,"text-anchor":"middle",class:"m"},"share of lines, worst first"),s("text",{x:12,y:m.t+ph/2,transform:`rotate(-90 12 ${m.t+ph/2})`,"text-anchor":"middle",class:"m"},"share of all character edits"));
  const rc=s("svg",{viewBox:`0 0 ${W} ${Hh}`,class:"chart"});const mxy=Math.max(A.base*1.12,.05);const Y2=v=>m.t+ph*(1-v/mxy),X2=v=>m.l+pw*v/.9;
  for(let k=0;k<=4;k++){const v=mxy*k/4;rc.append(s("line",{x1:m.l,x2:W-m.r,y1:Y2(v),y2:Y2(v),class:"gr"}),s("text",{x:m.l-7,y:Y2(v)+4,"text-anchor":"end",class:"m"},(v*100).toFixed(1)+"%"));}
  [0,.3,.6,.9].forEach(v=>rc.append(s("text",{x:X2(v),y:Hh-22,"text-anchor":"middle",class:"m"},Math.round(v*100)+"%")));
  rc.append(s("line",{x1:m.l,x2:W-m.r,y1:Y2(A.base),y2:Y2(A.base),stroke:"var(--muted)","stroke-dasharray":"2 4"}));
  const path=arr=>"M"+arr.map((v,i)=>X2(A.a[i]).toFixed(1)+" "+Y2(v).toFixed(1)).join("L");
  rc.append(s("path",{d:path(A.oracle),fill:"none",stroke:"var(--muted)","stroke-width":2,"stroke-dasharray":"6 4"}),s("path",{d:path(A.conf),fill:"none",stroke:"var(--cool)","stroke-width":2.8}));
  if(A.frag)rc.append(s("path",{d:path(A.frag),fill:"none",stroke:"var(--good)","stroke-width":2.8}),s("text",{x:m.l+8,y:m.t+ph-29,fill:"var(--good)","font-weight":600},"green: ranked by FRAG-RC expected character error"));
  const mk_=s("circle",{r:6,fill:"var(--accent)",stroke:"var(--card)","stroke-width":2});rc.append(mk_);
  rc.append(s("text",{x:m.l+8,y:m.t+ph-46,fill:"var(--cool)","font-weight":600},"blue: ranked by "+A.risk_name),s("text",{x:m.l+8,y:m.t+ph-12,class:"m"},"dashed: an oracle that knows the errors"),s("text",{x:m.l+pw/2,y:Hh-4,"text-anchor":"middle",class:"m"},"share of lines the system declines to read"),s("text",{x:12,y:m.t+ph/2,transform:`rotate(-90 12 ${m.t+ph/2})`,"text-anchor":"middle",class:"m"},"CER on the lines it does read"));
  const out=h("div",{class:"note",style:"font-size:14px;margin-top:8px"}),rng=h("input",{type:"range",min:0,max:90,value:20,"aria-label":"share of lines declined"});
  const upd=()=>{const k=+rng.value;mk_.setAttribute("cx",X2(A.a[k]));mk_.setAttribute("cy",Y2(A.conf[k]));out.innerHTML="Decline the <b>"+k+" %</b> least reliable lines &rarr; CER on the rest <b>"+pct(A.conf[k])+"</b>"+(A.frag?" by model doubt, <b>"+pct(A.frag[k])+"</b> by FRAG-RC risk":"")+" (oracle "+pct(A.oracle[k])+", no declining "+pct(A.base)+"). The declined lines hold <b>"+(A.share[k]*100).toFixed(0)+" %</b> of all edits.";};
  rng.addEventListener("input",upd);upd();
  return frame("abstain","Where the errors live",D.insights.abstain,[h("div",{class:"row2"},h("div",{class:"card"},h("h3",null,"Concentration of the errors"),lor),h("div",{class:"card"},h("h3",null,"Read less, err less"),rc,rng,out)),
    h("p",{class:"note"},"Area under the risk-coverage curve (lower is better): model doubt "+f3(A.aurc_conf,4)+(A.frag?", FRAG-RC expected error "+f3(A.aurc_frag,4):"")+", oracle "+f3(A.aurc_oracle,4)+".")],
    "<b>Left</b>: sort the lines from worst to best and add up their edits. <b>Right</b>: drag the slider - the system declines the lines it is least sure about and the remaining lines are read better.");}
function secBins(){
  const all=[...D.bins,D.lenbins].filter(b=>b&&b.rows&&b.rows.some(r=>r.cer!==null));if(!all.length)return null;
  const mx=Math.max(...all.flatMap(b=>b.rows.map(r=>r.ci?r.ci[1]:(r.cer||0))))*1.08;
  const grid=h("div",{class:"grid5"});
  all.forEach(b=>{const W=320,Hh=210,m={l:42,r:8,t:26,b:46},pw=W-m.l-m.r,ph=Hh-m.t-m.b,bw=pw/b.rows.length;const Y=v=>m.t+ph*(1-v/mx);
    const svg=s("svg",{viewBox:`0 0 ${W} ${Hh}`,class:"chart"});
    for(let k=0;k<=3;k++){const v=mx*k/3;svg.append(s("line",{x1:m.l,x2:W-m.r,y1:Y(v),y2:Y(v),class:"gr"}),s("text",{x:m.l-5,y:Y(v)+4,"text-anchor":"end",class:"m"},(v*100).toFixed(0)+"%"));}
    b.rows.forEach((r,k)=>{if(r.cer===null)return;const x=m.l+k*bw+bw*.16,w=bw*.68;
      svg.append(s("rect",{x,y:Y(r.cer),width:w,height:Math.max(1,m.t+ph-Y(r.cer)),rx:3,fill:"var(--accent)",opacity:.45+.5*(k/(b.rows.length-1||1)),onmousemove:ev=>showTip("<b>"+esc(b.label)+"</b> "+r.lo.toFixed(3)+" - "+(r.hi>1e8?"max":r.hi.toFixed(3))+"<br>"+r.n+" lines<br>CER "+pct(r.cer)+(r.ci?" ["+pct(r.ci[0],1)+", "+pct(r.ci[1],1)+"]":"")+(Math.abs(r.cer_final-r.cer)>1e-6?"<br>after validated options "+pct(r.cer_final):""),ev),onmouseleave:hideTip}));
      if(r.ci)svg.append(s("line",{x1:x+w/2,x2:x+w/2,y1:Y(r.ci[0]),y2:Y(r.ci[1]),stroke:"var(--ink)","stroke-width":1.5}),s("line",{x1:x+w/2-4,x2:x+w/2+4,y1:Y(r.ci[1]),y2:Y(r.ci[1]),stroke:"var(--ink)"}),s("line",{x1:x+w/2-4,x2:x+w/2+4,y1:Y(r.ci[0]),y2:Y(r.ci[0]),stroke:"var(--ink)"}));
      if(Math.abs(r.cer_final-r.cer)>1e-6)svg.append(s("path",{d:`M${x+w/2} ${Y(r.cer_final)-5}l5 5l-5 5l-5 -5z`,fill:"var(--card)",stroke:"var(--good)","stroke-width":2}));
      svg.append(s("text",{x:x+w/2,y:Hh-m.b+14,"text-anchor":"middle",class:"m"},b.key==="length"?(r.hi>1e8?">"+r.lo:(r.lo?(r.lo+1)+"-"+r.hi:"<="+r.hi)):(r.lo+(r.hi-r.lo)/2).toFixed(b.key==="slant_deg"?1:3)),s("text",{x:x+w/2,y:Hh-m.b+28,"text-anchor":"middle",class:"m"},"n="+r.n));});
    svg.append(s("text",{x:m.l,y:14,"font-weight":600},b.label+(fin(b.spearman)?"   rho = "+b.spearman.toFixed(2):"")),s("text",{x:W-m.r,y:Hh-4,"text-anchor":"end",class:"m"},b.direction+" (quintiles)"));
    grid.append(h("div",{class:"card"},svg));});
  return frame("bins","Who is hard to read",D.insights.bins,[grid],
    "Test lines are cut into quintiles of a <b>measured</b> property; bars are corpus CER with a form-clustered 95 % interval. A green diamond marks the CER after the options adopted on validation.");}
function secTrain(){
  const T=D.train;if(!T)return null;const S=T.series,xs=S.epoch.map(v=>Math.round(v)+ (S.epoch[0]===0?1:0));
  const perc=Object.keys(S).filter(k=>k.startsWith("val_cer_")).map((k,i)=>({name:"validation: "+k.slice(8),vals:S[k],color:["var(--purple)","var(--warn)","var(--good)","var(--muted)"][i%4],dash:"3 3"}));
  const cA=lineChart({xs,ylog:true,ymin:.002,ymax:1,yfmt:v=>(v*100).toFixed(v<.01?1:0)+"%",h:300,markers:[{x:T.best_epoch,label:"checkpoint"}],
    series:[{name:"labeled train subset",vals:S.train_eval_cer,color:"var(--good)"},{name:S.val_sel_cer?"val_sel (selects checkpoint)":"validation",vals:S.val_sel_cer||S.val_cer,color:"var(--accent)"},
      ...(S.val_gate_cer?[{name:"val_gate (never selects)",vals:S.val_gate_cer,color:"var(--cool)"}]:[]),...perc].filter(se=>se.vals)});
  const fk=Object.keys(S).filter(k=>k.startsWith("frag_unl_err_sel_s")).sort();
  let cB=null,capB="";
  if(S.train_frag_char_trust||fk.length){
    const ser=[];if(S.train_frag_char_trust)ser.push({name:"trusted share of characters (training batches)",vals:S.train_frag_char_trust,color:"var(--good)"});
    const cols=["var(--accent)","var(--warn)","var(--cool)","var(--purple)"];fk.forEach((k,i)=>ser.push({name:"error of trusted chars, cell "+k.slice(18)+" (unlabeled sample)",vals:S[k],color:cols[i%4],dash:"5 4"}));
    cB=lineChart({xs,ymin:0,ymax:Math.max(.12,...ser.flatMap(x=>x.vals.filter(fin)))*1.05,yfmt:v=>(v*100).toFixed(0)+"%",h:280,series:ser,markers:[{x:T.best_epoch,label:"checkpoint"}]});
    capB="<b>FRAG-RC</b>: the green line is the share of emitted characters the trust rule lets teach; the dashed lines are the realised error of those trusted characters on a sample of unlabeled lines (hidden text, diagnostic only) - they should stay near the target error.";}
  else if(S.pl_coverage){
    cB=lineChart({xs,ymin:0,ymax:Math.max(.2,...S.pl_coverage.filter(fin))*1.15,yfmt:v=>(v*100).toFixed(0)+"%",h:260,markers:[{x:T.best_epoch,label:"checkpoint"}],
      series:[{name:"line-level pseudo-label coverage",vals:S.pl_coverage,color:"var(--good)"},...(S.rc_calib_cov_raw?[{name:"raw coverage of the risk rule",vals:S.rc_calib_cov_raw,color:"var(--accent)",dash:"5 4"}]:[])]});
    capB="<b>Legacy line-level pseudo-labels</b>: the share of unlabeled lines that passed the rule.";}
  return frame("train","How it learned",D.insights.train,[h("div",{class:"card"},cA),cB?h("div",{class:"card"},cB):null],
    "<b>Top</b> (log scale): green = how well the model reads the labeled lines it trains on, orange/blue = lines it never trained on. "+(capB?"<br>"+capB:""));}
function secTrust(){
  const F=D.frag,S=D.train&&D.train.series;if(!F&&!(S&&Object.keys(S).some(k=>k.startsWith("frag_cal_chars"))))return null;
  const body=[];let ins="";
  if(F){
    ins=(F.trusted_share>0&&fin(F.err_trusted))
      ?"Applied unchanged to the evaluation split, the FRAG-RC rule trusts "+pct(F.trusted_share,1)+" of the emitted characters; the trusted ones are wrong "+pct(F.err_trusted,2)+" of the time, the doubtful ones "+pct(F.err_doubtful,1)+" (target "+pct(F.target,1)+"). AUROC of the learned risk "+f3(F.auroc_risk,3)+" vs peak probability "+f3(F.auroc_conf,3)+"."
      :"The FRAG-RC rule trusts none of the emitted characters on the evaluation split (the characters it reads are wrong "+pct(F.err_all,1)+" of the time, target "+pct(F.target,1)+"): either the model is not yet reliable enough for any threshold to meet the target, or the rule calibrated on the training writers does not transfer.";
    body.push(h("div",{class:"card"},h("table",null,h("thead",null,h("tr",null,["cell","lines","characters","trusted share","error of trusted","error of doubtful","target"].map((t,i)=>h("th",{class:i?"r":""},t)))),
      h("tbody",null,F.strata.map(r=>h("tr",null,h("td",null,String(r.stratum)),h("td",{class:"r"},r.lines),h("td",{class:"r"},r.chars),h("td",{class:"r"},pct(r.trusted_share,1)),h("td",{class:"r"},pct(r.err_trusted,2)),h("td",{class:"r"},pct(r.err_doubtful,1)),h("td",{class:"r"},pct(F.target,1)))))),
      h("p",{class:"note"},"Cell = corpus x hardness tertile (0 = least hard: thick strokes, wide gaps); cut points come from the training images.")));}
  if(S){const ks=Object.keys(S).filter(k=>k.startsWith("frag_cal_cov_s")).sort();const xs=S.epoch.map(v=>Math.round(v)+1);
    if(ks.length){const cols=["var(--accent)","var(--warn)","var(--cool)","var(--purple)"];
      const ch=lineChart({xs,ymin:0,ymax:1,yfmt:v=>(v*100).toFixed(0)+"%",h:260,series:ks.map((k,i)=>({name:"calibration coverage, cell "+k.slice(14),vals:S[k],color:cols[i%4]}))});
      if(ch)body.push(h("div",{class:"card"},h("h3",null,"Share of calibration characters the rule trusts, per cell"),ch));}}
  return frame("trust","What the model trusts",ins,body,"FRAG-RC replaces line-level pseudo-labels. A <b>character</b> is trusted when a label-free evidence score falls below a per-cell threshold chosen so that the error of the trusted characters stays under the target on held-out calibration lines.");}
function secWsd(){
  const W=D.wsd;if(!W)return null;const e=D.ledger.find(x=>x.id==="wsd");
  const slope=(title,by)=>{const keys=Object.keys(by||{});if(!keys.length)return null;const Wd=440,Hh=78+keys.length*26,m=110,xr=Wd-60;
    const vs=keys.flatMap(k=>[by[k].CER_before,by[k].CER_after]).filter(fin);if(!vs.length)return null;
    let lo=Math.min(...vs),hi=Math.max(...vs);const pad=Math.max((hi-lo)*0.18,0.002);lo=Math.max(0,lo-pad);hi=hi+pad;const X=v=>m+(xr-m)*(v-lo)/(hi-lo);
    const svg=s("svg",{viewBox:`0 0 ${Wd} ${Hh}`,class:"chart"});svg.append(s("text",{x:0,y:16,"font-weight":600},title),
      s("circle",{cx:m,cy:31,r:4,fill:"var(--muted)"}),s("text",{x:m+9,y:35,class:"m"},"before"),s("circle",{cx:m+70,cy:31,r:5,fill:"var(--good)"}),s("text",{x:m+80,y:35,class:"m"},"after (green = lower CER)"));
    keys.forEach((k,i)=>{const y=56+i*26,c=by[k],better=c.CER_after<c.CER_before;svg.append(s("text",{x:0,y:y+4,class:"m"},"cluster "+k+" ("+c.lines+")"),s("line",{x1:X(c.CER_before),x2:X(c.CER_after),y1:y,y2:y,stroke:better?"var(--good)":"var(--purple)","stroke-width":3}),
      s("circle",{cx:X(c.CER_before),cy:y,r:4,fill:"var(--muted)"}),s("circle",{cx:X(c.CER_after),cy:y,r:5,fill:better?"var(--good)":"var(--purple)"}),s("text",{x:Wd-52,y:y+4},pt(c.CER_after-c.CER_before)));});
    const ya=56+keys.length*26-6;svg.append(s("line",{x1:m,x2:xr,y1:ya,y2:ya,class:"ax"}),s("text",{x:m,y:ya+16,class:"m"},pct(lo,1)),s("text",{x:xr,y:ya+16,class:"m","text-anchor":"end"},pct(hi,1)),s("text",{x:(m+xr)/2,y:ya+16,class:"m","text-anchor":"middle"},"CER (axis does not start at 0)"));return svg;};
  const ins=e?("Writer self-distillation was "+(e.adopted?"ADOPTED":"not adopted")+" on val_gate (adjusted upper bound of the gain "+pt(e.val_hi_adj)+"). Test CER "+pct(e.cer)+(fin(e.test_gain)?" ("+pt(e.test_gain)+" vs. the reading it was compared with).":".")):"";
  return frame("wsd","Learning a hand",ins,[h("div",{class:"row2"},h("div",{class:"card"},slope("val_gate lines (decides adoption)",W.val_by_cluster)),h("div",{class:"card"},slope("test lines (reports only)",W.test_by_cluster)))],
    "The evaluation lines are grouped by <b>measured</b> pen style (k-means). For each group a copy of the model takes a few label-free steps on that group's own images. Steps: "+W.chosen_steps+"; transductive.");}
function secNbest(){
  const N=D.nbest;if(!N||!N.examples||!N.examples.length)return null;
  const cards=N.examples.map(e=>h("div",{class:"card"},h("div",{class:"note"},"line "+e.line+" - "+e.tag+" ("+(e.gain_edits>0?"-":"+")+Math.abs(e.gain_edits)+" edits)"),
      h("table",null,h("thead",null,h("tr",null,["CTC rank","hypothesis (differences from the truth marked)","CTC log-prob","LM log-prob","fused score","edits"].map((t,i)=>h("th",{class:i>1?"r":""},t)))),
        h("tbody",null,e.items.map((x,k)=>{const [,hy]=diff(e.gt,x.text);return h("tr",{style:k===e.chosen?"background:rgba(31,122,109,.13)":null},
          h("td",null,"#"+(x.rank+1)+(x.rank===0?" (top-1)":"")+(k===e.chosen?" - chosen":"")),h("td",{class:"txt",html:hy}),h("td",{class:"r"},f3(x.ctc,2)),h("td",{class:"r"},f3(x.lm,2)),
          h("td",{class:"r"},f3(x.fused,2)),h("td",{class:"r"},x.ed));}))),
      h("div",{class:"txt note",style:"margin-top:6px"},"truth: "+e.gt)));
  const ins="The "+(N.llm||"language model")+" "+(N.adopted?"was adopted":"was NOT adopted")+" on val_gate. It changed the top-1 reading of "+pct(N.changed_share,1)+" of the "+N.lines+" lines: it helped "+N.helped_lines+" lines, hurt "+N.hurt_lines+", net "+N.net_edits_saved+" edits saved (beta = "+N.beta+", rho = "+N.rho+").";
  return frame("nbest","How the language model chooses",ins,[...cards],"Each table is the recogniser's own N-best list for one line. The language model never writes anything; it only re-ranks. <b>Green</b> = the hypothesis the fused score picked.");}
function secInterp(){
  const I=D.interp;if(!I||!Object.keys(I).length)return null;const body=[];
  if(I.layer_probes&&I.layer_probes.layers){const L=I.layer_probes;body.push(h("div",{class:"card"},h("h3",null,"Where does the reading form? (linear probes per layer)"),h("table",null,h("thead",null,h("tr",null,["layer","character probe accuracy","slant probe R2","frames"].map((t,i)=>h("th",{class:i?"r":""},t)))),
    h("tbody",null,L.layers.map(r=>h("tr",null,h("td",null,r.name),h("td",{class:"r"},pct(r.char_probe_accuracy,1)),h("td",{class:"r"},f3(r.slant_probe_r2,2)),h("td",{class:"r"},r.n_frames))))),h("p",{class:"note"},"majority class accuracy "+pct(L.majority_class_accuracy,1)+". "+L.note)));}
  if(I.uncertainty_decomposition&&I.uncertainty_decomposition.signals){const U=I.uncertainty_decomposition;body.push(h("div",{class:"card"},h("h3",null,"Which uncertainty signal predicts a wrong line?"),h("table",null,h("thead",null,h("tr",null,["signal","AUROC (line has error)","CER of kept lines at 80 % coverage"].map((t,i)=>h("th",{class:i?"r":""},t)))),
    h("tbody",null,Object.entries(U.signals).map(([k,v])=>h("tr",null,h("td",null,k),h("td",{class:"r"},f3(v.auroc_line_has_error,3)),h("td",{class:"r"},pct(v.CER_of_kept_lines_at_80pct_coverage)))))),h("p",{class:"note"},U.n_lines+" lines, "+pct(U.line_error_rate,1)+" with at least one error.")));}
  if(I.longline_stress&&fin(I.longline_stress.CER_concatenated)){const S=I.longline_stress;body.push(h("div",{class:"card"},h("h3",null,"Long-line stress test"),h("p",null,S.pairs+" concatenated pairs (mean "+S.mean_length_chars.toFixed(0)+" chars): CER "+pct(S.CER_concatenated)+" vs "+pct(S.CER_same_lines_read_alone)+" for the same lines read alone (gap "+pt(S.generalisation_gap)+").")));}
  if(I.style_neighbours&&I.style_neighbours.neighbour_vs_random_descriptor_gap_ratio){const R=I.style_neighbours.neighbour_vs_random_descriptor_gap_ratio;body.push(h("div",{class:"card"},h("h3",null,"What the style space groups"),h("p",null,"Mean |descriptor difference| of embedding neighbours / random pairs: "+Object.entries(R).map(([k,v])=>k+" "+f3(v,2)).join(", ")+" (below 1 = neighbours have similar measured style).")));}
  if(!body.length)return null;
  return frame("interp","Inside the model",null,body,"Details, CSVs and figures: 10_explanations/, 02_uncertainty/, 07_robustness/, 05_style/ next to this report.");}
function secTraces(){
  const T=D.traces;if(!T||!T.length)return null;
  const cards=T.map(t=>{const wrap=h("div",{class:"imgwrap"},h("img",{src:t.image,alt:"line "+t.line}));
    t.chars.forEach(c=>{const d=h("div",{class:"ch "+(c.trusted?"t":"d")+(c.wrong?" w":""),style:`left:${(100*c.x0).toFixed(2)}%;width:${(100*(c.x1-c.x0)).toFixed(2)}%`});
      d.addEventListener("mousemove",ev=>showTip("<b>'"+esc(c.ch)+"'</b> "+(c.trusted?"trusted":"doubtful")+" - risk "+f3(c.risk,3)+"<br>peak prob "+f3(c.p,3)+(c.alt?"<br>runner-up '"+esc(c.alt)+"' "+f3(c.palt,3):"")+(c.wrong?"<br><b>wrong</b>":""),ev));d.addEventListener("mouseleave",hideTip);wrap.append(d);});
    return h("div",{class:"card trace"},h("div",{class:"note"},"line "+t.line+" - "+esc(t.tag)+" - CER "+pct(t.cer,1)+(t.rtl?" - right-to-left (text shown in logical order)":"")),wrap,h("div",{html:diffBlock(t.gt,t.pr,t.pw!==null&&t.pw!==undefined?{label:"after WSD",text:t.pw}:null)}));});
  return frame("traces","Reading traces","Every emitted character is drawn over the ink that produced it: green = the trust rule would let it teach, amber = doubtful; a red underline marks a character that is actually wrong.",[...cards],
    "Frames are 4 pixels wide. Hover a box for its probability and the runner-up. The gallery holds the worst lines, the longest perfectly-read lines, random lines and the lines the writer self-distillation helped / hurt most.");}
function secExplore(){
  const Ls=D.lines,n=Ls.idx.length;const st={sort:"cer",q:"",page:0,minlen:0,strat:"all",ling:true};const PS=15;const isLing=g=>/[\p{L}\p{N}]/u.test(g);
  const cells=[...new Set(Ls.strat)].sort((a,b)=>a-b);
  const sorts={cer:["worst CER",i=>-Ls.cer[i]],ed:["most edits",i=>-Ls.ed[i]],doubt:["most doubt",i=>-(fin(Ls.conf[i])?1-Ls.conf[i]:0)],gain:["biggest gain from options / WSD",i=>-(Ls.ed[i]-(Ls.edw?Ls.edw[i]:Ls.edf[i]))],loss:["biggest loss from options / WSD",i=>(Ls.ed[i]-(Ls.edw?Ls.edw[i]:Ls.edf[i]))],idx:["line index",i=>Ls.idx[i]]};
  const list=h("div"),pg=h("div",{class:"ctrl"});
  const ctrl=h("div",{class:"ctrl"},h("select",{onchange:e=>{st.sort=e.target.value;st.page=0;render();}},Object.keys(sorts).map(k=>h("option",{value:k},sorts[k][0]))),
    h("input",{type:"text",placeholder:"search a word or a line number",oninput:e=>{st.q=e.target.value.toLowerCase();st.page=0;render();}}),
    h("select",{onchange:e=>{st.strat=e.target.value;st.page=0;render();}},h("option",{value:"all"},"all cells"),...cells.map(k=>h("option",{value:k},"cell "+k))),
    h("select",{onchange:e=>{st.minlen=+e.target.value;st.page=0;render();}},h("option",{value:0},"any length"),h("option",{value:40},">= 40 chars"),h("option",{value:65},">= 65 chars")),
    h("select",{onchange:e=>{st.ling=e.target.value==="1";st.page=0;render();}},h("option",{value:1},"handwriting lines only"),h("option",{value:0},"include separator lines")));
  function render(){let ix=[];for(let i=0;i<n;i++){if(Ls.rl[i]<st.minlen)continue;if(st.ling&&!isLing(Ls.gt[i]))continue;if(st.strat!=="all"&&Ls.strat[i]!==+st.strat)continue;
      if(st.q&&!(String(Ls.idx[i])===st.q||Ls.gt[i].toLowerCase().includes(st.q)||Ls.pr[i].toLowerCase().includes(st.q)))continue;ix.push(i);}
    const f=sorts[st.sort][1];ix.sort((a,b)=>f(a)-f(b)||a-b);const np=Math.max(1,Math.ceil(ix.length/PS));st.page=Math.min(st.page,np-1);
    list.replaceChildren(...ix.slice(st.page*PS,(st.page+1)*PS).map(i=>h("div",{class:"card",style:"margin:8px 0;padding:10px 14px"},
      h("div",{class:"note"},"line "+Ls.idx[i]+(Ls.corpus[i]?" ["+esc(Ls.corpus[i])+"]":"")+" - "+Ls.ed[i]+" edits / "+Ls.rl[i]+" chars - CER "+pct(Ls.cer[i],1)+" - confidence "+f3(Ls.conf[i],2)+" - cell "+Ls.strat[i]+(Ls.edw?" - WSD edits "+Ls.edw[i]:"")),
      h("div",{html:diffBlock(Ls.gt[i],Ls.pr[i],Ls.pw&&Ls.pw[i]!==Ls.pr[i]?{label:"after WSD",text:Ls.pw[i]}:(Ls.pf[i]!==Ls.pr[i]?{label:"after options",text:Ls.pf[i]}:null))}))));
    pg.replaceChildren(h("button",{onclick:()=>{st.page=Math.max(0,st.page-1);render();}},"< prev"),h("span",{class:"note"},"page "+(st.page+1)+" / "+np+" - "+ix.length+" lines"),h("button",{onclick:()=>{st.page=Math.min(np-1,st.page+1);render();}},"next >"));}
  mounts.push(render);
  return frame("explore","Line explorer","Every test line, with the reading laid over the truth.",[h("div",{class:"card"},ctrl,list,pg)],
    "<span style='background:rgba(181,83,47,.22);padding:0 4px;border-radius:3px'>substitution</span> <span style='background:rgba(197,138,26,.35);padding:0 4px;border-radius:3px'>missing in the reading</span> <span style='background:rgba(122,79,163,.30);padding:0 4px;border-radius:3px'>extra in the reading</span>; a visible space mark (&#9251;) shows errors on word gaps.");}
function secIntegrity(){
  const I=D.integrity,P=I.protocol,body=[];
  if(P){body.push(h("div",{class:"card"},h("h3",null,"Which split is used for what"),h("table",null,h("tbody",null,Object.entries(P.roles||{}).map(([k,v])=>h("tr",null,h("td",{style:"width:32%;font-weight:600"},k),h("td",null,v))))),
    h("p",{class:"note"},"validation = "+(P.val_sel_lines)+" lines for checkpoint selection (val_sel) + "+(P.val_gate_lines)+" lines for every post-hoc decision (val_gate), disjoint whole blocks of "+P.val_block_lines+" lines"+(P.val_split_enabled?"":" - SPLIT SWITCHED OFF in this run")+"."+(P.corpora?" Corpora: "+P.corpora.join(", ")+".":""))));}
  else body.push(h("div",{class:"card"},h("p",null,"No protocol.json was found for this run (an older run).")));
  body.push(h("div",{class:"card"},h("h3",null,"Looks at the evaluation split"),(I.looks&&I.looks.length)?h("table",null,h("thead",null,h("tr",null,["time","split","checkpoint","epoch"].map(t=>h("th",null,t)))),h("tbody",null,I.looks.map(l=>h("tr",null,h("td",null,l.time),h("td",null,l.split),h("td",{class:"mono"},String(l.checkpoint||"")),h("td",null,String(l.epoch===null||l.epoch===undefined?"":l.epoch)))))):h("p",{class:"note"},"No log of looks available.")));
  body.push(h("div",{class:"card cav"},h("h3",null,"What this report does not establish"),h("ul",null,D.caveats.map(c=>h("li",null,c)))));
  return frame("integrity","Integrity",null,body,"Every panel above is computed from the files in this folder. The numbers behind each panel sit in the numbered sub-folders next to this page.");}
function build(){
  const meta=D.meta,H=D.head;const app=document.getElementById("app");
  const secs=[secLedger(),secTerrain(),secAbstain(),secBins(),secTrain(),secTrust(),secWsd(),secNbest(),secInterp(),secTraces(),secExplore(),secIntegrity()].filter(Boolean);
  const kpi=(v,l,sub)=>h("div",{class:"kpi"},h("div",{class:"v"},v),h("div",{class:"l"},l),sub?h("div",{class:"s"},sub):null);
  const top=h("header",{class:"top"},h("h1",null,meta.title),
    h("p",null,"A reading of one run: where the model is right, where it is not, what it believes about its own characters, and what each step of the pipeline contributed. Everything on this page is computed from the run's own files."),
    h("div",{class:"badges"},meta.synthetic?h("span",{class:"badge warn"},"SYNTHETIC SMOKE TEST - fake handwriting - NOT FOR PUBLICATION"):h("span",{class:"badge good"},"real data"),
      h("span",{class:"badge"},"split: "+meta.split+" ("+meta.n+" lines)"),h("span",{class:"badge"},meta.version),(meta.corpora&&meta.corpora.length)?h("span",{class:"badge"},"corpora: "+meta.corpora.join(", ")):null,
      H.looks?h("span",{class:"badge"},"looks at this split so far: "+H.looks):null,H.ckpt_epoch?h("span",{class:"badge"},"checkpoint epoch "+H.ckpt_epoch+(H.weights_from?" ("+H.weights_from+")":"")):null,h("span",{class:"badge"},"built "+meta.generated)),
    h("div",{class:"kpis"},kpi(pct(H.cer),"character error rate",H.ci?"95 % form-clustered CI "+pct(H.ci[0],1)+" - "+pct(H.ci[1],1):null),
      kpi(pct(H.final_cer),"system CER with every validated option"),kpi(pct(H.wer,1),"word error rate",H.wer_ci?"CI "+pct(H.wer_ci[0],1)+" - "+pct(H.wer_ci[1],1):null),
      kpi(pct(H.exact,1),"lines read exactly"),H.labeled_fraction!==null&&H.labeled_fraction!==undefined?kpi(Math.round(H.labeled_fraction*100)+" %","of the training lines labeled",H.calibration_fraction?"+ "+Math.round(H.calibration_fraction*100)+" % held out to calibrate trust":null):null));
  const nav=h("nav",null,h("div",{class:"brand"},"Reading Atlas"),h("div",{class:"sub"},meta.run),...SEC.map(([id,t],i)=>h("a",{href:"#"+id,"data-id":id},h("b",null,i+1),t)));
  app.append(h("div",{class:"layout"},nav,h("main",null,top,...secs)),tip);
  mounts.forEach(f=>f());
  const links=[...nav.querySelectorAll("a")];
  const io=new IntersectionObserver(es=>{es.forEach(e=>{if(e.isIntersecting){links.forEach(a=>a.classList.toggle("on",a.dataset.id===e.target.id));}});},{rootMargin:"-30% 0px -60% 0px"});
  SEC.forEach(([id])=>{const el=document.getElementById(id);if(el)io.observe(el);});
}
window.addEventListener("DOMContentLoaded",()=>{try{build();}catch(e){document.body.append(h("pre",{style:"color:#b00020;padding:20px"},"Report error: "+e.message+"\n"+e.stack));console.error(e);}});
})();
'''

_REPORT_HTML = r'''<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>InkLingua - Reading Atlas</title>
<style>/*__CSS__*/</style></head>
<body><div id="app"></div>
<script>window.__D = /*__DATA__*/null;</script>
<script>/*__JS__*/</script>
</body></html>
'''

In [ ]:
%%writefile inklingua/suite.py
"""
The experiment suite: train + evaluate every variant for every seed and write the seed-paired comparison.  This is the experiment that
decides what a paper can claim: compare `full` with `labeled_only` (same labeled lines, no unlabeled data), with `ctc_only` (plain CTC
Transformer) and with `legacy_pl` (the v9 line-level pseudo-labels), and read every v10/v11 component off its own ablation.

suite_summary.json / .md hold, per variant: greedy CER, WER, CER with the validated post-hoc options, the system CER (incl. LLM rescoring if
adopted), mean +- std over seeds; against the baseline: the form-clustered paired bootstrap of CER(variant) - CER(baseline) per seed, the mean
over seeds and how many seeds are (significantly) better; and a component-contribution table "full minus each ablation".
"""
from __future__ import annotations

import copy
import csv
import json
import os
from typing import Any, Dict, Optional, Tuple

import numpy as np

from .config import SUITE_PRESETS, Config
from .text import paired_bootstrap_delta
from .utils import LOG, json_safe


def _read_edits(path: str) -> Dict[int, Tuple[int, int, int]]:
    """line_index -> (edits, reference length, edits after the validated post-hoc options) from a predictions.csv."""
    out: Dict[int, Tuple[int, int, int]] = {}
    with open(path, newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            ed, rl = int(r["edits"]), int(r["ref_len"])
            edp = int(round(float(r["CER_posthoc"]) * max(rl, 1))) if r.get("CER_posthoc") not in (None, "") else ed
            out[int(r["line_index"])] = (ed, rl, edp)
    return out


def _read_metrics(pred_path: str) -> Dict[str, Any]:
    m = os.path.join(os.path.dirname(os.path.dirname(pred_path)), "00_summary", "metrics.json")
    if not os.path.exists(m):
        m = os.path.join(os.path.dirname(pred_path), "metrics.json")
    try:
        with open(m) as f:
            return json.load(f)
    except Exception:
        return {}


def summarise_suite(runs: Dict[str, Dict[int, str]], baseline: str = "labeled_only", n_boot: int = 1000, block: int = 20) -> Dict[str, Any]:
    """runs[variant][seed] = path of that run's predictions.csv (test split)."""
    edits = {v: {s: _read_edits(p) for s, p in d.items()} for v, d in runs.items()}
    mets = {v: {s: _read_metrics(p) for s, p in d.items()} for v, d in runs.items()}
    out: Dict[str, Any] = {"baseline": baseline, "variants": {}, "paired_vs_baseline": {}, "contribution_vs_full": {}}

    def agg(vals):
        a = np.array([v for v in vals if v is not None and np.isfinite(v)], np.float64)
        return {"mean": float(a.mean()) if len(a) else float("nan"), "std": float(a.std(ddof=1)) if len(a) > 1 else float("nan"), "n": int(len(a))}

    for v, d in edits.items():
        cers = {s: sum(e for e, _, _ in m.values()) / max(sum(l for _, l, _ in m.values()), 1) for s, m in d.items()}
        cerp = {s: sum(e for _, _, e in m.values()) / max(sum(l for _, l, _ in m.values()), 1) for s, m in d.items()}
        wers = {s: mets[v][s].get("WER") for s in d}
        sysc = {s: mets[v][s].get("CER_system_final", mets[v][s].get("CER_final_with_validated_posthoc")) for s in d}
        out["variants"][v] = {"CER_per_seed": cers, "CER": agg(cers.values()), "CER_posthoc_per_seed": cerp, "CER_posthoc": agg(cerp.values()),
                              "WER_per_seed": wers, "WER": agg(wers.values()), "CER_system_per_seed": sysc, "CER_system": agg(sysc.values()),
                              "CER_mean": agg(cers.values())["mean"], "CER_std": agg(cers.values())["std"], "n_seeds": len(cers)}

    def paired(v, ref):
        per, deltas = {}, []
        for s, m in edits[v].items():
            b = edits[ref].get(s)
            if b is None:
                continue
            ids = sorted(set(m) & set(b))
            ea, eb = np.array([m[i][0] for i in ids], np.float64), np.array([b[i][0] for i in ids], np.float64)
            rl = np.array([m[i][1] for i in ids], np.float64)
            per[s] = paired_bootstrap_delta(ea, eb, rl, n_boot, s, groups=np.asarray(ids) // max(1, int(block)))
            deltas.append(per[s]["delta"])
        if not deltas:
            return None
        return {"per_seed": per, "mean_delta": float(np.mean(deltas)), "seeds_better": int(sum(d_ < 0 for d_ in deltas)), "n_seeds": len(deltas),
                "seeds_significantly_better": int(sum(p["hi"] < 0 for p in per.values())), "seeds_significantly_worse": int(sum(p["lo"] > 0 for p in per.values()))}

    if baseline in edits:
        for v in edits:
            if v != baseline:
                r = paired(v, baseline)
                if r:
                    out["paired_vs_baseline"][v] = r
    if "full" in edits:
        for v in edits:
            if v != "full" and (v.startswith("no_") or v in ("legacy_pl", "ctc_only")):
                r = paired(v, "full")
                if r:
                    out["contribution_vs_full"][v] = {**r, "reading": "positive mean_delta = removing the component HURTS (the component helps)"}
    return out


def suite_markdown(summ: Dict[str, Any]) -> str:
    rows = ["| variant | test CER (mean +- std) | WER | CER + validated post-hoc | system CER | per seed (CER) | vs baseline: mean delta | seeds better / significant |",
            "|---|---|---|---|---|---|---|---|"]
    f = lambda a: ("n/a" if not np.isfinite(a["mean"]) else f"{a['mean'] * 100:.2f}" + (f" +- {a['std'] * 100:.2f}" if np.isfinite(a["std"]) else "") + " %")  # noqa: E731
    for v, d in summ["variants"].items():
        pv = summ["paired_vs_baseline"].get(v)
        ps = ", ".join(f"{s}: {c * 100:.2f}" for s, c in d["CER_per_seed"].items())
        rows.append(f"| {v} | {f(d['CER'])} | {f(d['WER'])} | {f(d['CER_posthoc'])} | {f(d['CER_system'])} | {ps} | "
                    + (f"{pv['mean_delta'] * 100:+.2f} pt | {pv['seeds_better']}/{pv['n_seeds']} better, {pv['seeds_significantly_better']} significant |"
                       if pv else "(baseline) | |"))
    md = "\n".join(rows) + f"\n\nbaseline = `{summ['baseline']}`; negative delta = better than the baseline; 'significant' = the 95% form-clustered paired-bootstrap interval excludes 0.\n"
    if summ.get("contribution_vs_full"):
        md += "\n### Component contributions (ablation minus full; positive = the removed component helps)\n\n| removed | mean delta CER | seeds where removal hurts / significant |\n|---|---|---|\n"
        for v, r in summ["contribution_vs_full"].items():
            md += f"| {v} | {r['mean_delta'] * 100:+.2f} pt | {r['n_seeds'] - r['seeds_better']}/{r['n_seeds']} , {r['seeds_significantly_worse']} significant |\n"
    return md


def suite_variants(cfg: Config) -> Tuple[str, ...]:
    if cfg.run.suite_preset == "custom":
        return tuple(cfg.run.suite_variants)
    if cfg.run.suite_preset not in SUITE_PRESETS:
        raise ValueError(f"unknown suite preset '{cfg.run.suite_preset}' (choose from {sorted(SUITE_PRESETS)} or 'custom')")
    return SUITE_PRESETS[cfg.run.suite_preset]


def _member_pred_path(c: Config) -> str:
    return os.path.join(c.infer_dir, "01_recognition", "predictions.csv")


def run_suite(cfg: Optional[Config] = None, train: bool = True) -> Dict[str, Any]:
    """Every run has its own directory and resumes if interrupted, so the suite can be re-launched until it is complete.  The self-tests run once."""
    from .main import main
    from .selftest import run_self_tests
    cfg = cfg or Config()
    base = copy.deepcopy(cfg)
    base.run.mode = "single"
    if cfg.run.self_test and train:
        run_self_tests()
    base.run.self_test = False
    runs: Dict[str, Dict[int, str]] = {}
    for v in suite_variants(cfg):
        for s in cfg.run.suite_seeds:
            c = Config.make_variant(v, base).with_seed(s)
            c.run.mode = "single"
            p = _member_pred_path(c)
            if train:
                LOG.info("=== suite: variant '%s', seed %d  (%s)", v, s, c.run.name)
                m = main(c)
                if m is None or not os.path.exists(p):
                    continue
            elif not os.path.exists(p):
                LOG.warning("suite_summary: no predictions for variant '%s' seed %d (%s)", v, s, p)
                continue
            runs.setdefault(v, {})[s] = p
    out_dir = os.path.join(cfg.run.work_dir, f"{cfg.run.name}_suite")
    os.makedirs(out_dir, exist_ok=True)
    baseline = "labeled_only" if "labeled_only" in runs else next(iter(runs), "labeled_only")
    summ = summarise_suite(runs, baseline, cfg.infer.bootstrap, cfg.infer.bootstrap_block)
    summ["variants_requested"] = list(suite_variants(cfg))
    summ["seeds"] = list(cfg.run.suite_seeds)
    with open(os.path.join(out_dir, "suite_summary.json"), "w") as f:
        json.dump(json_safe(summ), f, indent=2, allow_nan=False)
    md = suite_markdown(summ)
    with open(os.path.join(out_dir, "suite_summary.md"), "w") as f:
        f.write(md)
    LOG.info("suite summary (%s):\n%s", out_dir, md)
    return summ


def run_seeds(cfg: Optional[Config] = None, seeds=(1337, 1338, 1339)) -> Dict[str, Any]:
    """Run the full experiment for several seeds and report mean +- std of the test CER / WER."""
    from .main import main
    cfg = cfg or Config()
    rows = []
    for s in seeds:
        m = main(cfg.with_seed(s))
        if m:
            rows.append({"seed": s, "CER": m["CER"], "WER": m["WER"]})
    summ: Dict[str, Any] = {"runs": rows}
    for k in ("CER", "WER"):
        v = np.array([r[k] for r in rows], np.float64)
        if len(v):
            summ[k] = {"mean": float(v.mean()), "std": float(v.std(ddof=1)) if len(v) > 1 else float("nan")}
    path = os.path.join(cfg.run.work_dir, f"{cfg.run.name}_seeds.json")
    os.makedirs(cfg.run.work_dir, exist_ok=True)
    with open(path, "w") as f:
        json.dump(json_safe(summ), f, indent=2, allow_nan=False)
    LOG.info("multi-seed summary: %s  (written to %s)", summ.get("CER"), path)
    return summ

In [ ]:
%%writefile inklingua/selftest.py
"""
Fast invariant tests (CPU, no download, no real data) and the synthetic end-to-end smoke.

`run_self_tests()` runs before every real run (cfg.run.self_test) and guards what a reader of the paper has to be able to trust: decoding and
metrics are right, padding does not change a prediction, the style-equivariant targets follow the geometry, hidden transcriptions cannot reach
training, the alphabet is grapheme-cluster based and RTL-safe, the trust rules do what they say, warm starts preserve the function.

`run_smoke()` trains a tiny model on SYNTHETIC lines, resumes, evaluates, builds the report (minutes on CPU).  It is NOT part of run_self_tests
(v10 ran it inside the self-tests, which cost every real run minutes of synthetic training and redirected its log handlers into a temporary
directory that was then deleted); run it with `mode="smoke"` or `tests/test_smoke.py`.
"""
from __future__ import annotations

import copy
import csv
import json
import math
import os
import random
import shutil
import tempfile
import time
import unicodedata
from types import SimpleNamespace
from typing import Any, Dict, List

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image, ImageDraw, ImageFont

from .alphabet import CharGroups, extend_codec, extend_ctc_heads, resolve_variants, visual_base_of
from .config import SUITE_PRESETS, Config, DatasetSpec
from .data import Collator, HandwritingDataModule, ListSource, font_paths, preprocess_line, to_u8
from .decoding import ctc_prefix_beam_search, decode_with_space_bonus, greedy_decode, greedy_runs
from .explain import adapt_bn_state, bn_layers, code_char_stats, contrast_rollout, load_bn, snapshot_bn
from .frag import FragModel, cell_theta, char_evidence, char_wrong_labels, fragment_mask
from .llm import Hypothesis, LLMEngine, calibrate_word_mark, mark_uncertain
from .model import ClosedFormLeakage, FiLM, GPUAugment, InkLinguaNet, body_ratio_t, compute_losses, ctc_per_sample, frame_consistency, fragment_distillation, transform_slant_deg
from .protocol import log_test_look, write_protocol
from .pseudo import PseudoLabelBank, SelectionRule, reliability_features, reliability_views
from .render import _baseline_y, _draw_line, render_styled_text_info, style_mismatch
from .riskcontrol import anchor_audit, auroc, char_risk_threshold, domain_shift, pava_increasing, risk_control_threshold, risk_curve, transfer_risk_control, weighted_ks, weighted_risk_curve
from .style import NUM_DESC, STYLE_FLOORS, body_ratio_from_rows, compute_style_descriptors, estimate_slant_deg
from .suite import suite_markdown, summarise_suite
from .text import (CharCodec, bootstrap_cer_ci, cer, char_class, chrf, ctc_feasible, detokenize, edit_distance, grapheme_clusters, paired_bootstrap_delta,
                   ridge_cv_r2, to_logical, to_visual, wer)
from .trainer import Trainer, warm_start
from .utils import LOG, restore_logging

RELIABILITY_N = 10


def run_self_tests(verbose: bool = True) -> None:
    t0, done = time.time(), []

    def ok(name: str) -> None:
        done.append(name)
        if verbose:
            LOG.info("self-test ok: %s", name)

    # ---- CTC decoding --------------------------------------------------------------------------------------------------------
    V = 4
    path = [0, 1, 1, 0, 1, 2, 2, 0, 3]
    lp = np.log(np.full((len(path), V), 0.02))
    for t, c in enumerate(path):
        lp[t, c] = np.log(0.94)
    ids, cf, fr = greedy_decode(lp)
    assert ids == [1, 1, 2, 3] and len(cf) == 4 and fr[0] < fr[1] < fr[2] < fr[3], (ids, fr)
    ids_r, st_r, en_r = greedy_runs(lp)
    assert ids_r == [1, 1, 2, 3] and st_r.tolist() == [1, 4, 5, 8] and en_r.tolist() == [3, 5, 7, 9]
    nb = ctc_prefix_beam_search(lp, beam=4)
    assert list(nb[0][0]) == [1, 1, 2, 3] and nb[0][1] <= 0.0 and all(nb[i][1] >= nb[i + 1][1] for i in range(len(nb) - 1))
    ok("CTC greedy / runs / prefix beam search")

    # ---- text: metrics, grapheme codec, RTL, detokeniser, bootstrap -------------------------------------------------------------
    assert abs(cer("abcd", "abxd") - 0.25) < 1e-9 and wer("a b c", "a x c") == 1 / 3 and cer("", "") == 0.0
    assert chrf("hello world", "hello world") == 100.0 and chrf("abc", "xyz") == 0.0
    for raw_, want_ in [('He said , " go . "', 'He said, "go."'), ("it 's do n't", "it's don't"), ("( a ) b , c ;", "(a) b, c;"),
                        ("`` hi '' , he said", "``hi'', he said"), ('say " a " and " b "', 'say "a" and "b"')]:
        got_ = detokenize(raw_)
        assert got_ == want_, (raw_, got_, want_)
    assert mark_uncertain("ab cd", [0.9, 0.9, 0.0, 0.2, 0.3], 0.7) == "ab cd[?]"
    Eg = unicodedata.normalize("NFC", "ẹ́")
    assert grapheme_clusters("ẹ́a") == [Eg, "a"] and grapheme_clusters("क्ष") == ["क्", "ष"]
    cd = CharCodec.from_texts(["ab " + Eg, "b"])
    assert cd.units == [" ", "a", "b", Eg] and cd.num_classes == 5 and cd.decode_ids(cd.encode("b" + Eg + "a")) == "b" + Eg + "a"
    assert CharCodec(cd.chars) == cd and cd.encode("z") == []                                  # chars round trip; unknown units dropped
    assert ctc_feasible("ab", 2) and not ctc_feasible("aa", 2) and ctc_feasible("aa", 3) and not ctc_feasible("", 5) and ctc_feasible(Eg + Eg, 3)
    ar = "سلام عليكم"
    vis = to_visual(ar, True)
    assert vis != ar and to_logical(vis, True) == ar and to_visual(ar, False) == ar and len(vis) == len(ar)
    assert cer(ar, ar.replace("م", "ن", 1)) == cer(to_visual(ar, True), to_visual(ar.replace("م", "ن", 1), True))      # metrics invariant under reversal
    assert char_class("a") == "lower" and char_class("A") == "upper" and char_class("س") == "letter" and char_class("中") == "letter" and char_class("٣") == "digit"
    assert char_class(".") == "punct" and char_class(" ") == "space"
    ed = np.array([0, 1, 2, 0, 3.0])
    rl = np.array([10, 10, 10, 10, 10.0])
    lo, hi = bootstrap_cer_ci(ed, rl, 300, 0)
    assert lo <= ed.sum() / rl.sum() <= hi
    d = paired_bootstrap_delta(ed * 0.5, ed, rl, 300, 0)
    assert d["delta"] < 0 and d["hi"] <= 0.0 and d["p_a_better"] > 0.9
    rg = np.random.RandomState(3)
    base_ = np.repeat(rg.rand(60) * 6, 10)
    edc = rg.poisson(base_).astype(np.float64)
    rlc = np.full(600, 40.0)
    lo_l, hi_l = bootstrap_cer_ci(edc, rlc, 600, 0)
    lo_g, hi_g = bootstrap_cer_ci(edc, rlc, 600, 0, groups=np.arange(600) // 10)
    assert (hi_g - lo_g) > 1.5 * (hi_l - lo_l) and lo_g <= edc.sum() / rlc.sum() <= hi_g
    dg = paired_bootstrap_delta(edc * 0.5, edc, rlc, 400, 0, groups=np.arange(600) // 10, n_tests=4)
    assert dg["hi_adj"] >= dg["hi"] and dg["lo_adj"] <= dg["lo"] and dg["delta"] < 0
    Xq = np.random.RandomState(4).randn(200, 3)
    Yq = np.column_stack([Xq[:, 0] + 0.1 * np.random.RandomState(5).randn(200)])
    r_a, r_b = ridge_cv_r2(Xq, Yq, np.ones_like(Yq, bool)), ridge_cv_r2(Xq, Yq + 50.0, np.ones_like(Yq, bool))
    assert r_a[0] > 0.8 and abs(r_a[0] - r_b[0]) < 1e-4
    ok("text metrics, grapheme-cluster codec, RTL round trip (metrics invariant), Unicode character classes, form-clustered bootstrap")

    # ---- [CAL-MARK] and the renderer baseline ----------------------------------------------------------------------------------
    rm_ = random.Random(2)
    preds_m, gts_m, confs_m = [], [], []
    for _ in range(120):
        gw = ["".join(rm_.choice("abcdef") for _ in range(rm_.randint(3, 7))) for _ in range(6)]
        pw, cw = [], []
        for w in gw:
            bad = rm_.random() < 0.3
            pw.append(("z" + w[1:]) if bad else w)
            c = (0.93 + 0.05 * rm_.random()) if bad else (0.985 + 0.014 * rm_.random())
            cw += [c] * len(w)
            cw.append(1.0)
        preds_m.append(" ".join(pw)), gts_m.append(" ".join(gw)), confs_m.append(cw[:-1])
    cal_m = calibrate_word_mark(preds_m, gts_m, confs_m, 0.5)
    assert cal_m["threshold"] is not None and cal_m["precision"] >= 0.5 and cal_m["recall"] > 0.3 and cal_m["auroc"] > 0.9 and 0.93 < cal_m["threshold"] < 0.999
    assert calibrate_word_mark(preds_m[:2], gts_m[:2], confs_m[:2])["threshold"] is None
    if font_paths():
        Hs_t, sz_t = 128, 60
        fnt_t = ImageFont.truetype(font_paths()[0], sz_t)
        inkp = lambda t: _draw_line(t, font_paths()[0], sz_t, 0.0, 10, 0, 0.0, Hs_t)  # noqa: E731
        rows_of = lambda a: np.flatnonzero(a.max(1) > 0.3)  # noqa: E731
        base_t = _baseline_y(fnt_t, Hs_t)
        rd, rp, rx = rows_of(inkp(".")), rows_of(inkp("p")), rows_of(inkp("x"))
        assert abs(rd.max() - base_t) <= 3 and rp.max() > base_t + 0.12 * sz_t and abs(rx.max() - base_t) <= 3
    ok("calibrated [?] marks, full stops and descenders on one baseline")

    # ---- preprocessing ---------------------------------------------------------------------------------------------------------
    img = Image.fromarray((255 - 200 * (np.random.RandomState(0).rand(128, 800) > 0.9)).astype(np.uint8))
    ink = preprocess_line(img, 64, 2560, 32, 8)
    assert ink.shape == (64, 400 + 16) and ink[:, :8].max() == 0 and ink[:, -8:].max() == 0 and 0 <= ink.min() <= ink.max() <= 1
    assert preprocess_line(Image.fromarray(np.full((128, 9000), 255, np.uint8)), 64, 2560, 32, 8).shape[1] <= 2560
    ok("line preprocessing")

    # ---- model: padding invariance (train and eval), hidden states ------------------------------------------------------------
    mc = copy.deepcopy(Config.smoke().model)
    mc.dropout, mc.layer_drop = 0.0, 0.0
    torch.manual_seed(0)
    net = InkLinguaNet(mc, 30, NUM_DESC, 10, 32)
    x = torch.rand(4, 1, 32, 96)
    w = torch.tensor([96, 80, 60, 40])
    for i, ww in enumerate(w):
        x[i, :, :, ww:] = 0
    x2 = torch.cat([x, torch.zeros(4, 1, 32, 64)], -1)
    net.train()
    net(x, w, vq_lambda=0.7)
    for mode in ("train", "eval"):
        a, b = InkLinguaNet(mc, 30, NUM_DESC, 10, 32), InkLinguaNet(mc, 30, NUM_DESC, 10, 32)
        a.load_state_dict(net.state_dict()), b.load_state_dict(net.state_dict())
        getattr(a, mode)(), getattr(b, mode)()
        o1, o2 = a(x, w, vq_lambda=0.7), b(x2, w, vq_lambda=0.7)
        for k in range(4):
            L = int(o1["out_lens"][k])
            assert (o1["log_probs"][k, :L] - o2["log_probs"][k, :L]).abs().max().item() < 1e-3, f"padding changes the output in {mode} mode"
    net.eval()
    oh = net(x, w, keep_hidden=True, extras=True)
    assert len(oh["hidden"]) == mc.enc_layers + 1 and oh["style_att"].shape == (4, oh["log_probs"].shape[1])
    ok("padding invariance (train and eval), hidden states and style attention exposed")

    # ---- style descriptors -----------------------------------------------------------------------------------------------------
    H_, W_ = 64, 400

    def bars(angle: float, n: int = 10) -> np.ndarray:
        m = np.zeros((H_, W_), np.float32)
        t = math.tan(math.radians(angle))
        for k in range(n):
            for y in range(8, H_ - 8):
                xx = int(round(20 + 36 * k - t * (y - (H_ - 1) / 2.0)))
                m[y, xx:xx + 3] = 1.0
        return m

    for a_ in (-20, -8, 0, 12, 25):
        est = estimate_slant_deg((bars(a_) > 0.45).astype(np.uint8))
        assert abs(est - a_) < 1.5, (a_, est)
    assert math.isnan(estimate_slant_deg(np.zeros((H_, W_), np.uint8)))
    assert np.isnan(compute_style_descriptors(np.zeros((H_, W_), np.float32))).all()
    assert np.isfinite(compute_style_descriptors(bars(10))).all()
    xr = (torch.rand(5, 1, 64, 200, generator=torch.Generator().manual_seed(1)) > 0.8).float()
    for b_ in range(5):
        assert abs(float(body_ratio_t(xr)[b_]) - body_ratio_from_rows((xr[b_, 0] > 0.45).sum(1).numpy(), 64)) < 1e-6
    ok("style descriptors (slant estimator, NaN = not measurable, body ratio)")

    # ---- [SES] -----------------------------------------------------------------------------------------------------------------
    src = ListSource(["the quick brown fox", "jumps over the lazy dog", "handwriting recognition", "pack my box"] * 6, 3)
    inks, wid = [], []
    for i in range(len(src)):
        u = to_u8(preprocess_line(src.image(i), 64, 400, 32, 8)).astype(np.float32) / 255.0
        wid.append(min(u.shape[1], 384))
        inks.append(np.pad(u, ((0, 0), (0, max(0, 384 - u.shape[1]))))[:, :384])
    inks = np.stack(inks)
    X = torch.from_numpy(inks)[:, None]
    d0 = np.stack([compute_style_descriptors(i) for i in inks])
    B, _, Hh, Ww = X.shape
    for shear in (0.10, -0.10):
        sh, rot, vs = torch.full((B,), shear), torch.zeros(B), torch.ones(B)
        th = torch.zeros(B, 2, 3)
        th[:, 0, 0], th[:, 0, 1], th[:, 1, 1] = 1, (sh - rot) / (Ww / Hh), vs
        Y = F.grid_sample(X, F.affine_grid(th, list(X.shape), align_corners=False), padding_mode="zeros", align_corners=False)
        d1 = np.stack([compute_style_descriptors(y[0].numpy()) for y in Y])
        pred = transform_slant_deg(torch.from_numpy(d0[:, 0]), sh, rot, vs).numpy()
        fin = np.isfinite(d1[:, 0]) & np.isfinite(pred)
        assert fin.sum() >= B // 2 and abs((d1[fin, 0] - pred[fin]).mean()) < 1.5
    tc = copy.deepcopy(Config.smoke().train)
    tc.aug_geom_prob, tc.aug_hscale_prob, tc.aug_elastic_prob, tc.aug_morph_prob, tc.aug_blur_prob = 1.0, 1.0, 0.0, 0.0, 0.0
    with np.errstate(all="ignore"):
        mu_, sd_ = np.nanmean(d0, 0), np.maximum(np.nanstd(d0, 0), STYLE_FLOORS)
    aug = GPUAugment(tc, mu_, sd_)
    okd = np.isfinite(d0)
    z = torch.from_numpy(np.clip(np.nan_to_num((d0 - mu_) / sd_), -3, 3)).float()
    torch.manual_seed(0)
    xs_, xw_, wd_, st_, sm_, cok_ = aug(X, z, torch.from_numpy(okd), torch.tensor(wid), torch.full((B,), 300))
    assert xs_.shape == X.shape and float(xs_.min()) >= 0 and float(xs_.max()) <= 1 and int(wd_.max()) <= Ww and int(wd_.min()) >= 300
    assert (sm_[~torch.from_numpy(okd)] == 0).all()
    e_sl, e_body = [], []
    for b_ in range(B):
        dm_ = compute_style_descriptors(xw_[b_, 0, :, :int(wd_[b_])].numpy())
        if sm_[b_, 0] > 0 and np.isfinite(dm_[0]):
            e_sl.append(dm_[0] - float(st_[b_, 0] * sd_[0] + mu_[0]))
        if sm_[b_, 4] > 0 and abs(float(st_[b_, 4])) < 2.9:
            e_body.append(dm_[4] - float(st_[b_, 4] * sd_[4] + mu_[4]))
    assert len(e_sl) >= B // 2 and abs(np.mean(e_sl)) < 1.5 and np.mean(np.abs(e_sl)) < 3.0
    assert len(e_body) >= B // 2 and np.mean(np.abs(e_body)) < 0.03
    ok("style-equivariant targets (slant closed form, re-measured body ratio, widths, masks)")

    # ---- FiLM, consistency, leakage --------------------------------------------------------------------------------------------
    fl = FiLM(8, 16, 0.5)
    xf = torch.randn(2, 5, 16)
    yf, g_, b_ = fl(xf, torch.randn(2, 8))
    assert torch.allclose(yf, xf)
    torch.nn.init.normal_(fl.lin.weight, std=10.0)
    _, g_, b_ = fl(xf, torch.randn(2, 8) * 10)
    assert float(g_.detach().abs().max()) <= 0.5 + 1e-6
    lpa = F.log_softmax(torch.randn(2, 10, 5), -1)
    lpb = F.log_softmax(torch.randn(2, 10, 5), -1)
    lens_ = torch.tensor([10, 6])
    assert abs(float(frame_consistency(lpa, lpa, lens_))) < 1e-6 and float(frame_consistency(lpb, lpa, lens_)) > 0
    gen = torch.Generator().manual_seed(0)
    act = np.array([1, 1, 0, 0, 0], bool)
    for leaky, floor, expect in ((False, 0.0, "low"), (True, 0.0, "high"), (True, 0.95, "low")):
        cfl = ClosedFormLeakage(8, 5, 256, 0.3, 0.02, act, np.full(5, floor), torch.device("cpu"))
        pen = None
        for _ in range(40):
            yy = torch.randn(16, 5, generator=gen)
            cc = torch.randn(16, 8, generator=gen)
            if leaky:
                cc[:, 0] = yy[:, 0] + 0.1 * torch.randn(16, generator=gen)
            cc.requires_grad_(True)
            pen = cfl(cc, yy, torch.ones(16, 5))
        assert (float(pen) > 0.2) if expect == "high" else (float(pen) < 0.1), (leaky, floor, float(pen))
    ok("FiLM bounded, teacher-student consistency, closed-form leakage penalty")

    # ---- data module: hidden labels, calibration lines, alphabet, multi-corpus + RTL, provenance, synthetic guard ------------------
    texts = [f"{chr(0x100 + i) * 3} ab" for i in range(40)]
    tmp = tempfile.mkdtemp()
    c = Config.smoke(tmp)
    c.data.labeled_fraction, c.data.max_text_len = 0.5, 30
    dm = HandwritingDataModule(c)
    dm._load_sources = lambda: [{"train": ListSource(texts, 1), "val": ListSource(texts[:8], 2), "test": ListSource(texts[8:16], 3)}]   # type: ignore
    dm.prepare()
    tr = dm.train_ds
    assert abs(int(tr.labeled.sum() + tr.calib.sum()) - len(tr) / 2) <= 1 and tr.calib.sum() >= 2 and not (tr.labeled & tr.calib).any()
    for i in range(len(tr)):
        it = tr[i]
        if tr.labeled[i]:
            assert it["text"] == tr.oracle[i] != "" and it["weight"] == 1.0 and it["oracle"] == tr.oracle[i]
        else:
            assert it["text"] == "" and it["oracle"] == "" and it["weight"] == 0.0, "hidden / calibration text leaked into a batch"
            assert tr.reference(i) == (tr.oracle[i] if tr.calib[i] else "")
    n_tr = len(tr)
    tr.set_pseudo(["ab"] * n_tr, np.ones(n_tr, np.float32), include_calib=False)
    assert all(tr.tgt_weight[i] == 0.0 for i in np.flatnonzero(tr.calib))
    tr.set_pseudo(["ab"] * n_tr, np.ones(n_tr, np.float32), include_calib=True)
    assert all(tr.tgt_text[i] == "ab" and tr.tgt_weight[i] == 1.0 for i in np.flatnonzero(tr.calib)) and all(tr.tgt_text[i] == tr.oracle[i] for i in np.flatnonzero(tr.labeled))
    assert all(tr[i]["oracle"] == "" for i in np.flatnonzero(tr.calib))
    tr.set_pseudo([""] * n_tr, np.zeros(n_tr, np.float32), include_calib=True)
    pool = tr.labeled | tr.calib
    lab_chars = set("".join(t for t, l in zip(tr.oracle, pool) if l))
    unl_only = set("".join(t for t, l in zip(tr.oracle, pool) if not l)) - lab_chars
    assert unl_only and not (unl_only & set(dm.codec.units)), "alphabet leaks characters of unlabeled lines"
    batch = Collator(dm.codec)([tr[i] for i in range(len(tr))])
    assert int(batch["target_len"].sum()) == sum(len(dm.codec.encode(tr.oracle[i])) for i in np.flatnonzero(tr.labeled))
    assert all(o == "" for o, l in zip(batch["oracle"], batch["labeled"].tolist()) if not l)
    assert dm.val_ds.reference(0) == texts[0] and os.path.exists(os.path.join(c.run_dir, "provenance.json"))
    c2 = Config.smoke(tmp)
    c2.run.smoke_test = False
    try:
        HandwritingDataModule(c2)._load_sources()
        raise AssertionError("synthetic data must be refused outside Config.smoke()")
    except RuntimeError:
        pass
    # two corpora, one of them right-to-left: domains, visual order, display, per-domain strata, MixLine within a corpus only
    ar_lines = ["سلام عليكم يا صديقي", "كتاب على الطاولة", "ذهب الولد الى المدرسة", "القط ينام في الحديقة"] * 10
    en_lines = ["the cat sat on the mat", "a quick brown fox", "hello world again", "pack my box with jugs"] * 10
    c3 = Config.smoke(tmp)
    c3.data.sources, c3.data.labeled_fraction, c3.data.max_text_len = ("synthetic", "arabic_test"), 0.6, 40
    c3.data.custom_sources = (DatasetSpec(key="arabic_test", language="ar", script="Arabic", rtl=True),)
    c3.train.frag.strata = 2
    dm3 = HandwritingDataModule(c3)
    dm3._load_sources = lambda: [{"train": ListSource(en_lines, 1), "val": ListSource(en_lines[:12], 2), "test": ListSource(en_lines[:8], 3)},        # type: ignore
                                 {"train": ListSource(ar_lines, 4), "val": ListSource(ar_lines[:12], 5), "test": ListSource(ar_lines[:8], 6)}]
    dm3.prepare()
    t3 = dm3.train_ds
    assert set(t3.domain.tolist()) == {0, 1} and t3.domain_names == ["synthetic", "arabic_test"] and dm3.n_strata == 4
    ar_rows = np.flatnonzero(t3.domain == 1)
    assert all(t3.rtl[i] for i in ar_rows) and not t3.rtl[np.flatnonzero(t3.domain == 0)[0]]
    i_ar = int(ar_rows[0])
    assert t3.oracle[i_ar] == to_visual(ar_lines[int(t3.indices[i_ar])], True) and t3.display(i_ar) == ar_lines[int(t3.indices[i_ar])]
    assert set(t3.stratum[t3.domain == 0]) <= {0, 1} and set(t3.stratum[t3.domain == 1]) <= {2, 3}
    assert dm3.test_ds.domain_names == t3.domain_names and dm3.val_ds.rtl.sum() == 12 and "ar" in [s.language for s in dm3.specs]
    col3 = Collator(dm3.codec, mix_prob=1.0, max_w=4096, gap=(2, 2), seed=1)
    la, lb = [int(i) for i in np.flatnonzero(t3.labeled & (t3.domain == 0))[:1]], [int(i) for i in np.flatnonzero(t3.labeled & (t3.domain == 1))[:1]]
    bmix = col3([t3[la[0]], t3[lb[0]]])
    assert bmix["image"].shape[0] == 2, "MixLine must not concatenate lines of different corpora"
    assert bmix["domain"].tolist() == [0, 1]
    shutil.rmtree(tmp, ignore_errors=True)
    ok("data module: hidden labels / calibration / alphabet / synthetic guard; two corpora with one RTL (visual order, display, per-corpus cells, MixLine)")

    # ---- risk control, reliability model, pseudo-label gates ------------------------------------------------------------------
    rng_ = np.random.RandomState(0)
    n_ = 600
    s_ = rng_.rand(n_)
    ed_ = rng_.binomial(40, 0.4 * s_ ** 3).astype(np.float64)
    rl_ = np.full(n_, 40.0)
    assert np.allclose(pava_increasing(np.array([3.0, 1.0, 2.0, 5.0, 4.0]), np.ones(5)), [2, 2, 2, 4.5, 4.5])
    cv = risk_curve(s_, ed_, rl_, True)
    assert np.all(np.diff(cv["smooth"]) >= -1e-12)
    th1, k1, c1 = risk_control_threshold(s_, ed_, rl_, 0.02, 1.0, 15)
    th2, k2, _ = risk_control_threshold(s_, ed_, rl_, 0.08, 1.0, 15)
    assert k1 >= 15 and k2 > k1 and th2 >= th1 and risk_control_threshold(s_, ed_ + 5, rl_, 0.02, 1.0, 15)[1] == 0
    assert auroc(np.array([1, 2, 3, 4.0]), np.array([0, 0, 1, 1], bool)) == 1.0
    Xr = np.column_stack([s_ + 0.15 * rng_.randn(n_)] + [rng_.randn(n_) for _ in range(RELIABILITY_N - 1)])
    from .config import PseudoConfig
    rule = SelectionRule.fit(Xr, ed_, rl_, PseudoConfig(), seed=0)
    assert rule.info["calib_auroc_error_detection"] > 0.7 and rule.predict(Xr).shape == (n_,)
    pcg = PseudoConfig()
    free = SelectionRule.fit(Xr, ed_, rl_, pcg, seed=0)
    capped = SelectionRule.fit(Xr, ed_, rl_, pcg, seed=0, prev_cov=0.0)
    assert capped.info["calib_selected"] <= int(math.floor(pcg.max_cov_growth * n_)) and free.info["calib_selected"] > capped.info["calib_selected"]
    lp_ = F.log_softmax(torch.randn(30, 6), -1).numpy()
    ids_, cf_, fr_ = greedy_decode(lp_)
    assert reliability_features(lp_, ids_, cf_, fr_, [0.0, 0.2]).shape == (RELIABILITY_N,) and len(reliability_views(PseudoConfig())) == 3
    pc = PseudoConfig()
    pc.min_stability, pc.lops_drop, pc.fallback_tau_conf = 0, 0.0, 0.9
    bank = PseudoLabelBank(7, np.ones(7, bool), ["hello"] * 7, np.full(7, 50))
    bank.begin_round()
    bank.update_predictions(range(7), ["hello", "hello", "h", "hello", "hello", "x" * 100, "hello"], [0.99, 0.99, 0.99, 0.99, 0.50, 0.99, 0.62],
                            [0.0, 0.50, 0.0, 0.0, 0.0, 0.0, 0.0], [0.05, 0.05, 0.05, 0.05, 0.90, 0.05, 0.40])
    bank.llm_ok[6], bank.llm_text[6] = True, "hello"
    bank.llm_ok[1], bank.llm_text[1] = True, "hello"
    bank.llm_ok[4], bank.llm_text[4] = True, "hello"
    bank.select(pc, 0.30, 0.50)
    assert bank.sel.tolist() == [True, False, False, True, False, False, True] and bank.source.tolist() == [1, 0, 0, 1, 0, 0, 2]
    bank.select(pc, None)
    assert bank.sel.tolist() == [True, False, False, True, False, False, False]
    ok("line-level risk control, reliability model, pseudo-label gates (legacy baseline)")

    # ---- test-time adaptation helpers, space bonus -----------------------------------------------------------------------------
    src_ = {"a": (np.array([3.0]), np.array([1.0]))}
    acc_ = {"a": {"n": 10.0, "s1": np.array([10.0]), "s2": np.array([20.0])}}
    ad_ = adapt_bn_state(src_, acc_, 10, 10.0)
    assert abs(ad_["a"][0][0] - 2.0) < 1e-9 and abs(ad_["a"][1][0] - 2.0) < 1e-9
    lps_ = np.log(np.array([[0.1, 0.8, 0.1], [0.5, 0.1, 0.4], [0.9, 0.05, 0.05]]))
    assert decode_with_space_bonus(lps_, 2, 0.0) == [1] and decode_with_space_bonus(lps_, 2, 1.5) == [1, 2] and decode_with_space_bonus(lps_, None, 9.0) == [1]
    cm = Config.smoke().model
    net_ = InkLinguaNet(cm, 24, NUM_DESC, 4, 32).eval()
    for _, m_ in bn_layers(net_):
        m_.running_mean.normal_(0, 0.5), m_.running_var.uniform_(0.5, 2.0)
    x_, w_ = torch.rand(4, 1, 32, 64), torch.tensor([64, 60, 52, 64])
    snap_ = snapshot_bn(net_)
    with torch.no_grad():
        base_o = net_(x_, w_)["log_probs"].clone()
        for _, m_ in bn_layers(net_):
            m_.collect = {}
        net_(x_, w_)
        acc2_ = {n: m_.collect for n, m_ in bn_layers(net_)}
        for _, m_ in bn_layers(net_):
            m_.collect = None
        after_ = snapshot_bn(net_)
        assert all(np.array_equal(snap_[k][0], after_[k][0]) for k in snap_)
        load_bn(net_, adapt_bn_state(snap_, acc2_, 4, 0.0))
        moved_ = net_(x_, w_)["log_probs"]
        load_bn(net_, adapt_bn_state(snap_, acc2_, 4, 1e15))
        same_ = net_(x_, w_)["log_probs"]
        load_bn(net_, snap_)
    assert (moved_ - base_o).abs().max() > 1e-4 and (same_ - base_o).abs().max() < 1e-4
    ok("AdaBN helpers (collection pass leaves the model untouched), space bonus")

    # ---- closed-loop renderer ----------------------------------------------------------------------------------------------------
    if font_paths():
        tg_ = [([24.5, 0.061, 0.106, 0.356, 0.28], "Parfois ils ont parle du theme"), ([-4.2, 0.056, 0.094, 0.356, 0.35], "et esthetique aussi")]
        e0_, e1_ = 0.0, 0.0
        for d_, x_t in tg_:
            inf_ = render_styled_text_info(x_t, np.array(d_), 64, iters=6)
            assert inf_["mismatch"] <= inf_["mismatch_round0"] + 1e-12 and inf_["image"].shape[0] == 64
            e0_, e1_ = e0_ + inf_["mismatch_round0"], e1_ + inf_["mismatch"]
        assert e1_ < 0.6 * e0_, (e0_, e1_)
        assert style_mismatch(np.array([np.nan] * 5), np.array([1.0] * 5)) == float("inf")
        ok("closed-loop style renderer (re-measured, corrected, never worse than round 0)")

    # ---- post-hoc options on a fake model (validation-gated adoption) ------------------------------------------------------------
    from .inferencer import Inferencer

    class _DS:
        def __init__(self, texts):
            self.oracle = texts
            self.style = np.zeros((len(texts), NUM_DESC), np.float32)
        def __len__(self): return len(self.oracle)

    codec_ = CharCodec("ab ")
    rs4 = np.random.RandomState(4)

    def fake_lp(text: str) -> np.ndarray:
        rows = []
        for ch in text:
            p = np.full(4, 1e-3)
            if ch == " ":
                p[codec_.encode(" ")[0]], p[0] = 0.30, 0.69
            else:
                p[codec_.encode(ch)[0]], p[0] = 0.95, 0.049
            rows.append(p / p.sum())
            blank = np.full(4, 1e-3)
            blank[0] = 0.997
            rows.append(blank / blank.sum())
        return np.log(np.array(rows, np.float32))

    def rand_text() -> str:
        return " ".join("".join(rs4.choice(list("ab"), rs4.randint(2, 5))) for _ in range(rs4.randint(2, 4)))

    v_txt, t_txt = [rand_text() for _ in range(80)], [rand_text() for _ in range(80)]
    inf_ = object.__new__(Inferencer)
    inf_.ic = copy.deepcopy(Config().infer)
    inf_.ic.tta, inf_.ic.bootstrap = False, 200
    inf_.data = SimpleNamespace(val_ds=_DS(v_txt), codec=codec_, val_gate_idx=np.arange(len(v_txt)))               # type: ignore
    inf_.model, inf_.frag_model, inf_.frag_state = None, None, None
    inf_._lp_pass = lambda ds, ids, assign=None, states=None, view=None: {int(i): fake_lp(ds.oracle[int(i)]) for i in ids}   # type: ignore
    R_ = {"idx": list(range(80))}
    ph_ = inf_._posthoc_block(_DS(t_txt), R_)
    assert ph_["test_CER_base"] > 0.05 and ph_["space_bonus"]["adopted"] and ph_["space_bonus"]["chosen"] >= 1.0
    assert ph_["test_CER_final"] == 0.0 and [R_["pred_posthoc"][i] for i in range(80)] == t_txt
    v2, t2 = ["ab" * 3] * 40, ["ba" * 3] * 40
    inf_.data = SimpleNamespace(val_ds=_DS(v2), codec=codec_, val_gate_idx=np.arange(len(v2)))                  # type: ignore
    ph2_ = inf_._posthoc_block(_DS(t2), {"idx": list(range(40))})
    assert not ph2_["adopted"]["tta"] and ph2_["adopted"]["space_bonus"] == 0.0 and ph2_["view_ensemble"]["adopted"] is False and ph2_["test_CER_final"] == 0.0
    ok("post-hoc options (validation-gated adoption of the space bonus; no adoption without evidence)")

    # ---- [ALPHA-SPLIT] / [LEX-TIE] / warm start --------------------------------------------------------------------------------
    kn = set("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ .,")
    assert visual_base_of("é", kn) == "e" and visual_base_of(Eg, kn) == "e" and visual_base_of("ɛ̀", kn) == "e" and visual_base_of("ж", kn) is None
    uc0 = CharCodec(sorted(kn))
    uc1, gr = extend_codec(uc0, "éèà " + Eg + "ɛж́")
    assert uc1.units[:len(uc0.units)] == uc0.units and gr.orphans == [uc1.c2i["ж́"]]
    net8 = InkLinguaNet(Config.smoke().model, uc0.num_classes, NUM_DESC, 4, 32).eval()
    x8, w8 = torch.rand(2, 1, 32, 96), torch.tensor([96, 80])
    with torch.no_grad():
        lp_old = net8(x8, w8)["log_probs"].numpy()
    extend_ctc_heads(net8, gr)
    with torch.no_grad():
        lp_new = net8(x8, w8)["log_probs"].numpy()
    assert lp_new.shape[-1] == uc1.num_classes and np.abs(gr.collapse(lp_new)[..., :uc0.num_classes] - lp_old).max() < 1e-4
    assert np.abs(lp_new[..., uc1.c2i["e"]] - lp_new[..., uc1.c2i["é"]]).max() < 1e-5
    Vg = gr.V

    def frames_for(text, probs_fn):
        lp = np.full((2 * len(text), Vg), math.log(1e-6))
        lp[:, 0] = math.log(0.9)
        ids, fr = [], []
        for j, ch in enumerate(text):
            i, t = uc1.c2i[ch], 2 * j
            mem = gr.variants(i)
            lp[t, :] = math.log(1e-6)
            lp[t, mem] = np.log(probs_fn(ch, mem))
            ids.append(i), fr.append(t)
        return ids, fr, lp
    lex = {"été", "élève", "école"}
    lm_fake = lambda texts: (np.array([sum(-1.0 if w_ in lex else -9.0 for w_ in t_.split()) for t_ in texts]), np.array([len(t_) for t_ in texts], float))  # noqa: E731
    tied = lambda ch, mem: np.ones(len(mem)) / len(mem) * 0.99          # noqa: E731
    for word, want in (("ete", "été"), ("eleve", "élève"), ("ecole", "école")):
        ids_, fr_, lp_ = frames_for(word, tied)
        assert resolve_variants(ids_, fr_, lp_, gr, lm_fake, beam=4)[0] == want
    ids_, fr_, lp_ = frames_for("ete", tied)
    assert resolve_variants(ids_, fr_, lp_, gr, None)[0] == "ete" and resolve_variants(ids_, fr_, lp_, gr, lm_fake, beam=1)[0] == "été"
    # warm start of a model whose alphabet EXTENDS the checkpoint's: function preserved on the old units
    tmpw = tempfile.mkdtemp()
    small = InkLinguaNet(Config.smoke().model, uc0.num_classes, NUM_DESC, 4, 32).eval()
    torch.save({"weights": small.state_dict(), "units": uc0.units, "epoch": 1}, os.path.join(tmpw, "best.pt"))
    big = InkLinguaNet(Config.smoke().model, uc1.num_classes, NUM_DESC, 4, 32).eval()
    info_w = warm_start(big, os.path.join(tmpw, "best.pt"), uc1)
    with torch.no_grad():
        lp_s, lp_b = small(x8, w8)["log_probs"].numpy(), big(x8, w8)["log_probs"].numpy()
    assert info_w["appended_units"] == uc1.num_classes - uc0.num_classes and np.abs(gr.collapse(lp_b)[..., :uc0.num_classes] - lp_s).max() < 1e-4
    try:
        warm_start(InkLinguaNet(Config.smoke().model, uc1.num_classes, NUM_DESC, 4, 32), os.path.join(tmpw, "best.pt"), uc1, strict_alphabet=True)
        raise AssertionError("strict warm start must refuse a different alphabet")
    except RuntimeError:
        pass
    shutil.rmtree(tmpw, ignore_errors=True)
    ok("[ALPHA-SPLIT] alphabet growth preserves the function; [LEX-TIE] language-prior tie-breaking; warm start with alphabet extension")

    # ---- [XL-RC] ----------------------------------------------------------------------------------------------------------------
    rs8 = np.random.RandomState(0)
    Z0 = rs8.randn(400, 3)
    w0, au0, es0 = domain_shift(Z0[:200], Z0[200:])
    Zs, Zt = rs8.randn(400, 3), rs8.randn(400, 3) + np.array([1.2, 0.0, 0.0])
    w1, au1, es1 = domain_shift(Zs, Zt)
    assert au0 < 0.62 and au1 > 0.7 and abs(w1.mean() - 1) < 1e-9
    e8, l8 = rs8.poisson(0.05 * 40, 300).astype(float), np.full(300, 40.0)
    s8 = rs8.rand(300)
    cv1, cv0 = weighted_risk_curve(s8, e8, l8, np.ones(300)), risk_curve(s8, e8, l8, True)
    assert np.allclose(cv1["smooth"], cv0["smooth"]) and np.allclose(cv1["se"], cv0["se"])

    def xl_trial(seed, conc=1.0, n_anchor=0):
        r_ = np.random.RandomState(seed)
        xs, xt = r_.randn(500), r_.randn(900) + 1.0
        rate = lambda x_: 0.004 * np.exp(0.9 * x_)                                                  # noqa: E731
        ed_s, ed_t = r_.poisson(rate(xs) * 40).astype(float), r_.poisson(conc * rate(xt) * 40).astype(float)
        sa_ = ea_ = None
        if n_anchor:
            xa = r_.randn(n_anchor) + 1.0
            sa_, ea_ = xa, r_.poisson(conc * rate(xa) * 40).astype(float)
        res = transfer_risk_control(xs[:, None], xs, ed_s, np.full(500, 40.0), xt[:, None], xt, 0.01, 2.0, 15, 10.0, score_a=sa_, ed_a=ea_,
                                    rl_a=None if sa_ is None else np.full(n_anchor, 40.0), zero_anchor_inflation=1.0)
        th_, k_, _ = risk_control_threshold(xs, ed_s, np.full(500, 40.0), 0.01, 2.0, 15, True)
        out = {}
        for tag, (theta, k) in (("weighted", (res["theta"], res["k"])), ("plain", (th_, k_))):
            sel = xt <= theta if k > 0 else np.zeros(900, bool)
            out[tag] = (ed_t[sel].sum() / max(40.0 * sel.sum(), 1.0) if sel.any() else float("nan"), res)
        return out
    viol = {"weighted": [], "plain": []}
    for sd in range(40):
        o_ = xl_trial(sd)
        for tag in viol:
            if np.isfinite(o_[tag][0]):
                viol[tag].append(o_[tag][0] > 0.01)
    assert np.mean(viol["weighted"]) < np.mean(viol["plain"])
    f_noshift = np.median([xl_trial(sd, 1.0, 60)["weighted"][1]["factor"] for sd in range(10)])
    f_hard = np.median([xl_trial(sd, 2.5, 60)["weighted"][1]["factor"] for sd in range(10)])
    assert f_noshift < 1.35 and f_hard > 1.6, (f_noshift, f_hard)
    assert 0.0 <= weighted_ks(Zs[:, 0], Zt[:, 0]) <= 1.0 and weighted_ks(Zs[:, 0], Zs[:, 0]) == 0.0
    ok("[XL-RC] transfer-calibrated risk control (domain weights help under covariate shift, anchors detect a harder target)")

    # ---- FRAG-RC: evidence, labels, masks, character-level risk control, distillation --------------------------------------------
    Vf, path_f = 5, [0, 1, 1, 0, 2, 0, 3, 3]
    lpf = np.full((len(path_f), Vf), 0.01 / (Vf - 1))
    for t_, c_ in enumerate(path_f):
        lpf[t_, c_] = 0.99
    lpf = np.log(lpf)
    ids_f, st_f, en_f, Ff = char_evidence(lpf, lpf, lpf)
    assert ids_f == [1, 2, 3] and st_f.tolist() == [1, 4, 6] and en_f.tolist() == [3, 5, 8] and np.isfinite(Ff).all() and (Ff[:, 2] == 1.0).all()
    lp_bad = lpf.copy()
    lp_bad[4] = np.log(np.array([0.05, 0.1, 0.45, 0.3, 0.1]))
    lp_alt = lpf.copy()
    lp_alt[4] = np.log(np.array([0.05, 0.1, 0.1, 0.65, 0.1]))
    _, _, _, Fb = char_evidence(lp_bad, lp_alt, lp_bad)
    assert Fb[1, 0] < Ff[1, 0] - 2 and Fb[1, 3] == 0.0 and Fb[0, 3] == 1.0 and Fb[0, 4] < Ff[0, 4]
    assert char_wrong_labels("abcd", "abxd").tolist() == [0, 0, 1, 0] and char_wrong_labels("abcde", "abde").tolist() == [0, 1, 1, 0]
    m0 = fragment_mask(10, np.array([1, 3, 7]), np.array([3, 5, 9]), np.array([False, True, False]), guard=0)
    assert m0.tolist() == [True, True, True, False, False, False, False, True, True, True]
    m1 = fragment_mask(10, np.array([1, 3, 7]), np.array([3, 5, 9]), np.array([False, True, False]), guard=1)
    assert (not m1[2]) and m1[0] and m1[7]
    assert not fragment_mask(6, np.zeros(0, np.int64), np.zeros(0, np.int64), np.zeros(0, bool), 1).any()
    assert cell_theta([0.1, 0.2], 7) == 0.2 and cell_theta([], 0) == float("-inf")        # clipped cell index (audit A4)
    rg2 = np.random.RandomState(0)
    nch = 4000
    rk = rg2.rand(nch)
    ln_ = rg2.randint(0, 130, nch)
    wr = (rg2.rand(nch) < 0.002 + 0.45 * rk ** 4).astype(np.float64)
    th1, k1, e1 = char_risk_threshold(rk, wr, ln_, 0.02, 2.0, 100)
    th2, k2, e2 = char_risk_threshold(rk, wr, ln_, 0.05, 2.0, 100)
    assert k1 > 100 and e1 < 0.04 and k2 >= k1 and th2 >= th1 and char_risk_threshold(rk, np.ones(nch), ln_, 0.02, 2.0, 100)[1] == 0
    lines_c = np.repeat(np.arange(100), 30)
    rk_c = rg2.rand(3000)
    y_spread = (rg2.rand(3000) < 0.02).astype(np.float64)
    y_clust = np.isin(lines_c, [3, 71]).astype(np.float64)
    assert char_risk_threshold(rk_c, y_spread, lines_c, 0.04, 2.0, 400)[1] > char_risk_threshold(rk_c, y_clust, lines_c, 0.04, 2.0, 400)[1]
    fm_ = FragModel.fit(rg2.randn(500, Ff.shape[1]), (rg2.rand(500) < 0.3).astype(np.float64))
    xx_ = rg2.randn(20, Ff.shape[1])
    assert np.allclose(fm_.risk(xx_), FragModel.from_state(fm_.state()).risk(xx_))
    gB, gT, gV = 3, 7, 5
    t_lp_ = F.log_softmax(torch.randn(gB, gT, gV), -1)
    s_raw = torch.randn(gB, gT, gV, requires_grad=True)
    lens_g, trust_g, rows_g = torch.tensor([7, 5, 7]), torch.zeros(gB, gT, dtype=torch.bool), torch.tensor([True, True, False])
    trust_g[0, 1:4] = True
    fragment_distillation(F.log_softmax(s_raw, -1), t_lp_, lens_g, trust_g, rows_g, 0.6, 0.1).backward()
    gnorm = s_raw.grad.abs().sum(-1)
    assert float(gnorm[2].sum()) == 0.0 and float(gnorm[1, :5].min()) > 0 and float(gnorm[1, 5:].sum()) == 0.0
    s_same = t_lp_.clone().requires_grad_(True)
    assert abs(float(fragment_distillation(s_same, t_lp_, lens_g, torch.zeros(gB, gT, dtype=torch.bool), rows_g, 0.6, 0.1))) < 1e-5
    assert float(fragment_distillation(s_same, t_lp_, lens_g, torch.ones(gB, gT, dtype=torch.bool), rows_g, 0.6, 0.1)) > 1e-3
    assert float(fragment_distillation(s_same, t_lp_, lens_g, torch.ones(gB, gT, dtype=torch.bool), rows_g, 1.0, 0.1)) < 1e-5
    ok("[FRAG-RC] character evidence, correctness labels, fragment masks, cluster-robust character-level risk control, distillation gradients")

    # ---- protocol, strata, MixLine, lr schedule, suite, config ---------------------------------------------------------------------
    rs10 = random.Random(11)
    texts10 = ["".join(rs10.choice("abcdefgh") for _ in range(rs10.randint(8, 14))) for _ in range(400)]
    tmp10 = tempfile.mkdtemp()
    c10 = Config.smoke(tmp10)
    c10.data.labeled_fraction, c10.data.max_text_len, c10.data.calib_fraction, c10.data.val_block = 0.4, 30, 0.2, 8
    dm10 = HandwritingDataModule(c10)
    dm10._load_sources = lambda: [{"train": ListSource(texts10[:320], 1), "val": ListSource(texts10[320:372], 2), "test": ListSource(texts10[372:], 3)}]   # type: ignore
    dm10.prepare()
    nv10 = len(dm10.val_ds)
    sel10, gate10 = set(dm10.val_sel_idx.tolist()), set(dm10.val_gate_idx.tolist())
    assert not (sel10 & gate10) and (sel10 | gate10) == set(range(nv10)) and len(sel10) > 0 and len(gate10) > 0
    assert not ({i // c10.data.val_block for i in sel10} & {i // c10.data.val_block for i in gate10})
    col = Collator(dm10.codec, mix_prob=1.0, max_w=4096, gap=(2, 2), seed=1)
    lab10 = [int(i) for i in dm10.labeled_idx[:4]]
    unl10 = [int(i) for i in dm10.unlabeled_idx[:2]]
    items10 = [dm10.train_ds[i] for i in lab10 + unl10]
    b10 = col(items10)
    sep10 = " " if " " in dm10.codec.c2i else ""
    assert b10["image"].shape[0] == 4 and int(b10["width"][0]) == int(items10[0]["width"]) + 2 + int(items10[1]["width"])
    assert int(b10["target_len"][0]) == len(items10[0]["text"]) + len(sep10) + len(items10[1]["text"]) and (b10["style_ok"][:2] == 0).all()
    assert Collator(dm10.codec)(items10)["image"].shape[0] == 6
    pr10 = write_protocol(c10, dm10)
    assert pr10["val_sel_lines"] + pr10["val_gate_lines"] == nv10 and log_test_look(c10, {"path": "x.pt", "epoch": 3}, "test") == 1 and log_test_look(c10, {"path": "x.pt"}, "test") == 2
    # the training loop with a FORCED all-trusting rule: trusted fragments reach the loss; without FRAG-RC nothing does
    c12 = copy.deepcopy(c10)
    c12.train.aux_start_epoch, c12.train.aux_ramp_epochs = 0, 1
    dev12 = torch.device("cpu")
    trn12 = Trainer(c12, dm10, InkLinguaNet(c12.model, dm10.codec.num_classes, NUM_DESC, dm10.n_clusters, c12.data.img_height), LLMEngine(c12.llm, dev12), dev12)
    nf12 = Ff.shape[1]
    trn12._frag_set_state({"model": FragModel(np.zeros(nf12), np.ones(nf12), np.zeros(nf12 + 1)).state(), "theta": [1.0] * dm10.n_strata, "alpha": 0.1, "epoch": 0, "n_strata": dm10.n_strata})
    trn12.global_step = 2 * trn12.steps_per_epoch
    tr12 = trn12._train_epoch(0)
    assert tr12.get("frag_trust", 0.0) > 0.0 and tr12["frag"] > 0 and np.isfinite(tr12["total"])
    c13 = copy.deepcopy(c12)
    c13.train.frag.enabled = False
    trn13 = Trainer(c13, dm10, InkLinguaNet(c13.model, dm10.codec.num_classes, NUM_DESC, dm10.n_clusters, c13.data.img_height), LLMEngine(c13.llm, dev12), dev12)
    trn13.global_step = 2 * trn13.steps_per_epoch
    tr13 = trn13._train_epoch(0)
    assert tr13["frag"] == 0.0 and "frag_trust" not in tr13
    shutil.rmtree(tmp10, ignore_errors=True)
    tr_ = object.__new__(Trainer)
    tr_.tc, tr_.steps_per_epoch, tr_.total_steps = Config().train, 10, 1000
    tr_.cycles = [{"start": 0, "end": 1000, "peak": tr_.tc.lr, "warm": tr_.tc.warmup_steps, "lo": 0.0}]
    tr_.global_step = 1000
    lr_end = tr_._lr(999)
    tr_.total_steps = 5000
    tr_._extend_schedule(1000)
    assert tr_._lr(1000) < 2 * lr_end + 1e-6 and tr_._lr(4999) < 0.01 * tr_.tc.lr
    b_ = Config()
    lo_ = Config.make_variant("labeled_only", b_)
    assert not lo_.train.pseudo.enabled and not lo_.train.frag.enabled and lo_.data.calib_fraction == 0.0 and lo_.train.mixline_prob == b_.train.mixline_prob
    assert Config.make_variant("legacy_pl", b_).train.pseudo.enabled and not Config.make_variant("no_frag", b_).train.frag.enabled
    assert Config.make_variant("no_strata", b_).train.frag.strata == 1 and Config.make_variant("ctc_only", b_).model.film is False
    for kind in Config.ABLATIONS:
        Config.ablation(kind)
    assert all(v in Config.ABLATIONS or v in ("labeled_only", "full", "full_supervised") for p in SUITE_PRESETS.values() for v in p)
    assert Config.from_dict(json.loads(json.dumps(Config().to_dict()))) == Config()
    cm_ = Config.multi(("iam", "khatt"))
    assert Config.from_dict(json.loads(json.dumps(cm_.to_dict()))).data.sources == ("iam", "khatt")
    cc_ = Config()
    cc_.data.custom_sources = (DatasetSpec(key="x", hf_dataset="a/b", rtl=True),)
    assert Config.from_dict(json.loads(json.dumps(cc_.to_dict()))).data.custom_sources[0].rtl is True
    tmp3 = tempfile.mkdtemp()
    rs3, paths3 = np.random.RandomState(3), {"labeled_only": {}, "full": {}, "no_frag": {}}
    for s_ in (1, 2):
        base_e = rs3.binomial(40, 0.10, 300)
        for v_, e_ in (("labeled_only", base_e), ("full", np.maximum(base_e - rs3.binomial(1, 0.25, 300), 0)), ("no_frag", np.maximum(base_e - rs3.binomial(1, 0.1, 300), 0))):
            d_ = os.path.join(tmp3, f"{v_}_{s_}", "01_recognition")
            os.makedirs(d_)
            with open(os.path.join(d_, "predictions.csv"), "w", newline="", encoding="utf-8") as f_:
                w_ = csv.writer(f_)
                w_.writerow(["line_index", "edits", "ref_len", "CER_posthoc"])
                for i_, ee in enumerate(e_):
                    w_.writerow([i_, int(ee), 40, f"{ee / 40:.4f}"])
            paths3[v_][s_] = os.path.join(d_, "predictions.csv")
    sm_ = summarise_suite(paths3, "labeled_only", 200)
    pv_ = sm_["paired_vs_baseline"]["full"]
    assert pv_["mean_delta"] < 0 and pv_["seeds_better"] == 2 and sm_["contribution_vs_full"]["no_frag"]["mean_delta"] > 0 and "| full |" in suite_markdown(sm_)
    shutil.rmtree(tmp3, ignore_errors=True)
    # explainability helpers
    rs = np.random.RandomState(0)
    cc2 = np.zeros((60, 30))
    for k in range(60):
        cc2[k, k % 30] = 40
    assert code_char_stats(cc2, 20)["component_code_purity"] == 1.0
    shared = np.zeros((3, 40))
    shared[:, [5, 20]] = 0.4
    for r, f in enumerate((8, 15, 30)):
        shared[r, f] = 0.2
    res, sink = contrast_rollout(shared)
    assert 0.8 < sink < 0.9 and [int(np.argmax(res[r])) for r in range(3)] == [8, 15, 30]
    ok("protocol split, MixLine, FRAG-RC in the training loop, lr schedule on resume, suite aggregation, config round trips, explainability helpers")
    LOG.info("%d self-tests passed in %.1fs", len(done), time.time() - t0)


def run_smoke(work_dir: str = None, keep: bool = False) -> Dict[str, Any]:
    """The synthetic end-to-end smoke: train (FRAG-RC refresh + MixLine + val_sel), resume, evaluate (val_gate, WSD, interpretability), report.
    Returns the metrics; the directory is deleted unless `keep`."""
    from .main import main
    tmp = work_dir or tempfile.mkdtemp(prefix="inklingua_smoke_")
    c = Config.smoke(tmp)
    c.run.self_test = False
    c.train.epochs, c.train.frag.warmup_epochs, c.train.frag.refresh_every = 12, 4, 4
    c.train.aux_start_epoch, c.train.aux_ramp_epochs, c.train.vq_start_epoch, c.train.vq_anneal_epochs = 2, 2, 2, 2
    c.data.synthetic_sizes = (480, 64, 48)
    c.infer.posthoc, c.infer.wsd, c.infer.n_vis, c.infer.n_translate = True, True, 1, 1
    c.infer.pl_diag_samples = 20
    try:
        m = main(c)
        assert m is not None and m["SYNTHETIC_SMOKE_TEST_NOT_FOR_PUBLICATION"] is True and m["val_sel_lines"] > 0 and m["val_gate_lines"] > 0
        hist = json.load(open(os.path.join(c.curve_dir, "history.json")))
        assert "val_sel_cer" in hist and "val_gate_cer" in hist and "train_frag" in hist and any(k.startswith("frag_theta_s") for k in hist), "FRAG-RC never calibrated"
        wsd = (m.get("posthoc") or {}).get("wsd")
        assert wsd is not None and "adopted" in wsd
        for rel in ("00_summary/metrics.json", "01_recognition/predictions.csv", "01_recognition/alignments.jsonl", "04_posthoc/posthoc.json", "12_report/index.html",
                    "10_explanations/layer_probes.csv", "02_uncertainty/uncertainty_signals.csv", "03_error_structure/groups.csv", "07_robustness/longline_stress.json"):
            assert os.path.exists(os.path.join(c.infer_dir, rel)), rel
        assert os.path.exists(os.path.join(c.run_dir, "protocol", "protocol.json"))
        # resume: a second call with more epochs continues from the checkpoint (warm restart) instead of starting over
        c.train.epochs, c.run.do_infer = 14, False
        main(c)
        hist2 = json.load(open(os.path.join(c.curve_dir, "history.json")))
        assert len(hist2["epoch"]) == 14 and hist2["epoch"][:12] == hist["epoch"][:12]
        LOG.info("smoke test passed (%s)", tmp)
        return m
    finally:
        restore_logging()
        if not keep:
            shutil.rmtree(tmp, ignore_errors=True)

In [ ]:
%%writefile inklingua/main.py
"""Entry point.  `main(cfg)` dispatches on cfg.run.mode; `python inklingua.py` runs the default IAM experiment."""
from __future__ import annotations

import os
from typing import Any, Dict, Optional, Sequence

from .config import Config
from .data import HandwritingDataModule
from .llm import LLMEngine
from .model import InkLinguaNet
from .protocol import write_protocol
from .style import NUM_DESC
from .utils import LOG, resolve_device, resolve_llm_device, set_seed, setup_logging


def run_single(cfg: Config, base_units: Optional[Sequence[str]] = None) -> Optional[Dict[str, Any]]:
    """Train (resumable) and / or evaluate one configuration."""
    from . import __version__
    from .inferencer import Inferencer
    from .selftest import run_self_tests
    from .trainer import Trainer
    cfg.make_dirs()
    setup_logging(cfg)
    set_seed(cfg.run.seed)
    cfg.save()
    device = resolve_device(cfg.run.device)
    LOG.info("InkLingua v%s | run dir: %s | device: %s | seed %d | corpora %s", __version__, cfg.run_dir, device, cfg.run.seed, cfg.data.sources)
    if cfg.run.self_test:
        run_self_tests()
        set_seed(cfg.run.seed)
    data = HandwritingDataModule(cfg, base_units=base_units).prepare()
    if not cfg.llm.source_language:
        cfg.llm.source_language = data.primary_language
    write_protocol(cfg, data)
    llm_device = resolve_llm_device(cfg.llm.device, device)
    LOG.info("recogniser on %s, LLM on %s", device, llm_device)
    llm = LLMEngine(cfg.llm, llm_device)
    model = InkLinguaNet(cfg.model, data.codec.num_classes, NUM_DESC, data.n_clusters, cfg.data.img_height)
    best, metrics = None, None
    if cfg.run.do_train:
        trainer = Trainer(cfg, data, model, llm, device)
        trainer.fit()
        best = trainer.best_path if os.path.exists(trainer.best_path) else None
    if cfg.run.do_infer:
        infer = Inferencer(cfg, data, llm, device)
        infer.load(best)
        metrics = infer.run()
        LOG.info("All done. Open %s", os.path.join(cfg.infer_dir, "12_report", "index.html"))
    return metrics


def main(cfg: Optional[Config] = None) -> Optional[Dict[str, Any]]:
    cfg = cfg or Config()
    mode = cfg.run.mode
    if mode == "suite":
        from .suite import run_suite
        return run_suite(cfg)
    if mode == "suite_summary":
        from .suite import run_suite
        return run_suite(cfg, train=False)
    if mode == "finetune":
        from .adapt import run_finetune
        return run_finetune(cfg)
    if mode == "adapt":
        from .adapt import run_adapt
        return run_adapt(cfg)
    if mode == "adapt_curve":
        from .adapt import run_adapt_curve
        return run_adapt_curve(cfg)
    if mode == "source_calib":
        from .adapt import run_source_calib
        run_source_calib(cfg)
        return None
    if mode == "report":
        from .report import build_run_report
        cfg.make_dirs()
        setup_logging(cfg)
        build_run_report(cfg.infer_dir, cfg.run_dir, curve_dir=cfg.curve_dir)
        return None
    if mode == "smoke":
        from .selftest import run_smoke
        return run_smoke()
    if mode != "single":
        raise ValueError(f"unknown cfg.run.mode '{mode}'")
    return run_single(cfg)

## 3. Configuration

In [ ]:
# ---- 3. Configuration knobs ------------------------------------------------------------------------------------------------
# Everything else keeps the defaults of inklingua/config.py (documented there).

MODE = "single"            # "smoke"  : synthetic plumbing test, no download (minutes)
                           # "single" : train + evaluate ONE configuration (the default experiment)
                           # "suite"  : baselines + ablations x seeds -> suite_summary.md  (what a paper needs; long)
                           # "finetune": continue a checkpoint (INIT_FROM) on the corpora in SOURCES
                           # "report" : rebuild the report of an existing run from its files
RUN_NAME = "inklingua_iam"
SOURCES = ("iam",)         # registry keys; several = multi-corpus training, e.g. ("iam", "rimes") or ("iam", "khatt")
                           # registry: iam rimes belfort esposalles himanis home_alcar newseye_de norhand1 norhand3 popp casia khatt
LABELED_FRACTION = 0.30    # share of the training lines whose transcription is used
EPOCHS = 100
SEED = 1337
DATA_SOURCE = "hf"         # "hf" (download, Internet on) | "local" (LOCAL_DIR with train/validation/test .parquet)
LOCAL_DIR = ""             # e.g. "/kaggle/input/iam-line-parquet"
USE_LLM = True             # Qwen on the second GPU: translation + N-best rescoring.  False -> "dummy" backend (no download)
LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
TARGET_LANGUAGE = "fr"
SUITE_PRESET = "quick"     # for MODE="suite": "quick" (labeled_only, full) | "paper" (8 variants) | "all"
SUITE_SEEDS = (1337, 1338, 1339)
INIT_FROM = ""             # for MODE="finetune": path of best.pt / last.pt / adapted.pt
QUICK_TEST = True          # True: 1,200 train / 300 val / 300 test lines, 6 epochs, small LLM sample -> ~20 min dry run of every output

In [ ]:
# ---- 3b. Build the Config from the knobs ------------------------------------------------------------------------------------
from inklingua import Config, main
from inklingua.config import SUITE_PRESETS

if MODE == "smoke":
    cfg = Config.smoke(os.path.join(WORK_DIR, "smoke"))
else:
    cfg = Config()
    cfg.run.name, cfg.run.work_dir, cfg.run.seed, cfg.run.mode = RUN_NAME, WORK_DIR, SEED, MODE
    cfg.data.sources, cfg.data.labeled_fraction, cfg.data.source = tuple(SOURCES), LABELED_FRACTION, DATA_SOURCE
    if DATA_SOURCE == "local":
        cfg.data.local_dir = LOCAL_DIR
    cfg.train.epochs = EPOCHS
    cfg.llm.backend = "causal" if USE_LLM else "dummy"
    cfg.llm.model_name, cfg.llm.target_language = LLM_NAME, TARGET_LANGUAGE
    cfg.llm.device = "auto"                 # second GPU when available
    cfg.run.suite_preset, cfg.run.suite_seeds = SUITE_PRESET, tuple(SUITE_SEEDS)
    if MODE == "finetune":
        cfg.train.init_from = INIT_FROM
    if QUICK_TEST:
        cfg.run.name = RUN_NAME + "_quick"
        cfg.data.limit_train, cfg.data.limit_val, cfg.data.limit_test = 1200, 300, 300
        cfg.train.epochs, cfg.train.min_epochs, cfg.train.patience = 6, 0, 0
        cfg.train.frag.warmup_epochs, cfg.train.frag.refresh_every, cfg.train.frag.min_chars = 2, 2, 100
        cfg.train.aux_start_epoch, cfg.train.aux_ramp_epochs, cfg.train.vq_start_epoch, cfg.train.vq_anneal_epochs = 1, 1, 1, 1
        cfg.infer.n_translate, cfg.infer.rescore_val_lines, cfg.infer.selective_val_lines = 24, 150, 300
        cfg.infer.pl_diag_samples, cfg.infer.bootstrap, cfg.infer.wsd_steps = 300, 500, (10, 20)
        cfg.infer.probe_lines, cfg.infer.uncertainty_lines, cfg.infer.longline_pairs = 200, 200, 100
    cfg.run.self_test = True                # the fast invariant tests run before the data is touched (~30 s)

print("mode:", MODE, "| run dir:", cfg.run_dir, "| corpora:", cfg.data.sources, "| labeled:", cfg.data.labeled_fraction,
      "| epochs:", cfg.train.epochs, "| LLM:", cfg.llm.backend, cfg.llm.model_name if cfg.llm.backend != "dummy" else "")
if cfg.run.mode == "suite":
    print("suite variants:", SUITE_PRESETS[cfg.run.suite_preset], "x seeds", cfg.run.suite_seeds)

## 4. Self-tests

In [ ]:
# ---- 4. Fast self-tests (CPU, no data) ---------------------------------------------------------------------------------------
import logging, sys
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s", stream=sys.stdout, force=True)
from inklingua.selftest import run_self_tests
run_self_tests()

## 5. Train and evaluate

In [ ]:
# ---- 5. Train + evaluate (resumable: re-running this cell continues from the last checkpoint) ---------------------------------
metrics = main(cfg)
if isinstance(metrics, dict) and "CER" in metrics:
    ci = metrics.get("CER_ci95") or [float("nan"), float("nan")]
    print(f"\nCER {100 * metrics['CER']:.2f} %  [{100 * ci[0]:.2f}, {100 * ci[1]:.2f}]  |  WER {100 * metrics['WER']:.2f} %  |  "
          f"exact lines {100 * metrics['exact_line_accuracy']:.1f} %  |  system CER (validated options) "
          f"{100 * metrics.get('CER_system_final', metrics.get('CER_final_with_validated_posthoc', metrics['CER'])):.2f} %")

## 6. Results

In [ ]:
# ---- 6. Results ----------------------------------------------------------------------------------------------------------------
import glob, json
from IPython.display import HTML, IFrame, Image, Markdown, display

run_dir = cfg.run_dir if cfg.run.mode != "suite" else None
if run_dir and os.path.isdir(os.path.join(run_dir, "inference", cfg.infer.split)):
    inf = os.path.join(run_dir, "inference", cfg.infer.split)
    M = json.load(open(os.path.join(inf, "00_summary", "metrics.json")))
    rows = [("CER (greedy)", M["CER"]), ("CER 95 % CI", M["CER_ci95"]), ("WER", M["WER"]), ("exact lines", M["exact_line_accuracy"]),
            ("CER + validated post-hoc", M.get("CER_final_with_validated_posthoc")), ("system CER (incl. LLM if adopted)", M.get("CER_system_final")),
            ("looks at this split so far", M.get("evaluation_looks_so_far"))]
    display(Markdown("### Headline numbers\n\n| metric | value |\n|---|---|\n" + "\n".join(f"| {k} | {v} |" for k, v in rows)))
    if M.get("per_corpus"):
        display(Markdown("### Per corpus\n\n| corpus | lines | CER | WER |\n|---|---|---|---|\n" + "\n".join(
            f"| {k} | {v['lines']} | {100 * v['CER']:.2f} % | {100 * v['WER']:.2f} % |" for k, v in M["per_corpus"].items())))
    ph = M.get("posthoc") or {}
    if ph:
        display(Markdown("### Post-hoc options (adopted only on a significant val_gate gain)\n\n| option | adopted | test CER |\n|---|---|---|\n" + "\n".join(
            f"| {k} | {ph[k].get('adopted')} | {100 * ph[k]['test_CER']:.2f} % |" for k in ("view_ensemble", "tta", "wsd", "space_bonus")
            if isinstance(ph.get(k), dict) and ph[k].get("test_CER") is not None)))
    display(Markdown("### Training curves"))
    display(Image(os.path.join(run_dir, "curves", "curves_all.png"), width=1400))
    display(Markdown("### Where the errors come from / uncertainty / explanations"))
    for rel in ("03_error_structure/error_structure.png", "02_uncertainty/calibration.png", "02_uncertainty/uncertainty_decomposition.png",
                "10_explanations/layer_probes.png", "07_robustness/robustness_profile.png", "11_translation/translation_ablation.png"):
        p = os.path.join(inf, rel)
        if os.path.exists(p):
            display(Markdown(f"`{rel}`")), display(Image(p, width=1100))
    dash = sorted(glob.glob(os.path.join(inf, "10_explanations", "samples", "*", "dashboard.png")))
    for p in dash[:2]:
        display(Markdown(f"`{os.path.relpath(p, inf)}`")), display(Image(p, width=1100))
    rep = os.path.join(inf, "12_report", "index.html")
    if os.path.exists(rep):
        display(Markdown("### The interactive report (also saved as `12_report/index.html`)"))
        display(IFrame(src=os.path.relpath(rep, os.getcwd()), width="100%", height=900))
elif cfg.run.mode == "suite":
    p = os.path.join(cfg.run.work_dir, f"{cfg.run.name}_suite", "suite_summary.md")
    display(Markdown(open(p).read()) if os.path.exists(p) else Markdown("no suite summary yet"))
else:
    print("no inference folder yet for", run_dir)

## 7. Optional: suite, multi-corpus, fine-tuning, zero-shot adaptation

In [ ]:
# ---- 7. Optional recipes (set RUN_* to True and run) ------------------------------------------------------------------------
RUN_SUITE, RUN_MULTI, RUN_FINETUNE = False, False, False

if RUN_SUITE:         # the decisive experiment: does the unlabeled data help?  labeled_only vs full (preset "quick"), >= 3 seeds
    s = Config(); s.run.name, s.run.work_dir, s.run.mode = RUN_NAME, WORK_DIR, "suite"
    s.run.suite_preset, s.run.suite_seeds = "quick", (1337, 1338, 1339)
    s.llm.backend = "causal" if USE_LLM else "dummy"
    s.train.epochs = 60                       # validation CER is flat after ~30 epochs in every IAM run so far
    summary = main(s)

if RUN_MULTI:         # English + French + Arabic (right-to-left) at once; per-corpus FRAG-RC cells and per-corpus metrics
    m_ = Config.multi(("iam", "rimes", "khatt")); m_.run.work_dir = WORK_DIR
    m_.data.domain_balance = 0.5              # visit the smaller corpora more often
    m_.llm.backend = "causal" if USE_LLM else "dummy"
    metrics_multi = main(m_)

if RUN_FINETUNE:      # continue the IAM checkpoint on another corpus; the alphabet grows without changing the model's behaviour
    f_ = Config(); f_.run.name, f_.run.work_dir, f_.run.mode = RUN_NAME + "_ft_esposalles", WORK_DIR, "finetune"
    f_.train.init_from = os.path.join(WORK_DIR, RUN_NAME, "checkpoints", "best.pt")
    f_.data.sources, f_.train.epochs = ("esposalles",), 30
    f_.llm.backend = "causal" if USE_LLM else "dummy"
    metrics_ft = main(f_)

# Zero-shot adaptation to a FOLDER of line images in a new language (no or few labels):
#   a = Config(); a.run.name, a.run.work_dir, a.run.mode = RUN_NAME, WORK_DIR, "adapt"
#   a.adapt.lines_dir = "/kaggle/input/my-lines"; a.adapt.charset = "éèàùç"; a.adapt.target_languages = ("en", "fr")
#   a.adapt.anchors_file = "/kaggle/input/my-lines/anchors.tsv"   # optional "file<TAB>text"
#   main(a)                                   # -> <run>/adapt/adapt_results.csv, adapted.pt, adapt_metrics.json

## 8. Export

In [ ]:
# ---- 8. Export ---------------------------------------------------------------------------------------------------------------
import shutil
if ON_KAGGLE and os.path.isdir(cfg.run_dir):
    zip_path = shutil.make_archive(os.path.join("/kaggle/working", os.path.basename(cfg.run_dir) + "_outputs"), "zip",
                                   root_dir=cfg.run_dir, base_dir=".")
    print("zipped run directory ->", zip_path, f"({os.path.getsize(zip_path) / 2**20:.1f} MB)")
    print("checkpoints stay in", cfg.ckpt_dir, "(best.pt / last.pt); keep /kaggle/working to resume in a new session")